In [ ]:
import pyActigraphy
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

In [ ]:
#fpath = 'C:\\Users\\dc00955\\OneDrive - University of Surrey\\Desktop\\Sara_case_report\\'
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [ ]:
# first joined file (from 21/09/2022 to 25/08/2024)
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_24-09-23_22-09-21.mtn')

# second joined file (from 23/09/2024 to 03/12/2024 )
#raw = pyActigraphy.io.read_raw_mtn(fpath+'2.0_Joined_23-09-24_03-12-24.mtn')

# third joined file (from 03/12/2024 to 31/01/2025)
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_03-12-24_03-02-25.mtn')

#raw = pyActigraphy.io.read_raw_mtn(fpath+'\\Joined_21-09-22_25-08-24.mtn')
raw = pyActigraphy.io.read_raw_mtn(fpath+'\\1.0_Joined_23-09-24_03-02-25.mtn')

In [ ]:
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_03-12-24_03-02-25.mtn',
#                                   start_time = "2024-12-03 13:00:00",
#                                   period = "59 days") # dec-jan 2025

In [ ]:
raw.duration()

In [ ]:
raw.start_time

In [ ]:
raw.light.get_channel_list()

In [ ]:
layout = go.Layout(
    title="Light exposure data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title=" Lux(log scale)"),
    showlegend=False
)
fig1 = go.Figure(
    data=[go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='White light')
    ],
    layout=layout
)
fig1.show()

In [ ]:
# APPLY MASKING BEFORE CALCULATING METRICS

In [ ]:
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_24-09-23_22-09-21.mtn')

In [ ]:
raw.light.create_light_mask()

In [ ]:
# manually adding mask to more than one period (FOR FIRST JOINED FILED (SEPT 2022 TO AUGUST 2024))
periods = [{'start': '2022-12-28 12:00:00', 'stop': '2022-12-29 12:00:00'},{'start': '2023-03-04 22:40:00', 'stop': '2023-04-18 10:00:00'},{'start': '2023-05-14 22:30:00', 'stop': '2023-05-15 07:15:00'},
           {'start': '2023-05-16 09:00:00', 'stop': '2023-06-29 13:00:00'},{'start': '2023-07-31 19:15:00', 'stop': '2023-08-16 12:00:00'},{'start': '2023-08-18 10:00:00', 'stop': '2023-08-22 13:00:00'},
           {'start': '2024-06-27 10:52:00', 'stop': '2024-06-28 10:40:00'}, {'start': '2024-07-30 19:15:00', 'stop': '2024-08-04 19:15:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
# manually adding mask to more than one period
periods = [{'start': '2023-03-04 22:00:00', 'stop': '2023-04-18 09:15:00'},{'start': '2023-05-16 07:00:00', 'stop': '2023-06-29 16:00:00'},{'start': '2023-07-28 14:00:00', 'stop': '2023-08-22 16:00:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
# manually adding mask to more than one period (THIRD JOINED FILE)
#periods = [{'start': '2024-10-21 00:00:00', 'stop': '2024-10-22 00:00:00'},{'start': '2024-11-18 00:00:00', 'stop': '2024-11-19 00:00:00'},{'start': '2024-12-03 00:00:00', 'stop': '2024-12-04 00:00:00'},{'start': '2024-12-18 22:00:00', 'stop': '2024-12-19 07:10:00'},{'start': '2025-01-02 13:00:00', 'stop': '2025-01-03 09:00:00'},{'start': '2025-01-31 17:00:00', 'stop': '2025-02-03 13:15:00'}]
#for period in periods:
#    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
raw.light.apply_mask = True

In [ ]:
raw.light.apply_mask 

In [ ]:
# visualising masked periods
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title=" Lux(log scale)"),
    yaxis2=dict(title='Mask',overlaying='y',side='right'),
    showlegend=True
)
fig2 = go.Figure([
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='Light'),
    go.Scatter(
        x=raw.light.mask.index.astype(str),
        y=raw.light.mask.loc[:,'whitelight'],
        yaxis='y2', opacity=0.5,
        name='Mask')
], layout=layout)
fig2.show()

In [ ]:
# converting 250 melanopic lux to photopic lux (not that simple) - photopic lux = melanopic EDI/melanopic ratio
# standard office lighting (3500 K LED) with 300 lux at the task plane and 150 lux at the eye has a melanopic ratio (melanopic DER) of about 0.51 (which I used here to convert)
# REF: https://pmc.ncbi.nlm.nih.gov/articles/PMC8215265/
# also here: https://www.fagerhult.com/knowledge/light-and-people/melanopic-ratio/a-new-way-to-report-the-biological-impact-of-lighting/#:~:text=Melanopic%20Ratio%20(Melanopic%20Daylight%20Efficacy,the%20reference%20for%20Melanopic%20Ratio.
# https://journals.sagepub.com/doi/epub/10.1177/14771535221120350
# https://issuu.com/designinglighting/docs/dec_2022/s/18078622

# conversion: photopic lux = 250 melanopic EDI / 0.51 (melanopic DER) = 490 lux
# converted to log scale (to use it in the TAT treshold): np.log10(490+1) = 2.69

# treshold for outdoor light exposure = 2500 lux
# converted to log scale: 3.398

In [ ]:
np.log10(100+1)

In [ ]:
np.log10(500+1)

In [ ]:
np.log10(1000+1)

In [ ]:
np.log10(2500+1)

In [ ]:
raw.light.light_exposure_level(agg='median')

In [ ]:
raw.light.summary_statistics_per_time_bin()

In [ ]:
light_values = raw.light.summary_statistics_per_time_bin()

In [ ]:
# convert the index in column
light_values.reset_index(inplace=True)

In [ ]:
#list of the headers
light_values.columns

In [ ]:
#rename (     'index',       '') to 'date'
light_values.rename(columns={'index':'date'}, inplace=True)

In [ ]:
light_values.columns

In [ ]:
plt.figure(figsize=(28,9))
sns.scatterplot(x='date', y=('whitelight', 'mean'), data=light_values)
plt.title('Mean light exposure', fontsize=20)
plt.xlabel('Date', fontsize=16)
# x major ticks every month
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.ylabel('Lux(log)', fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()

In [ ]:
# Convert light values into a pandas DataFrame
light_summary = pd.DataFrame(light_values)
# Save to a CSV file
#light_summary.to_csv("daily_light_summary_part3.csv", index=False)


RANGE 7:00-17:00

In [ ]:
from datetime import datetime, timedelta

original_intervals = [
    ['2022-09-21 19:00:00', '2022-09-22 07:00:00'],
    ['2022-09-22 07:00:00', '2022-09-22 19:00:00'],
    ['2022-09-22 19:00:00', '2022-09-23 07:00:00'],
    ['2022-09-23 07:00:00', '2022-09-23 19:00:00'],
    ['2022-09-23 19:00:00', '2022-09-24 07:00:00'],
    ['2022-09-24 07:00:00', '2022-09-24 19:00:00'],
    ['2022-09-24 19:00:00', '2022-09-25 07:00:00'],
    ['2022-09-25 07:00:00', '2022-09-25 19:00:00'],
    ['2022-09-25 19:00:00', '2022-09-26 07:00:00'],
    ['2022-09-26 07:00:00', '2022-09-26 19:00:00'],
    ['2022-09-26 19:00:00', '2022-09-27 07:00:00'],
    ['2022-09-27 07:00:00', '2022-09-27 19:00:00'],
    ['2022-09-27 19:00:00', '2022-09-28 07:00:00'],
    ['2022-09-28 07:00:00', '2022-09-28 19:00:00'],
    ['2022-09-28 19:00:00', '2022-09-29 07:00:00'],
    ['2022-09-29 07:00:00', '2022-09-29 19:00:00'],
    ['2022-09-29 19:00:00', '2022-09-30 07:00:00'],
    ['2022-09-30 07:00:00', '2022-09-30 19:00:00'],
    ['2022-09-30 19:00:00', '2022-10-01 07:00:00'],
    ['2022-10-01 07:00:00', '2022-10-01 19:00:00'],
    ['2022-10-01 19:00:00', '2022-10-02 07:00:00'],
    ['2022-10-02 07:00:00', '2022-10-02 19:00:00'],
    ['2022-10-02 19:00:00', '2022-10-03 07:00:00'],
    ['2022-10-03 07:00:00', '2022-10-03 19:00:00'],
    ['2022-10-03 19:00:00', '2022-10-04 07:00:00'],
    ['2022-10-04 07:00:00', '2022-10-04 19:00:00'],
    ['2022-10-04 19:00:00', '2022-10-05 07:00:00'],
    ['2022-10-05 07:00:00', '2022-10-05 19:00:00'],
    ['2022-10-05 19:00:00', '2022-10-06 07:00:00'],
    ['2022-10-06 07:00:00', '2022-10-06 19:00:00'],
    ['2022-10-06 19:00:00', '2022-10-07 07:00:00'],
    ['2022-10-07 07:00:00', '2022-10-07 19:00:00'],
    ['2022-10-07 19:00:00', '2022-10-08 07:00:00'],
    ['2022-10-08 07:00:00', '2022-10-08 19:00:00'],
    ['2022-10-08 19:00:00', '2022-10-09 07:00:00'],
    ['2022-10-09 07:00:00', '2022-10-09 19:00:00'],
    ['2022-10-09 19:00:00', '2022-10-10 07:00:00'],
    ['2022-10-10 07:00:00', '2022-10-10 19:00:00'],
    ['2022-10-10 19:00:00', '2022-10-11 07:00:00'],
    ['2022-10-11 07:00:00', '2022-10-11 19:00:00'],
    ['2022-10-11 19:00:00', '2022-10-12 07:00:00'],
    ['2022-10-12 07:00:00', '2022-10-12 19:00:00'],
    ['2022-10-12 19:00:00', '2022-10-13 07:00:00'],
    ['2022-10-13 07:00:00', '2022-10-13 19:00:00'],
    ['2022-10-13 19:00:00', '2022-10-14 07:00:00'],
    ['2022-10-14 07:00:00', '2022-10-14 19:00:00'],
    ['2022-10-14 19:00:00', '2022-10-15 07:00:00'],
    ['2022-10-15 07:00:00', '2022-10-15 19:00:00'],
    ['2022-10-15 19:00:00', '2022-10-16 07:00:00'],
    ['2022-10-16 07:00:00', '2022-10-16 19:00:00'],
    ['2022-10-16 19:00:00', '2022-10-17 07:00:00'],
    ['2022-10-17 07:00:00', '2022-10-17 19:00:00'],
    ['2022-10-19 07:00:00', '2022-10-19 19:00:00'],  #here
    ['2022-10-19 19:00:00', '2022-10-20 07:00:00'],
    ['2022-10-20 07:00:00', '2022-10-20 19:00:00'],
    ['2022-10-20 19:00:00', '2022-10-21 07:00:00'],
    ['2022-10-21 07:00:00', '2022-10-21 19:00:00'],
    ['2022-10-21 19:00:00', '2022-10-22 07:00:00'],
    ['2022-10-22 07:00:00', '2022-10-22 19:00:00'],
    ['2022-10-22 19:00:00', '2022-10-23 07:00:00'],
    ['2022-10-23 07:00:00', '2022-10-23 19:00:00'],
    ['2022-10-23 19:00:00', '2022-10-24 07:00:00'],
    ['2022-10-24 07:00:00', '2022-10-24 19:00:00'],
    ['2022-10-24 19:00:00', '2022-10-25 07:00:00'],
    ['2022-10-25 07:00:00', '2022-10-25 19:00:00'],
    ['2022-10-25 19:00:00', '2022-10-26 07:00:00'],
    ['2022-10-26 07:00:00', '2022-10-26 19:00:00'],
    ['2022-10-26 19:00:00', '2022-10-27 07:00:00'],
    ['2022-10-27 07:00:00', '2022-10-27 19:00:00'],
    ['2022-10-27 19:00:00', '2022-10-28 07:00:00'],
    ['2022-10-28 07:00:00', '2022-10-28 19:00:00'],
    ['2022-10-28 19:00:00', '2022-10-29 07:00:00'],
    ['2022-10-29 07:00:00', '2022-10-29 19:00:00'],
    ['2022-10-29 19:00:00', '2022-10-30 07:00:00'],
    ['2022-10-30 07:00:00', '2022-10-30 19:00:00'],
    ['2022-10-30 19:00:00', '2022-10-31 07:00:00'],
    ['2022-10-31 07:00:00', '2022-10-31 19:00:00'],
    ['2022-10-31 19:00:00', '2022-11-01 07:00:00'],
    ['2022-11-01 07:00:00', '2022-11-01 19:00:00'],
    ['2022-11-01 19:00:00', '2022-11-02 07:00:00'],
    ['2022-11-02 07:00:00', '2022-11-02 19:00:00'],
    ['2022-11-02 19:00:00', '2022-11-03 07:00:00'],
    ['2022-11-03 07:00:00', '2022-11-03 19:00:00'],
    ['2022-11-03 19:00:00', '2022-11-04 07:00:00'],
    ['2022-11-04 07:00:00', '2022-11-04 19:00:00'],
    ['2022-11-04 19:00:00', '2022-11-05 07:00:00'],
    ['2022-11-05 07:00:00', '2022-11-05 19:00:00'],
    ['2022-11-05 19:00:00', '2022-11-06 07:00:00'],
    ['2022-11-06 07:00:00', '2022-11-06 19:00:00'],
    ['2022-11-06 19:00:00', '2022-11-07 07:00:00'],
    ['2022-11-07 07:00:00', '2022-11-07 19:00:00'],
    ['2022-11-07 19:00:00', '2022-11-08 07:00:00'],
    ['2022-11-08 07:00:00', '2022-11-08 19:00:00'],
    ['2022-11-08 19:00:00', '2022-11-09 07:00:00'],
    ['2022-11-09 07:00:00', '2022-11-09 19:00:00'],
    ['2022-11-09 19:00:00', '2022-11-10 07:00:00'],
    ['2022-11-10 07:00:00', '2022-11-10 19:00:00'],
    ['2022-11-10 19:00:00', '2022-11-11 07:00:00'],
    ['2022-11-11 07:00:00', '2022-11-11 19:00:00'],
    ['2022-11-11 19:00:00', '2022-11-12 07:00:00'],
    ['2022-11-12 07:00:00', '2022-11-12 19:00:00'],
    ['2022-11-12 19:00:00', '2022-11-13 07:00:00'],
    ['2022-11-13 07:00:00', '2022-11-13 19:00:00'],
    ['2022-11-13 19:00:00', '2022-11-14 07:00:00'],
    ['2022-11-14 07:00:00', '2022-11-14 19:00:00'],
    ['2022-11-14 19:00:00', '2022-11-15 07:00:00'],
    ['2022-11-15 07:00:00', '2022-11-15 19:00:00'],
    ['2022-11-15 19:00:00', '2022-11-16 07:00:00'],
    ['2022-11-16 07:00:00', '2022-11-16 19:00:00'],
    ['2022-11-16 19:00:00', '2022-11-17 07:00:00'],
    ['2022-11-17 07:00:00', '2022-11-17 19:00:00'],
    ['2022-11-19 07:00:00', '2022-11-19 19:00:00'],  #here
    ['2022-11-19 19:00:00', '2022-11-20 07:00:00'],
    ['2022-11-20 07:00:00', '2022-11-20 19:00:00'],
    ['2022-11-20 19:00:00', '2022-11-21 07:00:00'],
    ['2022-11-21 07:00:00', '2022-11-21 19:00:00'],
    ['2022-11-21 19:00:00', '2022-11-22 07:00:00'],
    ['2022-11-22 07:00:00', '2022-11-22 19:00:00'],
    ['2022-11-22 19:00:00', '2022-11-23 07:00:00'],
    ['2022-11-23 07:00:00', '2022-11-23 19:00:00'],
    ['2022-11-23 19:00:00', '2022-11-24 07:00:00'],
    ['2022-11-24 07:00:00', '2022-11-24 19:00:00'],
    ['2022-11-24 19:00:00', '2022-11-25 07:00:00'],
    ['2022-11-25 07:00:00', '2022-11-25 19:00:00'],
    ['2022-11-25 19:00:00', '2022-11-26 07:00:00'],
    ['2022-11-26 07:00:00', '2022-11-26 19:00:00'],
    ['2022-11-26 19:00:00', '2022-11-27 07:00:00'],
    ['2022-11-27 07:00:00', '2022-11-27 19:00:00'],
    ['2022-11-27 19:00:00', '2022-11-28 07:00:00'],
    ['2022-11-28 07:00:00', '2022-11-28 19:00:00'],
    ['2022-11-28 19:00:00', '2022-11-29 07:00:00'],
    ['2022-11-29 07:00:00', '2022-11-29 19:00:00'],
    ['2022-11-29 19:00:00', '2022-11-30 07:00:00'],
    ['2022-11-30 07:00:00', '2022-11-30 19:00:00'],
    ['2022-11-30 19:00:00', '2022-12-01 07:00:00'],
    ['2022-12-01 07:00:00', '2022-12-01 19:00:00'],
    ['2022-12-01 19:00:00', '2022-12-02 07:00:00'],
    ['2022-12-02 07:00:00', '2022-12-02 19:00:00'],
    ['2022-12-02 19:00:00', '2022-12-03 07:00:00'],
    ['2022-12-03 07:00:00', '2022-12-03 19:00:00'],
    ['2022-12-03 19:00:00', '2022-12-04 07:00:00'],
    ['2022-12-04 07:00:00', '2022-12-04 19:00:00'],
    ['2022-12-04 19:00:00', '2022-12-05 07:00:00'],
    ['2022-12-05 07:00:00', '2022-12-05 19:00:00'],
    ['2022-12-05 19:00:00', '2022-12-06 07:00:00'],
    ['2022-12-06 07:00:00', '2022-12-06 19:00:00'],
    ['2022-12-06 19:00:00', '2022-12-07 07:00:00'],
    ['2022-12-07 07:00:00', '2022-12-07 19:00:00'],
    ['2022-12-07 19:00:00', '2022-12-08 07:00:00'],
    ['2022-12-08 07:00:00', '2022-12-08 19:00:00'],
    ['2022-12-08 19:00:00', '2022-12-09 07:00:00'],
    ['2022-12-09 07:00:00', '2022-12-09 19:00:00'],
    ['2022-12-09 19:00:00', '2022-12-10 07:00:00'],
    ['2022-12-10 07:00:00', '2022-12-10 19:00:00'],
    ['2022-12-10 19:00:00', '2022-12-11 07:00:00'],
    ['2022-12-11 07:00:00', '2022-12-11 19:00:00'],
    ['2022-12-11 19:00:00', '2022-12-12 07:00:00'],
    ['2022-12-12 07:00:00', '2022-12-12 19:00:00'],
    ['2022-12-12 19:00:00', '2022-12-13 07:00:00'],
    ['2022-12-13 07:00:00', '2022-12-13 19:00:00'],
    ['2022-12-13 19:00:00', '2022-12-14 07:00:00'],
    ['2022-12-14 07:00:00', '2022-12-14 19:00:00'],
    ['2022-12-14 19:00:00', '2022-12-15 07:00:00'],
    ['2022-12-15 07:00:00', '2022-12-15 19:00:00'],
    ['2022-12-15 19:00:00', '2022-12-16 07:00:00'],
    ['2022-12-16 07:00:00', '2022-12-16 19:00:00'],
    ['2022-12-16 19:00:00', '2022-12-17 07:00:00'],
    ['2022-12-17 07:00:00', '2022-12-17 19:00:00'],
    ['2022-12-17 19:00:00', '2022-12-18 07:00:00'],
    ['2022-12-18 07:00:00', '2022-12-18 19:00:00'],
    ['2022-12-18 19:00:00', '2022-12-19 07:00:00'],
    ['2022-12-19 07:00:00', '2022-12-19 19:00:00'],
    ['2022-12-19 19:00:00', '2022-12-20 07:00:00'],
    ['2022-12-20 07:00:00', '2022-12-20 19:00:00'],
    ['2022-12-20 19:00:00', '2022-12-21 07:00:00'],
    ['2022-12-21 07:00:00', '2022-12-21 19:00:00'],
    ['2022-12-21 19:00:00', '2022-12-22 07:00:00'],
    ['2022-12-22 07:00:00', '2022-12-22 19:00:00'],
    ['2022-12-22 19:00:00', '2022-12-23 07:00:00'],
    ['2022-12-23 07:00:00', '2022-12-23 19:00:00'],
    ['2022-12-23 19:00:00', '2022-12-24 07:00:00'],
    ['2022-12-24 07:00:00', '2022-12-24 19:00:00'],
    ['2022-12-24 19:00:00', '2022-12-25 07:00:00'],
    ['2022-12-25 07:00:00', '2022-12-25 19:00:00'],
    ['2022-12-25 19:00:00', '2022-12-26 07:00:00'],
    ['2022-12-26 07:00:00', '2022-12-26 19:00:00'],
    ['2022-12-26 19:00:00', '2022-12-27 07:00:00'],
    ['2022-12-27 07:00:00', '2022-12-27 19:00:00'],
    ['2022-12-29 07:00:00', '2022-12-29 19:00:00'],  #here
    ['2022-12-29 19:00:00', '2022-12-30 07:00:00'],
    ['2022-12-30 07:00:00', '2022-12-30 19:00:00'],
    ['2022-12-30 19:00:00', '2022-12-31 07:00:00'],
    ['2022-12-31 07:00:00', '2022-12-31 19:00:00'],
    ['2022-12-31 19:00:00', '2023-01-01 07:00:00'],
    ['2023-01-01 07:00:00', '2023-01-01 19:00:00'],
    ['2023-01-01 19:00:00', '2023-01-02 07:00:00'],
    ['2023-01-02 07:00:00', '2023-01-02 19:00:00'],
    ['2023-01-02 19:00:00', '2023-01-03 07:00:00'],
    ['2023-01-03 07:00:00', '2023-01-03 19:00:00'],
    ['2023-01-03 19:00:00', '2023-01-04 07:00:00'],
    ['2023-01-04 07:00:00', '2023-01-04 19:00:00'],
    ['2023-01-04 19:00:00', '2023-01-05 07:00:00'],
    ['2023-01-05 07:00:00', '2023-01-05 19:00:00'],
    ['2023-01-05 19:00:00', '2023-01-06 07:00:00'],
    ['2023-01-06 07:00:00', '2023-01-06 19:00:00'],
    ['2023-01-06 19:00:00', '2023-01-07 07:00:00'],
    ['2023-01-07 07:00:00', '2023-01-07 19:00:00'],
    ['2023-01-07 19:00:00', '2023-01-08 07:00:00'],
    ['2023-01-08 07:00:00', '2023-01-08 19:00:00'],
    ['2023-01-08 19:00:00', '2023-01-09 07:00:00'],
    ['2023-01-09 07:00:00', '2023-01-09 19:00:00'],
    ['2023-01-09 19:00:00', '2023-01-10 07:00:00'],
    ['2023-01-10 07:00:00', '2023-01-10 19:00:00'],
    ['2023-01-10 19:00:00', '2023-01-11 07:00:00'],
    ['2023-01-11 07:00:00', '2023-01-11 19:00:00'],
    ['2023-01-11 19:00:00', '2023-01-12 07:00:00'],
    ['2023-01-12 07:00:00', '2023-01-12 19:00:00'],
    ['2023-01-12 19:00:00', '2023-01-13 07:00:00'],
    ['2023-01-13 07:00:00', '2023-01-13 19:00:00'],
    ['2023-01-13 19:00:00', '2023-01-14 07:00:00'],
    ['2023-01-14 07:00:00', '2023-01-14 19:00:00'],
    ['2023-01-14 19:00:00', '2023-01-15 07:00:00'],
    ['2023-01-15 07:00:00', '2023-01-15 19:00:00'],
    ['2023-01-15 19:00:00', '2023-01-16 07:00:00'],
    ['2023-01-16 07:00:00', '2023-01-16 19:00:00'],
    ['2023-01-16 19:00:00', '2023-01-17 07:00:00'],
    ['2023-01-17 07:00:00', '2023-01-17 19:00:00'],
    ['2023-01-19 07:00:00', '2023-01-19 19:00:00'],  #here
    ['2023-01-19 19:00:00', '2023-01-20 07:00:00'],
    ['2023-01-20 07:00:00', '2023-01-20 19:00:00'],
    ['2023-01-20 19:00:00', '2023-01-21 07:00:00'],
    ['2023-01-21 07:00:00', '2023-01-21 19:00:00'],
    ['2023-01-21 19:00:00', '2023-01-22 07:00:00'],
    ['2023-01-22 07:00:00', '2023-01-22 19:00:00'],
    ['2023-01-22 19:00:00', '2023-01-23 07:00:00'],
    ['2023-01-23 07:00:00', '2023-01-23 19:00:00'],
    ['2023-01-23 19:00:00', '2023-01-24 07:00:00'],
    ['2023-01-24 07:00:00', '2023-01-24 19:00:00'],
    ['2023-01-24 19:00:00', '2023-01-25 07:00:00'],
    ['2023-01-25 07:00:00', '2023-01-25 19:00:00'],
    ['2023-01-25 19:00:00', '2023-01-26 07:00:00'],
    ['2023-01-26 07:00:00', '2023-01-26 19:00:00'],
    ['2023-01-26 19:00:00', '2023-01-27 07:00:00'],
    ['2023-01-27 07:00:00', '2023-01-27 19:00:00'],
    ['2023-01-27 19:00:00', '2023-01-28 07:00:00'],
    ['2023-01-28 07:00:00', '2023-01-28 19:00:00'],
    ['2023-01-28 19:00:00', '2023-01-29 07:00:00'],
    ['2023-01-29 07:00:00', '2023-01-29 19:00:00'],
    ['2023-01-29 19:00:00', '2023-01-30 07:00:00'],
    ['2023-01-30 07:00:00', '2023-01-30 19:00:00'],
    ['2023-01-30 19:00:00', '2023-01-31 07:00:00'],
    ['2023-01-31 07:00:00', '2023-01-31 19:00:00'],
    ['2023-01-31 19:00:00', '2023-02-01 07:00:00'],
    ['2023-02-01 07:00:00', '2023-02-01 19:00:00'],
    ['2023-02-01 19:00:00', '2023-02-02 07:00:00'],
    ['2023-02-02 07:00:00', '2023-02-02 19:00:00'],
    ['2023-02-02 19:00:00', '2023-02-03 07:00:00'],
    ['2023-02-03 07:00:00', '2023-02-03 19:00:00'],
    ['2023-02-03 19:00:00', '2023-02-04 07:00:00'],
    ['2023-02-04 07:00:00', '2023-02-04 19:00:00'],
    ['2023-02-04 19:00:00', '2023-02-05 07:00:00'],
    ['2023-02-05 07:00:00', '2023-02-05 19:00:00'],
    ['2023-02-05 19:00:00', '2023-02-06 07:00:00'],
    ['2023-02-06 07:00:00', '2023-02-06 19:00:00'],
    ['2023-02-06 19:00:00', '2023-02-07 07:00:00'],
    ['2023-02-07 07:00:00', '2023-02-07 19:00:00'],
    ['2023-02-07 19:00:00', '2023-02-08 07:00:00'],
    ['2023-02-08 07:00:00', '2023-02-08 19:00:00'],
    ['2023-02-08 19:00:00', '2023-02-09 07:00:00'],
    ['2023-02-09 07:00:00', '2023-02-09 19:00:00'],
    ['2023-02-09 19:00:00', '2023-02-10 07:00:00'],
    ['2023-02-10 07:00:00', '2023-02-10 19:00:00'],
    ['2023-02-10 19:00:00', '2023-02-11 07:00:00'],
    ['2023-02-11 07:00:00', '2023-02-11 19:00:00'],
    ['2023-02-11 19:00:00', '2023-02-12 07:00:00'],
    ['2023-02-12 07:00:00', '2023-02-12 19:00:00'],
    ['2023-02-12 19:00:00', '2023-02-13 07:00:00'],
    ['2023-02-13 07:00:00', '2023-02-13 19:00:00'],
    ['2023-02-13 19:00:00', '2023-02-14 07:00:00'],
    ['2023-02-14 07:00:00', '2023-02-14 19:00:00'],
    ['2023-02-14 19:00:00', '2023-02-15 07:00:00'],
    ['2023-02-15 07:00:00', '2023-02-15 19:00:00'],
    ['2023-02-15 19:00:00', '2023-02-16 07:00:00'],
    ['2023-02-16 07:00:00', '2023-02-16 19:00:00'],
    ['2023-02-16 19:00:00', '2023-02-17 07:00:00'],
    ['2023-02-17 07:00:00', '2023-02-17 19:00:00'],
    ['2023-02-17 19:00:00', '2023-02-18 07:00:00'],
    ['2023-02-18 07:00:00', '2023-02-18 19:00:00'],
    ['2023-02-18 19:00:00', '2023-02-19 07:00:00'],
    ['2023-02-19 07:00:00', '2023-02-19 19:00:00'],
    ['2023-02-19 19:00:00', '2023-02-20 07:00:00'],
    ['2023-02-20 07:00:00', '2023-02-20 19:00:00'],
    ['2023-02-20 19:00:00', '2023-02-21 07:00:00'],
    ['2023-02-21 07:00:00', '2023-02-21 19:00:00'],
    ['2023-02-21 19:00:00', '2023-02-22 07:00:00'],
    ['2023-02-22 07:00:00', '2023-02-22 19:00:00'],
    ['2023-02-22 19:00:00', '2023-02-23 07:00:00'],
    ['2023-02-23 07:00:00', '2023-02-23 19:00:00'],
    ['2023-02-23 19:00:00', '2023-02-24 07:00:00'],
    ['2023-02-24 07:00:00', '2023-02-24 19:00:00'],
    ['2023-02-24 19:00:00', '2023-02-25 07:00:00'],
    ['2023-02-25 07:00:00', '2023-02-25 19:00:00'],
    ['2023-02-25 19:00:00', '2023-02-26 07:00:00'],
    ['2023-02-26 07:00:00', '2023-02-26 19:00:00'],
    ['2023-02-26 19:00:00', '2023-02-27 07:00:00'],
    ['2023-02-27 07:00:00', '2023-02-27 19:00:00'],
    ['2023-02-27 19:00:00', '2023-02-28 07:00:00'],
    ['2023-02-28 07:00:00', '2023-02-28 19:00:00'],
    ['2023-02-28 19:00:00', '2023-03-01 07:00:00'],
    ['2023-03-01 07:00:00', '2023-03-01 19:00:00'],
    ['2023-03-01 19:00:00', '2023-03-02 07:00:00'],
    ['2023-03-02 07:00:00', '2023-03-02 19:00:00'],
    ['2023-03-02 19:00:00', '2023-03-03 07:00:00'],
    ['2023-03-03 07:00:00', '2023-03-03 19:00:00'],
    ['2023-03-03 19:00:00', '2023-03-04 07:00:00'],  #here!!
    ['2023-04-19 07:00:00', '2023-04-19 19:00:00'],
    ['2023-04-19 19:00:00', '2023-04-20 07:00:00'],
    ['2023-04-20 07:00:00', '2023-04-20 19:00:00'],
    ['2023-04-20 19:00:00', '2023-04-21 07:00:00'],
    ['2023-04-21 07:00:00', '2023-04-21 19:00:00'],
    ['2023-04-21 19:00:00', '2023-04-22 07:00:00'],
    ['2023-04-22 07:00:00', '2023-04-22 19:00:00'],
    ['2023-04-22 19:00:00', '2023-04-23 07:00:00'],
    ['2023-04-23 07:00:00', '2023-04-23 19:00:00'],
    ['2023-04-23 19:00:00', '2023-04-24 07:00:00'],
    ['2023-04-24 07:00:00', '2023-04-24 19:00:00'],
    ['2023-04-24 19:00:00', '2023-04-25 07:00:00'],
    ['2023-04-25 07:00:00', '2023-04-25 19:00:00'],
    ['2023-04-25 19:00:00', '2023-04-26 07:00:00'],
    ['2023-04-26 07:00:00', '2023-04-26 19:00:00'],
    ['2023-04-26 19:00:00', '2023-04-27 07:00:00'],
    ['2023-04-27 07:00:00', '2023-04-27 19:00:00'],
    ['2023-04-27 19:00:00', '2023-04-28 07:00:00'],
    ['2023-04-28 07:00:00', '2023-04-28 19:00:00'],
    ['2023-04-28 19:00:00', '2023-04-29 07:00:00'],
    ['2023-04-29 07:00:00', '2023-04-29 19:00:00'],
    ['2023-04-29 19:00:00', '2023-04-30 07:00:00'],
    ['2023-04-30 07:00:00', '2023-04-30 19:00:00'],
    ['2023-04-30 19:00:00', '2023-05-01 07:00:00'],
    ['2023-05-01 07:00:00', '2023-05-01 19:00:00'],
    ['2023-05-01 19:00:00', '2023-05-02 07:00:00'],
    ['2023-05-02 07:00:00', '2023-05-02 19:00:00'],
    ['2023-05-02 19:00:00', '2023-05-03 07:00:00'],
    ['2023-05-03 07:00:00', '2023-05-03 19:00:00'],
    ['2023-05-03 19:00:00', '2023-05-04 07:00:00'],
    ['2023-05-04 07:00:00', '2023-05-04 19:00:00'],
    ['2023-05-04 19:00:00', '2023-05-05 07:00:00'],
    ['2023-05-05 07:00:00', '2023-05-05 19:00:00'],
    ['2023-05-05 19:00:00', '2023-05-06 07:00:00'],
    ['2023-05-06 07:00:00', '2023-05-06 19:00:00'],
    ['2023-05-06 19:00:00', '2023-05-07 07:00:00'],
    ['2023-05-07 07:00:00', '2023-05-07 19:00:00'],
    ['2023-05-07 19:00:00', '2023-05-08 07:00:00'],
    ['2023-05-08 07:00:00', '2023-05-08 19:00:00'],
    ['2023-05-08 19:00:00', '2023-05-09 07:00:00'],
    ['2023-05-09 07:00:00', '2023-05-09 19:00:00'],
    ['2023-05-09 19:00:00', '2023-05-10 07:00:00'],
    ['2023-05-10 07:00:00', '2023-05-10 19:00:00'],
    ['2023-05-10 19:00:00', '2023-05-11 07:00:00'],
    ['2023-05-11 07:00:00', '2023-05-11 19:00:00'],
    ['2023-05-11 19:00:00', '2023-05-12 07:00:00'],
    ['2023-05-12 07:00:00', '2023-05-12 19:00:00'],
    ['2023-05-12 19:00:00', '2023-05-13 07:00:00'],
    ['2023-05-13 07:00:00', '2023-05-13 19:00:00'],
    ['2023-05-13 19:00:00', '2023-05-14 07:00:00'],
    ['2023-05-14 07:00:00', '2023-05-14 19:00:00'],
    ['2023-05-14 19:00:00', '2023-05-15 07:00:00'],
    ['2023-05-15 07:00:00', '2023-05-15 19:00:00'],
    ['2023-05-15 19:00:00', '2023-05-16 07:00:00'],  #here
    ['2023-06-29 19:00:00', '2023-06-30 07:00:00'],
    ['2023-06-30 07:00:00', '2023-06-30 19:00:00'],
    ['2023-06-30 19:00:00', '2023-07-01 07:00:00'],
    ['2023-07-01 07:00:00', '2023-07-01 19:00:00'],
    ['2023-07-01 19:00:00', '2023-07-02 07:00:00'],
    ['2023-07-02 07:00:00', '2023-07-02 19:00:00'],
    ['2023-07-02 19:00:00', '2023-07-03 07:00:00'],
    ['2023-07-03 07:00:00', '2023-07-03 19:00:00'],
    ['2023-07-03 19:00:00', '2023-07-04 07:00:00'],
    ['2023-07-04 07:00:00', '2023-07-04 19:00:00'],
    ['2023-07-04 19:00:00', '2023-07-05 07:00:00'],
    ['2023-07-05 07:00:00', '2023-07-05 19:00:00'],
    ['2023-07-05 19:00:00', '2023-07-06 07:00:00'],
    ['2023-07-06 07:00:00', '2023-07-06 19:00:00'],
    ['2023-07-06 19:00:00', '2023-07-07 07:00:00'],
    ['2023-07-07 07:00:00', '2023-07-07 19:00:00'],
    ['2023-07-07 19:00:00', '2023-07-08 07:00:00'],
    ['2023-07-08 07:00:00', '2023-07-08 19:00:00'],
    ['2023-07-08 19:00:00', '2023-07-09 07:00:00'],
    ['2023-07-09 07:00:00', '2023-07-09 19:00:00'],
    ['2023-07-09 19:00:00', '2023-07-10 07:00:00'],
    ['2023-07-10 07:00:00', '2023-07-10 19:00:00'],
    ['2023-07-10 19:00:00', '2023-07-11 07:00:00'],
    ['2023-07-11 07:00:00', '2023-07-11 19:00:00'],
    ['2023-07-11 19:00:00', '2023-07-12 07:00:00'],
    ['2023-07-12 07:00:00', '2023-07-12 19:00:00'],
    ['2023-07-12 19:00:00', '2023-07-13 07:00:00'],
    ['2023-07-13 07:00:00', '2023-07-13 19:00:00'],
    ['2023-07-13 19:00:00', '2023-07-14 07:00:00'],
    ['2023-07-14 07:00:00', '2023-07-14 19:00:00'],
    ['2023-07-14 19:00:00', '2023-07-15 07:00:00'],
    ['2023-07-15 07:00:00', '2023-07-15 19:00:00'],
    ['2023-07-15 19:00:00', '2023-07-16 07:00:00'],
    ['2023-07-16 07:00:00', '2023-07-16 19:00:00'],  #here
    ['2023-07-18 07:00:00', '2023-07-18 19:00:00'],
    ['2023-07-18 19:00:00', '2023-07-19 07:00:00'],
    ['2023-07-19 07:00:00', '2023-07-19 19:00:00'],
    ['2023-07-19 19:00:00', '2023-07-20 07:00:00'],
    ['2023-07-20 07:00:00', '2023-07-20 19:00:00'],
    ['2023-07-20 19:00:00', '2023-07-21 07:00:00'],
    ['2023-07-21 07:00:00', '2023-07-21 19:00:00'],
    ['2023-07-21 19:00:00', '2023-07-22 07:00:00'],
    ['2023-07-22 07:00:00', '2023-07-22 19:00:00'],
    ['2023-07-22 19:00:00', '2023-07-23 07:00:00'],
    ['2023-07-23 07:00:00', '2023-07-23 19:00:00'],
    ['2023-07-23 19:00:00', '2023-07-24 07:00:00'],
    ['2023-07-24 07:00:00', '2023-07-24 19:00:00'],
    ['2023-07-24 19:00:00', '2023-07-25 07:00:00'],
    ['2023-07-25 07:00:00', '2023-07-25 19:00:00'],
    ['2023-07-25 19:00:00', '2023-07-26 07:00:00'],
    ['2023-07-26 07:00:00', '2023-07-26 19:00:00'],
    ['2023-07-26 19:00:00', '2023-07-27 07:00:00'],
    ['2023-07-27 07:00:00', '2023-07-27 19:00:00'],
    ['2023-07-27 19:00:00', '2023-07-28 07:00:00'],  #here
    ['2023-08-22 19:00:00', '2023-08-23 07:00:00'],
    ['2023-08-23 07:00:00', '2023-08-23 19:00:00'],
    ['2023-08-23 19:00:00', '2023-08-24 07:00:00'],
    ['2023-08-24 07:00:00', '2023-08-24 19:00:00'],
    ['2023-08-24 19:00:00', '2023-08-25 07:00:00'],
    ['2023-08-25 07:00:00', '2023-08-25 19:00:00'],
    ['2023-08-25 19:00:00', '2023-08-26 07:00:00'],
    ['2023-08-26 07:00:00', '2023-08-26 19:00:00'],
    ['2023-08-26 19:00:00', '2023-08-27 07:00:00'],
    ['2023-08-27 07:00:00', '2023-08-27 19:00:00'],
    ['2023-08-27 19:00:00', '2023-08-28 07:00:00'],
    ['2023-08-28 07:00:00', '2023-08-28 19:00:00'],
    ['2023-08-28 19:00:00', '2023-08-29 07:00:00'],
    ['2023-08-29 07:00:00', '2023-08-29 19:00:00'],
    ['2023-08-29 19:00:00', '2023-08-30 07:00:00'],
    ['2023-08-30 07:00:00', '2023-08-30 19:00:00'],  #here
    ['2023-09-02 07:00:00', '2023-09-02 19:00:00'],
    ['2023-09-02 19:00:00', '2023-09-03 07:00:00'],
    ['2023-09-03 07:00:00', '2023-09-03 19:00:00'],
    ['2023-09-03 19:00:00', '2023-09-04 07:00:00'],
    ['2023-09-04 07:00:00', '2023-09-04 19:00:00'],
    ['2023-09-04 19:00:00', '2023-09-05 07:00:00'],
    ['2023-09-05 07:00:00', '2023-09-05 19:00:00'],
    ['2023-09-05 19:00:00', '2023-09-06 07:00:00'],
    ['2023-09-06 07:00:00', '2023-09-06 19:00:00'],
    ['2023-09-06 19:00:00', '2023-09-07 07:00:00'],
    ['2023-09-07 07:00:00', '2023-09-07 19:00:00'],
    ['2023-09-07 19:00:00', '2023-09-08 07:00:00'],
    ['2023-09-08 07:00:00', '2023-09-08 19:00:00'],
    ['2023-09-08 19:00:00', '2023-09-09 07:00:00'],
    ['2023-09-09 07:00:00', '2023-09-09 19:00:00'],
    ['2023-09-09 19:00:00', '2023-09-10 07:00:00'],
    ['2023-09-10 07:00:00', '2023-09-10 19:00:00'],
    ['2023-09-10 19:00:00', '2023-09-11 07:00:00'],
    ['2023-09-11 07:00:00', '2023-09-11 19:00:00'],
    ['2023-09-11 19:00:00', '2023-09-12 07:00:00'],
    ['2023-09-12 07:00:00', '2023-09-12 19:00:00'],
    ['2023-09-12 19:00:00', '2023-09-13 07:00:00'],
    ['2023-09-13 07:00:00', '2023-09-13 19:00:00'],
    ['2023-09-13 19:00:00', '2023-09-14 07:00:00'],
    ['2023-09-14 07:00:00', '2023-09-14 19:00:00'],
    ['2023-09-14 19:00:00', '2023-09-15 07:00:00'],
    ['2023-09-15 07:00:00', '2023-09-15 19:00:00'],
    ['2023-09-15 19:00:00', '2023-09-16 07:00:00'],
    ['2023-09-16 07:00:00', '2023-09-16 19:00:00'],
    ['2023-09-16 19:00:00', '2023-09-17 07:00:00'],
    ['2023-09-17 07:00:00', '2023-09-17 19:00:00'],  #here
    ['2023-09-19 07:00:00', '2023-09-19 19:00:00'],
    ['2023-09-19 19:00:00', '2023-09-20 07:00:00'],
    ['2023-09-20 07:00:00', '2023-09-20 19:00:00'],
    ['2023-09-20 19:00:00', '2023-09-21 07:00:00'],
    ['2023-09-21 07:00:00', '2023-09-21 19:00:00'],
    ['2023-09-21 19:00:00', '2023-09-22 07:00:00'],
    ['2023-09-22 07:00:00', '2023-09-22 19:00:00'],
    ['2023-09-22 19:00:00', '2023-09-23 07:00:00'],
    ['2023-09-23 07:00:00', '2023-09-23 19:00:00'],
    ['2023-09-23 19:00:00', '2023-09-24 07:00:00'],
    ['2023-09-24 07:00:00', '2023-09-24 19:00:00'],
    ['2023-09-24 19:00:00', '2023-09-25 07:00:00'],
    ['2023-09-25 07:00:00', '2023-09-25 19:00:00'],
    ['2023-09-25 19:00:00', '2023-09-26 07:00:00'],
    ['2023-09-26 07:00:00', '2023-09-26 19:00:00'],
    ['2023-09-26 19:00:00', '2023-09-27 07:00:00'],
    ['2023-09-27 07:00:00', '2023-09-27 19:00:00'],
    ['2023-09-27 19:00:00', '2023-09-28 07:00:00'],
    ['2023-09-28 07:00:00', '2023-09-28 19:00:00'],
    ['2023-09-28 19:00:00', '2023-09-29 07:00:00'],
    ['2023-09-29 07:00:00', '2023-09-29 19:00:00'],
    ['2023-09-29 19:00:00', '2023-09-30 07:00:00'],
    ['2023-09-30 07:00:00', '2023-09-30 19:00:00'],
    ['2023-09-30 19:00:00', '2023-10-01 07:00:00'],
    ['2023-10-01 07:00:00', '2023-10-01 19:00:00'],
    ['2023-10-01 19:00:00', '2023-10-02 07:00:00'],
    ['2023-10-02 07:00:00', '2023-10-02 19:00:00'],
    ['2023-10-02 19:00:00', '2023-10-03 07:00:00'],
    ['2023-10-03 07:00:00', '2023-10-03 19:00:00'],
    ['2023-10-03 19:00:00', '2023-10-04 07:00:00'],
    ['2023-10-04 07:00:00', '2023-10-04 19:00:00'],
    ['2023-10-04 19:00:00', '2023-10-05 07:00:00'],
    ['2023-10-05 07:00:00', '2023-10-05 19:00:00'],
    ['2023-10-05 19:00:00', '2023-10-06 07:00:00'],
    ['2023-10-06 07:00:00', '2023-10-06 19:00:00'],
    ['2023-10-06 19:00:00', '2023-10-07 07:00:00'],
    ['2023-10-07 07:00:00', '2023-10-07 19:00:00'],
    ['2023-10-07 19:00:00', '2023-10-08 07:00:00'],
    ['2023-10-08 07:00:00', '2023-10-08 19:00:00'],
    ['2023-10-08 19:00:00', '2023-10-09 07:00:00'],
    ['2023-10-09 07:00:00', '2023-10-09 19:00:00'],
    ['2023-10-09 19:00:00', '2023-10-10 07:00:00'],
    ['2023-10-10 07:00:00', '2023-10-10 19:00:00'],
    ['2023-10-10 19:00:00', '2023-10-11 07:00:00'],
    ['2023-10-11 07:00:00', '2023-10-11 19:00:00'],
    ['2023-10-11 19:00:00', '2023-10-12 07:00:00'],
    ['2023-10-12 07:00:00', '2023-10-12 19:00:00'],
    ['2023-10-12 19:00:00', '2023-10-13 07:00:00'],
    ['2023-10-13 07:00:00', '2023-10-13 19:00:00'],
    ['2023-10-13 19:00:00', '2023-10-14 07:00:00'],
    ['2023-10-14 07:00:00', '2023-10-14 19:00:00'],
    ['2023-10-14 19:00:00', '2023-10-15 07:00:00'],
    ['2023-10-15 07:00:00', '2023-10-15 19:00:00'],
    ['2023-10-15 19:00:00', '2023-10-16 07:00:00'],
    ['2023-10-16 07:00:00', '2023-10-16 19:00:00'],  #here
    ['2023-10-18 07:00:00', '2023-10-18 19:00:00'],
    ['2023-10-18 19:00:00', '2023-10-19 07:00:00'],
    ['2023-10-19 07:00:00', '2023-10-19 19:00:00'],
    ['2023-10-19 19:00:00', '2023-10-20 07:00:00'],
    ['2023-10-20 07:00:00', '2023-10-20 19:00:00'],
    ['2023-10-20 19:00:00', '2023-10-21 07:00:00'],
    ['2023-10-21 07:00:00', '2023-10-21 19:00:00'],
    ['2023-10-21 19:00:00', '2023-10-22 07:00:00'],
    ['2023-10-22 07:00:00', '2023-10-22 19:00:00'],
    ['2023-10-22 19:00:00', '2023-10-23 07:00:00'],
    ['2023-10-23 07:00:00', '2023-10-23 19:00:00'],
    ['2023-10-23 19:00:00', '2023-10-24 07:00:00'],
    ['2023-10-24 07:00:00', '2023-10-24 19:00:00'],
    ['2023-10-24 19:00:00', '2023-10-25 07:00:00'],
    ['2023-10-25 07:00:00', '2023-10-25 19:00:00'],
    ['2023-10-25 19:00:00', '2023-10-26 07:00:00'],
    ['2023-10-26 07:00:00', '2023-10-26 19:00:00'],
    ['2023-10-26 19:00:00', '2023-10-27 07:00:00'],
    ['2023-10-27 07:00:00', '2023-10-27 19:00:00'],
    ['2023-10-27 19:00:00', '2023-10-28 07:00:00'],
    ['2023-10-28 07:00:00', '2023-10-28 19:00:00'],
    ['2023-10-28 19:00:00', '2023-10-29 07:00:00'],
    ['2023-10-29 07:00:00', '2023-10-29 19:00:00'],  #here
    ['2023-10-31 07:00:00', '2023-10-31 19:00:00'],
    ['2023-10-31 19:00:00', '2023-11-01 07:00:00'],
    ['2023-11-01 07:00:00', '2023-11-01 19:00:00'],
    ['2023-11-01 19:00:00', '2023-11-02 07:00:00'],
    ['2023-11-02 07:00:00', '2023-11-02 19:00:00'],
    ['2023-11-02 19:00:00', '2023-11-03 07:00:00'],
    ['2023-11-03 07:00:00', '2023-11-03 19:00:00'],
    ['2023-11-03 19:00:00', '2023-11-04 07:00:00'],
    ['2023-11-04 07:00:00', '2023-11-04 19:00:00'],
    ['2023-11-04 19:00:00', '2023-11-05 07:00:00'],
    ['2023-11-05 07:00:00', '2023-11-05 19:00:00'],
    ['2023-11-05 19:00:00', '2023-11-06 07:00:00'],
    ['2023-11-06 07:00:00', '2023-11-06 19:00:00'],
    ['2023-11-06 19:00:00', '2023-11-07 07:00:00'],
    ['2023-11-07 07:00:00', '2023-11-07 19:00:00'],
    ['2023-11-07 19:00:00', '2023-11-08 07:00:00'],
    ['2023-11-08 07:00:00', '2023-11-08 19:00:00'],
    ['2023-11-08 19:00:00', '2023-11-09 07:00:00'],
    ['2023-11-09 07:00:00', '2023-11-09 19:00:00'],
    ['2023-11-09 19:00:00', '2023-11-10 07:00:00'],
    ['2023-11-10 07:00:00', '2023-11-10 19:00:00'],
    ['2023-11-10 19:00:00', '2023-11-11 07:00:00'],
    ['2023-11-11 07:00:00', '2023-11-11 19:00:00'],
    ['2023-11-11 19:00:00', '2023-11-12 07:00:00'],
    ['2023-11-12 07:00:00', '2023-11-12 19:00:00'],
    ['2023-11-12 19:00:00', '2023-11-13 07:00:00'],
    ['2023-11-13 07:00:00', '2023-11-13 19:00:00'],
    ['2023-11-13 19:00:00', '2023-11-14 07:00:00'],
    ['2023-11-14 07:00:00', '2023-11-14 19:00:00'],
    ['2023-11-14 19:00:00', '2023-11-15 07:00:00'],
    ['2023-11-15 07:00:00', '2023-11-15 19:00:00'],
    ['2023-11-15 19:00:00', '2023-11-16 07:00:00'],
    ['2023-11-16 07:00:00', '2023-11-16 19:00:00'],
    ['2023-11-16 19:00:00', '2023-11-17 07:00:00'],
    ['2023-11-17 07:00:00', '2023-11-17 19:00:00'],
    ['2023-11-17 19:00:00', '2023-11-18 07:00:00'],
    ['2023-11-18 07:00:00', '2023-11-18 19:00:00'],
    ['2023-11-18 19:00:00', '2023-11-19 07:00:00'],
    ['2023-11-19 07:00:00', '2023-11-19 19:00:00'],
    ['2023-11-19 19:00:00', '2023-11-20 07:00:00'],
    ['2023-11-20 07:00:00', '2023-11-20 19:00:00'],  #here
    ['2023-11-22 07:00:00', '2023-11-22 19:00:00'],
    ['2023-11-22 19:00:00', '2023-11-23 07:00:00'],
    ['2023-11-23 07:00:00', '2023-11-23 19:00:00'],
    ['2023-11-23 19:00:00', '2023-11-24 07:00:00'],
    ['2023-11-24 07:00:00', '2023-11-24 19:00:00'],
    ['2023-11-24 19:00:00', '2023-11-25 07:00:00'],
    ['2023-11-25 07:00:00', '2023-11-25 19:00:00'],
    ['2023-11-25 19:00:00', '2023-11-26 07:00:00'],
    ['2023-11-26 07:00:00', '2023-11-26 19:00:00'],
    ['2023-11-26 19:00:00', '2023-11-27 07:00:00'],
    ['2023-11-27 07:00:00', '2023-11-27 19:00:00'],
    ['2023-11-27 19:00:00', '2023-11-28 07:00:00'],
    ['2023-11-28 07:00:00', '2023-11-28 19:00:00'],
    ['2023-11-28 19:00:00', '2023-11-29 07:00:00'],
    ['2023-11-29 07:00:00', '2023-11-29 19:00:00'],
    ['2023-11-29 19:00:00', '2023-11-30 07:00:00'],
    ['2023-11-30 07:00:00', '2023-11-30 19:00:00'],
    ['2023-11-30 19:00:00', '2023-12-01 07:00:00'],
    ['2023-12-01 07:00:00', '2023-12-01 19:00:00'],
    ['2023-12-01 19:00:00', '2023-12-02 07:00:00'],
    ['2023-12-02 07:00:00', '2023-12-02 19:00:00'],
    ['2023-12-02 19:00:00', '2023-12-03 07:00:00'],
    ['2023-12-03 07:00:00', '2023-12-03 19:00:00'],
    ['2023-12-03 19:00:00', '2023-12-04 07:00:00'],
    ['2023-12-04 07:00:00', '2023-12-04 19:00:00'],
    ['2023-12-04 19:00:00', '2023-12-05 07:00:00'],
    ['2023-12-05 07:00:00', '2023-12-05 19:00:00'],
    ['2023-12-05 19:00:00', '2023-12-06 07:00:00'],
    ['2023-12-06 07:00:00', '2023-12-06 19:00:00'],
    ['2023-12-06 19:00:00', '2023-12-07 07:00:00'],
    ['2023-12-07 07:00:00', '2023-12-07 19:00:00'],
    ['2023-12-07 19:00:00', '2023-12-08 07:00:00'],
    ['2023-12-08 07:00:00', '2023-12-08 19:00:00'],
    ['2023-12-08 19:00:00', '2023-12-09 07:00:00'],
    ['2023-12-09 07:00:00', '2023-12-09 19:00:00'],
    ['2023-12-09 19:00:00', '2023-12-10 07:00:00'],
    ['2023-12-10 07:00:00', '2023-12-10 19:00:00'],
    ['2023-12-10 19:00:00', '2023-12-11 07:00:00'],
    ['2023-12-11 07:00:00', '2023-12-11 19:00:00'],
    ['2023-12-11 19:00:00', '2023-12-12 07:00:00'],
    ['2023-12-12 07:00:00', '2023-12-12 19:00:00'],
    ['2023-12-12 19:00:00', '2023-12-13 07:00:00'],
    ['2023-12-13 07:00:00', '2023-12-13 19:00:00'],
    ['2023-12-13 19:00:00', '2023-12-14 07:00:00'],
    ['2023-12-14 07:00:00', '2023-12-14 19:00:00'],  #here
    ['2023-12-16 07:00:00', '2023-12-16 19:00:00'],
    ['2023-12-16 19:00:00', '2023-12-17 07:00:00'],
    ['2023-12-17 07:00:00', '2023-12-17 19:00:00'],  #here
    ['2023-12-19 07:00:00', '2023-12-19 19:00:00'],
    ['2023-12-19 19:00:00', '2023-12-20 07:00:00'],
    ['2023-12-20 07:00:00', '2023-12-20 19:00:00'],
    ['2023-12-20 19:00:00', '2023-12-21 07:00:00'],
    ['2023-12-21 07:00:00', '2023-12-21 19:00:00'],
    ['2023-12-21 19:00:00', '2023-12-22 07:00:00'],
    ['2023-12-22 07:00:00', '2023-12-22 19:00:00'],
    ['2023-12-22 19:00:00', '2023-12-23 07:00:00'],
    ['2023-12-23 07:00:00', '2023-12-23 19:00:00'],
    ['2023-12-23 19:00:00', '2023-12-24 07:00:00'],
    ['2023-12-24 07:00:00', '2023-12-24 19:00:00'],
    ['2023-12-24 19:00:00', '2023-12-25 07:00:00'],
    ['2023-12-25 07:00:00', '2023-12-25 19:00:00'],
    ['2023-12-25 19:00:00', '2023-12-26 07:00:00'],
    ['2023-12-26 07:00:00', '2023-12-26 19:00:00'],
    ['2023-12-26 19:00:00', '2023-12-27 07:00:00'],
    ['2023-12-27 07:00:00', '2023-12-27 19:00:00'],
    ['2023-12-27 19:00:00', '2023-12-28 07:00:00'],
    ['2023-12-28 07:00:00', '2023-12-28 19:00:00'],
    ['2023-12-28 19:00:00', '2023-12-29 07:00:00'],
    ['2023-12-29 07:00:00', '2023-12-29 19:00:00'],
    ['2023-12-29 19:00:00', '2023-12-30 07:00:00'],
    ['2023-12-30 07:00:00', '2023-12-30 19:00:00'],
    ['2023-12-30 19:00:00', '2023-12-31 07:00:00'],
    ['2023-12-31 07:00:00', '2023-12-31 19:00:00'],
    ['2023-12-31 19:00:00', '2024-01-01 07:00:00'],
    ['2024-01-01 07:00:00', '2024-01-01 19:00:00'],
    ['2024-01-01 19:00:00', '2024-01-02 07:00:00'],
    ['2024-01-02 07:00:00', '2024-01-02 19:00:00'],
    ['2024-01-02 19:00:00', '2024-01-03 07:00:00'],
    ['2024-01-03 07:00:00', '2024-01-03 19:00:00'],
    ['2024-01-03 19:00:00', '2024-01-04 07:00:00'],
    ['2024-01-04 07:00:00', '2024-01-04 19:00:00'],
    ['2024-01-04 19:00:00', '2024-01-05 07:00:00'],
    ['2024-01-05 07:00:00', '2024-01-05 19:00:00'],
    ['2024-01-05 19:00:00', '2024-01-06 07:00:00'],
    ['2024-01-06 07:00:00', '2024-01-06 19:00:00'],
    ['2024-01-06 19:00:00', '2024-01-07 07:00:00'],
    ['2024-01-07 07:00:00', '2024-01-07 19:00:00'],
    ['2024-01-07 19:00:00', '2024-01-08 07:00:00'],
    ['2024-01-08 07:00:00', '2024-01-08 19:00:00'],
    ['2024-01-08 19:00:00', '2024-01-09 07:00:00'],
    ['2024-01-09 07:00:00', '2024-01-09 19:00:00'],
    ['2024-01-09 19:00:00', '2024-01-10 07:00:00'],
    ['2024-01-10 07:00:00', '2024-01-10 19:00:00'],
    ['2024-01-10 19:00:00', '2024-01-11 07:00:00'],
    ['2024-01-11 07:00:00', '2024-01-11 19:00:00'],
    ['2024-01-11 19:00:00', '2024-01-12 07:00:00'],
    ['2024-01-12 07:00:00', '2024-01-12 19:00:00'],
    ['2024-01-12 19:00:00', '2024-01-13 07:00:00'],
    ['2024-01-13 07:00:00', '2024-01-13 19:00:00'],
    ['2024-01-13 19:00:00', '2024-01-14 07:00:00'],
    ['2024-01-14 07:00:00', '2024-01-14 19:00:00'],  #here
    ['2024-01-16 07:00:00', '2024-01-16 19:00:00'],
    ['2024-01-16 19:00:00', '2024-01-17 07:00:00'],
    ['2024-01-17 07:00:00', '2024-01-17 19:00:00'],
    ['2024-01-17 19:00:00', '2024-01-18 07:00:00'],
    ['2024-01-18 07:00:00', '2024-01-18 19:00:00'],
    ['2024-01-18 19:00:00', '2024-01-19 07:00:00'],
    ['2024-01-19 07:00:00', '2024-01-19 19:00:00'],
    ['2024-01-19 19:00:00', '2024-01-20 07:00:00'],
    ['2024-01-20 07:00:00', '2024-01-20 19:00:00'],
    ['2024-01-20 19:00:00', '2024-01-21 07:00:00'],
    ['2024-01-21 07:00:00', '2024-01-21 19:00:00'],
    ['2024-01-21 19:00:00', '2024-01-22 07:00:00'],
    ['2024-01-22 07:00:00', '2024-01-22 19:00:00'],
    ['2024-01-22 19:00:00', '2024-01-23 07:00:00'],
    ['2024-01-23 07:00:00', '2024-01-23 19:00:00'],
    ['2024-01-23 19:00:00', '2024-01-24 07:00:00'],
    ['2024-01-24 07:00:00', '2024-01-24 19:00:00'],
    ['2024-01-24 19:00:00', '2024-01-25 07:00:00'],
    ['2024-01-25 07:00:00', '2024-01-25 19:00:00'],
    ['2024-01-25 19:00:00', '2024-01-26 07:00:00'],
    ['2024-01-26 07:00:00', '2024-01-26 19:00:00'],
    ['2024-01-26 19:00:00', '2024-01-27 07:00:00'],
    ['2024-01-27 07:00:00', '2024-01-27 19:00:00'],
    ['2024-01-27 19:00:00', '2024-01-28 07:00:00'],
    ['2024-01-28 07:00:00', '2024-01-28 19:00:00'],
    ['2024-01-28 19:00:00', '2024-01-29 07:00:00'],
    ['2024-01-29 07:00:00', '2024-01-29 19:00:00'],
    ['2024-01-29 19:00:00', '2024-01-30 07:00:00'],
    ['2024-01-30 07:00:00', '2024-01-30 19:00:00'],
    ['2024-01-30 19:00:00', '2024-01-31 07:00:00'],
    ['2024-01-31 07:00:00', '2024-01-31 19:00:00'],
    ['2024-01-31 19:00:00', '2024-02-01 07:00:00'],
    ['2024-02-01 07:00:00', '2024-02-01 19:00:00'],
    ['2024-02-01 19:00:00', '2024-02-02 07:00:00'],
    ['2024-02-02 07:00:00', '2024-02-02 19:00:00'],
    ['2024-02-02 19:00:00', '2024-02-03 07:00:00'],
    ['2024-02-03 07:00:00', '2024-02-03 19:00:00'],
    ['2024-02-03 19:00:00', '2024-02-04 07:00:00'],
    ['2024-02-04 07:00:00', '2024-02-04 19:00:00'],
    ['2024-02-04 19:00:00', '2024-02-05 07:00:00'],
    ['2024-02-05 07:00:00', '2024-02-05 19:00:00'],
    ['2024-02-05 19:00:00', '2024-02-06 07:00:00'],
    ['2024-02-06 07:00:00', '2024-02-06 19:00:00'],
    ['2024-02-06 19:00:00', '2024-02-07 07:00:00'],
    ['2024-02-07 07:00:00', '2024-02-07 19:00:00'],
    ['2024-02-07 19:00:00', '2024-02-08 07:00:00'],
    ['2024-02-08 07:00:00', '2024-02-08 19:00:00'],
    ['2024-02-08 19:00:00', '2024-02-09 07:00:00'],
    ['2024-02-09 07:00:00', '2024-02-09 19:00:00'],
    ['2024-02-09 19:00:00', '2024-02-10 07:00:00'],
    ['2024-02-10 07:00:00', '2024-02-10 19:00:00'],
    ['2024-02-10 19:00:00', '2024-02-11 07:00:00'],
    ['2024-02-11 07:00:00', '2024-02-11 19:00:00'],
    ['2024-02-11 19:00:00', '2024-02-12 07:00:00'],
    ['2024-02-12 07:00:00', '2024-02-12 19:00:00'],
    ['2024-02-12 19:00:00', '2024-02-13 07:00:00'],
    ['2024-02-13 07:00:00', '2024-02-13 19:00:00'],
    ['2024-02-13 19:00:00', '2024-02-14 07:00:00'],
    ['2024-02-14 07:00:00', '2024-02-14 19:00:00'],
    ['2024-02-14 19:00:00', '2024-02-15 07:00:00'],
    ['2024-02-15 07:00:00', '2024-02-15 19:00:00'],
    ['2024-02-15 19:00:00', '2024-02-16 07:00:00'],
    ['2024-02-16 07:00:00', '2024-02-16 19:00:00'],
    ['2024-02-16 19:00:00', '2024-02-17 07:00:00'],
    ['2024-02-17 07:00:00', '2024-02-17 19:00:00'],
    ['2024-02-17 19:00:00', '2024-02-18 07:00:00'],
    ['2024-02-18 07:00:00', '2024-02-18 19:00:00'],
    ['2024-02-18 19:00:00', '2024-02-19 07:00:00'],
    ['2024-02-19 07:00:00', '2024-02-19 19:00:00'],
    ['2024-02-19 19:00:00', '2024-02-20 07:00:00'],
    ['2024-02-20 07:00:00', '2024-02-20 19:00:00'],
    ['2024-02-20 19:00:00', '2024-02-21 07:00:00'],
    ['2024-02-21 07:00:00', '2024-02-21 19:00:00'],
    ['2024-02-21 19:00:00', '2024-02-22 07:00:00'],
    ['2024-02-22 07:00:00', '2024-02-22 19:00:00'],
    ['2024-02-22 19:00:00', '2024-02-23 07:00:00'],
    ['2024-02-23 07:00:00', '2024-02-23 19:00:00'],
    ['2024-02-23 19:00:00', '2024-02-24 07:00:00'],
    ['2024-02-24 07:00:00', '2024-02-24 19:00:00'],
    ['2024-02-24 19:00:00', '2024-02-25 07:00:00'],
    ['2024-02-25 07:00:00', '2024-02-25 19:00:00'],  #here
    ['2024-02-27 07:00:00', '2024-02-27 19:00:00'],
    ['2024-02-27 19:00:00', '2024-02-28 07:00:00'],
    ['2024-02-28 07:00:00', '2024-02-28 19:00:00'],
    ['2024-02-28 19:00:00', '2024-02-29 07:00:00'],
    ['2024-02-29 07:00:00', '2024-02-29 19:00:00'],
    ['2024-02-29 19:00:00', '2024-03-01 07:00:00'],
    ['2024-03-01 07:00:00', '2024-03-01 19:00:00'],
    ['2024-03-01 19:00:00', '2024-03-02 07:00:00'],
    ['2024-03-02 07:00:00', '2024-03-02 19:00:00'],
    ['2024-03-02 19:00:00', '2024-03-03 07:00:00'],
    ['2024-03-03 07:00:00', '2024-03-03 19:00:00'],
    ['2024-03-03 19:00:00', '2024-03-04 07:00:00'],
    ['2024-03-04 07:00:00', '2024-03-04 19:00:00'],
    ['2024-03-04 19:00:00', '2024-03-05 07:00:00'],
    ['2024-03-05 07:00:00', '2024-03-05 19:00:00'],
    ['2024-03-05 19:00:00', '2024-03-06 07:00:00'],
    ['2024-03-06 07:00:00', '2024-03-06 19:00:00'],
    ['2024-03-06 19:00:00', '2024-03-07 07:00:00'],
    ['2024-03-07 07:00:00', '2024-03-07 19:00:00'],
    ['2024-03-07 19:00:00', '2024-03-08 07:00:00'],
    ['2024-03-08 07:00:00', '2024-03-08 19:00:00'],
    ['2024-03-08 19:00:00', '2024-03-09 07:00:00'],
    ['2024-03-09 07:00:00', '2024-03-09 19:00:00'],
    ['2024-03-09 19:00:00', '2024-03-10 07:00:00'],
    ['2024-03-10 07:00:00', '2024-03-10 19:00:00'],
    ['2024-03-10 19:00:00', '2024-03-11 07:00:00'],
    ['2024-03-11 07:00:00', '2024-03-11 19:00:00'],
    ['2024-03-11 19:00:00', '2024-03-12 07:00:00'],
    ['2024-03-12 07:00:00', '2024-03-12 19:00:00'],
    ['2024-03-12 19:00:00', '2024-03-13 07:00:00'],
    ['2024-03-13 07:00:00', '2024-03-13 19:00:00'],
    ['2024-03-13 19:00:00', '2024-03-14 07:00:00'],
    ['2024-03-14 07:00:00', '2024-03-14 19:00:00'],
    ['2024-03-14 19:00:00', '2024-03-15 07:00:00'],
    ['2024-03-15 07:00:00', '2024-03-15 19:00:00'],
    ['2024-03-15 19:00:00', '2024-03-16 07:00:00'],
    ['2024-03-16 07:00:00', '2024-03-16 19:00:00'],
    ['2024-03-16 19:00:00', '2024-03-17 07:00:00'],
    ['2024-03-17 07:00:00', '2024-03-17 19:00:00'],
    ['2024-03-17 19:00:00', '2024-03-18 07:00:00'],
    ['2024-03-18 07:00:00', '2024-03-18 19:00:00'],  #here
    ['2024-03-20 07:00:00', '2024-03-20 19:00:00'],
    ['2024-03-20 19:00:00', '2024-03-21 07:00:00'],
    ['2024-03-21 07:00:00', '2024-03-21 19:00:00'],
    ['2024-03-21 19:00:00', '2024-03-22 07:00:00'],
    ['2024-03-22 07:00:00', '2024-03-22 19:00:00'],
    ['2024-03-22 19:00:00', '2024-03-23 07:00:00'],
    ['2024-03-23 07:00:00', '2024-03-23 19:00:00'],
    ['2024-03-23 19:00:00', '2024-03-24 07:00:00'],
    ['2024-03-24 07:00:00', '2024-03-24 19:00:00'],
    ['2024-03-24 19:00:00', '2024-03-25 07:00:00'],
    ['2024-03-25 07:00:00', '2024-03-25 19:00:00'],  #here
    ['2024-03-27 07:00:00', '2024-03-27 19:00:00'],
    ['2024-03-27 19:00:00', '2024-03-28 07:00:00'],
    ['2024-03-28 07:00:00', '2024-03-28 19:00:00'],
    ['2024-03-28 19:00:00', '2024-03-29 07:00:00'],
    ['2024-03-29 07:00:00', '2024-03-29 19:00:00'],
    ['2024-03-29 19:00:00', '2024-03-30 07:00:00'],
    ['2024-03-30 07:00:00', '2024-03-30 19:00:00'],
    ['2024-04-01 07:00:00', '2024-04-01 19:00:00'],  #here
    ['2024-04-01 19:00:00', '2024-04-02 07:00:00'],
    ['2024-04-02 07:00:00', '2024-04-02 19:00:00'],
    ['2024-04-02 19:00:00', '2024-04-03 07:00:00'],
    ['2024-04-03 07:00:00', '2024-04-03 19:00:00'],
    ['2024-04-03 19:00:00', '2024-04-04 07:00:00'],
    ['2024-04-04 07:00:00', '2024-04-04 19:00:00'],
    ['2024-04-04 19:00:00', '2024-04-05 07:00:00'],
    ['2024-04-05 07:00:00', '2024-04-05 19:00:00'],
    ['2024-04-05 19:00:00', '2024-04-06 07:00:00'],
    ['2024-04-06 07:00:00', '2024-04-06 19:00:00'],
    ['2024-04-06 19:00:00', '2024-04-07 07:00:00'],
    ['2024-04-07 07:00:00', '2024-04-07 19:00:00'],
    ['2024-04-07 19:00:00', '2024-04-08 07:00:00'],
    ['2024-04-08 07:00:00', '2024-04-08 19:00:00'],
    ['2024-04-08 19:00:00', '2024-04-09 07:00:00'],
    ['2024-04-09 07:00:00', '2024-04-09 19:00:00'],
    ['2024-04-09 19:00:00', '2024-04-10 07:00:00'],
    ['2024-04-10 07:00:00', '2024-04-10 19:00:00'],
    ['2024-04-10 19:00:00', '2024-04-11 07:00:00'],
    ['2024-04-11 07:00:00', '2024-04-11 19:00:00'],
    ['2024-04-11 19:00:00', '2024-04-12 07:00:00'],
    ['2024-04-12 07:00:00', '2024-04-12 19:00:00'],
    ['2024-04-12 19:00:00', '2024-04-13 07:00:00'],
    ['2024-04-13 07:00:00', '2024-04-13 19:00:00'],
    ['2024-04-13 19:00:00', '2024-04-14 07:00:00'],
    ['2024-04-14 07:00:00', '2024-04-14 19:00:00'],
    ['2024-04-14 19:00:00', '2024-04-15 07:00:00'],
    ['2024-04-15 07:00:00', '2024-04-15 19:00:00'],
    ['2024-04-15 19:00:00', '2024-04-16 07:00:00'],
    ['2024-04-16 07:00:00', '2024-04-16 19:00:00'],
    ['2024-04-16 19:00:00', '2024-04-17 07:00:00'],
    ['2024-04-17 07:00:00', '2024-04-17 19:00:00'],  #here
    ['2024-04-19 07:00:00', '2024-04-19 19:00:00'],
    ['2024-04-19 19:00:00', '2024-04-20 07:00:00'],
    ['2024-04-20 07:00:00', '2024-04-20 19:00:00'],
    ['2024-04-20 19:00:00', '2024-04-21 07:00:00'],
    ['2024-04-21 07:00:00', '2024-04-21 19:00:00'],
    ['2024-04-21 19:00:00', '2024-04-22 07:00:00'],
    ['2024-04-22 07:00:00', '2024-04-22 19:00:00'],
    ['2024-04-22 19:00:00', '2024-04-23 07:00:00'],
    ['2024-04-23 07:00:00', '2024-04-23 19:00:00'],  #here
    ['2024-04-25 07:00:00', '2024-04-25 19:00:00'],
    ['2024-04-25 19:00:00', '2024-04-26 07:00:00'],
    ['2024-04-26 07:00:00', '2024-04-26 19:00:00'],
    ['2024-04-26 19:00:00', '2024-04-27 07:00:00'],
    ['2024-04-27 07:00:00', '2024-04-27 19:00:00'],
    ['2024-04-27 19:00:00', '2024-04-28 07:00:00'],
    ['2024-04-28 07:00:00', '2024-04-28 19:00:00'],
    ['2024-04-28 19:00:00', '2024-04-29 07:00:00'],
    ['2024-04-29 07:00:00', '2024-04-29 19:00:00'],
    ['2024-04-29 19:00:00', '2024-04-30 07:00:00'],
    ['2024-04-30 07:00:00', '2024-04-30 19:00:00'],
    ['2024-04-30 19:00:00', '2024-05-01 07:00:00'],
    ['2024-05-01 07:00:00', '2024-05-01 19:00:00'],
    ['2024-05-01 19:00:00', '2024-05-02 07:00:00'],
    ['2024-05-02 07:00:00', '2024-05-02 19:00:00'],
    ['2024-05-02 19:00:00', '2024-05-03 07:00:00'],
    ['2024-05-03 07:00:00', '2024-05-03 19:00:00'],
    ['2024-05-03 19:00:00', '2024-05-04 07:00:00'],
    ['2024-05-04 07:00:00', '2024-05-04 19:00:00'],
    ['2024-05-04 19:00:00', '2024-05-05 07:00:00'],
    ['2024-05-05 07:00:00', '2024-05-05 19:00:00'],
    ['2024-05-05 19:00:00', '2024-05-06 07:00:00'],
    ['2024-05-06 07:00:00', '2024-05-06 19:00:00'],
    ['2024-05-06 19:00:00', '2024-05-07 07:00:00'],
    ['2024-05-07 07:00:00', '2024-05-07 19:00:00'],
    ['2024-05-07 19:00:00', '2024-05-08 07:00:00'],
    ['2024-05-08 07:00:00', '2024-05-08 19:00:00'],
    ['2024-05-08 19:00:00', '2024-05-09 07:00:00'],
    ['2024-05-09 07:00:00', '2024-05-09 19:00:00'],
    ['2024-05-09 19:00:00', '2024-05-10 07:00:00'],
    ['2024-05-10 07:00:00', '2024-05-10 19:00:00'],
    ['2024-05-10 19:00:00', '2024-05-11 07:00:00'],
    ['2024-05-11 07:00:00', '2024-05-11 19:00:00'],
    ['2024-05-11 19:00:00', '2024-05-12 07:00:00'],
    ['2024-05-12 07:00:00', '2024-05-12 19:00:00'],
    ['2024-05-12 19:00:00', '2024-05-13 07:00:00'],
    ['2024-05-13 07:00:00', '2024-05-13 19:00:00'],  #here
    ['2024-05-16 07:00:00', '2024-05-16 19:00:00'],
    ['2024-05-16 19:00:00', '2024-05-17 07:00:00'],
    ['2024-05-17 07:00:00', '2024-05-17 19:00:00'],
    ['2024-05-17 19:00:00', '2024-05-18 07:00:00'],
    ['2024-05-18 07:00:00', '2024-05-18 19:00:00'],
    ['2024-05-18 19:00:00', '2024-05-19 07:00:00'],
    ['2024-05-19 07:00:00', '2024-05-19 19:00:00'],
    ['2024-05-19 19:00:00', '2024-05-20 07:00:00'],
    ['2024-05-20 07:00:00', '2024-05-20 19:00:00'],
    ['2024-05-20 19:00:00', '2024-05-21 07:00:00'],
    ['2024-05-21 07:00:00', '2024-05-21 19:00:00'],
    ['2024-05-21 19:00:00', '2024-05-22 07:00:00'],
    ['2024-05-22 07:00:00', '2024-05-22 19:00:00'],
    ['2024-05-22 19:00:00', '2024-05-23 07:00:00'],
    ['2024-05-23 07:00:00', '2024-05-23 19:00:00'],
    ['2024-05-23 19:00:00', '2024-05-24 07:00:00'],
    ['2024-05-24 07:00:00', '2024-05-24 19:00:00'],
    ['2024-05-24 19:00:00', '2024-05-25 07:00:00'],
    ['2024-05-25 07:00:00', '2024-05-25 19:00:00'],
    ['2024-05-25 19:00:00', '2024-05-26 07:00:00'],
    ['2024-05-26 07:00:00', '2024-05-26 19:00:00'],
    ['2024-05-26 19:00:00', '2024-05-27 07:00:00'],
    ['2024-05-27 07:00:00', '2024-05-27 19:00:00'],
    ['2024-05-27 19:00:00', '2024-05-28 07:00:00'],
    ['2024-05-28 07:00:00', '2024-05-28 19:00:00'],
    ['2024-05-28 19:00:00', '2024-05-29 07:00:00'],
    ['2024-05-29 07:00:00', '2024-05-29 19:00:00'],  #here
    ['2024-05-31 07:00:00', '2024-05-31 19:00:00'],
    ['2024-05-31 19:00:00', '2024-06-01 07:00:00'],
    ['2024-06-01 07:00:00', '2024-06-01 19:00:00'],
    ['2024-06-01 19:00:00', '2024-06-02 07:00:00'],
    ['2024-06-02 07:00:00', '2024-06-02 19:00:00'],
    ['2024-06-02 19:00:00', '2024-06-03 07:00:00'],
    ['2024-06-03 07:00:00', '2024-06-03 19:00:00'],
    ['2024-06-03 19:00:00', '2024-06-04 07:00:00'],
    ['2024-06-04 07:00:00', '2024-06-04 19:00:00'],
    ['2024-06-04 19:00:00', '2024-06-05 07:00:00'],
    ['2024-06-05 07:00:00', '2024-06-05 19:00:00'],
    ['2024-06-05 19:00:00', '2024-06-06 07:00:00'],
    ['2024-06-06 07:00:00', '2024-06-06 19:00:00'],
    ['2024-06-06 19:00:00', '2024-06-07 07:00:00'],
    ['2024-06-07 07:00:00', '2024-06-07 19:00:00'],
    ['2024-06-07 19:00:00', '2024-06-08 07:00:00'],
    ['2024-06-08 07:00:00', '2024-06-08 19:00:00'],
    ['2024-06-08 19:00:00', '2024-06-09 07:00:00'],
    ['2024-06-09 07:00:00', '2024-06-09 19:00:00'],
    ['2024-06-09 19:00:00', '2024-06-10 07:00:00'],
    ['2024-06-10 07:00:00', '2024-06-10 19:00:00'],
    ['2024-06-10 19:00:00', '2024-06-11 07:00:00'],
    ['2024-06-11 07:00:00', '2024-06-11 19:00:00'],
    ['2024-06-11 19:00:00', '2024-06-12 07:00:00'],
    ['2024-06-12 07:00:00', '2024-06-12 19:00:00'],
    ['2024-06-12 19:00:00', '2024-06-13 07:00:00'],
    ['2024-06-13 07:00:00', '2024-06-13 19:00:00'],
    ['2024-06-13 19:00:00', '2024-06-14 07:00:00'],
    ['2024-06-14 07:00:00', '2024-06-14 19:00:00'],
    ['2024-06-14 19:00:00', '2024-06-15 07:00:00'],
    ['2024-06-15 07:00:00', '2024-06-15 19:00:00'],
    ['2024-06-15 19:00:00', '2024-06-16 07:00:00'],
    ['2024-06-16 07:00:00', '2024-06-16 19:00:00'],
    ['2024-06-16 19:00:00', '2024-06-17 07:00:00'],
    ['2024-06-17 07:00:00', '2024-06-17 19:00:00'],  #here
    ['2024-06-19 07:00:00', '2024-06-19 19:00:00'],
    ['2024-06-19 19:00:00', '2024-06-20 07:00:00'],
    ['2024-06-20 07:00:00', '2024-06-20 19:00:00'],
    ['2024-06-20 19:00:00', '2024-06-21 07:00:00'],
    ['2024-06-21 07:00:00', '2024-06-21 19:00:00'],
    ['2024-06-21 19:00:00', '2024-06-22 07:00:00'],
    ['2024-06-22 07:00:00', '2024-06-22 19:00:00'],
    ['2024-06-22 19:00:00', '2024-06-23 07:00:00'],
    ['2024-06-23 07:00:00', '2024-06-23 19:00:00'],
    ['2024-06-23 19:00:00', '2024-06-24 07:00:00'],
    ['2024-06-24 07:00:00', '2024-06-24 19:00:00'],
    ['2024-06-24 19:00:00', '2024-06-25 07:00:00'],
    ['2024-06-25 07:00:00', '2024-06-25 19:00:00'],
    ['2024-06-25 19:00:00', '2024-06-26 07:00:00'],
    ['2024-06-28 19:00:00', '2024-06-29 07:00:00'],  #here
    ['2024-06-29 07:00:00', '2024-06-29 19:00:00'],
    ['2024-06-29 19:00:00', '2024-06-30 07:00:00'],
    ['2024-06-30 07:00:00', '2024-06-30 19:00:00'],
    ['2024-06-30 19:00:00', '2024-07-01 07:00:00'],
    ['2024-07-01 07:00:00', '2024-07-01 19:00:00'],
    ['2024-07-01 19:00:00', '2024-07-02 07:00:00'],
    ['2024-07-02 07:00:00', '2024-07-02 19:00:00'],
    ['2024-07-02 19:00:00', '2024-07-03 07:00:00'],
    ['2024-07-03 07:00:00', '2024-07-03 19:00:00'],
    ['2024-07-03 19:00:00', '2024-07-04 07:00:00'],
    ['2024-07-04 07:00:00', '2024-07-04 19:00:00'],
    ['2024-07-04 19:00:00', '2024-07-05 07:00:00'],
    ['2024-07-05 07:00:00', '2024-07-05 19:00:00'],
    ['2024-07-05 19:00:00', '2024-07-06 07:00:00'],
    ['2024-07-06 07:00:00', '2024-07-06 19:00:00'],
    ['2024-07-06 19:00:00', '2024-07-07 07:00:00'],
    ['2024-07-07 07:00:00', '2024-07-07 19:00:00'],
    ['2024-07-07 19:00:00', '2024-07-08 07:00:00'],
    ['2024-07-08 07:00:00', '2024-07-08 19:00:00'],
    ['2024-07-08 19:00:00', '2024-07-09 07:00:00'],
    ['2024-07-09 07:00:00', '2024-07-09 19:00:00'],
    ['2024-07-09 19:00:00', '2024-07-10 07:00:00'],
    ['2024-07-10 07:00:00', '2024-07-10 19:00:00'],
    ['2024-07-10 19:00:00', '2024-07-11 07:00:00'],
    ['2024-07-11 07:00:00', '2024-07-11 19:00:00'],
    ['2024-07-11 19:00:00', '2024-07-12 07:00:00'],
    ['2024-07-12 07:00:00', '2024-07-12 19:00:00'],
    ['2024-07-12 19:00:00', '2024-07-13 07:00:00'],
    ['2024-07-13 07:00:00', '2024-07-13 19:00:00'],
    ['2024-07-13 19:00:00', '2024-07-14 07:00:00'],
    ['2024-07-14 07:00:00', '2024-07-14 19:00:00'],
    ['2024-07-14 19:00:00', '2024-07-15 07:00:00'],
    ['2024-07-15 07:00:00', '2024-07-15 19:00:00'],
    ['2024-07-15 19:00:00', '2024-07-16 07:00:00'],
    ['2024-07-16 07:00:00', '2024-07-16 19:00:00'],
    ['2024-07-16 19:00:00', '2024-07-17 07:00:00'],
    ['2024-07-17 07:00:00', '2024-07-17 19:00:00'],  #here
    ['2024-07-19 07:00:00', '2024-07-19 19:00:00'],
    ['2024-07-19 19:00:00', '2024-07-20 07:00:00'],
    ['2024-07-20 07:00:00', '2024-07-20 19:00:00'],
    ['2024-07-20 19:00:00', '2024-07-21 07:00:00'],
    ['2024-07-21 07:00:00', '2024-07-21 19:00:00'],
    ['2024-07-21 19:00:00', '2024-07-22 07:00:00'],
    ['2024-07-22 07:00:00', '2024-07-22 19:00:00'],
    ['2024-07-22 19:00:00', '2024-07-23 07:00:00'],
    ['2024-07-23 07:00:00', '2024-07-23 19:00:00'],
    ['2024-07-23 19:00:00', '2024-07-24 07:00:00'],
    ['2024-07-24 07:00:00', '2024-07-24 19:00:00'],
    ['2024-07-24 19:00:00', '2024-07-25 07:00:00'],
    ['2024-07-25 07:00:00', '2024-07-25 19:00:00'],
    ['2024-07-25 19:00:00', '2024-07-26 07:00:00'],
    ['2024-07-26 07:00:00', '2024-07-26 19:00:00'],
    ['2024-07-26 19:00:00', '2024-07-27 07:00:00'],
    ['2024-07-27 07:00:00', '2024-07-27 19:00:00'],
    ['2024-07-27 19:00:00', '2024-07-28 07:00:00'],
    ['2024-07-28 07:00:00', '2024-07-28 19:00:00'],
    ['2024-07-28 19:00:00', '2024-07-29 07:00:00'],
    ['2024-07-29 07:00:00', '2024-07-29 19:00:00'],
    ['2024-07-29 19:00:00', '2024-07-30 07:00:00'],
    ['2024-07-30 07:00:00', '2024-07-30 19:00:00'],  #here
    ['2024-08-04 19:00:00', '2024-08-05 07:00:00'],
    ['2024-08-05 07:00:00', '2024-08-05 19:00:00'],
    ['2024-08-05 19:00:00', '2024-08-06 07:00:00'],
    ['2024-08-06 07:00:00', '2024-08-06 19:00:00'],
    ['2024-08-06 19:00:00', '2024-08-07 07:00:00'],
    ['2024-08-07 07:00:00', '2024-08-07 19:00:00'],
    ['2024-08-07 19:00:00', '2024-08-08 07:00:00'],
    ['2024-08-08 07:00:00', '2024-08-08 19:00:00'],
    ['2024-08-08 19:00:00', '2024-08-09 07:00:00'],
    ['2024-08-09 07:00:00', '2024-08-09 19:00:00'],
    ['2024-08-09 19:00:00', '2024-08-10 07:00:00'],
    ['2024-08-10 07:00:00', '2024-08-10 19:00:00'],
    ['2024-08-10 19:00:00', '2024-08-11 07:00:00'],
    ['2024-08-11 07:00:00', '2024-08-11 19:00:00'],
    ['2024-08-11 19:00:00', '2024-08-12 07:00:00'],
    ['2024-08-12 07:00:00', '2024-08-12 19:00:00'],
    ['2024-08-12 19:00:00', '2024-08-13 07:00:00'],
    ['2024-08-13 07:00:00', '2024-08-13 19:00:00'],
    ['2024-08-13 19:00:00', '2024-08-14 07:00:00'],
    ['2024-08-14 07:00:00', '2024-08-14 19:00:00'],
    ['2024-08-14 19:00:00', '2024-08-15 07:00:00'],
    ['2024-08-15 07:00:00', '2024-08-15 19:00:00'],
    ['2024-08-15 19:00:00', '2024-08-16 07:00:00'],
    ['2024-08-16 07:00:00', '2024-08-16 19:00:00'],
    ['2024-08-16 19:00:00', '2024-08-17 07:00:00'],
    ['2024-08-17 07:00:00', '2024-08-17 19:00:00'],
    ['2024-08-17 19:00:00', '2024-08-18 07:00:00'],
    ['2024-08-18 07:00:00', '2024-08-18 19:00:00'],  #here
    ['2024-08-20 07:00:00', '2024-08-20 19:00:00'],
    ['2024-08-20 19:00:00', '2024-08-21 07:00:00'],
    ['2024-08-21 07:00:00', '2024-08-21 19:00:00'],
    ['2024-08-21 19:00:00', '2024-08-22 07:00:00'],
    ['2024-08-22 07:00:00', '2024-08-22 19:00:00'],
    ['2024-08-22 19:00:00', '2024-08-23 07:00:00'],
    ['2024-08-23 07:00:00', '2024-08-23 19:00:00'],
    ['2024-08-23 19:00:00', '2024-08-24 07:00:00'],
    ['2024-08-24 07:00:00', '2024-08-24 19:00:00'],
    ['2024-08-24 19:00:00', '2024-08-25 07:00:00'],
    ['2024-08-25 07:00:00', '2024-08-25 19:00:00']
]

def convert_interval(start_str, end_str):
    """
    Ritorna la coppia (start_str, end_str) convertita
    sostituendo i range 07:00-19:00 -> 07:00-17:00
                      19:00-07:00 -> 23:00-05:00
    mantenendo inalterati data e day shift.
    """
    fmt = "%Y-%m-%d %H:%M:%S"
    start_dt = datetime.strptime(start_str, fmt)
    end_dt = datetime.strptime(end_str, fmt)
    
    start_hour = start_dt.hour
    end_hour   = end_dt.hour
    
    # Caso 07:00 -> 19:00 (stessa data)
    if start_hour == 7 and end_hour == 19 and start_dt.date() == end_dt.date():
        new_start = start_dt.replace(hour= 7, minute=0, second=0)
        new_end   = end_dt.replace(hour=17, minute=0, second=0)
        return [new_start.strftime(fmt), new_end.strftime(fmt)]
    
    # Caso 19:00 -> 07:00 (giorno successivo)
    # start: 19:00 su day X, end: 07:00 su day X+1
    # => 23:00 su day X, 05:00 su day X+1
    if start_hour == 19:
        # Verifichiamo che end_dt sia effettivamente il giorno dopo
        next_day = start_dt + timedelta(days=1)
        if end_hour == 7 and end_dt.date() == next_day.date():
            new_start = start_dt.replace(hour=23, minute=0, second=0)
            new_end   = end_dt.replace(hour=5,  minute=0, second=0)
            return [new_start.strftime(fmt), new_end.strftime(fmt)]
    
    # Se non rientra nei pattern, restituiamo l'intervallo invariato
    return [start_str, end_str]


converted = []
for interval in original_intervals:
    start_str, end_str = interval
    new_pair = convert_interval(start_str, end_str)
    converted.append(new_pair)

# Stampa finale del risultato
print("[")
for c in converted:
    print(f"    ['{c[0]}', '{c[1]}'],")
print("]")

In [ ]:
intervals = [
    ['2022-09-22 11:00:00', '2022-09-22 15:00:00'],
    ['2022-09-22 23:00:00', '2022-09-23 05:00:00'],
    ['2022-09-23 11:00:00', '2022-09-23 15:00:00'],
    ['2022-09-23 23:00:00', '2022-09-24 05:00:00'],
    ['2022-09-24 11:00:00', '2022-09-24 15:00:00'],
    ['2022-09-24 23:00:00', '2022-09-25 05:00:00'],
    ['2022-09-25 11:00:00', '2022-09-25 15:00:00'],
    ['2022-09-25 23:00:00', '2022-09-26 05:00:00'],
    ['2022-09-26 11:00:00', '2022-09-26 15:00:00'],
    ['2022-09-26 23:00:00', '2022-09-27 05:00:00'],
    ['2022-09-27 11:00:00', '2022-09-27 15:00:00'],
    ['2022-09-27 23:00:00', '2022-09-28 05:00:00'],
    ['2022-09-28 11:00:00', '2022-09-28 15:00:00'],
    ['2022-09-28 23:00:00', '2022-09-29 05:00:00'],
    ['2022-09-29 11:00:00', '2022-09-29 15:00:00'],
    ['2022-09-29 23:00:00', '2022-09-30 05:00:00'],
    ['2022-09-30 11:00:00', '2022-09-30 15:00:00'],
    ['2022-09-30 23:00:00', '2022-10-01 05:00:00'],
    ['2022-10-01 11:00:00', '2022-10-01 15:00:00'],
    ['2022-10-01 23:00:00', '2022-10-02 05:00:00'],
    ['2022-10-02 11:00:00', '2022-10-02 15:00:00'],
    ['2022-10-02 23:00:00', '2022-10-03 05:00:00'],
    ['2022-10-03 11:00:00', '2022-10-03 15:00:00'],
    ['2022-10-03 23:00:00', '2022-10-04 05:00:00'],
    ['2022-10-04 11:00:00', '2022-10-04 15:00:00'],
    ['2022-10-04 23:00:00', '2022-10-05 05:00:00'],
    ['2022-10-05 11:00:00', '2022-10-05 15:00:00'],
    ['2022-10-05 23:00:00', '2022-10-06 05:00:00'],
    ['2022-10-06 11:00:00', '2022-10-06 15:00:00'],
    ['2022-10-06 23:00:00', '2022-10-07 05:00:00'],
    ['2022-10-07 11:00:00', '2022-10-07 15:00:00'],
    ['2022-10-07 23:00:00', '2022-10-08 05:00:00'],
    ['2022-10-08 11:00:00', '2022-10-08 15:00:00'],
    ['2022-10-08 23:00:00', '2022-10-09 05:00:00'],
    ['2022-10-09 11:00:00', '2022-10-09 15:00:00'],
    ['2022-10-09 23:00:00', '2022-10-10 05:00:00'],
    ['2022-10-10 11:00:00', '2022-10-10 15:00:00'],
    ['2022-10-10 23:00:00', '2022-10-11 05:00:00'],
    ['2022-10-11 11:00:00', '2022-10-11 15:00:00'],
    ['2022-10-11 23:00:00', '2022-10-12 05:00:00'],
    ['2022-10-12 11:00:00', '2022-10-12 15:00:00'],
    ['2022-10-12 23:00:00', '2022-10-13 05:00:00'],
    ['2022-10-13 11:00:00', '2022-10-13 15:00:00'],
    ['2022-10-13 23:00:00', '2022-10-14 05:00:00'],
    ['2022-10-14 11:00:00', '2022-10-14 15:00:00'],
    ['2022-10-14 23:00:00', '2022-10-15 05:00:00'],
    ['2022-10-15 11:00:00', '2022-10-15 15:00:00'],
    ['2022-10-15 23:00:00', '2022-10-16 05:00:00'],
    ['2022-10-16 11:00:00', '2022-10-16 15:00:00'],
    ['2022-10-16 23:00:00', '2022-10-17 05:00:00'],
    ['2022-10-17 11:00:00', '2022-10-17 15:00:00'],
    ['2022-10-19 11:00:00', '2022-10-19 15:00:00'],
    ['2022-10-19 23:00:00', '2022-10-20 05:00:00'],
    ['2022-10-20 11:00:00', '2022-10-20 15:00:00'],
    ['2022-10-20 23:00:00', '2022-10-21 05:00:00'],
    ['2022-10-21 11:00:00', '2022-10-21 15:00:00'],
    ['2022-10-21 23:00:00', '2022-10-22 05:00:00'],
    ['2022-10-22 11:00:00', '2022-10-22 15:00:00'],
    ['2022-10-22 23:00:00', '2022-10-23 05:00:00'],
    ['2022-10-23 11:00:00', '2022-10-23 15:00:00'],
    ['2022-10-23 23:00:00', '2022-10-24 05:00:00'],
    ['2022-10-24 11:00:00', '2022-10-24 15:00:00'],
    ['2022-10-24 23:00:00', '2022-10-25 05:00:00'],
    ['2022-10-25 11:00:00', '2022-10-25 15:00:00'],
    ['2022-10-25 23:00:00', '2022-10-26 05:00:00'],
    ['2022-10-26 11:00:00', '2022-10-26 15:00:00'],
    ['2022-10-26 23:00:00', '2022-10-27 05:00:00'],
    ['2022-10-27 11:00:00', '2022-10-27 15:00:00'],
    ['2022-10-27 23:00:00', '2022-10-28 05:00:00'],
    ['2022-10-28 11:00:00', '2022-10-28 15:00:00'],
    ['2022-10-28 23:00:00', '2022-10-29 05:00:00'],
    ['2022-10-29 11:00:00', '2022-10-29 15:00:00'],
    ['2022-10-29 23:00:00', '2022-10-30 05:00:00'],
    ['2022-10-30 11:00:00', '2022-10-30 15:00:00'],
    ['2022-10-30 23:00:00', '2022-10-31 05:00:00'],
    ['2022-10-31 11:00:00', '2022-10-31 15:00:00'],
    ['2022-10-31 23:00:00', '2022-11-01 05:00:00'],
    ['2022-11-01 11:00:00', '2022-11-01 15:00:00'],
    ['2022-11-01 23:00:00', '2022-11-02 05:00:00'],
    ['2022-11-02 11:00:00', '2022-11-02 15:00:00'],
    ['2022-11-02 23:00:00', '2022-11-03 05:00:00'],
    ['2022-11-03 11:00:00', '2022-11-03 15:00:00'],
    ['2022-11-03 23:00:00', '2022-11-04 05:00:00'],
    ['2022-11-04 11:00:00', '2022-11-04 15:00:00'],
    ['2022-11-04 23:00:00', '2022-11-05 05:00:00'],
    ['2022-11-05 11:00:00', '2022-11-05 15:00:00'],
    ['2022-11-05 23:00:00', '2022-11-06 05:00:00'],
    ['2022-11-06 11:00:00', '2022-11-06 15:00:00'],
    ['2022-11-06 23:00:00', '2022-11-07 05:00:00'],
    ['2022-11-07 11:00:00', '2022-11-07 15:00:00'],
    ['2022-11-07 23:00:00', '2022-11-08 05:00:00'],
    ['2022-11-08 11:00:00', '2022-11-08 15:00:00'],
    ['2022-11-08 23:00:00', '2022-11-09 05:00:00'],
    ['2022-11-09 11:00:00', '2022-11-09 15:00:00'],
    ['2022-11-09 23:00:00', '2022-11-10 05:00:00'],
    ['2022-11-10 11:00:00', '2022-11-10 15:00:00'],
    ['2022-11-10 23:00:00', '2022-11-11 05:00:00'],
    ['2022-11-11 11:00:00', '2022-11-11 15:00:00'],
    ['2022-11-11 23:00:00', '2022-11-12 05:00:00'],
    ['2022-11-12 11:00:00', '2022-11-12 15:00:00'],
    ['2022-11-12 23:00:00', '2022-11-13 05:00:00'],
    ['2022-11-13 11:00:00', '2022-11-13 15:00:00'],
    ['2022-11-13 23:00:00', '2022-11-14 05:00:00'],
    ['2022-11-14 11:00:00', '2022-11-14 15:00:00'],
    ['2022-11-14 23:00:00', '2022-11-15 05:00:00'],
    ['2022-11-15 11:00:00', '2022-11-15 15:00:00'],
    ['2022-11-15 23:00:00', '2022-11-16 05:00:00'],
    ['2022-11-16 11:00:00', '2022-11-16 15:00:00'],
    ['2022-11-16 23:00:00', '2022-11-17 05:00:00'],
    ['2022-11-17 11:00:00', '2022-11-17 15:00:00'],
    ['2022-11-19 11:00:00', '2022-11-19 15:00:00'],
    ['2022-11-19 23:00:00', '2022-11-20 05:00:00'],
    ['2022-11-20 11:00:00', '2022-11-20 15:00:00'],
    ['2022-11-20 23:00:00', '2022-11-21 05:00:00'],
    ['2022-11-21 11:00:00', '2022-11-21 15:00:00'],
    ['2022-11-21 23:00:00', '2022-11-22 05:00:00'],
    ['2022-11-22 11:00:00', '2022-11-22 15:00:00'],
    ['2022-11-22 23:00:00', '2022-11-23 05:00:00'],
    ['2022-11-23 11:00:00', '2022-11-23 15:00:00'],
    ['2022-11-23 23:00:00', '2022-11-24 05:00:00'],
    ['2022-11-24 11:00:00', '2022-11-24 15:00:00'],
    ['2022-11-24 23:00:00', '2022-11-25 05:00:00'],
    ['2022-11-25 11:00:00', '2022-11-25 15:00:00'],
    ['2022-11-25 23:00:00', '2022-11-26 05:00:00'],
    ['2022-11-26 11:00:00', '2022-11-26 15:00:00'],
    ['2022-11-26 23:00:00', '2022-11-27 05:00:00'],
    ['2022-11-27 11:00:00', '2022-11-27 15:00:00'],
    ['2022-11-27 23:00:00', '2022-11-28 05:00:00'],
    ['2022-11-28 11:00:00', '2022-11-28 15:00:00'],
    ['2022-11-28 23:00:00', '2022-11-29 05:00:00'],
    ['2022-11-29 11:00:00', '2022-11-29 15:00:00'],
    ['2022-11-29 23:00:00', '2022-11-30 05:00:00'],
    ['2022-11-30 11:00:00', '2022-11-30 15:00:00'],
    ['2022-11-30 23:00:00', '2022-12-01 05:00:00'],
    ['2022-12-01 11:00:00', '2022-12-01 15:00:00'],
    ['2022-12-01 23:00:00', '2022-12-02 05:00:00'],
    ['2022-12-02 11:00:00', '2022-12-02 15:00:00'],
    ['2022-12-02 23:00:00', '2022-12-03 05:00:00'],
    ['2022-12-03 11:00:00', '2022-12-03 15:00:00'],
    ['2022-12-03 23:00:00', '2022-12-04 05:00:00'],
    ['2022-12-04 11:00:00', '2022-12-04 15:00:00'],
    ['2022-12-04 23:00:00', '2022-12-05 05:00:00'],
    ['2022-12-05 11:00:00', '2022-12-05 15:00:00'],
    ['2022-12-05 23:00:00', '2022-12-06 05:00:00'],
    ['2022-12-06 11:00:00', '2022-12-06 15:00:00'],
    ['2022-12-06 23:00:00', '2022-12-07 05:00:00'],
    ['2022-12-07 11:00:00', '2022-12-07 15:00:00'],
    ['2022-12-07 23:00:00', '2022-12-08 05:00:00'],
    ['2022-12-08 11:00:00', '2022-12-08 15:00:00'],
    ['2022-12-08 23:00:00', '2022-12-09 05:00:00'],
    ['2022-12-09 11:00:00', '2022-12-09 15:00:00'],
    ['2022-12-09 23:00:00', '2022-12-10 05:00:00'],
    ['2022-12-10 11:00:00', '2022-12-10 15:00:00'],
    ['2022-12-10 23:00:00', '2022-12-11 05:00:00'],
    ['2022-12-11 11:00:00', '2022-12-11 15:00:00'],
    ['2022-12-11 23:00:00', '2022-12-12 05:00:00'],
    ['2022-12-12 11:00:00', '2022-12-12 15:00:00'],
    ['2022-12-12 23:00:00', '2022-12-13 05:00:00'],
    ['2022-12-13 11:00:00', '2022-12-13 15:00:00'],
    ['2022-12-13 23:00:00', '2022-12-14 05:00:00'],
    ['2022-12-14 11:00:00', '2022-12-14 15:00:00'],
    ['2022-12-14 23:00:00', '2022-12-15 05:00:00'],
    ['2022-12-15 11:00:00', '2022-12-15 15:00:00'],
    ['2022-12-15 23:00:00', '2022-12-16 05:00:00'],
    ['2022-12-16 11:00:00', '2022-12-16 15:00:00'],
    ['2022-12-16 23:00:00', '2022-12-17 05:00:00'],
    ['2022-12-17 11:00:00', '2022-12-17 15:00:00'],
    ['2022-12-17 23:00:00', '2022-12-18 05:00:00'],
    ['2022-12-18 11:00:00', '2022-12-18 15:00:00'],
    ['2022-12-18 23:00:00', '2022-12-19 05:00:00'],
    ['2022-12-19 11:00:00', '2022-12-19 15:00:00'],
    ['2022-12-19 23:00:00', '2022-12-20 05:00:00'],
    ['2022-12-20 11:00:00', '2022-12-20 15:00:00'],
    ['2022-12-20 23:00:00', '2022-12-21 05:00:00'],
    ['2022-12-21 11:00:00', '2022-12-21 15:00:00'],
    ['2022-12-21 23:00:00', '2022-12-22 05:00:00'],
    ['2022-12-22 11:00:00', '2022-12-22 15:00:00'],
    ['2022-12-22 23:00:00', '2022-12-23 05:00:00'],
    ['2022-12-23 11:00:00', '2022-12-23 15:00:00'],
    ['2022-12-23 23:00:00', '2022-12-24 05:00:00'],
    ['2022-12-24 11:00:00', '2022-12-24 15:00:00'],
    ['2022-12-24 23:00:00', '2022-12-25 05:00:00'],
    ['2022-12-25 11:00:00', '2022-12-25 15:00:00'],
    ['2022-12-25 23:00:00', '2022-12-26 05:00:00'],
    ['2022-12-26 11:00:00', '2022-12-26 15:00:00'],
    ['2022-12-26 23:00:00', '2022-12-27 05:00:00'],
    ['2022-12-27 11:00:00', '2022-12-27 15:00:00'],
    ['2022-12-29 11:00:00', '2022-12-29 15:00:00'],
    ['2022-12-29 23:00:00', '2022-12-30 05:00:00'],
    ['2022-12-30 11:00:00', '2022-12-30 15:00:00'],
    ['2022-12-30 23:00:00', '2022-12-31 05:00:00'],
    ['2022-12-31 11:00:00', '2022-12-31 15:00:00'],
    ['2022-12-31 23:00:00', '2023-01-01 05:00:00'],
    ['2023-01-01 11:00:00', '2023-01-01 15:00:00'],
    ['2023-01-01 23:00:00', '2023-01-02 05:00:00'],
    ['2023-01-02 11:00:00', '2023-01-02 15:00:00'],
    ['2023-01-02 23:00:00', '2023-01-03 05:00:00'],
    ['2023-01-03 11:00:00', '2023-01-03 15:00:00'],
    ['2023-01-03 23:00:00', '2023-01-04 05:00:00'],
    ['2023-01-04 11:00:00', '2023-01-04 15:00:00'],
    ['2023-01-04 23:00:00', '2023-01-05 05:00:00'],
    ['2023-01-05 11:00:00', '2023-01-05 15:00:00'],
    ['2023-01-05 23:00:00', '2023-01-06 05:00:00'],
    ['2023-01-06 11:00:00', '2023-01-06 15:00:00'],
    ['2023-01-06 23:00:00', '2023-01-07 05:00:00'],
    ['2023-01-07 11:00:00', '2023-01-07 15:00:00'],
    ['2023-01-07 23:00:00', '2023-01-08 05:00:00'],
    ['2023-01-08 11:00:00', '2023-01-08 15:00:00'],
    ['2023-01-08 23:00:00', '2023-01-09 05:00:00'],
    ['2023-01-09 11:00:00', '2023-01-09 15:00:00'],
    ['2023-01-09 23:00:00', '2023-01-10 05:00:00'],
    ['2023-01-10 11:00:00', '2023-01-10 15:00:00'],
    ['2023-01-10 23:00:00', '2023-01-11 05:00:00'],
    ['2023-01-11 11:00:00', '2023-01-11 15:00:00'],
    ['2023-01-11 23:00:00', '2023-01-12 05:00:00'],
    ['2023-01-12 11:00:00', '2023-01-12 15:00:00'],
    ['2023-01-12 23:00:00', '2023-01-13 05:00:00'],
    ['2023-01-13 11:00:00', '2023-01-13 15:00:00'],
    ['2023-01-13 23:00:00', '2023-01-14 05:00:00'],
    ['2023-01-14 11:00:00', '2023-01-14 15:00:00'],
    ['2023-01-14 23:00:00', '2023-01-15 05:00:00'],
    ['2023-01-15 11:00:00', '2023-01-15 15:00:00'],
    ['2023-01-15 23:00:00', '2023-01-16 05:00:00'],
    ['2023-01-16 11:00:00', '2023-01-16 15:00:00'],
    ['2023-01-16 23:00:00', '2023-01-17 05:00:00'],
    ['2023-01-17 11:00:00', '2023-01-17 15:00:00'],
    ['2023-01-19 11:00:00', '2023-01-19 15:00:00'],
    ['2023-01-19 23:00:00', '2023-01-20 05:00:00'],
    ['2023-01-20 11:00:00', '2023-01-20 15:00:00'],
    ['2023-01-20 23:00:00', '2023-01-21 05:00:00'],
    ['2023-01-21 11:00:00', '2023-01-21 15:00:00'],
    ['2023-01-21 23:00:00', '2023-01-22 05:00:00'],
    ['2023-01-22 11:00:00', '2023-01-22 15:00:00'],
    ['2023-01-22 23:00:00', '2023-01-23 05:00:00'],
    ['2023-01-23 11:00:00', '2023-01-23 15:00:00'],
    ['2023-01-23 23:00:00', '2023-01-24 05:00:00'],
    ['2023-01-24 11:00:00', '2023-01-24 15:00:00'],
    ['2023-01-24 23:00:00', '2023-01-25 05:00:00'],
    ['2023-01-25 11:00:00', '2023-01-25 15:00:00'],
    ['2023-01-25 23:00:00', '2023-01-26 05:00:00'],
    ['2023-01-26 11:00:00', '2023-01-26 15:00:00'],
    ['2023-01-26 23:00:00', '2023-01-27 05:00:00'],
    ['2023-01-27 11:00:00', '2023-01-27 15:00:00'],
    ['2023-01-27 23:00:00', '2023-01-28 05:00:00'],
    ['2023-01-28 11:00:00', '2023-01-28 15:00:00'],
    ['2023-01-28 23:00:00', '2023-01-29 05:00:00'],
    ['2023-01-29 11:00:00', '2023-01-29 15:00:00'],
    ['2023-01-29 23:00:00', '2023-01-30 05:00:00'],
    ['2023-01-30 11:00:00', '2023-01-30 15:00:00'],
    ['2023-01-30 23:00:00', '2023-01-31 05:00:00'],
    ['2023-01-31 11:00:00', '2023-01-31 15:00:00'],
    ['2023-01-31 23:00:00', '2023-02-01 05:00:00'],
    ['2023-02-01 11:00:00', '2023-02-01 15:00:00'],
    ['2023-02-01 23:00:00', '2023-02-02 05:00:00'],
    ['2023-02-02 11:00:00', '2023-02-02 15:00:00'],
    ['2023-02-02 23:00:00', '2023-02-03 05:00:00'],
    ['2023-02-03 11:00:00', '2023-02-03 15:00:00'],
    ['2023-02-03 23:00:00', '2023-02-04 05:00:00'],
    ['2023-02-04 11:00:00', '2023-02-04 15:00:00'],
    ['2023-02-04 23:00:00', '2023-02-05 05:00:00'],
    ['2023-02-05 11:00:00', '2023-02-05 15:00:00'],
    ['2023-02-05 23:00:00', '2023-02-06 05:00:00'],
    ['2023-02-06 11:00:00', '2023-02-06 15:00:00'],
    ['2023-02-06 23:00:00', '2023-02-07 05:00:00'],
    ['2023-02-07 11:00:00', '2023-02-07 15:00:00'],
    ['2023-02-07 23:00:00', '2023-02-08 05:00:00'],
    ['2023-02-08 11:00:00', '2023-02-08 15:00:00'],
    ['2023-02-08 23:00:00', '2023-02-09 05:00:00'],
    ['2023-02-09 11:00:00', '2023-02-09 15:00:00'],
    ['2023-02-09 23:00:00', '2023-02-10 05:00:00'],
    ['2023-02-10 11:00:00', '2023-02-10 15:00:00'],
    ['2023-02-10 23:00:00', '2023-02-11 05:00:00'],
    ['2023-02-11 11:00:00', '2023-02-11 15:00:00'],
    ['2023-02-11 23:00:00', '2023-02-12 05:00:00'],
    ['2023-02-12 11:00:00', '2023-02-12 15:00:00'],
    ['2023-02-12 23:00:00', '2023-02-13 05:00:00'],
    ['2023-02-13 11:00:00', '2023-02-13 15:00:00'],
    ['2023-02-13 23:00:00', '2023-02-14 05:00:00'],
    ['2023-02-14 11:00:00', '2023-02-14 15:00:00'],
    ['2023-02-14 23:00:00', '2023-02-15 05:00:00'],
    ['2023-02-15 11:00:00', '2023-02-15 15:00:00'],
    ['2023-02-15 23:00:00', '2023-02-16 05:00:00'],
    ['2023-02-16 11:00:00', '2023-02-16 15:00:00'],
    ['2023-02-16 23:00:00', '2023-02-17 05:00:00'],
    ['2023-02-17 11:00:00', '2023-02-17 15:00:00'],
    ['2023-02-17 23:00:00', '2023-02-18 05:00:00'],
    ['2023-02-18 11:00:00', '2023-02-18 15:00:00'],
    ['2023-02-18 23:00:00', '2023-02-19 05:00:00'],
    ['2023-02-19 11:00:00', '2023-02-19 15:00:00'],
    ['2023-02-19 23:00:00', '2023-02-20 05:00:00'],
    ['2023-02-20 11:00:00', '2023-02-20 15:00:00'],
    ['2023-02-20 23:00:00', '2023-02-21 05:00:00'],
    ['2023-02-21 11:00:00', '2023-02-21 15:00:00'],
    ['2023-02-21 23:00:00', '2023-02-22 05:00:00'],
    ['2023-02-22 11:00:00', '2023-02-22 15:00:00'],
    ['2023-02-22 23:00:00', '2023-02-23 05:00:00'],
    ['2023-02-23 11:00:00', '2023-02-23 15:00:00'],
    ['2023-02-23 23:00:00', '2023-02-24 05:00:00'],
    ['2023-02-24 11:00:00', '2023-02-24 15:00:00'],
    ['2023-02-24 23:00:00', '2023-02-25 05:00:00'],
    ['2023-02-25 11:00:00', '2023-02-25 15:00:00'],
    ['2023-02-25 23:00:00', '2023-02-26 05:00:00'],
    ['2023-02-26 11:00:00', '2023-02-26 15:00:00'],
    ['2023-02-26 23:00:00', '2023-02-27 05:00:00'],
    ['2023-02-27 11:00:00', '2023-02-27 15:00:00'],
    ['2023-02-27 23:00:00', '2023-02-28 05:00:00'],
    ['2023-02-28 11:00:00', '2023-02-28 15:00:00'],
    ['2023-02-28 23:00:00', '2023-03-01 05:00:00'],
    ['2023-03-01 11:00:00', '2023-03-01 15:00:00'],
    ['2023-03-01 23:00:00', '2023-03-02 05:00:00'],
    ['2023-03-02 11:00:00', '2023-03-02 15:00:00'],
    ['2023-03-02 23:00:00', '2023-03-03 05:00:00'],
    ['2023-03-03 11:00:00', '2023-03-03 15:00:00'],
    ['2023-03-03 23:00:00', '2023-03-04 05:00:00'],
    ['2023-04-19 11:00:00', '2023-04-19 15:00:00'],
    ['2023-04-19 23:00:00', '2023-04-20 05:00:00'],
    ['2023-04-20 11:00:00', '2023-04-20 15:00:00'],
    ['2023-04-20 23:00:00', '2023-04-21 05:00:00'],
    ['2023-04-21 11:00:00', '2023-04-21 15:00:00'],
    ['2023-04-21 23:00:00', '2023-04-22 05:00:00'],
    ['2023-04-22 11:00:00', '2023-04-22 15:00:00'],
    ['2023-04-22 23:00:00', '2023-04-23 05:00:00'],
    ['2023-04-23 11:00:00', '2023-04-23 15:00:00'],
    ['2023-04-23 23:00:00', '2023-04-24 05:00:00'],
    ['2023-04-24 11:00:00', '2023-04-24 15:00:00'],
    ['2023-04-24 23:00:00', '2023-04-25 05:00:00'],
    ['2023-04-25 11:00:00', '2023-04-25 15:00:00'],
    ['2023-04-25 23:00:00', '2023-04-26 05:00:00'],
    ['2023-04-26 11:00:00', '2023-04-26 15:00:00'],
    ['2023-04-26 23:00:00', '2023-04-27 05:00:00'],
    ['2023-04-27 11:00:00', '2023-04-27 15:00:00'],
    ['2023-04-27 23:00:00', '2023-04-28 05:00:00'],
    ['2023-04-28 11:00:00', '2023-04-28 15:00:00'],
    ['2023-04-28 23:00:00', '2023-04-29 05:00:00'],
    ['2023-04-29 11:00:00', '2023-04-29 15:00:00'],
    ['2023-04-29 23:00:00', '2023-04-30 05:00:00'],
    ['2023-04-30 11:00:00', '2023-04-30 15:00:00'],
    ['2023-04-30 23:00:00', '2023-05-01 05:00:00'],
    ['2023-05-01 11:00:00', '2023-05-01 15:00:00'],
    ['2023-05-01 23:00:00', '2023-05-02 05:00:00'],
    ['2023-05-02 11:00:00', '2023-05-02 15:00:00'],
    ['2023-05-02 23:00:00', '2023-05-03 05:00:00'],
    ['2023-05-03 11:00:00', '2023-05-03 15:00:00'],
    ['2023-05-03 23:00:00', '2023-05-04 05:00:00'],
    ['2023-05-04 11:00:00', '2023-05-04 15:00:00'],
    ['2023-05-04 23:00:00', '2023-05-05 05:00:00'],
    ['2023-05-05 11:00:00', '2023-05-05 15:00:00'],
    ['2023-05-05 23:00:00', '2023-05-06 05:00:00'],
    ['2023-05-06 11:00:00', '2023-05-06 15:00:00'],
    ['2023-05-06 23:00:00', '2023-05-07 05:00:00'],
    ['2023-05-07 11:00:00', '2023-05-07 15:00:00'],
    ['2023-05-07 23:00:00', '2023-05-08 05:00:00'],
    ['2023-05-08 11:00:00', '2023-05-08 15:00:00'],
    ['2023-05-08 23:00:00', '2023-05-09 05:00:00'],
    ['2023-05-09 11:00:00', '2023-05-09 15:00:00'],
    ['2023-05-09 23:00:00', '2023-05-10 05:00:00'],
    ['2023-05-10 11:00:00', '2023-05-10 15:00:00'],
    ['2023-05-10 23:00:00', '2023-05-11 05:00:00'],
    ['2023-05-11 11:00:00', '2023-05-11 15:00:00'],
    ['2023-05-11 23:00:00', '2023-05-12 05:00:00'],
    ['2023-05-12 11:00:00', '2023-05-12 15:00:00'],
    ['2023-05-12 23:00:00', '2023-05-13 05:00:00'],
    ['2023-05-13 11:00:00', '2023-05-13 15:00:00'],
    ['2023-05-13 23:00:00', '2023-05-14 05:00:00'],
    ['2023-05-14 11:00:00', '2023-05-14 15:00:00'],
    ['2023-05-14 23:00:00', '2023-05-15 05:00:00'],
    ['2023-05-15 11:00:00', '2023-05-15 15:00:00'],
    ['2023-05-15 23:00:00', '2023-05-16 05:00:00'],
    ['2023-06-29 23:00:00', '2023-06-30 05:00:00'],
    ['2023-06-30 11:00:00', '2023-06-30 15:00:00'],
    ['2023-06-30 23:00:00', '2023-07-01 05:00:00'],
    ['2023-07-01 11:00:00', '2023-07-01 15:00:00'],
    ['2023-07-01 23:00:00', '2023-07-02 05:00:00'],
    ['2023-07-02 11:00:00', '2023-07-02 15:00:00'],
    ['2023-07-02 23:00:00', '2023-07-03 05:00:00'],
    ['2023-07-03 11:00:00', '2023-07-03 15:00:00'],
    ['2023-07-03 23:00:00', '2023-07-04 05:00:00'],
    ['2023-07-04 11:00:00', '2023-07-04 15:00:00'],
    ['2023-07-04 23:00:00', '2023-07-05 05:00:00'],
    ['2023-07-05 11:00:00', '2023-07-05 15:00:00'],
    ['2023-07-05 23:00:00', '2023-07-06 05:00:00'],
    ['2023-07-06 11:00:00', '2023-07-06 15:00:00'],
    ['2023-07-06 23:00:00', '2023-07-07 05:00:00'],
    ['2023-07-07 11:00:00', '2023-07-07 15:00:00'],
    ['2023-07-07 23:00:00', '2023-07-08 05:00:00'],
    ['2023-07-08 11:00:00', '2023-07-08 15:00:00'],
    #['2023-07-08 23:00:00', '2023-07-09 05:00:00'],
    ['2023-07-09 11:00:00', '2023-07-09 15:00:00'],
    ['2023-07-09 23:00:00', '2023-07-10 05:00:00'],
    ['2023-07-10 11:00:00', '2023-07-10 15:00:00'],
    ['2023-07-10 23:00:00', '2023-07-11 05:00:00'],
    ['2023-07-11 11:00:00', '2023-07-11 15:00:00'],
    ['2023-07-11 23:00:00', '2023-07-12 05:00:00'],
    ['2023-07-12 11:00:00', '2023-07-12 15:00:00'],
    ['2023-07-12 23:00:00', '2023-07-13 05:00:00'],
    ['2023-07-13 11:00:00', '2023-07-13 15:00:00'],
    ['2023-07-13 23:00:00', '2023-07-14 05:00:00'],
    ['2023-07-14 11:00:00', '2023-07-14 15:00:00'],
    ['2023-07-14 23:00:00', '2023-07-15 05:00:00'],
    ['2023-07-15 11:00:00', '2023-07-15 15:00:00'],
    ['2023-07-15 23:00:00', '2023-07-16 05:00:00'],
    ['2023-07-16 11:00:00', '2023-07-16 15:00:00'],
    ['2023-07-18 11:00:00', '2023-07-18 15:00:00'],
    ['2023-07-18 23:00:00', '2023-07-19 05:00:00'],
    ['2023-07-19 11:00:00', '2023-07-19 15:00:00'],
    ['2023-07-19 23:00:00', '2023-07-20 05:00:00'],
    ['2023-07-20 11:00:00', '2023-07-20 15:00:00'],
    ['2023-07-20 23:00:00', '2023-07-21 05:00:00'],
    ['2023-07-21 11:00:00', '2023-07-21 15:00:00'],
    ['2023-07-21 23:00:00', '2023-07-22 05:00:00'],
    ['2023-07-22 11:00:00', '2023-07-22 15:00:00'],
    ['2023-07-22 23:00:00', '2023-07-23 05:00:00'],
    ['2023-07-23 11:00:00', '2023-07-23 15:00:00'],
    ['2023-07-23 23:00:00', '2023-07-24 05:00:00'],
    ['2023-07-24 11:00:00', '2023-07-24 15:00:00'],
    ['2023-07-24 23:00:00', '2023-07-25 05:00:00'],
    ['2023-07-25 11:00:00', '2023-07-25 15:00:00'],
    ['2023-07-25 23:00:00', '2023-07-26 05:00:00'],
    ['2023-07-26 11:00:00', '2023-07-26 15:00:00'],
    ['2023-07-26 23:00:00', '2023-07-27 05:00:00'],
    ['2023-07-27 11:00:00', '2023-07-27 15:00:00'],
    ['2023-07-27 23:00:00', '2023-07-28 05:00:00'],
    ['2023-08-22 23:00:00', '2023-08-23 05:00:00'],
    ['2023-08-23 11:00:00', '2023-08-23 15:00:00'],
    ['2023-08-23 23:00:00', '2023-08-24 05:00:00'],
    ['2023-08-24 11:00:00', '2023-08-24 15:00:00'],
    ['2023-08-24 23:00:00', '2023-08-25 05:00:00'],
    ['2023-08-25 11:00:00', '2023-08-25 15:00:00'],
    ['2023-08-25 23:00:00', '2023-08-26 05:00:00'],
    ['2023-08-26 11:00:00', '2023-08-26 15:00:00'],
    ['2023-08-26 23:00:00', '2023-08-27 05:00:00'],
    ['2023-08-27 11:00:00', '2023-08-27 15:00:00'],
    ['2023-08-27 23:00:00', '2023-08-28 05:00:00'],
    ['2023-08-28 11:00:00', '2023-08-28 15:00:00'],
    ['2023-08-28 23:00:00', '2023-08-29 05:00:00'],
    ['2023-08-29 11:00:00', '2023-08-29 15:00:00'],
    ['2023-08-29 23:00:00', '2023-08-30 05:00:00'],
    ['2023-08-30 11:00:00', '2023-08-30 15:00:00'],
    ['2023-09-02 11:00:00', '2023-09-02 15:00:00'],
    ['2023-09-02 23:00:00', '2023-09-03 05:00:00'],
    ['2023-09-03 11:00:00', '2023-09-03 15:00:00'],
    ['2023-09-03 23:00:00', '2023-09-04 05:00:00'],
    ['2023-09-04 11:00:00', '2023-09-04 15:00:00'],
    ['2023-09-04 23:00:00', '2023-09-05 05:00:00'],
    ['2023-09-05 11:00:00', '2023-09-05 15:00:00'],
    ['2023-09-05 23:00:00', '2023-09-06 05:00:00'],
    ['2023-09-06 11:00:00', '2023-09-06 15:00:00'],
    ['2023-09-06 23:00:00', '2023-09-07 05:00:00'],
    ['2023-09-07 11:00:00', '2023-09-07 15:00:00'],
    ['2023-09-07 23:00:00', '2023-09-08 05:00:00'],
    ['2023-09-08 11:00:00', '2023-09-08 15:00:00'],
    ['2023-09-08 23:00:00', '2023-09-09 05:00:00'],
    ['2023-09-09 11:00:00', '2023-09-09 15:00:00'],
    ['2023-09-09 23:00:00', '2023-09-10 05:00:00'],
    ['2023-09-10 11:00:00', '2023-09-10 15:00:00'],
    ['2023-09-10 23:00:00', '2023-09-11 05:00:00'],
    ['2023-09-11 11:00:00', '2023-09-11 15:00:00'],
    ['2023-09-11 23:00:00', '2023-09-12 05:00:00'],
    ['2023-09-12 11:00:00', '2023-09-12 15:00:00'],
    ['2023-09-12 23:00:00', '2023-09-13 05:00:00'],
    ['2023-09-13 11:00:00', '2023-09-13 15:00:00'],
    ['2023-09-13 23:00:00', '2023-09-14 05:00:00'],
    ['2023-09-14 11:00:00', '2023-09-14 15:00:00'],
    ['2023-09-14 23:00:00', '2023-09-15 05:00:00'],
    ['2023-09-15 11:00:00', '2023-09-15 15:00:00'],
    ['2023-09-15 23:00:00', '2023-09-16 05:00:00'],
    ['2023-09-16 11:00:00', '2023-09-16 15:00:00'],
    ['2023-09-16 23:00:00', '2023-09-17 05:00:00'],
    ['2023-09-17 11:00:00', '2023-09-17 15:00:00'],
    ['2023-09-19 11:00:00', '2023-09-19 15:00:00'],
    ['2023-09-19 23:00:00', '2023-09-20 05:00:00'],
    ['2023-09-20 11:00:00', '2023-09-20 15:00:00'],
    ['2023-09-20 23:00:00', '2023-09-21 05:00:00'],
    ['2023-09-21 11:00:00', '2023-09-21 15:00:00'],
    ['2023-09-21 23:00:00', '2023-09-22 05:00:00'],
    ['2023-09-22 11:00:00', '2023-09-22 15:00:00'],
    ['2023-09-22 23:00:00', '2023-09-23 05:00:00'],
    ['2023-09-23 11:00:00', '2023-09-23 15:00:00'],
    ['2023-09-23 23:00:00', '2023-09-24 05:00:00'],
    ['2023-09-24 11:00:00', '2023-09-24 15:00:00'],
    ['2023-09-24 23:00:00', '2023-09-25 05:00:00'],
    ['2023-09-25 11:00:00', '2023-09-25 15:00:00'],
    ['2023-09-25 23:00:00', '2023-09-26 05:00:00'],
    ['2023-09-26 11:00:00', '2023-09-26 15:00:00'],
    ['2023-09-26 23:00:00', '2023-09-27 05:00:00'],
    ['2023-09-27 11:00:00', '2023-09-27 15:00:00'],
    ['2023-09-27 23:00:00', '2023-09-28 05:00:00'],
    ['2023-09-28 11:00:00', '2023-09-28 15:00:00'],
    ['2023-09-28 23:00:00', '2023-09-29 05:00:00'],
    ['2023-09-29 11:00:00', '2023-09-29 15:00:00'],
    ['2023-09-29 23:00:00', '2023-09-30 05:00:00'],
    ['2023-09-30 11:00:00', '2023-09-30 15:00:00'],
    ['2023-09-30 23:00:00', '2023-10-01 05:00:00'],
    ['2023-10-01 11:00:00', '2023-10-01 15:00:00'],
    ['2023-10-01 23:00:00', '2023-10-02 05:00:00'],
    ['2023-10-02 11:00:00', '2023-10-02 15:00:00'],
    ['2023-10-02 23:00:00', '2023-10-03 05:00:00'],
    ['2023-10-03 11:00:00', '2023-10-03 15:00:00'],
    ['2023-10-03 23:00:00', '2023-10-04 05:00:00'],
    ['2023-10-04 11:00:00', '2023-10-04 15:00:00'],
    ['2023-10-04 23:00:00', '2023-10-05 05:00:00'],
    ['2023-10-05 11:00:00', '2023-10-05 15:00:00'],
    ['2023-10-05 23:00:00', '2023-10-06 05:00:00'],
    ['2023-10-06 11:00:00', '2023-10-06 15:00:00'],
    ['2023-10-06 23:00:00', '2023-10-07 05:00:00'],
    ['2023-10-07 11:00:00', '2023-10-07 15:00:00'],
    ['2023-10-07 23:00:00', '2023-10-08 05:00:00'],
    ['2023-10-08 11:00:00', '2023-10-08 15:00:00'],
    ['2023-10-08 23:00:00', '2023-10-09 05:00:00'],
    ['2023-10-09 11:00:00', '2023-10-09 15:00:00'],
    ['2023-10-09 23:00:00', '2023-10-10 05:00:00'],
    ['2023-10-10 11:00:00', '2023-10-10 15:00:00'],
    ['2023-10-10 23:00:00', '2023-10-11 05:00:00'],
    ['2023-10-11 11:00:00', '2023-10-11 15:00:00'],
    ['2023-10-11 23:00:00', '2023-10-12 05:00:00'],
    ['2023-10-12 11:00:00', '2023-10-12 15:00:00'],
    ['2023-10-12 23:00:00', '2023-10-13 05:00:00'],
    ['2023-10-13 11:00:00', '2023-10-13 15:00:00'],
    ['2023-10-13 23:00:00', '2023-10-14 05:00:00'],
    ['2023-10-14 11:00:00', '2023-10-14 15:00:00'],
    ['2023-10-14 23:00:00', '2023-10-15 05:00:00'],
    ['2023-10-15 11:00:00', '2023-10-15 15:00:00'],
    ['2023-10-15 23:00:00', '2023-10-16 05:00:00'],
    ['2023-10-16 11:00:00', '2023-10-16 15:00:00'],
    ['2023-10-18 11:00:00', '2023-10-18 15:00:00'],
    ['2023-10-18 23:00:00', '2023-10-19 05:00:00'],
    ['2023-10-19 11:00:00', '2023-10-19 15:00:00'],
    ['2023-10-19 23:00:00', '2023-10-20 05:00:00'],
    ['2023-10-20 11:00:00', '2023-10-20 15:00:00'],
    ['2023-10-20 23:00:00', '2023-10-21 05:00:00'],
    ['2023-10-21 11:00:00', '2023-10-21 15:00:00'],
    ['2023-10-21 23:00:00', '2023-10-22 05:00:00'],
    ['2023-10-22 11:00:00', '2023-10-22 15:00:00'],
    ['2023-10-22 23:00:00', '2023-10-23 05:00:00'],
    ['2023-10-23 11:00:00', '2023-10-23 15:00:00'],
    ['2023-10-23 23:00:00', '2023-10-24 05:00:00'],
    ['2023-10-24 11:00:00', '2023-10-24 15:00:00'],
    ['2023-10-24 23:00:00', '2023-10-25 05:00:00'],
    ['2023-10-25 11:00:00', '2023-10-25 15:00:00'],
    ['2023-10-25 23:00:00', '2023-10-26 05:00:00'],
    ['2023-10-26 11:00:00', '2023-10-26 15:00:00'],
    ['2023-10-26 23:00:00', '2023-10-27 05:00:00'],
    ['2023-10-27 11:00:00', '2023-10-27 15:00:00'],
    ['2023-10-27 23:00:00', '2023-10-28 05:00:00'],
    ['2023-10-28 11:00:00', '2023-10-28 15:00:00'],
    ['2023-10-28 23:00:00', '2023-10-29 05:00:00'],
    ['2023-10-29 11:00:00', '2023-10-29 15:00:00'],
    ['2023-10-31 11:00:00', '2023-10-31 15:00:00'],
    ['2023-10-31 23:00:00', '2023-11-01 05:00:00'],
    ['2023-11-01 11:00:00', '2023-11-01 15:00:00'],
    ['2023-11-01 23:00:00', '2023-11-02 05:00:00'],
    ['2023-11-02 11:00:00', '2023-11-02 15:00:00'],
    ['2023-11-02 23:00:00', '2023-11-03 05:00:00'],
    ['2023-11-03 11:00:00', '2023-11-03 15:00:00'],
    ['2023-11-03 23:00:00', '2023-11-04 05:00:00'],
    ['2023-11-04 11:00:00', '2023-11-04 15:00:00'],
    ['2023-11-04 23:00:00', '2023-11-05 05:00:00'],
    ['2023-11-05 11:00:00', '2023-11-05 15:00:00'],
    ['2023-11-05 23:00:00', '2023-11-06 05:00:00'],
    ['2023-11-06 11:00:00', '2023-11-06 15:00:00'],
    ['2023-11-06 23:00:00', '2023-11-07 05:00:00'],
    ['2023-11-07 11:00:00', '2023-11-07 15:00:00'],
    ['2023-11-07 23:00:00', '2023-11-08 05:00:00'],
    ['2023-11-08 11:00:00', '2023-11-08 15:00:00'],
    ['2023-11-08 23:00:00', '2023-11-09 05:00:00'],
    ['2023-11-09 11:00:00', '2023-11-09 15:00:00'],
    ['2023-11-09 23:00:00', '2023-11-10 05:00:00'],
    ['2023-11-10 11:00:00', '2023-11-10 15:00:00'],
    ['2023-11-10 23:00:00', '2023-11-11 05:00:00'],
    ['2023-11-11 11:00:00', '2023-11-11 15:00:00'],
    ['2023-11-11 23:00:00', '2023-11-12 05:00:00'],
    ['2023-11-12 11:00:00', '2023-11-12 15:00:00'],
    ['2023-11-12 23:00:00', '2023-11-13 05:00:00'],
    ['2023-11-13 11:00:00', '2023-11-13 15:00:00'],
    ['2023-11-13 23:00:00', '2023-11-14 05:00:00'],
    ['2023-11-14 11:00:00', '2023-11-14 15:00:00'],
    ['2023-11-14 23:00:00', '2023-11-15 05:00:00'],
    ['2023-11-15 11:00:00', '2023-11-15 15:00:00'],
    ['2023-11-15 23:00:00', '2023-11-16 05:00:00'],
    ['2023-11-16 11:00:00', '2023-11-16 15:00:00'],
    ['2023-11-16 23:00:00', '2023-11-17 05:00:00'],
    ['2023-11-17 11:00:00', '2023-11-17 15:00:00'],
    ['2023-11-17 23:00:00', '2023-11-18 05:00:00'],
    ['2023-11-18 11:00:00', '2023-11-18 15:00:00'],
    ['2023-11-18 23:00:00', '2023-11-19 05:00:00'],
    ['2023-11-19 11:00:00', '2023-11-19 15:00:00'],
    ['2023-11-19 23:00:00', '2023-11-20 05:00:00'],
    ['2023-11-20 11:00:00', '2023-11-20 15:00:00'],
    ['2023-11-22 11:00:00', '2023-11-22 15:00:00'],
    ['2023-11-22 23:00:00', '2023-11-23 05:00:00'],
    ['2023-11-23 11:00:00', '2023-11-23 15:00:00'],
    ['2023-11-23 23:00:00', '2023-11-24 05:00:00'],
    ['2023-11-24 11:00:00', '2023-11-24 15:00:00'],
    ['2023-11-24 23:00:00', '2023-11-25 05:00:00'],
    ['2023-11-25 11:00:00', '2023-11-25 15:00:00'],
    ['2023-11-25 23:00:00', '2023-11-26 05:00:00'],
    ['2023-11-26 11:00:00', '2023-11-26 15:00:00'],
    ['2023-11-26 23:00:00', '2023-11-27 05:00:00'],
    ['2023-11-27 11:00:00', '2023-11-27 15:00:00'],
    ['2023-11-27 23:00:00', '2023-11-28 05:00:00'],
    ['2023-11-28 11:00:00', '2023-11-28 15:00:00'],
    ['2023-11-28 23:00:00', '2023-11-29 05:00:00'],
    ['2023-11-29 11:00:00', '2023-11-29 15:00:00'],
    ['2023-11-29 23:00:00', '2023-11-30 05:00:00'],
    ['2023-11-30 11:00:00', '2023-11-30 15:00:00'],
    ['2023-11-30 23:00:00', '2023-12-01 05:00:00'],
    ['2023-12-01 11:00:00', '2023-12-01 15:00:00'],
    ['2023-12-01 23:00:00', '2023-12-02 05:00:00'],
    ['2023-12-02 11:00:00', '2023-12-02 15:00:00'],
    ['2023-12-02 23:00:00', '2023-12-03 05:00:00'],
    ['2023-12-03 11:00:00', '2023-12-03 15:00:00'],
    ['2023-12-03 23:00:00', '2023-12-04 05:00:00'],
    ['2023-12-04 11:00:00', '2023-12-04 15:00:00'],
    ['2023-12-04 23:00:00', '2023-12-05 05:00:00'],
    ['2023-12-05 11:00:00', '2023-12-05 15:00:00'],
    ['2023-12-05 23:00:00', '2023-12-06 05:00:00'],
    ['2023-12-06 11:00:00', '2023-12-06 15:00:00'],
    ['2023-12-06 23:00:00', '2023-12-07 05:00:00'],
    ['2023-12-07 11:00:00', '2023-12-07 15:00:00'],
    ['2023-12-07 23:00:00', '2023-12-08 05:00:00'],
    ['2023-12-08 11:00:00', '2023-12-08 15:00:00'],
    ['2023-12-08 23:00:00', '2023-12-09 05:00:00'],
    ['2023-12-09 11:00:00', '2023-12-09 15:00:00'],
    ['2023-12-09 23:00:00', '2023-12-10 05:00:00'],
    ['2023-12-10 11:00:00', '2023-12-10 15:00:00'],
    ['2023-12-10 23:00:00', '2023-12-11 05:00:00'],
    ['2023-12-11 11:00:00', '2023-12-11 15:00:00'],
    ['2023-12-11 23:00:00', '2023-12-12 05:00:00'],
    ['2023-12-12 11:00:00', '2023-12-12 15:00:00'],
    ['2023-12-12 23:00:00', '2023-12-13 05:00:00'],
    ['2023-12-13 11:00:00', '2023-12-13 15:00:00'],
    ['2023-12-13 23:00:00', '2023-12-14 05:00:00'],
    ['2023-12-14 11:00:00', '2023-12-14 15:00:00'],
    ['2023-12-16 11:00:00', '2023-12-16 15:00:00'],
    ['2023-12-16 23:00:00', '2023-12-17 05:00:00'],
    ['2023-12-17 11:00:00', '2023-12-17 15:00:00'],
    ['2023-12-19 11:00:00', '2023-12-19 15:00:00'],
    ['2023-12-19 23:00:00', '2023-12-20 05:00:00'],
    ['2023-12-20 11:00:00', '2023-12-20 15:00:00'],
    ['2023-12-20 23:00:00', '2023-12-21 05:00:00'],
    ['2023-12-21 11:00:00', '2023-12-21 15:00:00'],
    ['2023-12-21 23:00:00', '2023-12-22 05:00:00'],
    ['2023-12-22 11:00:00', '2023-12-22 15:00:00'],
    ['2023-12-22 23:00:00', '2023-12-23 05:00:00'],
    ['2023-12-23 11:00:00', '2023-12-23 15:00:00'],
    ['2023-12-23 23:00:00', '2023-12-24 05:00:00'],
    ['2023-12-24 11:00:00', '2023-12-24 15:00:00'],
    ['2023-12-24 23:00:00', '2023-12-25 05:00:00'],
    ['2023-12-25 11:00:00', '2023-12-25 15:00:00'],
    ['2023-12-25 23:00:00', '2023-12-26 05:00:00'],
    ['2023-12-26 11:00:00', '2023-12-26 15:00:00'],
    ['2023-12-26 23:00:00', '2023-12-27 05:00:00'],
    ['2023-12-27 11:00:00', '2023-12-27 15:00:00'],
    ['2023-12-27 23:00:00', '2023-12-28 05:00:00'],
    ['2023-12-28 11:00:00', '2023-12-28 15:00:00'],
    ['2023-12-28 23:00:00', '2023-12-29 05:00:00'],
    ['2023-12-29 11:00:00', '2023-12-29 15:00:00'],
    ['2023-12-29 23:00:00', '2023-12-30 05:00:00'],
    ['2023-12-30 11:00:00', '2023-12-30 15:00:00'],
    ['2023-12-30 23:00:00', '2023-12-31 05:00:00'],
    ['2023-12-31 11:00:00', '2023-12-31 15:00:00'],
    ['2023-12-31 23:00:00', '2024-01-01 05:00:00'],
    ['2024-01-01 11:00:00', '2024-01-01 15:00:00'],
    ['2024-01-01 23:00:00', '2024-01-02 05:00:00'],
    ['2024-01-02 11:00:00', '2024-01-02 15:00:00'],
    ['2024-01-02 23:00:00', '2024-01-03 05:00:00'],
    ['2024-01-03 11:00:00', '2024-01-03 15:00:00'],
    ['2024-01-03 23:00:00', '2024-01-04 05:00:00'],
    ['2024-01-04 11:00:00', '2024-01-04 15:00:00'],
    ['2024-01-04 23:00:00', '2024-01-05 05:00:00'],
    ['2024-01-05 11:00:00', '2024-01-05 15:00:00'],
    ['2024-01-05 23:00:00', '2024-01-06 05:00:00'],
    ['2024-01-06 11:00:00', '2024-01-06 15:00:00'],
    ['2024-01-06 23:00:00', '2024-01-07 05:00:00'],
    ['2024-01-07 11:00:00', '2024-01-07 15:00:00'],
    ['2024-01-07 23:00:00', '2024-01-08 05:00:00'],
    ['2024-01-08 11:00:00', '2024-01-08 15:00:00'],
    ['2024-01-08 23:00:00', '2024-01-09 05:00:00'],
    ['2024-01-09 11:00:00', '2024-01-09 15:00:00'],
    ['2024-01-09 23:00:00', '2024-01-10 05:00:00'],
    ['2024-01-10 11:00:00', '2024-01-10 15:00:00'],
    ['2024-01-10 23:00:00', '2024-01-11 05:00:00'],
    ['2024-01-11 11:00:00', '2024-01-11 15:00:00'],
    ['2024-01-11 23:00:00', '2024-01-12 05:00:00'],
    ['2024-01-12 11:00:00', '2024-01-12 15:00:00'],
    ['2024-01-12 23:00:00', '2024-01-13 05:00:00'],
    ['2024-01-13 11:00:00', '2024-01-13 15:00:00'],
    ['2024-01-13 23:00:00', '2024-01-14 05:00:00'],
    ['2024-01-14 11:00:00', '2024-01-14 15:00:00'],
    ['2024-01-16 11:00:00', '2024-01-16 15:00:00'],
    ['2024-01-16 23:00:00', '2024-01-17 05:00:00'],
    ['2024-01-17 11:00:00', '2024-01-17 15:00:00'],
    ['2024-01-17 23:00:00', '2024-01-18 05:00:00'],
    ['2024-01-18 11:00:00', '2024-01-18 15:00:00'],
    ['2024-01-18 23:00:00', '2024-01-19 05:00:00'],
    ['2024-01-19 11:00:00', '2024-01-19 15:00:00'],
    ['2024-01-19 23:00:00', '2024-01-20 05:00:00'],
    ['2024-01-20 11:00:00', '2024-01-20 15:00:00'],
    ['2024-01-20 23:00:00', '2024-01-21 05:00:00'],
    ['2024-01-21 11:00:00', '2024-01-21 15:00:00'],
    ['2024-01-21 23:00:00', '2024-01-22 05:00:00'],
    ['2024-01-22 11:00:00', '2024-01-22 15:00:00'],
    ['2024-01-22 23:00:00', '2024-01-23 05:00:00'],
    ['2024-01-23 11:00:00', '2024-01-23 15:00:00'],
    ['2024-01-23 23:00:00', '2024-01-24 05:00:00'],
    ['2024-01-24 11:00:00', '2024-01-24 15:00:00'],
    ['2024-01-24 23:00:00', '2024-01-25 05:00:00'],
    ['2024-01-25 11:00:00', '2024-01-25 15:00:00'],
    ['2024-01-25 23:00:00', '2024-01-26 05:00:00'],
    ['2024-01-26 11:00:00', '2024-01-26 15:00:00'],
    ['2024-01-26 23:00:00', '2024-01-27 05:00:00'],
    ['2024-01-27 11:00:00', '2024-01-27 15:00:00'],
    ['2024-01-27 23:00:00', '2024-01-28 05:00:00'],
    ['2024-01-28 11:00:00', '2024-01-28 15:00:00'],
    ['2024-01-28 23:00:00', '2024-01-29 05:00:00'],
    ['2024-01-29 11:00:00', '2024-01-29 15:00:00'],
    ['2024-01-29 23:00:00', '2024-01-30 05:00:00'],
    ['2024-01-30 11:00:00', '2024-01-30 15:00:00'],
    ['2024-01-30 23:00:00', '2024-01-31 05:00:00'],
    ['2024-01-31 11:00:00', '2024-01-31 15:00:00'],
    ['2024-01-31 23:00:00', '2024-02-01 05:00:00'],
    ['2024-02-01 11:00:00', '2024-02-01 15:00:00'],
    ['2024-02-01 23:00:00', '2024-02-02 05:00:00'],
    ['2024-02-02 11:00:00', '2024-02-02 15:00:00'],
    ['2024-02-02 23:00:00', '2024-02-03 05:00:00'],
    ['2024-02-03 11:00:00', '2024-02-03 15:00:00'],
    ['2024-02-03 23:00:00', '2024-02-04 05:00:00'],
    ['2024-02-04 11:00:00', '2024-02-04 15:00:00'],
    ['2024-02-04 23:00:00', '2024-02-05 05:00:00'],
    ['2024-02-05 11:00:00', '2024-02-05 15:00:00'],
    ['2024-02-05 23:00:00', '2024-02-06 05:00:00'],
    ['2024-02-06 11:00:00', '2024-02-06 15:00:00'],
    ['2024-02-06 23:00:00', '2024-02-07 05:00:00'],
    ['2024-02-07 11:00:00', '2024-02-07 15:00:00'],
    ['2024-02-07 23:00:00', '2024-02-08 05:00:00'],
    ['2024-02-08 11:00:00', '2024-02-08 15:00:00'],
    ['2024-02-08 23:00:00', '2024-02-09 05:00:00'],
    ['2024-02-09 11:00:00', '2024-02-09 15:00:00'],
    ['2024-02-09 23:00:00', '2024-02-10 05:00:00'],
    ['2024-02-10 11:00:00', '2024-02-10 15:00:00'],
    ['2024-02-10 23:00:00', '2024-02-11 05:00:00'],
    ['2024-02-11 11:00:00', '2024-02-11 15:00:00'],
    ['2024-02-11 23:00:00', '2024-02-12 05:00:00'],
    ['2024-02-12 11:00:00', '2024-02-12 15:00:00'],
    ['2024-02-12 23:00:00', '2024-02-13 05:00:00'],
    ['2024-02-13 11:00:00', '2024-02-13 15:00:00'],
    ['2024-02-13 23:00:00', '2024-02-14 05:00:00'],
    ['2024-02-14 11:00:00', '2024-02-14 15:00:00'],
    ['2024-02-14 23:00:00', '2024-02-15 05:00:00'],
    ['2024-02-15 11:00:00', '2024-02-15 15:00:00'],
    ['2024-02-15 23:00:00', '2024-02-16 05:00:00'],
    ['2024-02-16 11:00:00', '2024-02-16 15:00:00'],
    ['2024-02-16 23:00:00', '2024-02-17 05:00:00'],
    ['2024-02-17 11:00:00', '2024-02-17 15:00:00'],
    ['2024-02-17 23:00:00', '2024-02-18 05:00:00'],
    ['2024-02-18 11:00:00', '2024-02-18 15:00:00'],
    ['2024-02-18 23:00:00', '2024-02-19 05:00:00'],
    ['2024-02-19 11:00:00', '2024-02-19 15:00:00'],
    ['2024-02-19 23:00:00', '2024-02-20 05:00:00'],
    ['2024-02-20 11:00:00', '2024-02-20 15:00:00'],
    ['2024-02-20 23:00:00', '2024-02-21 05:00:00'],
    ['2024-02-21 11:00:00', '2024-02-21 15:00:00'],
    ['2024-02-21 23:00:00', '2024-02-22 05:00:00'],
    ['2024-02-22 11:00:00', '2024-02-22 15:00:00'],
    ['2024-02-22 23:00:00', '2024-02-23 05:00:00'],
    ['2024-02-23 11:00:00', '2024-02-23 15:00:00'],
    ['2024-02-23 23:00:00', '2024-02-24 05:00:00'],
    ['2024-02-24 11:00:00', '2024-02-24 15:00:00'],
    ['2024-02-24 23:00:00', '2024-02-25 05:00:00'],
    ['2024-02-25 11:00:00', '2024-02-25 15:00:00'],
    ['2024-02-27 11:00:00', '2024-02-27 15:00:00'],
    ['2024-02-27 23:00:00', '2024-02-28 05:00:00'],
    ['2024-02-28 11:00:00', '2024-02-28 15:00:00'],
    ['2024-02-28 23:00:00', '2024-02-29 05:00:00'],
    ['2024-02-29 11:00:00', '2024-02-29 15:00:00'],
    ['2024-02-29 23:00:00', '2024-03-01 05:00:00'],
    ['2024-03-01 11:00:00', '2024-03-01 15:00:00'],
    ['2024-03-01 23:00:00', '2024-03-02 05:00:00'],
    ['2024-03-02 11:00:00', '2024-03-02 15:00:00'],
    ['2024-03-02 23:00:00', '2024-03-03 05:00:00'],
    ['2024-03-03 11:00:00', '2024-03-03 15:00:00'],
    ['2024-03-03 23:00:00', '2024-03-04 05:00:00'],
    ['2024-03-04 11:00:00', '2024-03-04 15:00:00'],
    ['2024-03-04 23:00:00', '2024-03-05 05:00:00'],
    ['2024-03-05 11:00:00', '2024-03-05 15:00:00'],
    ['2024-03-05 23:00:00', '2024-03-06 05:00:00'],
    ['2024-03-06 11:00:00', '2024-03-06 15:00:00'],
    ['2024-03-06 23:00:00', '2024-03-07 05:00:00'],
    ['2024-03-07 11:00:00', '2024-03-07 15:00:00'],
    ['2024-03-07 23:00:00', '2024-03-08 05:00:00'],
    ['2024-03-08 11:00:00', '2024-03-08 15:00:00'],
    ['2024-03-08 23:00:00', '2024-03-09 05:00:00'],
    ['2024-03-09 11:00:00', '2024-03-09 15:00:00'],
    ['2024-03-09 23:00:00', '2024-03-10 05:00:00'],
    ['2024-03-10 11:00:00', '2024-03-10 15:00:00'],
    ['2024-03-10 23:00:00', '2024-03-11 05:00:00'],
    ['2024-03-11 11:00:00', '2024-03-11 15:00:00'],
    ['2024-03-11 23:00:00', '2024-03-12 05:00:00'],
    ['2024-03-12 11:00:00', '2024-03-12 15:00:00'],
    ['2024-03-12 23:00:00', '2024-03-13 05:00:00'],
    ['2024-03-13 11:00:00', '2024-03-13 15:00:00'],
    ['2024-03-13 23:00:00', '2024-03-14 05:00:00'],
    ['2024-03-14 11:00:00', '2024-03-14 15:00:00'],
    #['2024-03-14 23:00:00', '2024-03-15 05:00:00'],
    ['2024-03-15 11:00:00', '2024-03-15 15:00:00'],
    ['2024-03-15 23:00:00', '2024-03-16 05:00:00'],
    ['2024-03-16 11:00:00', '2024-03-16 15:00:00'],
    ['2024-03-16 23:00:00', '2024-03-17 05:00:00'],
    ['2024-03-17 11:00:00', '2024-03-17 15:00:00'],
    ['2024-03-17 23:00:00', '2024-03-18 05:00:00'],
    ['2024-03-18 11:00:00', '2024-03-18 15:00:00'],
    ['2024-03-20 11:00:00', '2024-03-20 15:00:00'],
    ['2024-03-20 23:00:00', '2024-03-21 05:00:00'],
    ['2024-03-21 11:00:00', '2024-03-21 15:00:00'],
    ['2024-03-21 23:00:00', '2024-03-22 05:00:00'],
    ['2024-03-22 11:00:00', '2024-03-22 15:00:00'],
    ['2024-03-22 23:00:00', '2024-03-23 05:00:00'],
    ['2024-03-23 11:00:00', '2024-03-23 15:00:00'],
    ['2024-03-23 23:00:00', '2024-03-24 05:00:00'],
    ['2024-03-24 11:00:00', '2024-03-24 15:00:00'],
    ['2024-03-24 23:00:00', '2024-03-25 05:00:00'],
    ['2024-03-25 11:00:00', '2024-03-25 15:00:00'],
    ['2024-03-27 11:00:00', '2024-03-27 15:00:00'],
    ['2024-03-27 23:00:00', '2024-03-28 05:00:00'],
    ['2024-03-28 11:00:00', '2024-03-28 15:00:00'],
    ['2024-03-28 23:00:00', '2024-03-29 05:00:00'],
    ['2024-03-29 11:00:00', '2024-03-29 15:00:00'],
    ['2024-03-29 23:00:00', '2024-03-30 05:00:00'],
    ['2024-03-30 11:00:00', '2024-03-30 15:00:00'],
    ['2024-04-01 11:00:00', '2024-04-01 15:00:00'],
    ['2024-04-01 23:00:00', '2024-04-02 05:00:00'],
    ['2024-04-02 11:00:00', '2024-04-02 15:00:00'],
    ['2024-04-02 23:00:00', '2024-04-03 05:00:00'],
    ['2024-04-03 11:00:00', '2024-04-03 15:00:00'],
    ['2024-04-03 23:00:00', '2024-04-04 05:00:00'],
    ['2024-04-04 11:00:00', '2024-04-04 15:00:00'],
    ['2024-04-04 23:00:00', '2024-04-05 05:00:00'],
    ['2024-04-05 11:00:00', '2024-04-05 15:00:00'],
    ['2024-04-05 23:00:00', '2024-04-06 05:00:00'],
    ['2024-04-06 11:00:00', '2024-04-06 15:00:00'],
    ['2024-04-06 23:00:00', '2024-04-07 05:00:00'],
    ['2024-04-07 11:00:00', '2024-04-07 15:00:00'],
    ['2024-04-07 23:00:00', '2024-04-08 05:00:00'],
    ['2024-04-08 11:00:00', '2024-04-08 15:00:00'],
    ['2024-04-08 23:00:00', '2024-04-09 05:00:00'],
    ['2024-04-09 11:00:00', '2024-04-09 15:00:00'],
    ['2024-04-09 23:00:00', '2024-04-10 05:00:00'],
    ['2024-04-10 11:00:00', '2024-04-10 15:00:00'],
    ['2024-04-10 23:00:00', '2024-04-11 05:00:00'],
    ['2024-04-11 11:00:00', '2024-04-11 15:00:00'],
    ['2024-04-11 23:00:00', '2024-04-12 05:00:00'],
    ['2024-04-12 11:00:00', '2024-04-12 15:00:00'],
    ['2024-04-12 23:00:00', '2024-04-13 05:00:00'],
    ['2024-04-13 11:00:00', '2024-04-13 15:00:00'],
    ['2024-04-13 23:00:00', '2024-04-14 05:00:00'],
    ['2024-04-14 11:00:00', '2024-04-14 15:00:00'],
    ['2024-04-14 23:00:00', '2024-04-15 05:00:00'],
    ['2024-04-15 11:00:00', '2024-04-15 15:00:00'],
    ['2024-04-15 23:00:00', '2024-04-16 05:00:00'],
    ['2024-04-16 11:00:00', '2024-04-16 15:00:00'],
    ['2024-04-16 23:00:00', '2024-04-17 05:00:00'],
    ['2024-04-17 11:00:00', '2024-04-17 15:00:00'],
    ['2024-04-19 11:00:00', '2024-04-19 15:00:00'],
    ['2024-04-19 23:00:00', '2024-04-20 05:00:00'],
    ['2024-04-20 11:00:00', '2024-04-20 15:00:00'],
    ['2024-04-20 23:00:00', '2024-04-21 05:00:00'],
    ['2024-04-21 11:00:00', '2024-04-21 15:00:00'],
    ['2024-04-21 23:00:00', '2024-04-22 05:00:00'],
    ['2024-04-22 11:00:00', '2024-04-22 15:00:00'],
    ['2024-04-22 23:00:00', '2024-04-23 05:00:00'],
    ['2024-04-23 11:00:00', '2024-04-23 15:00:00'],
    ['2024-04-25 11:00:00', '2024-04-25 15:00:00'],
    ['2024-04-25 23:00:00', '2024-04-26 05:00:00'],
    ['2024-04-26 11:00:00', '2024-04-26 15:00:00'],
    ['2024-04-26 23:00:00', '2024-04-27 05:00:00'],
    ['2024-04-27 11:00:00', '2024-04-27 15:00:00'],
    ['2024-04-27 23:00:00', '2024-04-28 05:00:00'],
    ['2024-04-28 11:00:00', '2024-04-28 15:00:00'],
    ['2024-04-28 23:00:00', '2024-04-29 05:00:00'],
    ['2024-04-29 11:00:00', '2024-04-29 15:00:00'],
    ['2024-04-29 23:00:00', '2024-04-30 05:00:00'],
    ['2024-04-30 11:00:00', '2024-04-30 15:00:00'],
    ['2024-04-30 23:00:00', '2024-05-01 05:00:00'],
    ['2024-05-01 11:00:00', '2024-05-01 15:00:00'],
    ['2024-05-01 23:00:00', '2024-05-02 05:00:00'],
    ['2024-05-02 11:00:00', '2024-05-02 15:00:00'],
    ['2024-05-02 23:00:00', '2024-05-03 05:00:00'],
    ['2024-05-03 11:00:00', '2024-05-03 15:00:00'],
    ['2024-05-03 23:00:00', '2024-05-04 05:00:00'],
    ['2024-05-04 11:00:00', '2024-05-04 15:00:00'],
    ['2024-05-04 23:00:00', '2024-05-05 05:00:00'],
    ['2024-05-05 11:00:00', '2024-05-05 15:00:00'],
    ['2024-05-05 23:00:00', '2024-05-06 05:00:00'],
    ['2024-05-06 11:00:00', '2024-05-06 15:00:00'],
    ['2024-05-06 23:00:00', '2024-05-07 05:00:00'],
    ['2024-05-07 11:00:00', '2024-05-07 15:00:00'],
    ['2024-05-07 23:00:00', '2024-05-08 05:00:00'],
    ['2024-05-08 11:00:00', '2024-05-08 15:00:00'],
    ['2024-05-08 23:00:00', '2024-05-09 05:00:00'],
    ['2024-05-09 11:00:00', '2024-05-09 15:00:00'],
    ['2024-05-09 23:00:00', '2024-05-10 05:00:00'],
    ['2024-05-10 11:00:00', '2024-05-10 15:00:00'],
    ['2024-05-10 23:00:00', '2024-05-11 05:00:00'],
    ['2024-05-11 11:00:00', '2024-05-11 15:00:00'],
    ['2024-05-11 23:00:00', '2024-05-12 05:00:00'],
    ['2024-05-12 11:00:00', '2024-05-12 15:00:00'],
    ['2024-05-12 23:00:00', '2024-05-13 05:00:00'],
    ['2024-05-13 11:00:00', '2024-05-13 15:00:00'],
    ['2024-05-16 11:00:00', '2024-05-16 15:00:00'],
    ['2024-05-16 23:00:00', '2024-05-17 05:00:00'],
    ['2024-05-17 11:00:00', '2024-05-17 15:00:00'],
    ['2024-05-17 23:00:00', '2024-05-18 05:00:00'],
    ['2024-05-18 11:00:00', '2024-05-18 15:00:00'],
    ['2024-05-18 23:00:00', '2024-05-19 05:00:00'],
    ['2024-05-19 11:00:00', '2024-05-19 15:00:00'],
    ['2024-05-19 23:00:00', '2024-05-20 05:00:00'],
    ['2024-05-20 11:00:00', '2024-05-20 15:00:00'],
    ['2024-05-20 23:00:00', '2024-05-21 05:00:00'],
    ['2024-05-21 11:00:00', '2024-05-21 15:00:00'],
    ['2024-05-21 23:00:00', '2024-05-22 05:00:00'],
    ['2024-05-22 11:00:00', '2024-05-22 15:00:00'],
    ['2024-05-22 23:00:00', '2024-05-23 05:00:00'],
    #['2024-05-23 11:00:00', '2024-05-23 15:00:00'],
    ['2024-05-23 23:00:00', '2024-05-24 05:00:00'],
    ['2024-05-24 11:00:00', '2024-05-24 15:00:00'],
    ['2024-05-24 23:00:00', '2024-05-25 05:00:00'],
    ['2024-05-25 11:00:00', '2024-05-25 15:00:00'],
    ['2024-05-25 23:00:00', '2024-05-26 05:00:00'],
    ['2024-05-26 11:00:00', '2024-05-26 15:00:00'],
    ['2024-05-26 23:00:00', '2024-05-27 05:00:00'],
    ['2024-05-27 11:00:00', '2024-05-27 15:00:00'],
    ['2024-05-27 23:00:00', '2024-05-28 05:00:00'],
    ['2024-05-28 11:00:00', '2024-05-28 15:00:00'],
    ['2024-05-28 23:00:00', '2024-05-29 05:00:00'],
    ['2024-05-29 11:00:00', '2024-05-29 15:00:00'],
    ['2024-05-31 11:00:00', '2024-05-31 15:00:00'],
    ['2024-05-31 23:00:00', '2024-06-01 05:00:00'],
    ['2024-06-01 11:00:00', '2024-06-01 15:00:00'],
    ['2024-06-01 23:00:00', '2024-06-02 05:00:00'],
    ['2024-06-02 11:00:00', '2024-06-02 15:00:00'],
    ['2024-06-02 23:00:00', '2024-06-03 05:00:00'],
    ['2024-06-03 11:00:00', '2024-06-03 15:00:00'],
    ['2024-06-03 23:00:00', '2024-06-04 05:00:00'],
    ['2024-06-04 11:00:00', '2024-06-04 15:00:00'],
    ['2024-06-04 23:00:00', '2024-06-05 05:00:00'],
    ['2024-06-05 11:00:00', '2024-06-05 15:00:00'],
    ['2024-06-05 23:00:00', '2024-06-06 05:00:00'],
    ['2024-06-06 11:00:00', '2024-06-06 15:00:00'],
    ['2024-06-06 23:00:00', '2024-06-07 05:00:00'],
    ['2024-06-07 11:00:00', '2024-06-07 15:00:00'],
    ['2024-06-07 23:00:00', '2024-06-08 05:00:00'],
    ['2024-06-08 11:00:00', '2024-06-08 15:00:00'],
    ['2024-06-08 23:00:00', '2024-06-09 05:00:00'],
    ['2024-06-09 11:00:00', '2024-06-09 15:00:00'],
    ['2024-06-09 23:00:00', '2024-06-10 05:00:00'],
    ['2024-06-10 11:00:00', '2024-06-10 15:00:00'],
    ['2024-06-10 23:00:00', '2024-06-11 05:00:00'],
    ['2024-06-11 11:00:00', '2024-06-11 15:00:00'],
    ['2024-06-11 23:00:00', '2024-06-12 05:00:00'],
    ['2024-06-12 11:00:00', '2024-06-12 15:00:00'],
    ['2024-06-12 23:00:00', '2024-06-13 05:00:00'],
    ['2024-06-13 11:00:00', '2024-06-13 15:00:00'],
    ['2024-06-13 23:00:00', '2024-06-14 05:00:00'],
    ['2024-06-14 11:00:00', '2024-06-14 15:00:00'],
    ['2024-06-14 23:00:00', '2024-06-15 05:00:00'],
    ['2024-06-15 11:00:00', '2024-06-15 15:00:00'],
    ['2024-06-15 23:00:00', '2024-06-16 05:00:00'],
    ['2024-06-16 11:00:00', '2024-06-16 15:00:00'],
    ['2024-06-16 23:00:00', '2024-06-17 05:00:00'],
    ['2024-06-17 11:00:00', '2024-06-17 15:00:00'],
    ['2024-06-19 11:00:00', '2024-06-19 15:00:00'],
    ['2024-06-19 23:00:00', '2024-06-20 05:00:00'],
    ['2024-06-20 11:00:00', '2024-06-20 15:00:00'],
    ['2024-06-20 23:00:00', '2024-06-21 05:00:00'],
    ['2024-06-21 11:00:00', '2024-06-21 15:00:00'],
    ['2024-06-21 23:00:00', '2024-06-22 05:00:00'],
    ['2024-06-22 11:00:00', '2024-06-22 15:00:00'],
    ['2024-06-22 23:00:00', '2024-06-23 05:00:00'],
    ['2024-06-23 11:00:00', '2024-06-23 15:00:00'],
    ['2024-06-23 23:00:00', '2024-06-24 05:00:00'],
    ['2024-06-24 11:00:00', '2024-06-24 15:00:00'],
    ['2024-06-24 23:00:00', '2024-06-25 05:00:00'],
    ['2024-06-25 11:00:00', '2024-06-25 15:00:00'],
    ['2024-06-25 23:00:00', '2024-06-26 05:00:00'],
    ['2024-06-28 23:00:00', '2024-06-29 05:00:00'],
    ['2024-06-29 11:00:00', '2024-06-29 15:00:00'],
    ['2024-06-29 23:00:00', '2024-06-30 05:00:00'],
    ['2024-06-30 11:00:00', '2024-06-30 15:00:00'],
    ['2024-06-30 23:00:00', '2024-07-01 05:00:00'],
    ['2024-07-01 11:00:00', '2024-07-01 15:00:00'],
    ['2024-07-01 23:00:00', '2024-07-02 05:00:00'],
    ['2024-07-02 11:00:00', '2024-07-02 15:00:00'],
    ['2024-07-02 23:00:00', '2024-07-03 05:00:00'],
    ['2024-07-03 11:00:00', '2024-07-03 15:00:00'],
    ['2024-07-03 23:00:00', '2024-07-04 05:00:00'],
    ['2024-07-04 11:00:00', '2024-07-04 15:00:00'],
    ['2024-07-04 23:00:00', '2024-07-05 05:00:00'],
    ['2024-07-05 11:00:00', '2024-07-05 15:00:00'],
    ['2024-07-05 23:00:00', '2024-07-06 05:00:00'],
    ['2024-07-06 11:00:00', '2024-07-06 15:00:00'],
    ['2024-07-06 23:00:00', '2024-07-07 05:00:00'],
    ['2024-07-07 11:00:00', '2024-07-07 15:00:00'],
    ['2024-07-07 23:00:00', '2024-07-08 05:00:00'],
    ['2024-07-08 11:00:00', '2024-07-08 15:00:00'],
    ['2024-07-08 23:00:00', '2024-07-09 05:00:00'],
    ['2024-07-09 11:00:00', '2024-07-09 15:00:00'],
    ['2024-07-09 23:00:00', '2024-07-10 05:00:00'],
    ['2024-07-10 11:00:00', '2024-07-10 15:00:00'],
    ['2024-07-10 23:00:00', '2024-07-11 05:00:00'],
    ['2024-07-11 11:00:00', '2024-07-11 15:00:00'],
    ['2024-07-11 23:00:00', '2024-07-12 05:00:00'],
    ['2024-07-12 11:00:00', '2024-07-12 15:00:00'],
    ['2024-07-12 23:00:00', '2024-07-13 05:00:00'],
    ['2024-07-13 11:00:00', '2024-07-13 15:00:00'],
    ['2024-07-13 23:00:00', '2024-07-14 05:00:00'],
    ['2024-07-14 11:00:00', '2024-07-14 15:00:00'],
    ['2024-07-14 23:00:00', '2024-07-15 05:00:00'],
    ['2024-07-15 11:00:00', '2024-07-15 15:00:00'],
    ['2024-07-15 23:00:00', '2024-07-16 05:00:00'],
    ['2024-07-16 11:00:00', '2024-07-16 15:00:00'],
    ['2024-07-16 23:00:00', '2024-07-17 05:00:00'],
    ['2024-07-17 11:00:00', '2024-07-17 15:00:00'],
    ['2024-07-19 11:00:00', '2024-07-19 15:00:00'],
    ['2024-07-19 23:00:00', '2024-07-20 05:00:00'],
    ['2024-07-20 11:00:00', '2024-07-20 15:00:00'],
    ['2024-07-20 23:00:00', '2024-07-21 05:00:00'],
    ['2024-07-21 11:00:00', '2024-07-21 15:00:00'],
    ['2024-07-21 23:00:00', '2024-07-22 05:00:00'],
    ['2024-07-22 11:00:00', '2024-07-22 15:00:00'],
    ['2024-07-22 23:00:00', '2024-07-23 05:00:00'],
    ['2024-07-23 11:00:00', '2024-07-23 15:00:00'],
    ['2024-07-23 23:00:00', '2024-07-24 05:00:00'],
    ['2024-07-24 11:00:00', '2024-07-24 15:00:00'],
    ['2024-07-24 23:00:00', '2024-07-25 05:00:00'],
    ['2024-07-25 11:00:00', '2024-07-25 15:00:00'],
    ['2024-07-25 23:00:00', '2024-07-26 05:00:00'],
    ['2024-07-26 11:00:00', '2024-07-26 15:00:00'],
    ['2024-07-26 23:00:00', '2024-07-27 05:00:00'],
    ['2024-07-27 11:00:00', '2024-07-27 15:00:00'],
    ['2024-07-27 23:00:00', '2024-07-28 05:00:00'],
    ['2024-07-28 11:00:00', '2024-07-28 15:00:00'],
    ['2024-07-28 23:00:00', '2024-07-29 05:00:00'],
    ['2024-07-29 11:00:00', '2024-07-29 15:00:00'],
    ['2024-07-29 23:00:00', '2024-07-30 05:00:00'],
    ['2024-07-30 11:00:00', '2024-07-30 15:00:00'],
    ['2024-08-04 23:00:00', '2024-08-05 05:00:00'],
    ['2024-08-05 11:00:00', '2024-08-05 15:00:00'],
    ['2024-08-05 23:00:00', '2024-08-06 05:00:00'],
    ['2024-08-06 11:00:00', '2024-08-06 15:00:00'],
    ['2024-08-06 23:00:00', '2024-08-07 05:00:00'],
    ['2024-08-07 11:00:00', '2024-08-07 15:00:00'],
    ['2024-08-07 23:00:00', '2024-08-08 05:00:00'],
    ['2024-08-08 11:00:00', '2024-08-08 15:00:00'],
    ['2024-08-08 23:00:00', '2024-08-09 05:00:00'],
    ['2024-08-09 11:00:00', '2024-08-09 15:00:00'],
    ['2024-08-09 23:00:00', '2024-08-10 05:00:00'],
    ['2024-08-10 11:00:00', '2024-08-10 15:00:00'],
    ['2024-08-10 23:00:00', '2024-08-11 05:00:00'],
    ['2024-08-11 11:00:00', '2024-08-11 15:00:00'],
    ['2024-08-11 23:00:00', '2024-08-12 05:00:00'],
    ['2024-08-12 11:00:00', '2024-08-12 15:00:00'],
    ['2024-08-12 23:00:00', '2024-08-13 05:00:00'],
    ['2024-08-13 11:00:00', '2024-08-13 15:00:00'],
    ['2024-08-13 23:00:00', '2024-08-14 05:00:00'],
    ['2024-08-14 11:00:00', '2024-08-14 15:00:00'],
    ['2024-08-14 23:00:00', '2024-08-15 05:00:00'],
    ['2024-08-15 11:00:00', '2024-08-15 15:00:00'],
    ['2024-08-15 23:00:00', '2024-08-16 05:00:00'],
    ['2024-08-16 11:00:00', '2024-08-16 15:00:00'],
    ['2024-08-16 23:00:00', '2024-08-17 05:00:00'],
    ['2024-08-17 11:00:00', '2024-08-17 15:00:00'],
    ['2024-08-17 23:00:00', '2024-08-18 05:00:00'],
    ['2024-08-18 11:00:00', '2024-08-18 15:00:00'],
    ['2024-08-20 11:00:00', '2024-08-20 15:00:00'],
    ['2024-08-20 23:00:00', '2024-08-21 05:00:00'],
    ['2024-08-21 11:00:00', '2024-08-21 15:00:00'],
    ['2024-08-21 23:00:00', '2024-08-22 05:00:00'],
    ['2024-08-22 11:00:00', '2024-08-22 15:00:00'],
    ['2024-08-22 23:00:00', '2024-08-23 05:00:00'],
    ['2024-08-23 11:00:00', '2024-08-23 15:00:00'],
    ['2024-08-23 23:00:00', '2024-08-24 05:00:00'],
    ['2024-08-24 11:00:00', '2024-08-24 15:00:00'],
    ['2024-08-24 23:00:00', '2024-08-25 05:00:00'],
    ['2024-08-25 11:00:00', '2024-08-25 15:00:00']
]

filtered_intervals = [
    iv for iv in intervals
    # Keep only if not 19->07
    if not (iv[0].endswith('23:00:00') and iv[1].endswith('05:00:00'))
]

filtered_intervals

In [ ]:
x=raw.light.summary_statistics_per_time_bin(
    bins=[
 ['2022-09-22 07:00:00', '2022-09-22 17:00:00'],
 ['2022-09-23 07:00:00', '2022-09-23 17:00:00'],
 ['2022-09-24 07:00:00', '2022-09-24 17:00:00'],
 ['2022-09-25 07:00:00', '2022-09-25 17:00:00'],
 ['2022-09-26 07:00:00', '2022-09-26 17:00:00'],
 ['2022-09-27 07:00:00', '2022-09-27 17:00:00'],
 ['2022-09-28 07:00:00', '2022-09-28 17:00:00'],
 ['2022-09-29 07:00:00', '2022-09-29 17:00:00'],
 ['2022-09-30 07:00:00', '2022-09-30 17:00:00'],
 ['2022-10-01 07:00:00', '2022-10-01 17:00:00'],
 ['2022-10-02 07:00:00', '2022-10-02 17:00:00'],
 ['2022-10-03 07:00:00', '2022-10-03 17:00:00'],
 ['2022-10-04 07:00:00', '2022-10-04 17:00:00'],
 ['2022-10-05 07:00:00', '2022-10-05 17:00:00'],
 ['2022-10-06 07:00:00', '2022-10-06 17:00:00'],
 ['2022-10-07 07:00:00', '2022-10-07 17:00:00'],
 ['2022-10-08 07:00:00', '2022-10-08 17:00:00'],
 ['2022-10-09 07:00:00', '2022-10-09 17:00:00'],
 ['2022-10-10 07:00:00', '2022-10-10 17:00:00'],
 ['2022-10-11 07:00:00', '2022-10-11 17:00:00'],
 ['2022-10-12 07:00:00', '2022-10-12 17:00:00'],
 ['2022-10-13 07:00:00', '2022-10-13 17:00:00'],
 ['2022-10-14 07:00:00', '2022-10-14 17:00:00'],
 ['2022-10-15 07:00:00', '2022-10-15 17:00:00'],
 ['2022-10-16 07:00:00', '2022-10-16 17:00:00'],
 ['2022-10-17 07:00:00', '2022-10-17 17:00:00'],
 ['2022-10-19 07:00:00', '2022-10-19 17:00:00'],
 ['2022-10-20 07:00:00', '2022-10-20 17:00:00'],
 ['2022-10-21 07:00:00', '2022-10-21 17:00:00'],
 ['2022-10-22 07:00:00', '2022-10-22 17:00:00'],
 ['2022-10-23 07:00:00', '2022-10-23 17:00:00'],
 ['2022-10-24 07:00:00', '2022-10-24 17:00:00'],
 ['2022-10-25 07:00:00', '2022-10-25 17:00:00'],
 ['2022-10-26 07:00:00', '2022-10-26 17:00:00'],
 ['2022-10-27 07:00:00', '2022-10-27 17:00:00'],
 ['2022-10-28 07:00:00', '2022-10-28 17:00:00'],
 ['2022-10-29 07:00:00', '2022-10-29 17:00:00'],
 ['2022-10-30 07:00:00', '2022-10-30 17:00:00'],
 ['2022-10-31 07:00:00', '2022-10-31 17:00:00'],
 ['2022-11-01 07:00:00', '2022-11-01 17:00:00'],
 ['2022-11-02 07:00:00', '2022-11-02 17:00:00'],
 ['2022-11-03 07:00:00', '2022-11-03 17:00:00'],
 ['2022-11-04 07:00:00', '2022-11-04 17:00:00'],
 ['2022-11-05 07:00:00', '2022-11-05 17:00:00'],
 ['2022-11-06 07:00:00', '2022-11-06 17:00:00'],
 ['2022-11-07 07:00:00', '2022-11-07 17:00:00'],
 ['2022-11-08 07:00:00', '2022-11-08 17:00:00'],
 ['2022-11-09 07:00:00', '2022-11-09 17:00:00'],
 ['2022-11-10 07:00:00', '2022-11-10 17:00:00'],
 ['2022-11-11 07:00:00', '2022-11-11 17:00:00'],
 ['2022-11-12 07:00:00', '2022-11-12 17:00:00'],
 ['2022-11-13 07:00:00', '2022-11-13 17:00:00'],
 ['2022-11-14 07:00:00', '2022-11-14 17:00:00'],
 ['2022-11-15 07:00:00', '2022-11-15 17:00:00'],
 ['2022-11-16 07:00:00', '2022-11-16 17:00:00'],
 ['2022-11-17 07:00:00', '2022-11-17 17:00:00'],
 ['2022-11-19 07:00:00', '2022-11-19 17:00:00'],
 ['2022-11-20 07:00:00', '2022-11-20 17:00:00'],
 ['2022-11-21 07:00:00', '2022-11-21 17:00:00'],
 ['2022-11-22 07:00:00', '2022-11-22 17:00:00'],
 ['2022-11-23 07:00:00', '2022-11-23 17:00:00'],
 ['2022-11-24 07:00:00', '2022-11-24 17:00:00'],
 ['2022-11-25 07:00:00', '2022-11-25 17:00:00'],
 ['2022-11-26 07:00:00', '2022-11-26 17:00:00'],
 ['2022-11-27 07:00:00', '2022-11-27 17:00:00'],
 ['2022-11-28 07:00:00', '2022-11-28 17:00:00'],
 ['2022-11-29 07:00:00', '2022-11-29 17:00:00'],
 ['2022-11-30 07:00:00', '2022-11-30 17:00:00'],
 ['2022-12-01 07:00:00', '2022-12-01 17:00:00'],
 ['2022-12-02 07:00:00', '2022-12-02 17:00:00'],
 ['2022-12-03 07:00:00', '2022-12-03 17:00:00'],
 ['2022-12-04 07:00:00', '2022-12-04 17:00:00'],
 ['2022-12-05 07:00:00', '2022-12-05 17:00:00'],
 ['2022-12-06 07:00:00', '2022-12-06 17:00:00'],
 ['2022-12-07 07:00:00', '2022-12-07 17:00:00'],
 ['2022-12-08 07:00:00', '2022-12-08 17:00:00'],
 ['2022-12-09 07:00:00', '2022-12-09 17:00:00'],
 ['2022-12-10 07:00:00', '2022-12-10 17:00:00'],
 ['2022-12-11 07:00:00', '2022-12-11 17:00:00'],
 ['2022-12-12 07:00:00', '2022-12-12 17:00:00'],
 ['2022-12-13 07:00:00', '2022-12-13 17:00:00'],
 ['2022-12-14 07:00:00', '2022-12-14 17:00:00'],
 ['2022-12-15 07:00:00', '2022-12-15 17:00:00'],
 ['2022-12-16 07:00:00', '2022-12-16 17:00:00'],
 ['2022-12-17 07:00:00', '2022-12-17 17:00:00'],
 ['2022-12-18 07:00:00', '2022-12-18 17:00:00'],
 ['2022-12-19 07:00:00', '2022-12-19 17:00:00'],
 ['2022-12-20 07:00:00', '2022-12-20 17:00:00'],
 ['2022-12-21 07:00:00', '2022-12-21 17:00:00'],
 ['2022-12-22 07:00:00', '2022-12-22 17:00:00'],
 ['2022-12-23 07:00:00', '2022-12-23 17:00:00'],
 ['2022-12-24 07:00:00', '2022-12-24 17:00:00'],
 ['2022-12-25 07:00:00', '2022-12-25 17:00:00'],
 ['2022-12-26 07:00:00', '2022-12-26 17:00:00'],
 ['2022-12-27 07:00:00', '2022-12-27 17:00:00'],
 ['2022-12-29 07:00:00', '2022-12-29 17:00:00'],
 ['2022-12-30 07:00:00', '2022-12-30 17:00:00'],
 ['2022-12-31 07:00:00', '2022-12-31 17:00:00'],
 ['2023-01-01 07:00:00', '2023-01-01 17:00:00'],
 ['2023-01-02 07:00:00', '2023-01-02 17:00:00'],
 ['2023-01-03 07:00:00', '2023-01-03 17:00:00'],
 ['2023-01-04 07:00:00', '2023-01-04 17:00:00'],
 ['2023-01-05 07:00:00', '2023-01-05 17:00:00'],
 ['2023-01-06 07:00:00', '2023-01-06 17:00:00'],
 ['2023-01-07 07:00:00', '2023-01-07 17:00:00'],
 ['2023-01-08 07:00:00', '2023-01-08 17:00:00'],
 ['2023-01-09 07:00:00', '2023-01-09 17:00:00'],
 ['2023-01-10 07:00:00', '2023-01-10 17:00:00'],
 ['2023-01-11 07:00:00', '2023-01-11 17:00:00'],
 ['2023-01-12 07:00:00', '2023-01-12 17:00:00'],
 ['2023-01-13 07:00:00', '2023-01-13 17:00:00'],
 ['2023-01-14 07:00:00', '2023-01-14 17:00:00'],
 ['2023-01-15 07:00:00', '2023-01-15 17:00:00'],
 ['2023-01-16 07:00:00', '2023-01-16 17:00:00'],
 ['2023-01-17 07:00:00', '2023-01-17 17:00:00'],
 ['2023-01-19 07:00:00', '2023-01-19 17:00:00'],
 ['2023-01-20 07:00:00', '2023-01-20 17:00:00'],
 ['2023-01-21 07:00:00', '2023-01-21 17:00:00'],
 ['2023-01-22 07:00:00', '2023-01-22 17:00:00'],
 ['2023-01-23 07:00:00', '2023-01-23 17:00:00'],
 ['2023-01-24 07:00:00', '2023-01-24 17:00:00'],
 ['2023-01-25 07:00:00', '2023-01-25 17:00:00'],
 ['2023-01-26 07:00:00', '2023-01-26 17:00:00'],
 ['2023-01-27 07:00:00', '2023-01-27 17:00:00'],
 ['2023-01-28 07:00:00', '2023-01-28 17:00:00'],
 ['2023-01-29 07:00:00', '2023-01-29 17:00:00'],
 ['2023-01-30 07:00:00', '2023-01-30 17:00:00'],
 ['2023-01-31 07:00:00', '2023-01-31 17:00:00'],
 ['2023-02-01 07:00:00', '2023-02-01 17:00:00'],
 ['2023-02-02 07:00:00', '2023-02-02 17:00:00'],
 ['2023-02-03 07:00:00', '2023-02-03 17:00:00'],
 ['2023-02-04 07:00:00', '2023-02-04 17:00:00'],
 ['2023-02-05 07:00:00', '2023-02-05 17:00:00'],
 ['2023-02-06 07:00:00', '2023-02-06 17:00:00'],
 ['2023-02-07 07:00:00', '2023-02-07 17:00:00'],
 ['2023-02-08 07:00:00', '2023-02-08 17:00:00'],
 ['2023-02-09 07:00:00', '2023-02-09 17:00:00'],
 ['2023-02-10 07:00:00', '2023-02-10 17:00:00'],
 ['2023-02-11 07:00:00', '2023-02-11 17:00:00'],
 ['2023-02-12 07:00:00', '2023-02-12 17:00:00'],
 ['2023-02-13 07:00:00', '2023-02-13 17:00:00'],
 ['2023-02-14 07:00:00', '2023-02-14 17:00:00'],
 ['2023-02-15 07:00:00', '2023-02-15 17:00:00'],
 ['2023-02-16 07:00:00', '2023-02-16 17:00:00'],
 ['2023-02-17 07:00:00', '2023-02-17 17:00:00'],
 ['2023-02-18 07:00:00', '2023-02-18 17:00:00'],
 ['2023-02-19 07:00:00', '2023-02-19 17:00:00'],
 ['2023-02-20 07:00:00', '2023-02-20 17:00:00'],
 ['2023-02-21 07:00:00', '2023-02-21 17:00:00'],
 ['2023-02-22 07:00:00', '2023-02-22 17:00:00'],
 ['2023-02-23 07:00:00', '2023-02-23 17:00:00'],
 ['2023-02-24 07:00:00', '2023-02-24 17:00:00'],
 ['2023-02-25 07:00:00', '2023-02-25 17:00:00'],
 ['2023-02-26 07:00:00', '2023-02-26 17:00:00'],
 ['2023-02-27 07:00:00', '2023-02-27 17:00:00'],
 ['2023-02-28 07:00:00', '2023-02-28 17:00:00'],
 ['2023-03-01 07:00:00', '2023-03-01 17:00:00'],
 ['2023-03-02 07:00:00', '2023-03-02 17:00:00'],
 ['2023-03-03 07:00:00', '2023-03-03 17:00:00'],
 ['2023-04-19 07:00:00', '2023-04-19 17:00:00'],
 ['2023-04-20 07:00:00', '2023-04-20 17:00:00'],
 ['2023-04-21 07:00:00', '2023-04-21 17:00:00'],
 ['2023-04-22 07:00:00', '2023-04-22 17:00:00'],
 ['2023-04-23 07:00:00', '2023-04-23 17:00:00'],
 ['2023-04-24 07:00:00', '2023-04-24 17:00:00'],
 ['2023-04-25 07:00:00', '2023-04-25 17:00:00'],
 ['2023-04-26 07:00:00', '2023-04-26 17:00:00'],
 ['2023-04-27 07:00:00', '2023-04-27 17:00:00'],
 ['2023-04-28 07:00:00', '2023-04-28 17:00:00'],
 ['2023-04-29 07:00:00', '2023-04-29 17:00:00'],
 ['2023-04-30 07:00:00', '2023-04-30 17:00:00'],
 ['2023-05-01 07:00:00', '2023-05-01 17:00:00'],
 ['2023-05-02 07:00:00', '2023-05-02 17:00:00'],
 ['2023-05-03 07:00:00', '2023-05-03 17:00:00'],
 ['2023-05-04 07:00:00', '2023-05-04 17:00:00'],
 ['2023-05-05 07:00:00', '2023-05-05 17:00:00'],
 ['2023-05-06 07:00:00', '2023-05-06 17:00:00'],
 ['2023-05-07 07:00:00', '2023-05-07 17:00:00'],
 ['2023-05-08 07:00:00', '2023-05-08 17:00:00'],
 ['2023-05-09 07:00:00', '2023-05-09 17:00:00'],
 ['2023-05-10 07:00:00', '2023-05-10 17:00:00'],
 ['2023-05-11 07:00:00', '2023-05-11 17:00:00'],
 ['2023-05-12 07:00:00', '2023-05-12 17:00:00'],
 ['2023-05-13 07:00:00', '2023-05-13 17:00:00'],
 ['2023-05-14 07:00:00', '2023-05-14 17:00:00'],
 ['2023-05-15 07:00:00', '2023-05-15 17:00:00'],
 ['2023-06-30 07:00:00', '2023-06-30 17:00:00'],
 ['2023-07-01 07:00:00', '2023-07-01 17:00:00'],
 ['2023-07-02 07:00:00', '2023-07-02 17:00:00'],
 ['2023-07-03 07:00:00', '2023-07-03 17:00:00'],
 ['2023-07-04 07:00:00', '2023-07-04 17:00:00'],
 ['2023-07-05 07:00:00', '2023-07-05 17:00:00'],
 ['2023-07-06 07:00:00', '2023-07-06 17:00:00'],
 ['2023-07-07 07:00:00', '2023-07-07 17:00:00'],
 ['2023-07-08 07:00:00', '2023-07-08 17:00:00'],
 ['2023-07-09 07:00:00', '2023-07-09 17:00:00'],
 ['2023-07-10 07:00:00', '2023-07-10 17:00:00'],
 ['2023-07-11 07:00:00', '2023-07-11 17:00:00'],
 ['2023-07-12 07:00:00', '2023-07-12 17:00:00'],
 ['2023-07-13 07:00:00', '2023-07-13 17:00:00'],
 ['2023-07-14 07:00:00', '2023-07-14 17:00:00'],
 ['2023-07-15 07:00:00', '2023-07-15 17:00:00'],
 ['2023-07-16 07:00:00', '2023-07-16 17:00:00'],
 ['2023-07-18 07:00:00', '2023-07-18 17:00:00'],
 ['2023-07-19 07:00:00', '2023-07-19 17:00:00'],
 ['2023-07-20 07:00:00', '2023-07-20 17:00:00'],
 ['2023-07-21 07:00:00', '2023-07-21 17:00:00'],
 ['2023-07-22 07:00:00', '2023-07-22 17:00:00'],
 ['2023-07-23 07:00:00', '2023-07-23 17:00:00'],
 ['2023-07-24 07:00:00', '2023-07-24 17:00:00'],
 ['2023-07-25 07:00:00', '2023-07-25 17:00:00'],
 ['2023-07-26 07:00:00', '2023-07-26 17:00:00'],
 ['2023-07-27 07:00:00', '2023-07-27 17:00:00'],
 ['2023-08-23 07:00:00', '2023-08-23 17:00:00'],
 ['2023-08-24 07:00:00', '2023-08-24 17:00:00'],
 ['2023-08-25 07:00:00', '2023-08-25 17:00:00'],
 ['2023-08-26 07:00:00', '2023-08-26 17:00:00'],
 ['2023-08-27 07:00:00', '2023-08-27 17:00:00'],
 ['2023-08-28 07:00:00', '2023-08-28 17:00:00'],
 ['2023-08-29 07:00:00', '2023-08-29 17:00:00'],
 ['2023-08-30 07:00:00', '2023-08-30 17:00:00'],
 ['2023-09-02 07:00:00', '2023-09-02 17:00:00'],
 ['2023-09-03 07:00:00', '2023-09-03 17:00:00'],
 ['2023-09-04 07:00:00', '2023-09-04 17:00:00'],
 ['2023-09-05 07:00:00', '2023-09-05 17:00:00'],
 ['2023-09-06 07:00:00', '2023-09-06 17:00:00'],
 ['2023-09-07 07:00:00', '2023-09-07 17:00:00'],
 ['2023-09-08 07:00:00', '2023-09-08 17:00:00'],
 ['2023-09-09 07:00:00', '2023-09-09 17:00:00'],
 ['2023-09-10 07:00:00', '2023-09-10 17:00:00'],
 ['2023-09-11 07:00:00', '2023-09-11 17:00:00'],
 ['2023-09-12 07:00:00', '2023-09-12 17:00:00'],
 ['2023-09-13 07:00:00', '2023-09-13 17:00:00'],
 ['2023-09-14 07:00:00', '2023-09-14 17:00:00'],
 ['2023-09-15 07:00:00', '2023-09-15 17:00:00'],
 ['2023-09-16 07:00:00', '2023-09-16 17:00:00'],
 ['2023-09-17 07:00:00', '2023-09-17 17:00:00'],
 ['2023-09-19 07:00:00', '2023-09-19 17:00:00'],
 ['2023-09-20 07:00:00', '2023-09-20 17:00:00'],
 ['2023-09-21 07:00:00', '2023-09-21 17:00:00'],
 ['2023-09-22 07:00:00', '2023-09-22 17:00:00'],
 ['2023-09-23 07:00:00', '2023-09-23 17:00:00'],
 ['2023-09-24 07:00:00', '2023-09-24 17:00:00'],
 ['2023-09-25 07:00:00', '2023-09-25 17:00:00'],
 ['2023-09-26 07:00:00', '2023-09-26 17:00:00'],
 ['2023-09-27 07:00:00', '2023-09-27 17:00:00'],
 ['2023-09-28 07:00:00', '2023-09-28 17:00:00'],
 ['2023-09-29 07:00:00', '2023-09-29 17:00:00'],
 ['2023-09-30 07:00:00', '2023-09-30 17:00:00'],
 ['2023-10-01 07:00:00', '2023-10-01 17:00:00'],
 ['2023-10-02 07:00:00', '2023-10-02 17:00:00'],
 ['2023-10-03 07:00:00', '2023-10-03 17:00:00'],
 ['2023-10-04 07:00:00', '2023-10-04 17:00:00'],
 ['2023-10-05 07:00:00', '2023-10-05 17:00:00'],
 ['2023-10-06 07:00:00', '2023-10-06 17:00:00'],
 ['2023-10-07 07:00:00', '2023-10-07 17:00:00'],
 ['2023-10-08 07:00:00', '2023-10-08 17:00:00'],
 ['2023-10-09 07:00:00', '2023-10-09 17:00:00'],
 ['2023-10-10 07:00:00', '2023-10-10 17:00:00'],
 ['2023-10-11 07:00:00', '2023-10-11 17:00:00'],
 ['2023-10-12 07:00:00', '2023-10-12 17:00:00'],
 ['2023-10-13 07:00:00', '2023-10-13 17:00:00'],
 ['2023-10-14 07:00:00', '2023-10-14 17:00:00'],
 ['2023-10-15 07:00:00', '2023-10-15 17:00:00'],
 ['2023-10-16 07:00:00', '2023-10-16 17:00:00'],
 ['2023-10-18 07:00:00', '2023-10-18 17:00:00'],
 ['2023-10-19 07:00:00', '2023-10-19 17:00:00'],
 ['2023-10-20 07:00:00', '2023-10-20 17:00:00'],
 ['2023-10-21 07:00:00', '2023-10-21 17:00:00'],
 ['2023-10-22 07:00:00', '2023-10-22 17:00:00'],
 ['2023-10-23 07:00:00', '2023-10-23 17:00:00'],
 ['2023-10-24 07:00:00', '2023-10-24 17:00:00'],
 ['2023-10-25 07:00:00', '2023-10-25 17:00:00'],
 ['2023-10-26 07:00:00', '2023-10-26 17:00:00'],
 ['2023-10-27 07:00:00', '2023-10-27 17:00:00'],
 ['2023-10-28 07:00:00', '2023-10-28 17:00:00'],
 ['2023-10-29 07:00:00', '2023-10-29 17:00:00'],
 ['2023-10-31 07:00:00', '2023-10-31 17:00:00'],
 ['2023-11-01 07:00:00', '2023-11-01 17:00:00'],
 ['2023-11-02 07:00:00', '2023-11-02 17:00:00'],
 ['2023-11-03 07:00:00', '2023-11-03 17:00:00'],
 ['2023-11-04 07:00:00', '2023-11-04 17:00:00'],
 ['2023-11-05 07:00:00', '2023-11-05 17:00:00'],
 ['2023-11-06 07:00:00', '2023-11-06 17:00:00'],
 ['2023-11-07 07:00:00', '2023-11-07 17:00:00'],
 ['2023-11-08 07:00:00', '2023-11-08 17:00:00'],
 ['2023-11-09 07:00:00', '2023-11-09 17:00:00'],
 ['2023-11-10 07:00:00', '2023-11-10 17:00:00'],
 ['2023-11-11 07:00:00', '2023-11-11 17:00:00'],
 ['2023-11-12 07:00:00', '2023-11-12 17:00:00'],
 ['2023-11-13 07:00:00', '2023-11-13 17:00:00'],
 ['2023-11-14 07:00:00', '2023-11-14 17:00:00'],
 ['2023-11-15 07:00:00', '2023-11-15 17:00:00'],
 ['2023-11-16 07:00:00', '2023-11-16 17:00:00'],
 ['2023-11-17 07:00:00', '2023-11-17 17:00:00'],
 ['2023-11-18 07:00:00', '2023-11-18 17:00:00'],
 ['2023-11-19 07:00:00', '2023-11-19 17:00:00'],
 ['2023-11-20 07:00:00', '2023-11-20 17:00:00'],
 ['2023-11-22 07:00:00', '2023-11-22 17:00:00'],
 ['2023-11-23 07:00:00', '2023-11-23 17:00:00'],
 ['2023-11-24 07:00:00', '2023-11-24 17:00:00'],
 ['2023-11-25 07:00:00', '2023-11-25 17:00:00'],
 ['2023-11-26 07:00:00', '2023-11-26 17:00:00'],
 ['2023-11-27 07:00:00', '2023-11-27 17:00:00'],
 ['2023-11-28 07:00:00', '2023-11-28 17:00:00'],
 ['2023-11-29 07:00:00', '2023-11-29 17:00:00'],
 ['2023-11-30 07:00:00', '2023-11-30 17:00:00'],
 ['2023-12-01 07:00:00', '2023-12-01 17:00:00'],
 ['2023-12-02 07:00:00', '2023-12-02 17:00:00'],
 ['2023-12-03 07:00:00', '2023-12-03 17:00:00'],
 ['2023-12-04 07:00:00', '2023-12-04 17:00:00'],
 ['2023-12-05 07:00:00', '2023-12-05 17:00:00'],
 ['2023-12-06 07:00:00', '2023-12-06 17:00:00'],
 ['2023-12-07 07:00:00', '2023-12-07 17:00:00'],
 ['2023-12-08 07:00:00', '2023-12-08 17:00:00'],
 ['2023-12-09 07:00:00', '2023-12-09 17:00:00'],
 ['2023-12-10 07:00:00', '2023-12-10 17:00:00'],
 ['2023-12-11 07:00:00', '2023-12-11 17:00:00'],
 ['2023-12-12 07:00:00', '2023-12-12 17:00:00'],
 ['2023-12-13 07:00:00', '2023-12-13 17:00:00'],
 ['2023-12-14 07:00:00', '2023-12-14 17:00:00'],
 ['2023-12-16 07:00:00', '2023-12-16 17:00:00'],
 ['2023-12-17 07:00:00', '2023-12-17 17:00:00'],
 ['2023-12-19 07:00:00', '2023-12-19 17:00:00'],
 ['2023-12-20 07:00:00', '2023-12-20 17:00:00'],
 ['2023-12-21 07:00:00', '2023-12-21 17:00:00'],
 ['2023-12-22 07:00:00', '2023-12-22 17:00:00'],
 ['2023-12-23 07:00:00', '2023-12-23 17:00:00'],
 ['2023-12-24 07:00:00', '2023-12-24 17:00:00'],
 ['2023-12-25 07:00:00', '2023-12-25 17:00:00'],
 ['2023-12-26 07:00:00', '2023-12-26 17:00:00'],
 ['2023-12-27 07:00:00', '2023-12-27 17:00:00'],
 ['2023-12-28 07:00:00', '2023-12-28 17:00:00'],
 ['2023-12-29 07:00:00', '2023-12-29 17:00:00'],
 ['2023-12-30 07:00:00', '2023-12-30 17:00:00'],
 ['2023-12-31 07:00:00', '2023-12-31 17:00:00'],
 ['2024-01-01 07:00:00', '2024-01-01 17:00:00'],
 ['2024-01-02 07:00:00', '2024-01-02 17:00:00'],
 ['2024-01-03 07:00:00', '2024-01-03 17:00:00'],
 ['2024-01-04 07:00:00', '2024-01-04 17:00:00'],
 ['2024-01-05 07:00:00', '2024-01-05 17:00:00'],
 ['2024-01-06 07:00:00', '2024-01-06 17:00:00'],
 ['2024-01-07 07:00:00', '2024-01-07 17:00:00'],
 ['2024-01-08 07:00:00', '2024-01-08 17:00:00'],
 ['2024-01-09 07:00:00', '2024-01-09 17:00:00'],
 ['2024-01-10 07:00:00', '2024-01-10 17:00:00'],
 ['2024-01-11 07:00:00', '2024-01-11 17:00:00'],
 ['2024-01-12 07:00:00', '2024-01-12 17:00:00'],
 ['2024-01-13 07:00:00', '2024-01-13 17:00:00'],
 ['2024-01-14 07:00:00', '2024-01-14 17:00:00'],
 ['2024-01-16 07:00:00', '2024-01-16 17:00:00'],
 ['2024-01-17 07:00:00', '2024-01-17 17:00:00'],
 ['2024-01-18 07:00:00', '2024-01-18 17:00:00'],
 ['2024-01-19 07:00:00', '2024-01-19 17:00:00'],
 ['2024-01-20 07:00:00', '2024-01-20 17:00:00'],
 ['2024-01-21 07:00:00', '2024-01-21 17:00:00'],
 ['2024-01-22 07:00:00', '2024-01-22 17:00:00'],
 ['2024-01-23 07:00:00', '2024-01-23 17:00:00'],
 ['2024-01-24 07:00:00', '2024-01-24 17:00:00'],
 ['2024-01-25 07:00:00', '2024-01-25 17:00:00'],
 ['2024-01-26 07:00:00', '2024-01-26 17:00:00'],
 ['2024-01-27 07:00:00', '2024-01-27 17:00:00'],
 ['2024-01-28 07:00:00', '2024-01-28 17:00:00'],
 ['2024-01-29 07:00:00', '2024-01-29 17:00:00'],
 ['2024-01-30 07:00:00', '2024-01-30 17:00:00'],
 ['2024-01-31 07:00:00', '2024-01-31 17:00:00'],
 ['2024-02-01 07:00:00', '2024-02-01 17:00:00'],
 ['2024-02-02 07:00:00', '2024-02-02 17:00:00'],
 ['2024-02-03 07:00:00', '2024-02-03 17:00:00'],
 ['2024-02-04 07:00:00', '2024-02-04 17:00:00'],
 ['2024-02-05 07:00:00', '2024-02-05 17:00:00'],
 ['2024-02-06 07:00:00', '2024-02-06 17:00:00'],
 ['2024-02-07 07:00:00', '2024-02-07 17:00:00'],
 ['2024-02-08 07:00:00', '2024-02-08 17:00:00'],
 ['2024-02-09 07:00:00', '2024-02-09 17:00:00'],
 ['2024-02-10 07:00:00', '2024-02-10 17:00:00'],
 ['2024-02-11 07:00:00', '2024-02-11 17:00:00'],
 ['2024-02-12 07:00:00', '2024-02-12 17:00:00'],
 ['2024-02-13 07:00:00', '2024-02-13 17:00:00'],
 ['2024-02-14 07:00:00', '2024-02-14 17:00:00'],
 ['2024-02-15 07:00:00', '2024-02-15 17:00:00'],
 ['2024-02-16 07:00:00', '2024-02-16 17:00:00'],
 ['2024-02-17 07:00:00', '2024-02-17 17:00:00'],
 ['2024-02-18 07:00:00', '2024-02-18 17:00:00'],
 ['2024-02-19 07:00:00', '2024-02-19 17:00:00'],
 ['2024-02-20 07:00:00', '2024-02-20 17:00:00'],
 ['2024-02-21 07:00:00', '2024-02-21 17:00:00'],
 ['2024-02-22 07:00:00', '2024-02-22 17:00:00'],
 ['2024-02-23 07:00:00', '2024-02-23 17:00:00'],
 ['2024-02-24 07:00:00', '2024-02-24 17:00:00'],
 ['2024-02-25 07:00:00', '2024-02-25 17:00:00'],
 ['2024-02-27 07:00:00', '2024-02-27 17:00:00'],
 ['2024-02-28 07:00:00', '2024-02-28 17:00:00'],
 ['2024-02-29 07:00:00', '2024-02-29 17:00:00'],
 ['2024-03-01 07:00:00', '2024-03-01 17:00:00'],
 ['2024-03-02 07:00:00', '2024-03-02 17:00:00'],
 ['2024-03-03 07:00:00', '2024-03-03 17:00:00'],
 ['2024-03-04 07:00:00', '2024-03-04 17:00:00'],
 ['2024-03-05 07:00:00', '2024-03-05 17:00:00'],
 ['2024-03-06 07:00:00', '2024-03-06 17:00:00'],
 ['2024-03-07 07:00:00', '2024-03-07 17:00:00'],
 ['2024-03-08 07:00:00', '2024-03-08 17:00:00'],
 ['2024-03-09 07:00:00', '2024-03-09 17:00:00'],
 ['2024-03-10 07:00:00', '2024-03-10 17:00:00'],
 ['2024-03-11 07:00:00', '2024-03-11 17:00:00'],
 ['2024-03-12 07:00:00', '2024-03-12 17:00:00'],
 ['2024-03-13 07:00:00', '2024-03-13 17:00:00'],
 ['2024-03-14 07:00:00', '2024-03-14 17:00:00'],
 ['2024-03-15 07:00:00', '2024-03-15 17:00:00'],
 ['2024-03-16 07:00:00', '2024-03-16 17:00:00'],
 ['2024-03-17 07:00:00', '2024-03-17 17:00:00'],
 ['2024-03-18 07:00:00', '2024-03-18 17:00:00'],
 ['2024-03-20 07:00:00', '2024-03-20 17:00:00'],
 ['2024-03-21 07:00:00', '2024-03-21 17:00:00'],
 ['2024-03-22 07:00:00', '2024-03-22 17:00:00'],
 ['2024-03-23 07:00:00', '2024-03-23 17:00:00'],
 ['2024-03-24 07:00:00', '2024-03-24 17:00:00'],
 ['2024-03-25 07:00:00', '2024-03-25 17:00:00'],
 ['2024-03-27 07:00:00', '2024-03-27 17:00:00'],
 ['2024-03-28 07:00:00', '2024-03-28 17:00:00'],
 ['2024-03-29 07:00:00', '2024-03-29 17:00:00'],
 ['2024-03-30 07:00:00', '2024-03-30 17:00:00'],
 ['2024-04-01 07:00:00', '2024-04-01 17:00:00'],
 ['2024-04-02 07:00:00', '2024-04-02 17:00:00'],
 ['2024-04-03 07:00:00', '2024-04-03 17:00:00'],
 ['2024-04-04 07:00:00', '2024-04-04 17:00:00'],
 ['2024-04-05 07:00:00', '2024-04-05 17:00:00'],
 ['2024-04-06 07:00:00', '2024-04-06 17:00:00'],
 ['2024-04-07 07:00:00', '2024-04-07 17:00:00'],
 ['2024-04-08 07:00:00', '2024-04-08 17:00:00'],
 ['2024-04-09 07:00:00', '2024-04-09 17:00:00'],
 ['2024-04-10 07:00:00', '2024-04-10 17:00:00'],
 ['2024-04-11 07:00:00', '2024-04-11 17:00:00'],
 ['2024-04-12 07:00:00', '2024-04-12 17:00:00'],
 ['2024-04-13 07:00:00', '2024-04-13 17:00:00'],
 ['2024-04-14 07:00:00', '2024-04-14 17:00:00'],
 ['2024-04-15 07:00:00', '2024-04-15 17:00:00'],
 ['2024-04-16 07:00:00', '2024-04-16 17:00:00'],
 ['2024-04-17 07:00:00', '2024-04-17 17:00:00'],
 ['2024-04-19 07:00:00', '2024-04-19 17:00:00'],
 ['2024-04-20 07:00:00', '2024-04-20 17:00:00'],
 ['2024-04-21 07:00:00', '2024-04-21 17:00:00'],
 ['2024-04-22 07:00:00', '2024-04-22 17:00:00'],
 ['2024-04-23 07:00:00', '2024-04-23 17:00:00'],
 ['2024-04-25 07:00:00', '2024-04-25 17:00:00'],
 ['2024-04-26 07:00:00', '2024-04-26 17:00:00'],
 ['2024-04-27 07:00:00', '2024-04-27 17:00:00'],
 ['2024-04-28 07:00:00', '2024-04-28 17:00:00'],
 ['2024-04-29 07:00:00', '2024-04-29 17:00:00'],
 ['2024-04-30 07:00:00', '2024-04-30 17:00:00'],
 ['2024-05-01 07:00:00', '2024-05-01 17:00:00'],
 ['2024-05-02 07:00:00', '2024-05-02 17:00:00'],
 ['2024-05-03 07:00:00', '2024-05-03 17:00:00'],
 ['2024-05-04 07:00:00', '2024-05-04 17:00:00'],
 ['2024-05-05 07:00:00', '2024-05-05 17:00:00'],
 ['2024-05-06 07:00:00', '2024-05-06 17:00:00'],
 ['2024-05-07 07:00:00', '2024-05-07 17:00:00'],
 ['2024-05-08 07:00:00', '2024-05-08 17:00:00'],
 ['2024-05-09 07:00:00', '2024-05-09 17:00:00'],
 ['2024-05-10 07:00:00', '2024-05-10 17:00:00'],
 ['2024-05-11 07:00:00', '2024-05-11 17:00:00'],
 ['2024-05-12 07:00:00', '2024-05-12 17:00:00'],
 ['2024-05-13 07:00:00', '2024-05-13 17:00:00'],
 ['2024-05-16 07:00:00', '2024-05-16 17:00:00'],
 ['2024-05-17 07:00:00', '2024-05-17 17:00:00'],
 ['2024-05-18 07:00:00', '2024-05-18 17:00:00'],
 ['2024-05-19 07:00:00', '2024-05-19 17:00:00'],
 ['2024-05-20 07:00:00', '2024-05-20 17:00:00'],
 ['2024-05-21 07:00:00', '2024-05-21 17:00:00'],
 ['2024-05-22 07:00:00', '2024-05-22 17:00:00'],
 ['2024-05-24 07:00:00', '2024-05-24 17:00:00'],
 ['2024-05-25 07:00:00', '2024-05-25 17:00:00'],
 ['2024-05-26 07:00:00', '2024-05-26 17:00:00'],
 ['2024-05-27 07:00:00', '2024-05-27 17:00:00'],
 ['2024-05-28 07:00:00', '2024-05-28 17:00:00'],
 ['2024-05-29 07:00:00', '2024-05-29 17:00:00'],
 ['2024-05-31 07:00:00', '2024-05-31 17:00:00'],
 ['2024-06-01 07:00:00', '2024-06-01 17:00:00'],
 ['2024-06-02 07:00:00', '2024-06-02 17:00:00'],
 ['2024-06-03 07:00:00', '2024-06-03 17:00:00'],
 ['2024-06-04 07:00:00', '2024-06-04 17:00:00'],
 ['2024-06-05 07:00:00', '2024-06-05 17:00:00'],
 ['2024-06-06 07:00:00', '2024-06-06 17:00:00'],
 ['2024-06-07 07:00:00', '2024-06-07 17:00:00'],
 ['2024-06-08 07:00:00', '2024-06-08 17:00:00'],
 ['2024-06-09 07:00:00', '2024-06-09 17:00:00'],
 ['2024-06-10 07:00:00', '2024-06-10 17:00:00'],
 ['2024-06-11 07:00:00', '2024-06-11 17:00:00'],
 ['2024-06-12 07:00:00', '2024-06-12 17:00:00'],
 ['2024-06-13 07:00:00', '2024-06-13 17:00:00'],
 ['2024-06-14 07:00:00', '2024-06-14 17:00:00'],
 ['2024-06-15 07:00:00', '2024-06-15 17:00:00'],
 ['2024-06-16 07:00:00', '2024-06-16 17:00:00'],
 ['2024-06-17 07:00:00', '2024-06-17 17:00:00'],
 ['2024-06-19 07:00:00', '2024-06-19 17:00:00'],
 ['2024-06-20 07:00:00', '2024-06-20 17:00:00'],
 ['2024-06-21 07:00:00', '2024-06-21 17:00:00'],
 ['2024-06-22 07:00:00', '2024-06-22 17:00:00'],
 ['2024-06-23 07:00:00', '2024-06-23 17:00:00'],
 ['2024-06-24 07:00:00', '2024-06-24 17:00:00'],
 ['2024-06-25 07:00:00', '2024-06-25 17:00:00'],
 ['2024-06-29 07:00:00', '2024-06-29 17:00:00'],
 ['2024-06-30 07:00:00', '2024-06-30 17:00:00'],
 ['2024-07-01 07:00:00', '2024-07-01 17:00:00'],
 ['2024-07-02 07:00:00', '2024-07-02 17:00:00'],
 ['2024-07-03 07:00:00', '2024-07-03 17:00:00'],
 ['2024-07-04 07:00:00', '2024-07-04 17:00:00'],
 ['2024-07-05 07:00:00', '2024-07-05 17:00:00'],
 ['2024-07-06 07:00:00', '2024-07-06 17:00:00'],
 ['2024-07-07 07:00:00', '2024-07-07 17:00:00'],
 ['2024-07-08 07:00:00', '2024-07-08 17:00:00'],
 ['2024-07-09 07:00:00', '2024-07-09 17:00:00'],
 ['2024-07-10 07:00:00', '2024-07-10 17:00:00'],
 ['2024-07-11 07:00:00', '2024-07-11 17:00:00'],
 ['2024-07-12 07:00:00', '2024-07-12 17:00:00'],
 ['2024-07-13 07:00:00', '2024-07-13 17:00:00'],
 ['2024-07-14 07:00:00', '2024-07-14 17:00:00'],
 ['2024-07-15 07:00:00', '2024-07-15 17:00:00'],
 ['2024-07-16 07:00:00', '2024-07-16 17:00:00'],
 ['2024-07-17 07:00:00', '2024-07-17 17:00:00'],
 ['2024-07-19 07:00:00', '2024-07-19 17:00:00'],
 ['2024-07-20 07:00:00', '2024-07-20 17:00:00'],
 ['2024-07-21 07:00:00', '2024-07-21 17:00:00'],
 ['2024-07-22 07:00:00', '2024-07-22 17:00:00'],
 ['2024-07-23 07:00:00', '2024-07-23 17:00:00'],
 ['2024-07-24 07:00:00', '2024-07-24 17:00:00'],
 ['2024-07-25 07:00:00', '2024-07-25 17:00:00'],
 ['2024-07-26 07:00:00', '2024-07-26 17:00:00'],
 ['2024-07-27 07:00:00', '2024-07-27 17:00:00'],
 ['2024-07-28 07:00:00', '2024-07-28 17:00:00'],
 ['2024-07-29 07:00:00', '2024-07-29 17:00:00'],
 ['2024-07-30 07:00:00', '2024-07-30 17:00:00'],
 ['2024-08-05 07:00:00', '2024-08-05 17:00:00'],
 ['2024-08-06 07:00:00', '2024-08-06 17:00:00'],
 ['2024-08-07 07:00:00', '2024-08-07 17:00:00'],
 ['2024-08-08 07:00:00', '2024-08-08 17:00:00'],
 ['2024-08-09 07:00:00', '2024-08-09 17:00:00'],
 ['2024-08-10 07:00:00', '2024-08-10 17:00:00'],
 ['2024-08-11 07:00:00', '2024-08-11 17:00:00'],
 ['2024-08-12 07:00:00', '2024-08-12 17:00:00'],
 ['2024-08-13 07:00:00', '2024-08-13 17:00:00'],
 ['2024-08-14 07:00:00', '2024-08-14 17:00:00'],
 ['2024-08-15 07:00:00', '2024-08-15 17:00:00'],
 ['2024-08-16 07:00:00', '2024-08-16 17:00:00'],
 ['2024-08-17 07:00:00', '2024-08-17 17:00:00'],
 ['2024-08-18 07:00:00', '2024-08-18 17:00:00'],
 ['2024-08-20 07:00:00', '2024-08-20 17:00:00'],
 ['2024-08-21 07:00:00', '2024-08-21 17:00:00'],
 ['2024-08-22 07:00:00', '2024-08-22 17:00:00'],
 ['2024-08-23 07:00:00', '2024-08-23 17:00:00'],
 ['2024-08-24 07:00:00', '2024-08-24 17:00:00'],
 ['2024-08-25 07:00:00', '2024-08-25 17:00:00']
    ]
    #agg_func=['mean', 'std']
 )


In [ ]:
list_x=[
 ['2022-09-22 07:00:00', '2022-09-22 17:00:00'],
 ['2022-09-23 07:00:00', '2022-09-23 17:00:00'],
 ['2022-09-24 07:00:00', '2022-09-24 17:00:00'],
 ['2022-09-25 07:00:00', '2022-09-25 17:00:00'],
 ['2022-09-26 07:00:00', '2022-09-26 17:00:00'],
 ['2022-09-27 07:00:00', '2022-09-27 17:00:00'],
 ['2022-09-28 07:00:00', '2022-09-28 17:00:00'],
 ['2022-09-29 07:00:00', '2022-09-29 17:00:00'],
 ['2022-09-30 07:00:00', '2022-09-30 17:00:00'],
 ['2022-10-01 07:00:00', '2022-10-01 17:00:00'],
 ['2022-10-02 07:00:00', '2022-10-02 17:00:00'],
 ['2022-10-03 07:00:00', '2022-10-03 17:00:00'],
 ['2022-10-04 07:00:00', '2022-10-04 17:00:00'],
 ['2022-10-05 07:00:00', '2022-10-05 17:00:00'],
 ['2022-10-06 07:00:00', '2022-10-06 17:00:00'],
 ['2022-10-07 07:00:00', '2022-10-07 17:00:00'],
 ['2022-10-08 07:00:00', '2022-10-08 17:00:00'],
 ['2022-10-09 07:00:00', '2022-10-09 17:00:00'],
 ['2022-10-10 07:00:00', '2022-10-10 17:00:00'],
 ['2022-10-11 07:00:00', '2022-10-11 17:00:00'],
 ['2022-10-12 07:00:00', '2022-10-12 17:00:00'],
 ['2022-10-13 07:00:00', '2022-10-13 17:00:00'],
 ['2022-10-14 07:00:00', '2022-10-14 17:00:00'],
 ['2022-10-15 07:00:00', '2022-10-15 17:00:00'],
 ['2022-10-16 07:00:00', '2022-10-16 17:00:00'],
 ['2022-10-17 07:00:00', '2022-10-17 17:00:00'],
 ['2022-10-19 07:00:00', '2022-10-19 17:00:00'],
 ['2022-10-20 07:00:00', '2022-10-20 17:00:00'],
 ['2022-10-21 07:00:00', '2022-10-21 17:00:00'],
 ['2022-10-22 07:00:00', '2022-10-22 17:00:00'],
 ['2022-10-23 07:00:00', '2022-10-23 17:00:00'],
 ['2022-10-24 07:00:00', '2022-10-24 17:00:00'],
 ['2022-10-25 07:00:00', '2022-10-25 17:00:00'],
 ['2022-10-26 07:00:00', '2022-10-26 17:00:00'],
 ['2022-10-27 07:00:00', '2022-10-27 17:00:00'],
 ['2022-10-28 07:00:00', '2022-10-28 17:00:00'],
 ['2022-10-29 07:00:00', '2022-10-29 17:00:00'],
 ['2022-10-30 07:00:00', '2022-10-30 17:00:00'],
 ['2022-10-31 07:00:00', '2022-10-31 17:00:00'],
 ['2022-11-01 07:00:00', '2022-11-01 17:00:00'],
 ['2022-11-02 07:00:00', '2022-11-02 17:00:00'],
 ['2022-11-03 07:00:00', '2022-11-03 17:00:00'],
 ['2022-11-04 07:00:00', '2022-11-04 17:00:00'],
 ['2022-11-05 07:00:00', '2022-11-05 17:00:00'],
 ['2022-11-06 07:00:00', '2022-11-06 17:00:00'],
 ['2022-11-07 07:00:00', '2022-11-07 17:00:00'],
 ['2022-11-08 07:00:00', '2022-11-08 17:00:00'],
 ['2022-11-09 07:00:00', '2022-11-09 17:00:00'],
 ['2022-11-10 07:00:00', '2022-11-10 17:00:00'],
 ['2022-11-11 07:00:00', '2022-11-11 17:00:00'],
 ['2022-11-12 07:00:00', '2022-11-12 17:00:00'],
 ['2022-11-13 07:00:00', '2022-11-13 17:00:00'],
 ['2022-11-14 07:00:00', '2022-11-14 17:00:00'],
 ['2022-11-15 07:00:00', '2022-11-15 17:00:00'],
 ['2022-11-16 07:00:00', '2022-11-16 17:00:00'],
 ['2022-11-17 07:00:00', '2022-11-17 17:00:00'],
 ['2022-11-19 07:00:00', '2022-11-19 17:00:00'],
 ['2022-11-20 07:00:00', '2022-11-20 17:00:00'],
 ['2022-11-21 07:00:00', '2022-11-21 17:00:00'],
 ['2022-11-22 07:00:00', '2022-11-22 17:00:00'],
 ['2022-11-23 07:00:00', '2022-11-23 17:00:00'],
 ['2022-11-24 07:00:00', '2022-11-24 17:00:00'],
 ['2022-11-25 07:00:00', '2022-11-25 17:00:00'],
 ['2022-11-26 07:00:00', '2022-11-26 17:00:00'],
 ['2022-11-27 07:00:00', '2022-11-27 17:00:00'],
 ['2022-11-28 07:00:00', '2022-11-28 17:00:00'],
 ['2022-11-29 07:00:00', '2022-11-29 17:00:00'],
 ['2022-11-30 07:00:00', '2022-11-30 17:00:00'],
 ['2022-12-01 07:00:00', '2022-12-01 17:00:00'],
 ['2022-12-02 07:00:00', '2022-12-02 17:00:00'],
 ['2022-12-03 07:00:00', '2022-12-03 17:00:00'],
 ['2022-12-04 07:00:00', '2022-12-04 17:00:00'],
 ['2022-12-05 07:00:00', '2022-12-05 17:00:00'],
 ['2022-12-06 07:00:00', '2022-12-06 17:00:00'],
 ['2022-12-07 07:00:00', '2022-12-07 17:00:00'],
 ['2022-12-08 07:00:00', '2022-12-08 17:00:00'],
 ['2022-12-09 07:00:00', '2022-12-09 17:00:00'],
 ['2022-12-10 07:00:00', '2022-12-10 17:00:00'],
 ['2022-12-11 07:00:00', '2022-12-11 17:00:00'],
 ['2022-12-12 07:00:00', '2022-12-12 17:00:00'],
 ['2022-12-13 07:00:00', '2022-12-13 17:00:00'],
 ['2022-12-14 07:00:00', '2022-12-14 17:00:00'],
 ['2022-12-15 07:00:00', '2022-12-15 17:00:00'],
 ['2022-12-16 07:00:00', '2022-12-16 17:00:00'],
 ['2022-12-17 07:00:00', '2022-12-17 17:00:00'],
 ['2022-12-18 07:00:00', '2022-12-18 17:00:00'],
 ['2022-12-19 07:00:00', '2022-12-19 17:00:00'],
 ['2022-12-20 07:00:00', '2022-12-20 17:00:00'],
 ['2022-12-21 07:00:00', '2022-12-21 17:00:00'],
 ['2022-12-22 07:00:00', '2022-12-22 17:00:00'],
 ['2022-12-23 07:00:00', '2022-12-23 17:00:00'],
 ['2022-12-24 07:00:00', '2022-12-24 17:00:00'],
 ['2022-12-25 07:00:00', '2022-12-25 17:00:00'],
 ['2022-12-26 07:00:00', '2022-12-26 17:00:00'],
 ['2022-12-27 07:00:00', '2022-12-27 17:00:00'],
 ['2022-12-29 07:00:00', '2022-12-29 17:00:00'],
 ['2022-12-30 07:00:00', '2022-12-30 17:00:00'],
 ['2022-12-31 07:00:00', '2022-12-31 17:00:00'],
 ['2023-01-01 07:00:00', '2023-01-01 17:00:00'],
 ['2023-01-02 07:00:00', '2023-01-02 17:00:00'],
 ['2023-01-03 07:00:00', '2023-01-03 17:00:00'],
 ['2023-01-04 07:00:00', '2023-01-04 17:00:00'],
 ['2023-01-05 07:00:00', '2023-01-05 17:00:00'],
 ['2023-01-06 07:00:00', '2023-01-06 17:00:00'],
 ['2023-01-07 07:00:00', '2023-01-07 17:00:00'],
 ['2023-01-08 07:00:00', '2023-01-08 17:00:00'],
 ['2023-01-09 07:00:00', '2023-01-09 17:00:00'],
 ['2023-01-10 07:00:00', '2023-01-10 17:00:00'],
 ['2023-01-11 07:00:00', '2023-01-11 17:00:00'],
 ['2023-01-12 07:00:00', '2023-01-12 17:00:00'],
 ['2023-01-13 07:00:00', '2023-01-13 17:00:00'],
 ['2023-01-14 07:00:00', '2023-01-14 17:00:00'],
 ['2023-01-15 07:00:00', '2023-01-15 17:00:00'],
 ['2023-01-16 07:00:00', '2023-01-16 17:00:00'],
 ['2023-01-17 07:00:00', '2023-01-17 17:00:00'],
 ['2023-01-19 07:00:00', '2023-01-19 17:00:00'],
 ['2023-01-20 07:00:00', '2023-01-20 17:00:00'],
 ['2023-01-21 07:00:00', '2023-01-21 17:00:00'],
 ['2023-01-22 07:00:00', '2023-01-22 17:00:00'],
 ['2023-01-23 07:00:00', '2023-01-23 17:00:00'],
 ['2023-01-24 07:00:00', '2023-01-24 17:00:00'],
 ['2023-01-25 07:00:00', '2023-01-25 17:00:00'],
 ['2023-01-26 07:00:00', '2023-01-26 17:00:00'],
 ['2023-01-27 07:00:00', '2023-01-27 17:00:00'],
 ['2023-01-28 07:00:00', '2023-01-28 17:00:00'],
 ['2023-01-29 07:00:00', '2023-01-29 17:00:00'],
 ['2023-01-30 07:00:00', '2023-01-30 17:00:00'],
 ['2023-01-31 07:00:00', '2023-01-31 17:00:00'],
 ['2023-02-01 07:00:00', '2023-02-01 17:00:00'],
 ['2023-02-02 07:00:00', '2023-02-02 17:00:00'],
 ['2023-02-03 07:00:00', '2023-02-03 17:00:00'],
 ['2023-02-04 07:00:00', '2023-02-04 17:00:00'],
 ['2023-02-05 07:00:00', '2023-02-05 17:00:00'],
 ['2023-02-06 07:00:00', '2023-02-06 17:00:00'],
 ['2023-02-07 07:00:00', '2023-02-07 17:00:00'],
 ['2023-02-08 07:00:00', '2023-02-08 17:00:00'],
 ['2023-02-09 07:00:00', '2023-02-09 17:00:00'],
 ['2023-02-10 07:00:00', '2023-02-10 17:00:00'],
 ['2023-02-11 07:00:00', '2023-02-11 17:00:00'],
 ['2023-02-12 07:00:00', '2023-02-12 17:00:00'],
 ['2023-02-13 07:00:00', '2023-02-13 17:00:00'],
 ['2023-02-14 07:00:00', '2023-02-14 17:00:00'],
 ['2023-02-15 07:00:00', '2023-02-15 17:00:00'],
 ['2023-02-16 07:00:00', '2023-02-16 17:00:00'],
 ['2023-02-17 07:00:00', '2023-02-17 17:00:00'],
 ['2023-02-18 07:00:00', '2023-02-18 17:00:00'],
 ['2023-02-19 07:00:00', '2023-02-19 17:00:00'],
 ['2023-02-20 07:00:00', '2023-02-20 17:00:00'],
 ['2023-02-21 07:00:00', '2023-02-21 17:00:00'],
 ['2023-02-22 07:00:00', '2023-02-22 17:00:00'],
 ['2023-02-23 07:00:00', '2023-02-23 17:00:00'],
 ['2023-02-24 07:00:00', '2023-02-24 17:00:00'],
 ['2023-02-25 07:00:00', '2023-02-25 17:00:00'],
 ['2023-02-26 07:00:00', '2023-02-26 17:00:00'],
 ['2023-02-27 07:00:00', '2023-02-27 17:00:00'],
 ['2023-02-28 07:00:00', '2023-02-28 17:00:00'],
 ['2023-03-01 07:00:00', '2023-03-01 17:00:00'],
 ['2023-03-02 07:00:00', '2023-03-02 17:00:00'],
 ['2023-03-03 07:00:00', '2023-03-03 17:00:00'],
 ['2023-04-19 07:00:00', '2023-04-19 17:00:00'],
 ['2023-04-20 07:00:00', '2023-04-20 17:00:00'],
 ['2023-04-21 07:00:00', '2023-04-21 17:00:00'],
 ['2023-04-22 07:00:00', '2023-04-22 17:00:00'],
 ['2023-04-23 07:00:00', '2023-04-23 17:00:00'],
 ['2023-04-24 07:00:00', '2023-04-24 17:00:00'],
 ['2023-04-25 07:00:00', '2023-04-25 17:00:00'],
 ['2023-04-26 07:00:00', '2023-04-26 17:00:00'],
 ['2023-04-27 07:00:00', '2023-04-27 17:00:00'],
 ['2023-04-28 07:00:00', '2023-04-28 17:00:00'],
 ['2023-04-29 07:00:00', '2023-04-29 17:00:00'],
 ['2023-04-30 07:00:00', '2023-04-30 17:00:00'],
 ['2023-05-01 07:00:00', '2023-05-01 17:00:00'],
 ['2023-05-02 07:00:00', '2023-05-02 17:00:00'],
 ['2023-05-03 07:00:00', '2023-05-03 17:00:00'],
 ['2023-05-04 07:00:00', '2023-05-04 17:00:00'],
 ['2023-05-05 07:00:00', '2023-05-05 17:00:00'],
 ['2023-05-06 07:00:00', '2023-05-06 17:00:00'],
 ['2023-05-07 07:00:00', '2023-05-07 17:00:00'],
 ['2023-05-08 07:00:00', '2023-05-08 17:00:00'],
 ['2023-05-09 07:00:00', '2023-05-09 17:00:00'],
 ['2023-05-10 07:00:00', '2023-05-10 17:00:00'],
 ['2023-05-11 07:00:00', '2023-05-11 17:00:00'],
 ['2023-05-12 07:00:00', '2023-05-12 17:00:00'],
 ['2023-05-13 07:00:00', '2023-05-13 17:00:00'],
 ['2023-05-14 07:00:00', '2023-05-14 17:00:00'],
 ['2023-05-15 07:00:00', '2023-05-15 17:00:00'],
 ['2023-06-30 07:00:00', '2023-06-30 17:00:00'],
 ['2023-07-01 07:00:00', '2023-07-01 17:00:00'],
 ['2023-07-02 07:00:00', '2023-07-02 17:00:00'],
 ['2023-07-03 07:00:00', '2023-07-03 17:00:00'],
 ['2023-07-04 07:00:00', '2023-07-04 17:00:00'],
 ['2023-07-05 07:00:00', '2023-07-05 17:00:00'],
 ['2023-07-06 07:00:00', '2023-07-06 17:00:00'],
 ['2023-07-07 07:00:00', '2023-07-07 17:00:00'],
 ['2023-07-08 07:00:00', '2023-07-08 17:00:00'],
 ['2023-07-09 07:00:00', '2023-07-09 17:00:00'],
 ['2023-07-10 07:00:00', '2023-07-10 17:00:00'],
 ['2023-07-11 07:00:00', '2023-07-11 17:00:00'],
 ['2023-07-12 07:00:00', '2023-07-12 17:00:00'],
 ['2023-07-13 07:00:00', '2023-07-13 17:00:00'],
 ['2023-07-14 07:00:00', '2023-07-14 17:00:00'],
 ['2023-07-15 07:00:00', '2023-07-15 17:00:00'],
 ['2023-07-16 07:00:00', '2023-07-16 17:00:00'],
 ['2023-07-18 07:00:00', '2023-07-18 17:00:00'],
 ['2023-07-19 07:00:00', '2023-07-19 17:00:00'],
 ['2023-07-20 07:00:00', '2023-07-20 17:00:00'],
 ['2023-07-21 07:00:00', '2023-07-21 17:00:00'],
 ['2023-07-22 07:00:00', '2023-07-22 17:00:00'],
 ['2023-07-23 07:00:00', '2023-07-23 17:00:00'],
 ['2023-07-24 07:00:00', '2023-07-24 17:00:00'],
 ['2023-07-25 07:00:00', '2023-07-25 17:00:00'],
 ['2023-07-26 07:00:00', '2023-07-26 17:00:00'],
 ['2023-07-27 07:00:00', '2023-07-27 17:00:00'],
 ['2023-08-23 07:00:00', '2023-08-23 17:00:00'],
 ['2023-08-24 07:00:00', '2023-08-24 17:00:00'],
 ['2023-08-25 07:00:00', '2023-08-25 17:00:00'],
 ['2023-08-26 07:00:00', '2023-08-26 17:00:00'],
 ['2023-08-27 07:00:00', '2023-08-27 17:00:00'],
 ['2023-08-28 07:00:00', '2023-08-28 17:00:00'],
 ['2023-08-29 07:00:00', '2023-08-29 17:00:00'],
 ['2023-08-30 07:00:00', '2023-08-30 17:00:00'],
 ['2023-09-02 07:00:00', '2023-09-02 17:00:00'],
 ['2023-09-03 07:00:00', '2023-09-03 17:00:00'],
 ['2023-09-04 07:00:00', '2023-09-04 17:00:00'],
 ['2023-09-05 07:00:00', '2023-09-05 17:00:00'],
 ['2023-09-06 07:00:00', '2023-09-06 17:00:00'],
 ['2023-09-07 07:00:00', '2023-09-07 17:00:00'],
 ['2023-09-08 07:00:00', '2023-09-08 17:00:00'],
 ['2023-09-09 07:00:00', '2023-09-09 17:00:00'],
 ['2023-09-10 07:00:00', '2023-09-10 17:00:00'],
 ['2023-09-11 07:00:00', '2023-09-11 17:00:00'],
 ['2023-09-12 07:00:00', '2023-09-12 17:00:00'],
 ['2023-09-13 07:00:00', '2023-09-13 17:00:00'],
 ['2023-09-14 07:00:00', '2023-09-14 17:00:00'],
 ['2023-09-15 07:00:00', '2023-09-15 17:00:00'],
 ['2023-09-16 07:00:00', '2023-09-16 17:00:00'],
 ['2023-09-17 07:00:00', '2023-09-17 17:00:00'],
 ['2023-09-19 07:00:00', '2023-09-19 17:00:00'],
 ['2023-09-20 07:00:00', '2023-09-20 17:00:00'],
 ['2023-09-21 07:00:00', '2023-09-21 17:00:00'],
 ['2023-09-22 07:00:00', '2023-09-22 17:00:00'],
 ['2023-09-23 07:00:00', '2023-09-23 17:00:00'],
 ['2023-09-24 07:00:00', '2023-09-24 17:00:00'],
 ['2023-09-25 07:00:00', '2023-09-25 17:00:00'],
 ['2023-09-26 07:00:00', '2023-09-26 17:00:00'],
 ['2023-09-27 07:00:00', '2023-09-27 17:00:00'],
 ['2023-09-28 07:00:00', '2023-09-28 17:00:00'],
 ['2023-09-29 07:00:00', '2023-09-29 17:00:00'],
 ['2023-09-30 07:00:00', '2023-09-30 17:00:00'],
 ['2023-10-01 07:00:00', '2023-10-01 17:00:00'],
 ['2023-10-02 07:00:00', '2023-10-02 17:00:00'],
 ['2023-10-03 07:00:00', '2023-10-03 17:00:00'],
 ['2023-10-04 07:00:00', '2023-10-04 17:00:00'],
 ['2023-10-05 07:00:00', '2023-10-05 17:00:00'],
 ['2023-10-06 07:00:00', '2023-10-06 17:00:00'],
 ['2023-10-07 07:00:00', '2023-10-07 17:00:00'],
 ['2023-10-08 07:00:00', '2023-10-08 17:00:00'],
 ['2023-10-09 07:00:00', '2023-10-09 17:00:00'],
 ['2023-10-10 07:00:00', '2023-10-10 17:00:00'],
 ['2023-10-11 07:00:00', '2023-10-11 17:00:00'],
 ['2023-10-12 07:00:00', '2023-10-12 17:00:00'],
 ['2023-10-13 07:00:00', '2023-10-13 17:00:00'],
 ['2023-10-14 07:00:00', '2023-10-14 17:00:00'],
 ['2023-10-15 07:00:00', '2023-10-15 17:00:00'],
 ['2023-10-16 07:00:00', '2023-10-16 17:00:00'],
 ['2023-10-18 07:00:00', '2023-10-18 17:00:00'],
 ['2023-10-19 07:00:00', '2023-10-19 17:00:00'],
 ['2023-10-20 07:00:00', '2023-10-20 17:00:00'],
 ['2023-10-21 07:00:00', '2023-10-21 17:00:00'],
 ['2023-10-22 07:00:00', '2023-10-22 17:00:00'],
 ['2023-10-23 07:00:00', '2023-10-23 17:00:00'],
 ['2023-10-24 07:00:00', '2023-10-24 17:00:00'],
 ['2023-10-25 07:00:00', '2023-10-25 17:00:00'],
 ['2023-10-26 07:00:00', '2023-10-26 17:00:00'],
 ['2023-10-27 07:00:00', '2023-10-27 17:00:00'],
 ['2023-10-28 07:00:00', '2023-10-28 17:00:00'],
 ['2023-10-29 07:00:00', '2023-10-29 17:00:00'],
 ['2023-10-31 07:00:00', '2023-10-31 17:00:00'],
 ['2023-11-01 07:00:00', '2023-11-01 17:00:00'],
 ['2023-11-02 07:00:00', '2023-11-02 17:00:00'],
 ['2023-11-03 07:00:00', '2023-11-03 17:00:00'],
 ['2023-11-04 07:00:00', '2023-11-04 17:00:00'],
 ['2023-11-05 07:00:00', '2023-11-05 17:00:00'],
 ['2023-11-06 07:00:00', '2023-11-06 17:00:00'],
 ['2023-11-07 07:00:00', '2023-11-07 17:00:00'],
 ['2023-11-08 07:00:00', '2023-11-08 17:00:00'],
 ['2023-11-09 07:00:00', '2023-11-09 17:00:00'],
 ['2023-11-10 07:00:00', '2023-11-10 17:00:00'],
 ['2023-11-11 07:00:00', '2023-11-11 17:00:00'],
 ['2023-11-12 07:00:00', '2023-11-12 17:00:00'],
 ['2023-11-13 07:00:00', '2023-11-13 17:00:00'],
 ['2023-11-14 07:00:00', '2023-11-14 17:00:00'],
 ['2023-11-15 07:00:00', '2023-11-15 17:00:00'],
 ['2023-11-16 07:00:00', '2023-11-16 17:00:00'],
 ['2023-11-17 07:00:00', '2023-11-17 17:00:00'],
 ['2023-11-18 07:00:00', '2023-11-18 17:00:00'],
 ['2023-11-19 07:00:00', '2023-11-19 17:00:00'],
 ['2023-11-20 07:00:00', '2023-11-20 17:00:00'],
 ['2023-11-22 07:00:00', '2023-11-22 17:00:00'],
 ['2023-11-23 07:00:00', '2023-11-23 17:00:00'],
 ['2023-11-24 07:00:00', '2023-11-24 17:00:00'],
 ['2023-11-25 07:00:00', '2023-11-25 17:00:00'],
 ['2023-11-26 07:00:00', '2023-11-26 17:00:00'],
 ['2023-11-27 07:00:00', '2023-11-27 17:00:00'],
 ['2023-11-28 07:00:00', '2023-11-28 17:00:00'],
 ['2023-11-29 07:00:00', '2023-11-29 17:00:00'],
 ['2023-11-30 07:00:00', '2023-11-30 17:00:00'],
 ['2023-12-01 07:00:00', '2023-12-01 17:00:00'],
 ['2023-12-02 07:00:00', '2023-12-02 17:00:00'],
 ['2023-12-03 07:00:00', '2023-12-03 17:00:00'],
 ['2023-12-04 07:00:00', '2023-12-04 17:00:00'],
 ['2023-12-05 07:00:00', '2023-12-05 17:00:00'],
 ['2023-12-06 07:00:00', '2023-12-06 17:00:00'],
 ['2023-12-07 07:00:00', '2023-12-07 17:00:00'],
 ['2023-12-08 07:00:00', '2023-12-08 17:00:00'],
 ['2023-12-09 07:00:00', '2023-12-09 17:00:00'],
 ['2023-12-10 07:00:00', '2023-12-10 17:00:00'],
 ['2023-12-11 07:00:00', '2023-12-11 17:00:00'],
 ['2023-12-12 07:00:00', '2023-12-12 17:00:00'],
 ['2023-12-13 07:00:00', '2023-12-13 17:00:00'],
 ['2023-12-14 07:00:00', '2023-12-14 17:00:00'],
 ['2023-12-16 07:00:00', '2023-12-16 17:00:00'],
 ['2023-12-17 07:00:00', '2023-12-17 17:00:00'],
 ['2023-12-19 07:00:00', '2023-12-19 17:00:00'],
 ['2023-12-20 07:00:00', '2023-12-20 17:00:00'],
 ['2023-12-21 07:00:00', '2023-12-21 17:00:00'],
 ['2023-12-22 07:00:00', '2023-12-22 17:00:00'],
 ['2023-12-23 07:00:00', '2023-12-23 17:00:00'],
 ['2023-12-24 07:00:00', '2023-12-24 17:00:00'],
 ['2023-12-25 07:00:00', '2023-12-25 17:00:00'],
 ['2023-12-26 07:00:00', '2023-12-26 17:00:00'],
 ['2023-12-27 07:00:00', '2023-12-27 17:00:00'],
 ['2023-12-28 07:00:00', '2023-12-28 17:00:00'],
 ['2023-12-29 07:00:00', '2023-12-29 17:00:00'],
 ['2023-12-30 07:00:00', '2023-12-30 17:00:00'],
 ['2023-12-31 07:00:00', '2023-12-31 17:00:00'],
 ['2024-01-01 07:00:00', '2024-01-01 17:00:00'],
 ['2024-01-02 07:00:00', '2024-01-02 17:00:00'],
 ['2024-01-03 07:00:00', '2024-01-03 17:00:00'],
 ['2024-01-04 07:00:00', '2024-01-04 17:00:00'],
 ['2024-01-05 07:00:00', '2024-01-05 17:00:00'],
 ['2024-01-06 07:00:00', '2024-01-06 17:00:00'],
 ['2024-01-07 07:00:00', '2024-01-07 17:00:00'],
 ['2024-01-08 07:00:00', '2024-01-08 17:00:00'],
 ['2024-01-09 07:00:00', '2024-01-09 17:00:00'],
 ['2024-01-10 07:00:00', '2024-01-10 17:00:00'],
 ['2024-01-11 07:00:00', '2024-01-11 17:00:00'],
 ['2024-01-12 07:00:00', '2024-01-12 17:00:00'],
 ['2024-01-13 07:00:00', '2024-01-13 17:00:00'],
 ['2024-01-14 07:00:00', '2024-01-14 17:00:00'],
 ['2024-01-16 07:00:00', '2024-01-16 17:00:00'],
 ['2024-01-17 07:00:00', '2024-01-17 17:00:00'],
 ['2024-01-18 07:00:00', '2024-01-18 17:00:00'],
 ['2024-01-19 07:00:00', '2024-01-19 17:00:00'],
 ['2024-01-20 07:00:00', '2024-01-20 17:00:00'],
 ['2024-01-21 07:00:00', '2024-01-21 17:00:00'],
 ['2024-01-22 07:00:00', '2024-01-22 17:00:00'],
 ['2024-01-23 07:00:00', '2024-01-23 17:00:00'],
 ['2024-01-24 07:00:00', '2024-01-24 17:00:00'],
 ['2024-01-25 07:00:00', '2024-01-25 17:00:00'],
 ['2024-01-26 07:00:00', '2024-01-26 17:00:00'],
 ['2024-01-27 07:00:00', '2024-01-27 17:00:00'],
 ['2024-01-28 07:00:00', '2024-01-28 17:00:00'],
 ['2024-01-29 07:00:00', '2024-01-29 17:00:00'],
 ['2024-01-30 07:00:00', '2024-01-30 17:00:00'],
 ['2024-01-31 07:00:00', '2024-01-31 17:00:00'],
 ['2024-02-01 07:00:00', '2024-02-01 17:00:00'],
 ['2024-02-02 07:00:00', '2024-02-02 17:00:00'],
 ['2024-02-03 07:00:00', '2024-02-03 17:00:00'],
 ['2024-02-04 07:00:00', '2024-02-04 17:00:00'],
 ['2024-02-05 07:00:00', '2024-02-05 17:00:00'],
 ['2024-02-06 07:00:00', '2024-02-06 17:00:00'],
 ['2024-02-07 07:00:00', '2024-02-07 17:00:00'],
 ['2024-02-08 07:00:00', '2024-02-08 17:00:00'],
 ['2024-02-09 07:00:00', '2024-02-09 17:00:00'],
 ['2024-02-10 07:00:00', '2024-02-10 17:00:00'],
 ['2024-02-11 07:00:00', '2024-02-11 17:00:00'],
 ['2024-02-12 07:00:00', '2024-02-12 17:00:00'],
 ['2024-02-13 07:00:00', '2024-02-13 17:00:00'],
 ['2024-02-14 07:00:00', '2024-02-14 17:00:00'],
 ['2024-02-15 07:00:00', '2024-02-15 17:00:00'],
 ['2024-02-16 07:00:00', '2024-02-16 17:00:00'],
 ['2024-02-17 07:00:00', '2024-02-17 17:00:00'],
 ['2024-02-18 07:00:00', '2024-02-18 17:00:00'],
 ['2024-02-19 07:00:00', '2024-02-19 17:00:00'],
 ['2024-02-20 07:00:00', '2024-02-20 17:00:00'],
 ['2024-02-21 07:00:00', '2024-02-21 17:00:00'],
 ['2024-02-22 07:00:00', '2024-02-22 17:00:00'],
 ['2024-02-23 07:00:00', '2024-02-23 17:00:00'],
 ['2024-02-24 07:00:00', '2024-02-24 17:00:00'],
 ['2024-02-25 07:00:00', '2024-02-25 17:00:00'],
 ['2024-02-27 07:00:00', '2024-02-27 17:00:00'],
 ['2024-02-28 07:00:00', '2024-02-28 17:00:00'],
 ['2024-02-29 07:00:00', '2024-02-29 17:00:00'],
 ['2024-03-01 07:00:00', '2024-03-01 17:00:00'],
 ['2024-03-02 07:00:00', '2024-03-02 17:00:00'],
 ['2024-03-03 07:00:00', '2024-03-03 17:00:00'],
 ['2024-03-04 07:00:00', '2024-03-04 17:00:00'],
 ['2024-03-05 07:00:00', '2024-03-05 17:00:00'],
 ['2024-03-06 07:00:00', '2024-03-06 17:00:00'],
 ['2024-03-07 07:00:00', '2024-03-07 17:00:00'],
 ['2024-03-08 07:00:00', '2024-03-08 17:00:00'],
 ['2024-03-09 07:00:00', '2024-03-09 17:00:00'],
 ['2024-03-10 07:00:00', '2024-03-10 17:00:00'],
 ['2024-03-11 07:00:00', '2024-03-11 17:00:00'],
 ['2024-03-12 07:00:00', '2024-03-12 17:00:00'],
 ['2024-03-13 07:00:00', '2024-03-13 17:00:00'],
 ['2024-03-14 07:00:00', '2024-03-14 17:00:00'],
 ['2024-03-15 07:00:00', '2024-03-15 17:00:00'],
 ['2024-03-16 07:00:00', '2024-03-16 17:00:00'],
 ['2024-03-17 07:00:00', '2024-03-17 17:00:00'],
 ['2024-03-18 07:00:00', '2024-03-18 17:00:00'],
 ['2024-03-20 07:00:00', '2024-03-20 17:00:00'],
 ['2024-03-21 07:00:00', '2024-03-21 17:00:00'],
 ['2024-03-22 07:00:00', '2024-03-22 17:00:00'],
 ['2024-03-23 07:00:00', '2024-03-23 17:00:00'],
 ['2024-03-24 07:00:00', '2024-03-24 17:00:00'],
 ['2024-03-25 07:00:00', '2024-03-25 17:00:00'],
 ['2024-03-27 07:00:00', '2024-03-27 17:00:00'],
 ['2024-03-28 07:00:00', '2024-03-28 17:00:00'],
 ['2024-03-29 07:00:00', '2024-03-29 17:00:00'],
 ['2024-03-30 07:00:00', '2024-03-30 17:00:00'],
 ['2024-04-01 07:00:00', '2024-04-01 17:00:00'],
 ['2024-04-02 07:00:00', '2024-04-02 17:00:00'],
 ['2024-04-03 07:00:00', '2024-04-03 17:00:00'],
 ['2024-04-04 07:00:00', '2024-04-04 17:00:00'],
 ['2024-04-05 07:00:00', '2024-04-05 17:00:00'],
 ['2024-04-06 07:00:00', '2024-04-06 17:00:00'],
 ['2024-04-07 07:00:00', '2024-04-07 17:00:00'],
 ['2024-04-08 07:00:00', '2024-04-08 17:00:00'],
 ['2024-04-09 07:00:00', '2024-04-09 17:00:00'],
 ['2024-04-10 07:00:00', '2024-04-10 17:00:00'],
 ['2024-04-11 07:00:00', '2024-04-11 17:00:00'],
 ['2024-04-12 07:00:00', '2024-04-12 17:00:00'],
 ['2024-04-13 07:00:00', '2024-04-13 17:00:00'],
 ['2024-04-14 07:00:00', '2024-04-14 17:00:00'],
 ['2024-04-15 07:00:00', '2024-04-15 17:00:00'],
 ['2024-04-16 07:00:00', '2024-04-16 17:00:00'],
 ['2024-04-17 07:00:00', '2024-04-17 17:00:00'],
 ['2024-04-19 07:00:00', '2024-04-19 17:00:00'],
 ['2024-04-20 07:00:00', '2024-04-20 17:00:00'],
 ['2024-04-21 07:00:00', '2024-04-21 17:00:00'],
 ['2024-04-22 07:00:00', '2024-04-22 17:00:00'],
 ['2024-04-23 07:00:00', '2024-04-23 17:00:00'],
 ['2024-04-25 07:00:00', '2024-04-25 17:00:00'],
 ['2024-04-26 07:00:00', '2024-04-26 17:00:00'],
 ['2024-04-27 07:00:00', '2024-04-27 17:00:00'],
 ['2024-04-28 07:00:00', '2024-04-28 17:00:00'],
 ['2024-04-29 07:00:00', '2024-04-29 17:00:00'],
 ['2024-04-30 07:00:00', '2024-04-30 17:00:00'],
 ['2024-05-01 07:00:00', '2024-05-01 17:00:00'],
 ['2024-05-02 07:00:00', '2024-05-02 17:00:00'],
 ['2024-05-03 07:00:00', '2024-05-03 17:00:00'],
 ['2024-05-04 07:00:00', '2024-05-04 17:00:00'],
 ['2024-05-05 07:00:00', '2024-05-05 17:00:00'],
 ['2024-05-06 07:00:00', '2024-05-06 17:00:00'],
 ['2024-05-07 07:00:00', '2024-05-07 17:00:00'],
 ['2024-05-08 07:00:00', '2024-05-08 17:00:00'],
 ['2024-05-09 07:00:00', '2024-05-09 17:00:00'],
 ['2024-05-10 07:00:00', '2024-05-10 17:00:00'],
 ['2024-05-11 07:00:00', '2024-05-11 17:00:00'],
 ['2024-05-12 07:00:00', '2024-05-12 17:00:00'],
 ['2024-05-13 07:00:00', '2024-05-13 17:00:00'],
 ['2024-05-16 07:00:00', '2024-05-16 17:00:00'],
 ['2024-05-17 07:00:00', '2024-05-17 17:00:00'],
 ['2024-05-18 07:00:00', '2024-05-18 17:00:00'],
 ['2024-05-19 07:00:00', '2024-05-19 17:00:00'],
 ['2024-05-20 07:00:00', '2024-05-20 17:00:00'],
 ['2024-05-21 07:00:00', '2024-05-21 17:00:00'],
 ['2024-05-22 07:00:00', '2024-05-22 17:00:00'],
 ['2024-05-24 07:00:00', '2024-05-24 17:00:00'],
 ['2024-05-25 07:00:00', '2024-05-25 17:00:00'],
 ['2024-05-26 07:00:00', '2024-05-26 17:00:00'],
 ['2024-05-27 07:00:00', '2024-05-27 17:00:00'],
 ['2024-05-28 07:00:00', '2024-05-28 17:00:00'],
 ['2024-05-29 07:00:00', '2024-05-29 17:00:00'],
 ['2024-05-31 07:00:00', '2024-05-31 17:00:00'],
 ['2024-06-01 07:00:00', '2024-06-01 17:00:00'],
 ['2024-06-02 07:00:00', '2024-06-02 17:00:00'],
 ['2024-06-03 07:00:00', '2024-06-03 17:00:00'],
 ['2024-06-04 07:00:00', '2024-06-04 17:00:00'],
 ['2024-06-05 07:00:00', '2024-06-05 17:00:00'],
 ['2024-06-06 07:00:00', '2024-06-06 17:00:00'],
 ['2024-06-07 07:00:00', '2024-06-07 17:00:00'],
 ['2024-06-08 07:00:00', '2024-06-08 17:00:00'],
 ['2024-06-09 07:00:00', '2024-06-09 17:00:00'],
 ['2024-06-10 07:00:00', '2024-06-10 17:00:00'],
 ['2024-06-11 07:00:00', '2024-06-11 17:00:00'],
 ['2024-06-12 07:00:00', '2024-06-12 17:00:00'],
 ['2024-06-13 07:00:00', '2024-06-13 17:00:00'],
 ['2024-06-14 07:00:00', '2024-06-14 17:00:00'],
 ['2024-06-15 07:00:00', '2024-06-15 17:00:00'],
 ['2024-06-16 07:00:00', '2024-06-16 17:00:00'],
 ['2024-06-17 07:00:00', '2024-06-17 17:00:00'],
 ['2024-06-19 07:00:00', '2024-06-19 17:00:00'],
 ['2024-06-20 07:00:00', '2024-06-20 17:00:00'],
 ['2024-06-21 07:00:00', '2024-06-21 17:00:00'],
 ['2024-06-22 07:00:00', '2024-06-22 17:00:00'],
 ['2024-06-23 07:00:00', '2024-06-23 17:00:00'],
 ['2024-06-24 07:00:00', '2024-06-24 17:00:00'],
 ['2024-06-25 07:00:00', '2024-06-25 17:00:00'],
 ['2024-06-29 07:00:00', '2024-06-29 17:00:00'],
 ['2024-06-30 07:00:00', '2024-06-30 17:00:00'],
 ['2024-07-01 07:00:00', '2024-07-01 17:00:00'],
 ['2024-07-02 07:00:00', '2024-07-02 17:00:00'],
 ['2024-07-03 07:00:00', '2024-07-03 17:00:00'],
 ['2024-07-04 07:00:00', '2024-07-04 17:00:00'],
 ['2024-07-05 07:00:00', '2024-07-05 17:00:00'],
 ['2024-07-06 07:00:00', '2024-07-06 17:00:00'],
 ['2024-07-07 07:00:00', '2024-07-07 17:00:00'],
 ['2024-07-08 07:00:00', '2024-07-08 17:00:00'],
 ['2024-07-09 07:00:00', '2024-07-09 17:00:00'],
 ['2024-07-10 07:00:00', '2024-07-10 17:00:00'],
 ['2024-07-11 07:00:00', '2024-07-11 17:00:00'],
 ['2024-07-12 07:00:00', '2024-07-12 17:00:00'],
 ['2024-07-13 07:00:00', '2024-07-13 17:00:00'],
 ['2024-07-14 07:00:00', '2024-07-14 17:00:00'],
 ['2024-07-15 07:00:00', '2024-07-15 17:00:00'],
 ['2024-07-16 07:00:00', '2024-07-16 17:00:00'],
 ['2024-07-17 07:00:00', '2024-07-17 17:00:00'],
 ['2024-07-19 07:00:00', '2024-07-19 17:00:00'],
 ['2024-07-20 07:00:00', '2024-07-20 17:00:00'],
 ['2024-07-21 07:00:00', '2024-07-21 17:00:00'],
 ['2024-07-22 07:00:00', '2024-07-22 17:00:00'],
 ['2024-07-23 07:00:00', '2024-07-23 17:00:00'],
 ['2024-07-24 07:00:00', '2024-07-24 17:00:00'],
 ['2024-07-25 07:00:00', '2024-07-25 17:00:00'],
 ['2024-07-26 07:00:00', '2024-07-26 17:00:00'],
 ['2024-07-27 07:00:00', '2024-07-27 17:00:00'],
 ['2024-07-28 07:00:00', '2024-07-28 17:00:00'],
 ['2024-07-29 07:00:00', '2024-07-29 17:00:00'],
 ['2024-07-30 07:00:00', '2024-07-30 17:00:00'],
 ['2024-08-05 07:00:00', '2024-08-05 17:00:00'],
 ['2024-08-06 07:00:00', '2024-08-06 17:00:00'],
 ['2024-08-07 07:00:00', '2024-08-07 17:00:00'],
 ['2024-08-08 07:00:00', '2024-08-08 17:00:00'],
 ['2024-08-09 07:00:00', '2024-08-09 17:00:00'],
 ['2024-08-10 07:00:00', '2024-08-10 17:00:00'],
 ['2024-08-11 07:00:00', '2024-08-11 17:00:00'],
 ['2024-08-12 07:00:00', '2024-08-12 17:00:00'],
 ['2024-08-13 07:00:00', '2024-08-13 17:00:00'],
 ['2024-08-14 07:00:00', '2024-08-14 17:00:00'],
 ['2024-08-15 07:00:00', '2024-08-15 17:00:00'],
 ['2024-08-16 07:00:00', '2024-08-16 17:00:00'],
 ['2024-08-17 07:00:00', '2024-08-17 17:00:00'],
 ['2024-08-18 07:00:00', '2024-08-18 17:00:00'],
 ['2024-08-20 07:00:00', '2024-08-20 17:00:00'],
 ['2024-08-21 07:00:00', '2024-08-21 17:00:00'],
 ['2024-08-22 07:00:00', '2024-08-22 17:00:00'],
 ['2024-08-23 07:00:00', '2024-08-23 17:00:00'],
 ['2024-08-24 07:00:00', '2024-08-24 17:00:00'],
 ['2024-08-25 07:00:00', '2024-08-25 17:00:00']]

In [ ]:
# convert "list" into a dataframe
df = pd.DataFrame(list_x, columns = ['start', 'end'])
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])

In [ ]:
# convert datetime to string
df['start'] = df['start'].dt.strftime('%Y-%m-%d')
df['end'] = df['end'].dt.strftime('%Y-%m-%d')

In [ ]:
df.head()

In [ ]:
df['range'] = df['start'] + '/' + df['end']

In [ ]:
#drop the start and end columns
df = df.drop(['start', 'end'], axis=1)

In [ ]:
# trasform the range column into a list
list_x = df['range'].tolist()

In [ ]:
x['date'] = list_x

In [ ]:
x.head()

In [ ]:
x.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_7_17_sept22_ago24.csv')

In [ ]:
from datetime import datetime, timedelta

original_intervals = [
        ['2024-09-23 07:00:00','2024-09-23 19:00:00'],
        ['2024-09-23 19:00:00','2024-09-24 07:00:00'],
        ['2024-09-24 07:00:00','2024-09-24 19:00:00'],
        ['2024-09-24 19:00:00','2024-09-25 07:00:00'],
        ['2024-09-25 07:00:00','2024-09-25 19:00:00'],
        ['2024-09-25 19:00:00','2024-09-26 07:00:00'],
        ['2024-09-26 07:00:00','2024-09-26 19:00:00'],
        ['2024-09-26 19:00:00','2024-09-27 07:00:00'],
        ['2024-09-27 07:00:00','2024-09-27 19:00:00'],
        ['2024-09-27 19:00:00','2024-09-28 07:00:00'],
        ['2024-09-28 07:00:00','2024-09-28 19:00:00'],
        ['2024-09-28 19:00:00','2024-09-29 07:00:00'],
        ['2024-09-29 07:00:00','2024-09-29 19:00:00'],
        ['2024-09-29 19:00:00','2024-09-30 07:00:00'],
        ['2024-09-30 07:00:00','2024-09-30 19:00:00'],
        ['2024-09-30 19:00:00','2024-10-01 07:00:00'],
        ['2024-10-01 07:00:00','2024-10-01 19:00:00'],
        ['2024-10-01 19:00:00','2024-10-02 07:00:00'],
        ['2024-10-02 07:00:00','2024-10-02 19:00:00'],
        ['2024-10-02 19:00:00','2024-10-03 07:00:00'],
        ['2024-10-03 07:00:00','2024-10-03 19:00:00'],
        ['2024-10-03 19:00:00','2024-10-04 07:00:00'],
        ['2024-10-04 07:00:00','2024-10-04 19:00:00'],
        ['2024-10-04 19:00:00','2024-10-05 07:00:00'],
        ['2024-10-05 07:00:00','2024-10-05 19:00:00'],
        ['2024-10-05 19:00:00','2024-10-06 07:00:00'],
        ['2024-10-06 07:00:00','2024-10-06 19:00:00'],
        ['2024-10-06 19:00:00','2024-10-07 07:00:00'],
        ['2024-10-07 07:00:00','2024-10-07 19:00:00'],
        ['2024-10-07 19:00:00','2024-10-08 07:00:00'],
        ['2024-10-08 07:00:00','2024-10-08 19:00:00'],
        ['2024-10-08 19:00:00','2024-10-09 07:00:00'],
        ['2024-10-09 07:00:00','2024-10-09 19:00:00'],
        ['2024-10-09 19:00:00','2024-10-10 07:00:00'],
        ['2024-10-10 07:00:00','2024-10-10 19:00:00'],
        ['2024-10-10 19:00:00','2024-10-11 07:00:00'],
        ['2024-10-11 07:00:00','2024-10-11 19:00:00'],
        ['2024-10-11 19:00:00','2024-10-12 07:00:00'],
        ['2024-10-12 07:00:00','2024-10-12 19:00:00'],
        ['2024-10-12 19:00:00','2024-10-13 07:00:00'],
        ['2024-10-13 07:00:00','2024-10-13 19:00:00'],
        ['2024-10-13 19:00:00','2024-10-14 07:00:00'],
        ['2024-10-14 07:00:00','2024-10-14 19:00:00'],
        ['2024-10-14 19:00:00','2024-10-15 07:00:00'],
        ['2024-10-15 07:00:00','2024-10-15 19:00:00'],
        ['2024-10-15 19:00:00','2024-10-16 07:00:00'],
        ['2024-10-16 07:00:00','2024-10-16 19:00:00'],
        ['2024-10-16 19:00:00','2024-10-17 07:00:00'],
        ['2024-10-17 07:00:00','2024-10-17 19:00:00'],
        ['2024-10-17 19:00:00','2024-10-18 07:00:00'],
        ['2024-10-18 07:00:00','2024-10-18 19:00:00'],
        ['2024-10-18 19:00:00','2024-10-19 07:00:00'],
        ['2024-10-19 07:00:00','2024-10-19 19:00:00'],
        ['2024-10-19 19:00:00','2024-10-20 07:00:00'],
        ['2024-10-20 07:00:00','2024-10-20 19:00:00'],
        ['2024-10-22 07:00:00','2024-10-22 19:00:00'],
        ['2024-10-22 19:00:00','2024-10-23 07:00:00'],
        ['2024-10-23 07:00:00','2024-10-23 19:00:00'],
        ['2024-10-23 19:00:00','2024-10-24 07:00:00'],
        ['2024-10-24 07:00:00','2024-10-24 19:00:00'],
        ['2024-10-24 19:00:00','2024-10-25 07:00:00'],
        ['2024-10-25 07:00:00','2024-10-25 19:00:00'],
        ['2024-10-25 19:00:00','2024-10-26 07:00:00'],
        ['2024-10-26 07:00:00','2024-10-26 19:00:00'],
        ['2024-10-26 19:00:00','2024-10-27 07:00:00'],
        ['2024-10-27 07:00:00','2024-10-27 19:00:00'],
        ['2024-10-27 19:00:00','2024-10-28 07:00:00'],
        ['2024-10-28 07:00:00','2024-10-28 19:00:00'],
        ['2024-10-28 19:00:00','2024-10-29 07:00:00'],
        ['2024-10-29 07:00:00','2024-10-29 19:00:00'],
        ['2024-10-29 19:00:00','2024-10-30 07:00:00'],
        ['2024-10-30 07:00:00','2024-10-30 19:00:00'],
        ['2024-10-30 19:00:00','2024-10-31 07:00:00'],
        ['2024-10-31 07:00:00','2024-10-31 19:00:00'],
        ['2024-10-31 19:00:00','2024-11-01 07:00:00'],
        ['2024-11-01 07:00:00','2024-11-01 19:00:00'],
        ['2024-11-01 19:00:00','2024-11-02 07:00:00'],
        ['2024-11-02 07:00:00','2024-11-02 19:00:00'],
        ['2024-11-02 19:00:00','2024-11-03 07:00:00'],
        ['2024-11-03 07:00:00','2024-11-03 19:00:00'],
        ['2024-11-03 19:00:00','2024-11-04 07:00:00'],
        ['2024-11-04 07:00:00','2024-11-04 19:00:00'],
        ['2024-11-04 19:00:00','2024-11-05 07:00:00'],
        ['2024-11-05 07:00:00','2024-11-05 19:00:00'],
        ['2024-11-05 19:00:00','2024-11-06 07:00:00'],
        ['2024-11-06 07:00:00','2024-11-06 19:00:00'],
        ['2024-11-06 19:00:00','2024-11-07 07:00:00'],
        ['2024-11-07 07:00:00','2024-11-07 19:00:00'],
        ['2024-11-07 19:00:00','2024-11-08 07:00:00'],
        ['2024-11-08 07:00:00','2024-11-08 19:00:00'],
        ['2024-11-08 19:00:00','2024-11-09 07:00:00'],
        ['2024-11-09 07:00:00','2024-11-09 19:00:00'],
        ['2024-11-09 19:00:00','2024-11-10 07:00:00'],
        ['2024-11-10 07:00:00','2024-11-10 19:00:00'],
        ['2024-11-10 19:00:00','2024-11-11 07:00:00'],
        ['2024-11-11 07:00:00','2024-11-11 19:00:00'],
        ['2024-11-11 19:00:00','2024-11-12 07:00:00'],
        ['2024-11-12 07:00:00','2024-11-12 19:00:00'],
        ['2024-11-12 19:00:00','2024-11-13 07:00:00'],
        ['2024-11-13 07:00:00','2024-11-13 19:00:00'],
        ['2024-11-13 19:00:00','2024-11-14 07:00:00'],
        ['2024-11-14 07:00:00','2024-11-14 19:00:00'],
        ['2024-11-14 19:00:00','2024-11-15 07:00:00'],
        ['2024-11-15 07:00:00','2024-11-15 19:00:00'],
        ['2024-11-15 19:00:00','2024-11-16 07:00:00'],
        ['2024-11-16 07:00:00','2024-11-16 19:00:00'],
        ['2024-11-16 19:00:00','2024-11-17 07:00:00'],
        ['2024-11-17 07:00:00','2024-11-17 19:00:00'],
        ['2024-11-19 07:00:00','2024-11-19 19:00:00'],
        ['2024-11-19 19:00:00','2024-11-20 07:00:00'],
        ['2024-11-20 07:00:00','2024-11-20 19:00:00'],
        ['2024-11-20 19:00:00','2024-11-21 07:00:00'],
        ['2024-11-21 07:00:00','2024-11-21 19:00:00'],
        ['2024-11-21 19:00:00','2024-11-22 07:00:00'],
        ['2024-11-22 07:00:00','2024-11-22 19:00:00'],
        ['2024-11-22 19:00:00','2024-11-23 07:00:00'],
        ['2024-11-23 07:00:00','2024-11-23 19:00:00'],
        ['2024-11-23 19:00:00','2024-11-24 07:00:00'],
        ['2024-11-24 07:00:00','2024-11-24 19:00:00'],
        ['2024-11-24 19:00:00','2024-11-25 07:00:00'],
        ['2024-11-25 07:00:00','2024-11-25 19:00:00'],
        ['2024-11-25 19:00:00','2024-11-26 07:00:00'],
        ['2024-11-26 07:00:00','2024-11-26 19:00:00'],
        ['2024-11-26 19:00:00','2024-11-27 07:00:00'],
        ['2024-11-27 07:00:00','2024-11-27 19:00:00'],
        ['2024-11-27 19:00:00','2024-11-28 07:00:00'],
        ['2024-11-28 07:00:00','2024-11-28 19:00:00'],
        ['2024-11-28 19:00:00','2024-11-29 07:00:00'],
        ['2024-11-29 07:00:00','2024-11-29 19:00:00'],
        ['2024-11-29 19:00:00','2024-11-30 07:00:00'],
        ['2024-11-30 07:00:00','2024-11-30 19:00:00'],
        ['2024-11-30 19:00:00','2024-12-01 07:00:00'],
        ['2024-12-01 07:00:00','2024-12-01 19:00:00'],
        ['2024-12-01 19:00:00','2024-12-02 07:00:00'],
        ['2024-12-02 07:00:00','2024-12-02 19:00:00'],
        ['2024-12-04 07:00:00','2024-12-04 19:00:00'],
        ['2024-12-04 19:00:00','2024-12-05 07:00:00'],
        ['2024-12-05 07:00:00','2024-12-05 19:00:00'],
        ['2024-12-05 19:00:00','2024-12-06 07:00:00'],
        ['2024-12-06 07:00:00','2024-12-06 19:00:00'],
        ['2024-12-06 19:00:00','2024-12-07 07:00:00'],
        ['2024-12-07 07:00:00','2024-12-07 19:00:00'],
        ['2024-12-07 19:00:00','2024-12-08 07:00:00'],
        ['2024-12-08 07:00:00','2024-12-08 19:00:00'],
        ['2024-12-08 19:00:00','2024-12-09 07:00:00'],
        ['2024-12-09 07:00:00','2024-12-09 19:00:00'],
        ['2024-12-09 19:00:00','2024-12-10 07:00:00'],
        ['2024-12-10 07:00:00','2024-12-10 19:00:00'],
        ['2024-12-10 19:00:00','2024-12-11 07:00:00'],
        ['2024-12-11 07:00:00','2024-12-11 19:00:00'],
        ['2024-12-11 19:00:00','2024-12-12 07:00:00'],
        ['2024-12-12 07:00:00','2024-12-12 19:00:00'],
        ['2024-12-12 19:00:00','2024-12-13 07:00:00'],
        ['2024-12-13 07:00:00','2024-12-13 19:00:00'],
        ['2024-12-13 19:00:00','2024-12-14 07:00:00'],
        ['2024-12-14 07:00:00','2024-12-14 19:00:00'],
        ['2024-12-14 19:00:00','2024-12-15 07:00:00'],
        ['2024-12-15 07:00:00','2024-12-15 19:00:00'],
        ['2024-12-15 19:00:00','2024-12-16 07:00:00'],
        ['2024-12-16 07:00:00','2024-12-16 19:00:00'],
        ['2024-12-16 19:00:00','2024-12-17 07:00:00'],
        ['2024-12-17 07:00:00','2024-12-17 19:00:00'],
        ['2024-12-17 19:00:00','2024-12-18 07:00:00'],
        ['2024-12-18 07:00:00','2024-12-18 19:00:00'],
        ['2024-12-18 19:00:00','2024-12-19 07:00:00'],
        ['2024-12-19 07:00:00','2024-12-19 19:00:00'],
        ['2024-12-19 19:00:00','2024-12-20 07:00:00'],
        ['2024-12-20 07:00:00','2024-12-20 19:00:00'],
        ['2024-12-20 19:00:00','2024-12-21 07:00:00'],
        ['2024-12-21 07:00:00','2024-12-21 19:00:00'],
        ['2024-12-21 19:00:00','2024-12-22 07:00:00'],
        ['2024-12-22 07:00:00','2024-12-22 19:00:00'],
        ['2024-12-22 19:00:00','2024-12-23 07:00:00'],
        ['2024-12-23 07:00:00','2024-12-23 19:00:00'],
        ['2024-12-23 19:00:00','2024-12-24 07:00:00'],
        ['2024-12-24 07:00:00','2024-12-24 19:00:00'],
        ['2024-12-24 19:00:00','2024-12-25 07:00:00'],
        ['2024-12-25 07:00:00','2024-12-25 19:00:00'],
        ['2024-12-25 19:00:00','2024-12-26 07:00:00'],
        ['2024-12-26 07:00:00','2024-12-26 19:00:00'],
        ['2024-12-26 19:00:00','2024-12-27 07:00:00'],
        ['2024-12-27 07:00:00','2024-12-27 19:00:00'],
        ['2024-12-27 19:00:00','2024-12-28 07:00:00'],
        ['2024-12-28 07:00:00','2024-12-28 19:00:00'],
        ['2024-12-28 19:00:00','2024-12-29 07:00:00'],
        ['2024-12-29 07:00:00','2024-12-29 19:00:00'],
        ['2024-12-29 19:00:00','2024-12-30 07:00:00'],
        ['2024-12-30 07:00:00','2024-12-30 19:00:00'],
        ['2024-12-30 19:00:00','2024-12-31 07:00:00']
]

def convert_interval(start_str, end_str):
    """
    Ritorna la coppia (start_str, end_str) convertita
    sostituendo i range 07:00-19:00 -> 07:00-17:00
                      19:00-07:00 -> 23:00-05:00
    mantenendo inalterati data e day shift.
    """
    fmt = "%Y-%m-%d %H:%M:%S"
    start_dt = datetime.strptime(start_str, fmt)
    end_dt = datetime.strptime(end_str, fmt)
    
    start_hour = start_dt.hour
    end_hour   = end_dt.hour
    
    # Caso 07:00 -> 19:00 (stessa data)
    if start_hour == 7 and end_hour == 19 and start_dt.date() == end_dt.date():
        new_start = start_dt.replace(hour= 7, minute=0, second=0)
        new_end   = end_dt.replace(hour=17, minute=0, second=0)
        return [new_start.strftime(fmt), new_end.strftime(fmt)]
    
    # Caso 19:00 -> 07:00 (giorno successivo)
    # start: 19:00 su day X, end: 07:00 su day X+1
    # => 23:00 su day X, 05:00 su day X+1
    if start_hour == 19:
        # Verifichiamo che end_dt sia effettivamente il giorno dopo
        next_day = start_dt + timedelta(days=1)
        if end_hour == 7 and end_dt.date() == next_day.date():
            new_start = start_dt.replace(hour=23, minute=0, second=0)
            new_end   = end_dt.replace(hour=5,  minute=0, second=0)
            return [new_start.strftime(fmt), new_end.strftime(fmt)]
    
    # Se non rientra nei pattern, restituiamo l'intervallo invariato
    return [start_str, end_str]


converted = []
for interval in original_intervals:
    start_str, end_str = interval
    new_pair = convert_interval(start_str, end_str)
    converted.append(new_pair)

# Stampa finale del risultato
print("[")
for c in converted:
    print(f"    ['{c[0]}', '{c[1]}'],")
print("]")

In [ ]:
intervals = [
    ['2024-09-23 07:00:00', '2024-09-23 17:00:00'],
    ['2024-09-23 23:00:00', '2024-09-24 05:00:00'],
    ['2024-09-24 07:00:00', '2024-09-24 17:00:00'],
    ['2024-09-24 23:00:00', '2024-09-25 05:00:00'],
    ['2024-09-25 07:00:00', '2024-09-25 17:00:00'],
    ['2024-09-25 23:00:00', '2024-09-26 05:00:00'],
    ['2024-09-26 07:00:00', '2024-09-26 17:00:00'],
    ['2024-09-26 23:00:00', '2024-09-27 05:00:00'],
    ['2024-09-27 07:00:00', '2024-09-27 17:00:00'],
    ['2024-09-27 23:00:00', '2024-09-28 05:00:00'],
    ['2024-09-28 07:00:00', '2024-09-28 17:00:00'],
    ['2024-09-28 23:00:00', '2024-09-29 05:00:00'],
    ['2024-09-29 07:00:00', '2024-09-29 17:00:00'],
    ['2024-09-29 23:00:00', '2024-09-30 05:00:00'],
    ['2024-09-30 07:00:00', '2024-09-30 17:00:00'],
    ['2024-09-30 23:00:00', '2024-10-01 05:00:00'],
    ['2024-10-01 07:00:00', '2024-10-01 17:00:00'],
    ['2024-10-01 23:00:00', '2024-10-02 05:00:00'],
    ['2024-10-02 07:00:00', '2024-10-02 17:00:00'],
    ['2024-10-02 23:00:00', '2024-10-03 05:00:00'],
    ['2024-10-03 07:00:00', '2024-10-03 17:00:00'],
    ['2024-10-03 23:00:00', '2024-10-04 05:00:00'],
    ['2024-10-04 07:00:00', '2024-10-04 17:00:00'],
    ['2024-10-04 23:00:00', '2024-10-05 05:00:00'],
    ['2024-10-05 07:00:00', '2024-10-05 17:00:00'],
    ['2024-10-05 23:00:00', '2024-10-06 05:00:00'],
    ['2024-10-06 07:00:00', '2024-10-06 17:00:00'],
    ['2024-10-06 23:00:00', '2024-10-07 05:00:00'],
    ['2024-10-07 07:00:00', '2024-10-07 17:00:00'],
    ['2024-10-07 23:00:00', '2024-10-08 05:00:00'],
    ['2024-10-08 07:00:00', '2024-10-08 17:00:00'],
    ['2024-10-08 23:00:00', '2024-10-09 05:00:00'],
    ['2024-10-09 07:00:00', '2024-10-09 17:00:00'],
    ['2024-10-09 23:00:00', '2024-10-10 05:00:00'],
    ['2024-10-10 07:00:00', '2024-10-10 17:00:00'],
    ['2024-10-10 23:00:00', '2024-10-11 05:00:00'],
    ['2024-10-11 07:00:00', '2024-10-11 17:00:00'],
    ['2024-10-11 23:00:00', '2024-10-12 05:00:00'],
    ['2024-10-12 07:00:00', '2024-10-12 17:00:00'],
    ['2024-10-12 23:00:00', '2024-10-13 05:00:00'],
    ['2024-10-13 07:00:00', '2024-10-13 17:00:00'],
    ['2024-10-13 23:00:00', '2024-10-14 05:00:00'],
    ['2024-10-14 07:00:00', '2024-10-14 17:00:00'],
    ['2024-10-14 23:00:00', '2024-10-15 05:00:00'],
    ['2024-10-15 07:00:00', '2024-10-15 17:00:00'],
    ['2024-10-15 23:00:00', '2024-10-16 05:00:00'],
    ['2024-10-16 07:00:00', '2024-10-16 17:00:00'],
    ['2024-10-16 23:00:00', '2024-10-17 05:00:00'],
    ['2024-10-17 07:00:00', '2024-10-17 17:00:00'],
    ['2024-10-17 23:00:00', '2024-10-18 05:00:00'],
    ['2024-10-18 07:00:00', '2024-10-18 17:00:00'],
    ['2024-10-18 23:00:00', '2024-10-19 05:00:00'],
    ['2024-10-19 07:00:00', '2024-10-19 17:00:00'],
    ['2024-10-19 23:00:00', '2024-10-20 05:00:00'],
    ['2024-10-20 07:00:00', '2024-10-20 17:00:00'],
    ['2024-10-22 07:00:00', '2024-10-22 17:00:00'],
    ['2024-10-22 23:00:00', '2024-10-23 05:00:00'],
    ['2024-10-23 07:00:00', '2024-10-23 17:00:00'],
    ['2024-10-23 23:00:00', '2024-10-24 05:00:00'],
    ['2024-10-24 07:00:00', '2024-10-24 17:00:00'],
    ['2024-10-24 23:00:00', '2024-10-25 05:00:00'],
    ['2024-10-25 07:00:00', '2024-10-25 17:00:00'],
    ['2024-10-25 23:00:00', '2024-10-26 05:00:00'],
    ['2024-10-26 07:00:00', '2024-10-26 17:00:00'],
    ['2024-10-26 23:00:00', '2024-10-27 05:00:00'],
    ['2024-10-27 07:00:00', '2024-10-27 17:00:00'],
    ['2024-10-27 23:00:00', '2024-10-28 05:00:00'],
    ['2024-10-28 07:00:00', '2024-10-28 17:00:00'],
    ['2024-10-28 23:00:00', '2024-10-29 05:00:00'],
    ['2024-10-29 07:00:00', '2024-10-29 17:00:00'],
    ['2024-10-29 23:00:00', '2024-10-30 05:00:00'],
    ['2024-10-30 07:00:00', '2024-10-30 17:00:00'],
    ['2024-10-30 23:00:00', '2024-10-31 05:00:00'],
    ['2024-10-31 07:00:00', '2024-10-31 17:00:00'],
    ['2024-10-31 23:00:00', '2024-11-01 05:00:00'],
    ['2024-11-01 07:00:00', '2024-11-01 17:00:00'],
    ['2024-11-01 23:00:00', '2024-11-02 05:00:00'],
    ['2024-11-02 07:00:00', '2024-11-02 17:00:00'],
    ['2024-11-02 23:00:00', '2024-11-03 05:00:00'],
    ['2024-11-03 07:00:00', '2024-11-03 17:00:00'],
    ['2024-11-03 23:00:00', '2024-11-04 05:00:00'],
    ['2024-11-04 07:00:00', '2024-11-04 17:00:00'],
    ['2024-11-04 23:00:00', '2024-11-05 05:00:00'],
    ['2024-11-05 07:00:00', '2024-11-05 17:00:00'],
    ['2024-11-05 23:00:00', '2024-11-06 05:00:00'],
    ['2024-11-06 07:00:00', '2024-11-06 17:00:00'],
    ['2024-11-06 23:00:00', '2024-11-07 05:00:00'],
    ['2024-11-07 07:00:00', '2024-11-07 17:00:00'],
    ['2024-11-07 23:00:00', '2024-11-08 05:00:00'],
    ['2024-11-08 07:00:00', '2024-11-08 17:00:00'],
    ['2024-11-08 23:00:00', '2024-11-09 05:00:00'],
    ['2024-11-09 07:00:00', '2024-11-09 17:00:00'],
    ['2024-11-09 23:00:00', '2024-11-10 05:00:00'],
    ['2024-11-10 07:00:00', '2024-11-10 17:00:00'],
    ['2024-11-10 23:00:00', '2024-11-11 05:00:00'],
    ['2024-11-11 07:00:00', '2024-11-11 17:00:00'],
    ['2024-11-11 23:00:00', '2024-11-12 05:00:00'],
    ['2024-11-12 07:00:00', '2024-11-12 17:00:00'],
    ['2024-11-12 23:00:00', '2024-11-13 05:00:00'],
    ['2024-11-13 07:00:00', '2024-11-13 17:00:00'],
    ['2024-11-13 23:00:00', '2024-11-14 05:00:00'],
    ['2024-11-14 07:00:00', '2024-11-14 17:00:00'],
    ['2024-11-14 23:00:00', '2024-11-15 05:00:00'],
    ['2024-11-15 07:00:00', '2024-11-15 17:00:00'],
    ['2024-11-15 23:00:00', '2024-11-16 05:00:00'],
    ['2024-11-16 07:00:00', '2024-11-16 17:00:00'],
    ['2024-11-16 23:00:00', '2024-11-17 05:00:00'],
    ['2024-11-17 07:00:00', '2024-11-17 17:00:00'],
    ['2024-11-19 07:00:00', '2024-11-19 17:00:00'],
    ['2024-11-19 23:00:00', '2024-11-20 05:00:00'],
    ['2024-11-20 07:00:00', '2024-11-20 17:00:00'],
    ['2024-11-20 23:00:00', '2024-11-21 05:00:00'],
    ['2024-11-21 07:00:00', '2024-11-21 17:00:00'],
    ['2024-11-21 23:00:00', '2024-11-22 05:00:00'],
    ['2024-11-22 07:00:00', '2024-11-22 17:00:00'],
    ['2024-11-22 23:00:00', '2024-11-23 05:00:00'],
    ['2024-11-23 07:00:00', '2024-11-23 17:00:00'],
    ['2024-11-23 23:00:00', '2024-11-24 05:00:00'],
    ['2024-11-24 07:00:00', '2024-11-24 17:00:00'],
    ['2024-11-24 23:00:00', '2024-11-25 05:00:00'],
    ['2024-11-25 07:00:00', '2024-11-25 17:00:00'],
    ['2024-11-25 23:00:00', '2024-11-26 05:00:00'],
    ['2024-11-26 07:00:00', '2024-11-26 17:00:00'],
    ['2024-11-26 23:00:00', '2024-11-27 05:00:00'],
    ['2024-11-27 07:00:00', '2024-11-27 17:00:00'],
    ['2024-11-27 23:00:00', '2024-11-28 05:00:00'],
    ['2024-11-28 07:00:00', '2024-11-28 17:00:00'],
    ['2024-11-28 23:00:00', '2024-11-29 05:00:00'],
    ['2024-11-29 07:00:00', '2024-11-29 17:00:00'],
    ['2024-11-29 23:00:00', '2024-11-30 05:00:00'],
    ['2024-11-30 07:00:00', '2024-11-30 17:00:00'],
    ['2024-11-30 23:00:00', '2024-12-01 05:00:00'],
    ['2024-12-01 07:00:00', '2024-12-01 17:00:00'],
    ['2024-12-01 23:00:00', '2024-12-02 05:00:00'],
    ['2024-12-02 07:00:00', '2024-12-02 17:00:00'],
    ['2024-12-04 07:00:00', '2024-12-04 17:00:00'],
    ['2024-12-04 23:00:00', '2024-12-05 05:00:00'],
    ['2024-12-05 07:00:00', '2024-12-05 17:00:00'],
    ['2024-12-05 23:00:00', '2024-12-06 05:00:00'],
    ['2024-12-06 07:00:00', '2024-12-06 17:00:00'],
    ['2024-12-06 23:00:00', '2024-12-07 05:00:00'],
    ['2024-12-07 07:00:00', '2024-12-07 17:00:00'],
    ['2024-12-07 23:00:00', '2024-12-08 05:00:00'],
    ['2024-12-08 07:00:00', '2024-12-08 17:00:00'],
    ['2024-12-08 23:00:00', '2024-12-09 05:00:00'],
    ['2024-12-09 07:00:00', '2024-12-09 17:00:00'],
    ['2024-12-09 23:00:00', '2024-12-10 05:00:00'],
    ['2024-12-10 07:00:00', '2024-12-10 17:00:00'],
    ['2024-12-10 23:00:00', '2024-12-11 05:00:00'],
    ['2024-12-11 07:00:00', '2024-12-11 17:00:00'],
    ['2024-12-11 23:00:00', '2024-12-12 05:00:00'],
    ['2024-12-12 07:00:00', '2024-12-12 17:00:00'],
    ['2024-12-12 23:00:00', '2024-12-13 05:00:00'],
    ['2024-12-13 07:00:00', '2024-12-13 17:00:00'],
    ['2024-12-13 23:00:00', '2024-12-14 05:00:00'],
    ['2024-12-14 07:00:00', '2024-12-14 17:00:00'],
    ['2024-12-14 23:00:00', '2024-12-15 05:00:00'],
    ['2024-12-15 07:00:00', '2024-12-15 17:00:00'],
    ['2024-12-15 23:00:00', '2024-12-16 05:00:00'],
    ['2024-12-16 07:00:00', '2024-12-16 17:00:00'],
    ['2024-12-16 23:00:00', '2024-12-17 05:00:00'],
    ['2024-12-17 07:00:00', '2024-12-17 17:00:00'],
    ['2024-12-17 23:00:00', '2024-12-18 05:00:00'],
    ['2024-12-18 07:00:00', '2024-12-18 17:00:00'],
    ['2024-12-18 23:00:00', '2024-12-19 05:00:00'],
    ['2024-12-19 07:00:00', '2024-12-19 17:00:00'],
    ['2024-12-19 23:00:00', '2024-12-20 05:00:00'],
    ['2024-12-20 07:00:00', '2024-12-20 17:00:00'],
    ['2024-12-20 23:00:00', '2024-12-21 05:00:00'],
    ['2024-12-21 07:00:00', '2024-12-21 17:00:00'],
    ['2024-12-21 23:00:00', '2024-12-22 05:00:00'],
    ['2024-12-22 07:00:00', '2024-12-22 17:00:00'],
    ['2024-12-22 23:00:00', '2024-12-23 05:00:00'],
    ['2024-12-23 07:00:00', '2024-12-23 17:00:00'],
    ['2024-12-23 23:00:00', '2024-12-24 05:00:00'],
    ['2024-12-24 07:00:00', '2024-12-24 17:00:00'],
    ['2024-12-24 23:00:00', '2024-12-25 05:00:00'],
    ['2024-12-25 07:00:00', '2024-12-25 17:00:00'],
    ['2024-12-25 23:00:00', '2024-12-26 05:00:00'],
    ['2024-12-26 07:00:00', '2024-12-26 17:00:00'],
    ['2024-12-26 23:00:00', '2024-12-27 05:00:00'],
    ['2024-12-27 07:00:00', '2024-12-27 17:00:00'],
    ['2024-12-27 23:00:00', '2024-12-28 05:00:00'],
    ['2024-12-28 07:00:00', '2024-12-28 17:00:00'],
    ['2024-12-28 23:00:00', '2024-12-29 05:00:00'],
    ['2024-12-29 07:00:00', '2024-12-29 17:00:00'],
    ['2024-12-29 23:00:00', '2024-12-30 05:00:00'],
    ['2024-12-30 07:00:00', '2024-12-30 17:00:00'],
    ['2024-12-30 23:00:00', '2024-12-31 05:00:00']
]

filtered_intervals = [
    iv for iv in intervals
    # Keep only if not 19->07
    if not (iv[0].endswith('23:00:00') and iv[1].endswith('05:00:00'))
]

filtered_intervals

In [ ]:
y=raw.light.summary_statistics_per_time_bin(
    bins=[
        ['2024-09-23 07:00:00', '2024-09-23 17:00:00'],
 ['2024-09-24 07:00:00', '2024-09-24 17:00:00'],
 ['2024-09-25 07:00:00', '2024-09-25 17:00:00'],
 ['2024-09-26 07:00:00', '2024-09-26 17:00:00'],
 ['2024-09-27 07:00:00', '2024-09-27 17:00:00'],
 ['2024-09-28 07:00:00', '2024-09-28 17:00:00'],
 ['2024-09-29 07:00:00', '2024-09-29 17:00:00'],
 ['2024-09-30 07:00:00', '2024-09-30 17:00:00'],
 ['2024-10-01 07:00:00', '2024-10-01 17:00:00'],
 ['2024-10-02 07:00:00', '2024-10-02 17:00:00'],
 ['2024-10-03 07:00:00', '2024-10-03 17:00:00'],
 ['2024-10-04 07:00:00', '2024-10-04 17:00:00'],
 ['2024-10-05 07:00:00', '2024-10-05 17:00:00'],
 ['2024-10-06 07:00:00', '2024-10-06 17:00:00'],
 ['2024-10-07 07:00:00', '2024-10-07 17:00:00'],
 ['2024-10-08 07:00:00', '2024-10-08 17:00:00'],
 ['2024-10-09 07:00:00', '2024-10-09 17:00:00'],
 ['2024-10-10 07:00:00', '2024-10-10 17:00:00'],
 ['2024-10-11 07:00:00', '2024-10-11 17:00:00'],
 ['2024-10-12 07:00:00', '2024-10-12 17:00:00'],
 ['2024-10-13 07:00:00', '2024-10-13 17:00:00'],
 ['2024-10-14 07:00:00', '2024-10-14 17:00:00'],
 ['2024-10-15 07:00:00', '2024-10-15 17:00:00'],
 ['2024-10-16 07:00:00', '2024-10-16 17:00:00'],
 ['2024-10-17 07:00:00', '2024-10-17 17:00:00'],
 ['2024-10-18 07:00:00', '2024-10-18 17:00:00'],
 ['2024-10-19 07:00:00', '2024-10-19 17:00:00'],
 ['2024-10-20 07:00:00', '2024-10-20 17:00:00'],
 ['2024-10-22 07:00:00', '2024-10-22 17:00:00'],
 ['2024-10-23 07:00:00', '2024-10-23 17:00:00'],
 ['2024-10-24 07:00:00', '2024-10-24 17:00:00'],
 ['2024-10-25 07:00:00', '2024-10-25 17:00:00'],
 ['2024-10-26 07:00:00', '2024-10-26 17:00:00'],
 ['2024-10-27 07:00:00', '2024-10-27 17:00:00'],
 ['2024-10-28 07:00:00', '2024-10-28 17:00:00'],
 ['2024-10-29 07:00:00', '2024-10-29 17:00:00'],
 ['2024-10-30 07:00:00', '2024-10-30 17:00:00'],
 ['2024-10-31 07:00:00', '2024-10-31 17:00:00'],
 ['2024-11-01 07:00:00', '2024-11-01 17:00:00'],
 ['2024-11-02 07:00:00', '2024-11-02 17:00:00'],
 ['2024-11-03 07:00:00', '2024-11-03 17:00:00'],
 ['2024-11-04 07:00:00', '2024-11-04 17:00:00'],
 ['2024-11-05 07:00:00', '2024-11-05 17:00:00'],
 ['2024-11-06 07:00:00', '2024-11-06 17:00:00'],
 ['2024-11-07 07:00:00', '2024-11-07 17:00:00'],
 ['2024-11-08 07:00:00', '2024-11-08 17:00:00'],
 ['2024-11-09 07:00:00', '2024-11-09 17:00:00'],
 ['2024-11-10 07:00:00', '2024-11-10 17:00:00'],
 ['2024-11-11 07:00:00', '2024-11-11 17:00:00'],
 ['2024-11-12 07:00:00', '2024-11-12 17:00:00'],
 ['2024-11-13 07:00:00', '2024-11-13 17:00:00'],
 ['2024-11-14 07:00:00', '2024-11-14 17:00:00'],
 ['2024-11-15 07:00:00', '2024-11-15 17:00:00'],
 ['2024-11-16 07:00:00', '2024-11-16 17:00:00'],
 ['2024-11-17 07:00:00', '2024-11-17 17:00:00'],
 ['2024-11-19 07:00:00', '2024-11-19 17:00:00'],
 ['2024-11-20 07:00:00', '2024-11-20 17:00:00'],
 ['2024-11-21 07:00:00', '2024-11-21 17:00:00'],
 ['2024-11-22 07:00:00', '2024-11-22 17:00:00'],
 ['2024-11-23 07:00:00', '2024-11-23 17:00:00'],
 ['2024-11-24 07:00:00', '2024-11-24 17:00:00'],
 ['2024-11-25 07:00:00', '2024-11-25 17:00:00'],
 ['2024-11-26 07:00:00', '2024-11-26 17:00:00'],
 ['2024-11-27 07:00:00', '2024-11-27 17:00:00'],
 ['2024-11-28 07:00:00', '2024-11-28 17:00:00'],
 ['2024-11-29 07:00:00', '2024-11-29 17:00:00'],
 ['2024-11-30 07:00:00', '2024-11-30 17:00:00'],
 ['2024-12-01 07:00:00', '2024-12-01 17:00:00'],
 ['2024-12-02 07:00:00', '2024-12-02 17:00:00'],
 ['2024-12-04 07:00:00', '2024-12-04 17:00:00'],
 ['2024-12-05 07:00:00', '2024-12-05 17:00:00'],
 ['2024-12-06 07:00:00', '2024-12-06 17:00:00'],
 ['2024-12-07 07:00:00', '2024-12-07 17:00:00'],
 ['2024-12-08 07:00:00', '2024-12-08 17:00:00'],
 ['2024-12-09 07:00:00', '2024-12-09 17:00:00'],
 ['2024-12-10 07:00:00', '2024-12-10 17:00:00'],
 ['2024-12-11 07:00:00', '2024-12-11 17:00:00'],
 ['2024-12-12 07:00:00', '2024-12-12 17:00:00'],
 ['2024-12-13 07:00:00', '2024-12-13 17:00:00'],
 ['2024-12-14 07:00:00', '2024-12-14 17:00:00'],
 ['2024-12-15 07:00:00', '2024-12-15 17:00:00'],
 ['2024-12-16 07:00:00', '2024-12-16 17:00:00'],
 ['2024-12-17 07:00:00', '2024-12-17 17:00:00'],
 ['2024-12-18 07:00:00', '2024-12-18 17:00:00'],
 ['2024-12-19 07:00:00', '2024-12-19 17:00:00'],
 ['2024-12-20 07:00:00', '2024-12-20 17:00:00'],
 ['2024-12-21 07:00:00', '2024-12-21 17:00:00'],
 ['2024-12-22 07:00:00', '2024-12-22 17:00:00'],
 ['2024-12-23 07:00:00', '2024-12-23 17:00:00'],
 ['2024-12-24 07:00:00', '2024-12-24 17:00:00'],
 ['2024-12-25 07:00:00', '2024-12-25 17:00:00'],
 ['2024-12-26 07:00:00', '2024-12-26 17:00:00'],
 ['2024-12-27 07:00:00', '2024-12-27 17:00:00'],
 ['2024-12-28 07:00:00', '2024-12-28 17:00:00'],
 ['2024-12-29 07:00:00', '2024-12-29 17:00:00'],
 ['2024-12-30 07:00:00', '2024-12-30 17:00:00']
    ], 
    #agg_func=['mean', 'min', 'max']
 )

In [ ]:
list_y=[
 ['2024-09-23 07:00:00', '2024-09-23 17:00:00'],
 ['2024-09-24 07:00:00', '2024-09-24 17:00:00'],
 ['2024-09-25 07:00:00', '2024-09-25 17:00:00'],
 ['2024-09-26 07:00:00', '2024-09-26 17:00:00'],
 ['2024-09-27 07:00:00', '2024-09-27 17:00:00'],
 ['2024-09-28 07:00:00', '2024-09-28 17:00:00'],
 ['2024-09-29 07:00:00', '2024-09-29 17:00:00'],
 ['2024-09-30 07:00:00', '2024-09-30 17:00:00'],
 ['2024-10-01 07:00:00', '2024-10-01 17:00:00'],
 ['2024-10-02 07:00:00', '2024-10-02 17:00:00'],
 ['2024-10-03 07:00:00', '2024-10-03 17:00:00'],
 ['2024-10-04 07:00:00', '2024-10-04 17:00:00'],
 ['2024-10-05 07:00:00', '2024-10-05 17:00:00'],
 ['2024-10-06 07:00:00', '2024-10-06 17:00:00'],
 ['2024-10-07 07:00:00', '2024-10-07 17:00:00'],
 ['2024-10-08 07:00:00', '2024-10-08 17:00:00'],
 ['2024-10-09 07:00:00', '2024-10-09 17:00:00'],
 ['2024-10-10 07:00:00', '2024-10-10 17:00:00'],
 ['2024-10-11 07:00:00', '2024-10-11 17:00:00'],
 ['2024-10-12 07:00:00', '2024-10-12 17:00:00'],
 ['2024-10-13 07:00:00', '2024-10-13 17:00:00'],
 ['2024-10-14 07:00:00', '2024-10-14 17:00:00'],
 ['2024-10-15 07:00:00', '2024-10-15 17:00:00'],
 ['2024-10-16 07:00:00', '2024-10-16 17:00:00'],
 ['2024-10-17 07:00:00', '2024-10-17 17:00:00'],
 ['2024-10-18 07:00:00', '2024-10-18 17:00:00'],
 ['2024-10-19 07:00:00', '2024-10-19 17:00:00'],
 ['2024-10-20 07:00:00', '2024-10-20 17:00:00'],
 ['2024-10-22 07:00:00', '2024-10-22 17:00:00'],
 ['2024-10-23 07:00:00', '2024-10-23 17:00:00'],
 ['2024-10-24 07:00:00', '2024-10-24 17:00:00'],
 ['2024-10-25 07:00:00', '2024-10-25 17:00:00'],
 ['2024-10-26 07:00:00', '2024-10-26 17:00:00'],
 ['2024-10-27 07:00:00', '2024-10-27 17:00:00'],
 ['2024-10-28 07:00:00', '2024-10-28 17:00:00'],
 ['2024-10-29 07:00:00', '2024-10-29 17:00:00'],
 ['2024-10-30 07:00:00', '2024-10-30 17:00:00'],
 ['2024-10-31 07:00:00', '2024-10-31 17:00:00'],
 ['2024-11-01 07:00:00', '2024-11-01 17:00:00'],
 ['2024-11-02 07:00:00', '2024-11-02 17:00:00'],
 ['2024-11-03 07:00:00', '2024-11-03 17:00:00'],
 ['2024-11-04 07:00:00', '2024-11-04 17:00:00'],
 ['2024-11-05 07:00:00', '2024-11-05 17:00:00'],
 ['2024-11-06 07:00:00', '2024-11-06 17:00:00'],
 ['2024-11-07 07:00:00', '2024-11-07 17:00:00'],
 ['2024-11-08 07:00:00', '2024-11-08 17:00:00'],
 ['2024-11-09 07:00:00', '2024-11-09 17:00:00'],
 ['2024-11-10 07:00:00', '2024-11-10 17:00:00'],
 ['2024-11-11 07:00:00', '2024-11-11 17:00:00'],
 ['2024-11-12 07:00:00', '2024-11-12 17:00:00'],
 ['2024-11-13 07:00:00', '2024-11-13 17:00:00'],
 ['2024-11-14 07:00:00', '2024-11-14 17:00:00'],
 ['2024-11-15 07:00:00', '2024-11-15 17:00:00'],
 ['2024-11-16 07:00:00', '2024-11-16 17:00:00'],
 ['2024-11-17 07:00:00', '2024-11-17 17:00:00'],
 ['2024-11-19 07:00:00', '2024-11-19 17:00:00'],
 ['2024-11-20 07:00:00', '2024-11-20 17:00:00'],
 ['2024-11-21 07:00:00', '2024-11-21 17:00:00'],
 ['2024-11-22 07:00:00', '2024-11-22 17:00:00'],
 ['2024-11-23 07:00:00', '2024-11-23 17:00:00'],
 ['2024-11-24 07:00:00', '2024-11-24 17:00:00'],
 ['2024-11-25 07:00:00', '2024-11-25 17:00:00'],
 ['2024-11-26 07:00:00', '2024-11-26 17:00:00'],
 ['2024-11-27 07:00:00', '2024-11-27 17:00:00'],
 ['2024-11-28 07:00:00', '2024-11-28 17:00:00'],
 ['2024-11-29 07:00:00', '2024-11-29 17:00:00'],
 ['2024-11-30 07:00:00', '2024-11-30 17:00:00'],
 ['2024-12-01 07:00:00', '2024-12-01 17:00:00'],
 ['2024-12-02 07:00:00', '2024-12-02 17:00:00'],
 ['2024-12-04 07:00:00', '2024-12-04 17:00:00'],
 ['2024-12-05 07:00:00', '2024-12-05 17:00:00'],
 ['2024-12-06 07:00:00', '2024-12-06 17:00:00'],
 ['2024-12-07 07:00:00', '2024-12-07 17:00:00'],
 ['2024-12-08 07:00:00', '2024-12-08 17:00:00'],
 ['2024-12-09 07:00:00', '2024-12-09 17:00:00'],
 ['2024-12-10 07:00:00', '2024-12-10 17:00:00'],
 ['2024-12-11 07:00:00', '2024-12-11 17:00:00'],
 ['2024-12-12 07:00:00', '2024-12-12 17:00:00'],
 ['2024-12-13 07:00:00', '2024-12-13 17:00:00'],
 ['2024-12-14 07:00:00', '2024-12-14 17:00:00'],
 ['2024-12-15 07:00:00', '2024-12-15 17:00:00'],
 ['2024-12-16 07:00:00', '2024-12-16 17:00:00'],
 ['2024-12-17 07:00:00', '2024-12-17 17:00:00'],
 ['2024-12-18 07:00:00', '2024-12-18 17:00:00'],
 ['2024-12-19 07:00:00', '2024-12-19 17:00:00'],
 ['2024-12-20 07:00:00', '2024-12-20 17:00:00'],
 ['2024-12-21 07:00:00', '2024-12-21 17:00:00'],
 ['2024-12-22 07:00:00', '2024-12-22 17:00:00'],
 ['2024-12-23 07:00:00', '2024-12-23 17:00:00'],
 ['2024-12-24 07:00:00', '2024-12-24 17:00:00'],
 ['2024-12-25 07:00:00', '2024-12-25 17:00:00'],
 ['2024-12-26 07:00:00', '2024-12-26 17:00:00'],
 ['2024-12-27 07:00:00', '2024-12-27 17:00:00'],
 ['2024-12-28 07:00:00', '2024-12-28 17:00:00'],
 ['2024-12-29 07:00:00', '2024-12-29 17:00:00'],
 ['2024-12-30 07:00:00', '2024-12-30 17:00:00']]

In [ ]:
# convert "list" into a dataframe
df = pd.DataFrame(list_y, columns = ['start', 'end'])
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])

In [ ]:
# convert datetime to string
df['start'] = df['start'].dt.strftime('%Y-%m-%d')
df['end'] = df['end'].dt.strftime('%Y-%m-%d')

In [ ]:
df.head()

In [ ]:
df['range'] = df['start'] + '/' + df['end']

In [ ]:
#drop the start and end columns
df = df.drop(['start', 'end'], axis=1)

In [ ]:
# trasform the range column into a list
list_y = df['range'].tolist()

In [ ]:
y['date'] = list_y

In [ ]:
y.head()

In [ ]:
y.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_7_17_sept24_dec24.csv')

RANGE 19:00-7:00/7:00-19:00

In [ ]:
a=raw.light.summary_statistics_per_time_bin(
    bins=[
    ['2022-09-21 19:00:00', '2022-09-22 07:00:00'],
    ['2022-09-22 07:00:00', '2022-09-22 19:00:00'],
    ['2022-09-22 19:00:00', '2022-09-23 07:00:00'],
    ['2022-09-23 07:00:00', '2022-09-23 19:00:00'],
    ['2022-09-23 19:00:00', '2022-09-24 07:00:00'],
    ['2022-09-24 07:00:00', '2022-09-24 19:00:00'],
    ['2022-09-24 19:00:00', '2022-09-25 07:00:00'],
    ['2022-09-25 07:00:00', '2022-09-25 19:00:00'],
    ['2022-09-25 19:00:00', '2022-09-26 07:00:00'],
    ['2022-09-26 07:00:00', '2022-09-26 19:00:00'],
    ['2022-09-26 19:00:00', '2022-09-27 07:00:00'],
    ['2022-09-27 07:00:00', '2022-09-27 19:00:00'],
    ['2022-09-27 19:00:00', '2022-09-28 07:00:00'],
    ['2022-09-28 07:00:00', '2022-09-28 19:00:00'],
    ['2022-09-28 19:00:00', '2022-09-29 07:00:00'],
    ['2022-09-29 07:00:00', '2022-09-29 19:00:00'],
    ['2022-09-29 19:00:00', '2022-09-30 07:00:00'],
    ['2022-09-30 07:00:00', '2022-09-30 19:00:00'],
    ['2022-09-30 19:00:00', '2022-10-01 07:00:00'],
    ['2022-10-01 07:00:00', '2022-10-01 19:00:00'],
    ['2022-10-01 19:00:00', '2022-10-02 07:00:00'],
    ['2022-10-02 07:00:00', '2022-10-02 19:00:00'],
    ['2022-10-02 19:00:00', '2022-10-03 07:00:00'],
    ['2022-10-03 07:00:00', '2022-10-03 19:00:00'],
    ['2022-10-03 19:00:00', '2022-10-04 07:00:00'],
    ['2022-10-04 07:00:00', '2022-10-04 19:00:00'],
    ['2022-10-04 19:00:00', '2022-10-05 07:00:00'],
    ['2022-10-05 07:00:00', '2022-10-05 19:00:00'],
    ['2022-10-05 19:00:00', '2022-10-06 07:00:00'],
    ['2022-10-06 07:00:00', '2022-10-06 19:00:00'],
    ['2022-10-06 19:00:00', '2022-10-07 07:00:00'],
    ['2022-10-07 07:00:00', '2022-10-07 19:00:00'],
    ['2022-10-07 19:00:00', '2022-10-08 07:00:00'],
    ['2022-10-08 07:00:00', '2022-10-08 19:00:00'],
    ['2022-10-08 19:00:00', '2022-10-09 07:00:00'],
    ['2022-10-09 07:00:00', '2022-10-09 19:00:00'],
    ['2022-10-09 19:00:00', '2022-10-10 07:00:00'],
    ['2022-10-10 07:00:00', '2022-10-10 19:00:00'],
    ['2022-10-10 19:00:00', '2022-10-11 07:00:00'],
    ['2022-10-11 07:00:00', '2022-10-11 19:00:00'],
    ['2022-10-11 19:00:00', '2022-10-12 07:00:00'],
    ['2022-10-12 07:00:00', '2022-10-12 19:00:00'],
    ['2022-10-12 19:00:00', '2022-10-13 07:00:00'],
    ['2022-10-13 07:00:00', '2022-10-13 19:00:00'],
    ['2022-10-13 19:00:00', '2022-10-14 07:00:00'],
    ['2022-10-14 07:00:00', '2022-10-14 19:00:00'],
    ['2022-10-14 19:00:00', '2022-10-15 07:00:00'],
    ['2022-10-15 07:00:00', '2022-10-15 19:00:00'],
    ['2022-10-15 19:00:00', '2022-10-16 07:00:00'],
    ['2022-10-16 07:00:00', '2022-10-16 19:00:00'],
    ['2022-10-16 19:00:00', '2022-10-17 07:00:00'],
    ['2022-10-17 07:00:00', '2022-10-17 19:00:00'],
    ['2022-10-19 07:00:00', '2022-10-19 19:00:00'],#here
    ['2022-10-19 19:00:00', '2022-10-20 07:00:00'],
    ['2022-10-20 07:00:00', '2022-10-20 19:00:00'],
    ['2022-10-20 19:00:00', '2022-10-21 07:00:00'],
    ['2022-10-21 07:00:00', '2022-10-21 19:00:00'],
    ['2022-10-21 19:00:00', '2022-10-22 07:00:00'],
    ['2022-10-22 07:00:00', '2022-10-22 19:00:00'],
    ['2022-10-22 19:00:00', '2022-10-23 07:00:00'],
    ['2022-10-23 07:00:00', '2022-10-23 19:00:00'],
    ['2022-10-23 19:00:00', '2022-10-24 07:00:00'],
    ['2022-10-24 07:00:00', '2022-10-24 19:00:00'],
    ['2022-10-24 19:00:00', '2022-10-25 07:00:00'],
    ['2022-10-25 07:00:00', '2022-10-25 19:00:00'],
    ['2022-10-25 19:00:00', '2022-10-26 07:00:00'],
    ['2022-10-26 07:00:00', '2022-10-26 19:00:00'],
    ['2022-10-26 19:00:00', '2022-10-27 07:00:00'],
    ['2022-10-27 07:00:00', '2022-10-27 19:00:00'],
    ['2022-10-27 19:00:00', '2022-10-28 07:00:00'],
    ['2022-10-28 07:00:00', '2022-10-28 19:00:00'],
    ['2022-10-28 19:00:00', '2022-10-29 07:00:00'],
    ['2022-10-29 07:00:00', '2022-10-29 19:00:00'],
    ['2022-10-29 19:00:00', '2022-10-30 07:00:00'],
    ['2022-10-30 07:00:00', '2022-10-30 19:00:00'],
    ['2022-10-30 19:00:00', '2022-10-31 07:00:00'],
    ['2022-10-31 07:00:00', '2022-10-31 19:00:00'],
    ['2022-10-31 19:00:00', '2022-11-01 07:00:00'],
    ['2022-11-01 07:00:00', '2022-11-01 19:00:00'],
    ['2022-11-01 19:00:00', '2022-11-02 07:00:00'],
    ['2022-11-02 07:00:00', '2022-11-02 19:00:00'],
    ['2022-11-02 19:00:00', '2022-11-03 07:00:00'],
    ['2022-11-03 07:00:00', '2022-11-03 19:00:00'],
    ['2022-11-03 19:00:00', '2022-11-04 07:00:00'],
    ['2022-11-04 07:00:00', '2022-11-04 19:00:00'],
    ['2022-11-04 19:00:00', '2022-11-05 07:00:00'],
    ['2022-11-05 07:00:00', '2022-11-05 19:00:00'],
    ['2022-11-05 19:00:00', '2022-11-06 07:00:00'],
    ['2022-11-06 07:00:00', '2022-11-06 19:00:00'],
    ['2022-11-06 19:00:00', '2022-11-07 07:00:00'],
    ['2022-11-07 07:00:00', '2022-11-07 19:00:00'],
    ['2022-11-07 19:00:00', '2022-11-08 07:00:00'],
    ['2022-11-08 07:00:00', '2022-11-08 19:00:00'],
    ['2022-11-08 19:00:00', '2022-11-09 07:00:00'],
    ['2022-11-09 07:00:00', '2022-11-09 19:00:00'],
    ['2022-11-09 19:00:00', '2022-11-10 07:00:00'],
    ['2022-11-10 07:00:00', '2022-11-10 19:00:00'],
    ['2022-11-10 19:00:00', '2022-11-11 07:00:00'],
    ['2022-11-11 07:00:00', '2022-11-11 19:00:00'],
    ['2022-11-11 19:00:00', '2022-11-12 07:00:00'],
    ['2022-11-12 07:00:00', '2022-11-12 19:00:00'],
    ['2022-11-12 19:00:00', '2022-11-13 07:00:00'],
    ['2022-11-13 07:00:00', '2022-11-13 19:00:00'],
    ['2022-11-13 19:00:00', '2022-11-14 07:00:00'],
    ['2022-11-14 07:00:00', '2022-11-14 19:00:00'],
    ['2022-11-14 19:00:00', '2022-11-15 07:00:00'],
    ['2022-11-15 07:00:00', '2022-11-15 19:00:00'],
    ['2022-11-15 19:00:00', '2022-11-16 07:00:00'],
    ['2022-11-16 07:00:00', '2022-11-16 19:00:00'],
    ['2022-11-16 19:00:00', '2022-11-17 07:00:00'],
    ['2022-11-17 07:00:00', '2022-11-17 19:00:00'],
    ['2022-11-19 07:00:00', '2022-11-19 19:00:00'],#here
    ['2022-11-19 19:00:00', '2022-11-20 07:00:00'],
    ['2022-11-20 07:00:00', '2022-11-20 19:00:00'],
    ['2022-11-20 19:00:00', '2022-11-21 07:00:00'],
    ['2022-11-21 07:00:00', '2022-11-21 19:00:00'],
    ['2022-11-21 19:00:00', '2022-11-22 07:00:00'],
    ['2022-11-22 07:00:00', '2022-11-22 19:00:00'],
    ['2022-11-22 19:00:00', '2022-11-23 07:00:00'],
    ['2022-11-23 07:00:00', '2022-11-23 19:00:00'],
    ['2022-11-23 19:00:00', '2022-11-24 07:00:00'],
    ['2022-11-24 07:00:00', '2022-11-24 19:00:00'],
    ['2022-11-24 19:00:00', '2022-11-25 07:00:00'],
    ['2022-11-25 07:00:00', '2022-11-25 19:00:00'],
    ['2022-11-25 19:00:00', '2022-11-26 07:00:00'],
    ['2022-11-26 07:00:00', '2022-11-26 19:00:00'],
    ['2022-11-26 19:00:00', '2022-11-27 07:00:00'],
    ['2022-11-27 07:00:00', '2022-11-27 19:00:00'],
    ['2022-11-27 19:00:00', '2022-11-28 07:00:00'],
    ['2022-11-28 07:00:00', '2022-11-28 19:00:00'],
    ['2022-11-28 19:00:00', '2022-11-29 07:00:00'],
    ['2022-11-29 07:00:00', '2022-11-29 19:00:00'],
    ['2022-11-29 19:00:00', '2022-11-30 07:00:00'],
    ['2022-11-30 07:00:00', '2022-11-30 19:00:00'],
    ['2022-11-30 19:00:00', '2022-12-01 07:00:00'],
    ['2022-12-01 07:00:00', '2022-12-01 19:00:00'],
    ['2022-12-01 19:00:00', '2022-12-02 07:00:00'],
    ['2022-12-02 07:00:00', '2022-12-02 19:00:00'],
    ['2022-12-02 19:00:00', '2022-12-03 07:00:00'],
    ['2022-12-03 07:00:00', '2022-12-03 19:00:00'],
    ['2022-12-03 19:00:00', '2022-12-04 07:00:00'],
    ['2022-12-04 07:00:00', '2022-12-04 19:00:00'],
    ['2022-12-04 19:00:00', '2022-12-05 07:00:00'],
    ['2022-12-05 07:00:00', '2022-12-05 19:00:00'],
    ['2022-12-05 19:00:00', '2022-12-06 07:00:00'],
    ['2022-12-06 07:00:00', '2022-12-06 19:00:00'],
    ['2022-12-06 19:00:00', '2022-12-07 07:00:00'],
    ['2022-12-07 07:00:00', '2022-12-07 19:00:00'],
    ['2022-12-07 19:00:00', '2022-12-08 07:00:00'],
    ['2022-12-08 07:00:00', '2022-12-08 19:00:00'],
    ['2022-12-08 19:00:00', '2022-12-09 07:00:00'],
    ['2022-12-09 07:00:00', '2022-12-09 19:00:00'],
    ['2022-12-09 19:00:00', '2022-12-10 07:00:00'],
    ['2022-12-10 07:00:00', '2022-12-10 19:00:00'],
    ['2022-12-10 19:00:00', '2022-12-11 07:00:00'],
    ['2022-12-11 07:00:00', '2022-12-11 19:00:00'],
    ['2022-12-11 19:00:00', '2022-12-12 07:00:00'],
    ['2022-12-12 07:00:00', '2022-12-12 19:00:00'],
    ['2022-12-12 19:00:00', '2022-12-13 07:00:00'],
    ['2022-12-13 07:00:00', '2022-12-13 19:00:00'],
    ['2022-12-13 19:00:00', '2022-12-14 07:00:00'],
    ['2022-12-14 07:00:00', '2022-12-14 19:00:00'],
    ['2022-12-14 19:00:00', '2022-12-15 07:00:00'],
    ['2022-12-15 07:00:00', '2022-12-15 19:00:00'],
    ['2022-12-15 19:00:00', '2022-12-16 07:00:00'],
    ['2022-12-16 07:00:00', '2022-12-16 19:00:00'],
    ['2022-12-16 19:00:00', '2022-12-17 07:00:00'],
    ['2022-12-17 07:00:00', '2022-12-17 19:00:00'],
    ['2022-12-17 19:00:00', '2022-12-18 07:00:00'],
    ['2022-12-18 07:00:00', '2022-12-18 19:00:00'],
    ['2022-12-18 19:00:00', '2022-12-19 07:00:00'],
    ['2022-12-19 07:00:00', '2022-12-19 19:00:00'],
    ['2022-12-19 19:00:00', '2022-12-20 07:00:00'],
    ['2022-12-20 07:00:00', '2022-12-20 19:00:00'],
    ['2022-12-20 19:00:00', '2022-12-21 07:00:00'],
    ['2022-12-21 07:00:00', '2022-12-21 19:00:00'],
    ['2022-12-21 19:00:00', '2022-12-22 07:00:00'],
    ['2022-12-22 07:00:00', '2022-12-22 19:00:00'],
    ['2022-12-22 19:00:00', '2022-12-23 07:00:00'],
    ['2022-12-23 07:00:00', '2022-12-23 19:00:00'],
    ['2022-12-23 19:00:00', '2022-12-24 07:00:00'],
    ['2022-12-24 07:00:00', '2022-12-24 19:00:00'],
    ['2022-12-24 19:00:00', '2022-12-25 07:00:00'],
    ['2022-12-25 07:00:00', '2022-12-25 19:00:00'],
    ['2022-12-25 19:00:00', '2022-12-26 07:00:00'],
    ['2022-12-26 07:00:00', '2022-12-26 19:00:00'],
    ['2022-12-26 19:00:00', '2022-12-27 07:00:00'],
    ['2022-12-27 07:00:00', '2022-12-27 19:00:00'],
    ['2022-12-29 07:00:00', '2022-12-29 19:00:00'],#here
    ['2022-12-29 19:00:00', '2022-12-30 07:00:00'],
    ['2022-12-30 07:00:00', '2022-12-30 19:00:00'],
    ['2022-12-30 19:00:00', '2022-12-31 07:00:00'],
    ['2022-12-31 07:00:00', '2022-12-31 19:00:00'],
    ['2022-12-31 19:00:00', '2023-01-01 07:00:00'],
    ['2023-01-01 07:00:00', '2023-01-01 19:00:00'],
    ['2023-01-01 19:00:00', '2023-01-02 07:00:00'],
    ['2023-01-02 07:00:00', '2023-01-02 19:00:00'],
    ['2023-01-02 19:00:00', '2023-01-03 07:00:00'],
    ['2023-01-03 07:00:00', '2023-01-03 19:00:00'],
    ['2023-01-03 19:00:00', '2023-01-04 07:00:00'],
    ['2023-01-04 07:00:00', '2023-01-04 19:00:00'],
    ['2023-01-04 19:00:00', '2023-01-05 07:00:00'],
    ['2023-01-05 07:00:00', '2023-01-05 19:00:00'],
    ['2023-01-05 19:00:00', '2023-01-06 07:00:00'],
    ['2023-01-06 07:00:00', '2023-01-06 19:00:00'],
    ['2023-01-06 19:00:00', '2023-01-07 07:00:00'],
    ['2023-01-07 07:00:00', '2023-01-07 19:00:00'],
    ['2023-01-07 19:00:00', '2023-01-08 07:00:00'],
    ['2023-01-08 07:00:00', '2023-01-08 19:00:00'],
    ['2023-01-08 19:00:00', '2023-01-09 07:00:00'],
    ['2023-01-09 07:00:00', '2023-01-09 19:00:00'],
    ['2023-01-09 19:00:00', '2023-01-10 07:00:00'],
    ['2023-01-10 07:00:00', '2023-01-10 19:00:00'],
    ['2023-01-10 19:00:00', '2023-01-11 07:00:00'],
    ['2023-01-11 07:00:00', '2023-01-11 19:00:00'],
    ['2023-01-11 19:00:00', '2023-01-12 07:00:00'],
    ['2023-01-12 07:00:00', '2023-01-12 19:00:00'],
    ['2023-01-12 19:00:00', '2023-01-13 07:00:00'],
    ['2023-01-13 07:00:00', '2023-01-13 19:00:00'],
    ['2023-01-13 19:00:00', '2023-01-14 07:00:00'],
    ['2023-01-14 07:00:00', '2023-01-14 19:00:00'],
    ['2023-01-14 19:00:00', '2023-01-15 07:00:00'],
    ['2023-01-15 07:00:00', '2023-01-15 19:00:00'],
    ['2023-01-15 19:00:00', '2023-01-16 07:00:00'],
    ['2023-01-16 07:00:00', '2023-01-16 19:00:00'],
    ['2023-01-16 19:00:00', '2023-01-17 07:00:00'],
    ['2023-01-17 07:00:00', '2023-01-17 19:00:00'],
    ['2023-01-19 07:00:00', '2023-01-19 19:00:00'],#here
    ['2023-01-19 19:00:00', '2023-01-20 07:00:00'],
    ['2023-01-20 07:00:00', '2023-01-20 19:00:00'],
    ['2023-01-20 19:00:00', '2023-01-21 07:00:00'],
    ['2023-01-21 07:00:00', '2023-01-21 19:00:00'],
    ['2023-01-21 19:00:00', '2023-01-22 07:00:00'],
    ['2023-01-22 07:00:00', '2023-01-22 19:00:00'],
    ['2023-01-22 19:00:00', '2023-01-23 07:00:00'],
    ['2023-01-23 07:00:00', '2023-01-23 19:00:00'],
    ['2023-01-23 19:00:00', '2023-01-24 07:00:00'],
    ['2023-01-24 07:00:00', '2023-01-24 19:00:00'],
    ['2023-01-24 19:00:00', '2023-01-25 07:00:00'],
    ['2023-01-25 07:00:00', '2023-01-25 19:00:00'],
    ['2023-01-25 19:00:00', '2023-01-26 07:00:00'],
    ['2023-01-26 07:00:00', '2023-01-26 19:00:00'],
    ['2023-01-26 19:00:00', '2023-01-27 07:00:00'],
    ['2023-01-27 07:00:00', '2023-01-27 19:00:00'],
    ['2023-01-27 19:00:00', '2023-01-28 07:00:00'],
    ['2023-01-28 07:00:00', '2023-01-28 19:00:00'],
    ['2023-01-28 19:00:00', '2023-01-29 07:00:00'],
    ['2023-01-29 07:00:00', '2023-01-29 19:00:00'],
    ['2023-01-29 19:00:00', '2023-01-30 07:00:00'],
    ['2023-01-30 07:00:00', '2023-01-30 19:00:00'],
    ['2023-01-30 19:00:00', '2023-01-31 07:00:00'],
    ['2023-01-31 07:00:00', '2023-01-31 19:00:00'],
    ['2023-01-31 19:00:00', '2023-02-01 07:00:00'],
    ['2023-02-01 07:00:00', '2023-02-01 19:00:00'],
    ['2023-02-01 19:00:00', '2023-02-02 07:00:00'],
    ['2023-02-02 07:00:00', '2023-02-02 19:00:00'],
    ['2023-02-02 19:00:00', '2023-02-03 07:00:00'],
    ['2023-02-03 07:00:00', '2023-02-03 19:00:00'],
    ['2023-02-03 19:00:00', '2023-02-04 07:00:00'],
    ['2023-02-04 07:00:00', '2023-02-04 19:00:00'],
    ['2023-02-04 19:00:00', '2023-02-05 07:00:00'],
    ['2023-02-05 07:00:00', '2023-02-05 19:00:00'],
    ['2023-02-05 19:00:00', '2023-02-06 07:00:00'],
    ['2023-02-06 07:00:00', '2023-02-06 19:00:00'],
    ['2023-02-06 19:00:00', '2023-02-07 07:00:00'],
    ['2023-02-07 07:00:00', '2023-02-07 19:00:00'],
    ['2023-02-07 19:00:00', '2023-02-08 07:00:00'],
    ['2023-02-08 07:00:00', '2023-02-08 19:00:00'],
    ['2023-02-08 19:00:00', '2023-02-09 07:00:00'],
    ['2023-02-09 07:00:00', '2023-02-09 19:00:00'],
    ['2023-02-09 19:00:00', '2023-02-10 07:00:00'],
    ['2023-02-10 07:00:00', '2023-02-10 19:00:00'],
    ['2023-02-10 19:00:00', '2023-02-11 07:00:00'],
    ['2023-02-11 07:00:00', '2023-02-11 19:00:00'],
    ['2023-02-11 19:00:00', '2023-02-12 07:00:00'],
    ['2023-02-12 07:00:00', '2023-02-12 19:00:00'],
    ['2023-02-12 19:00:00', '2023-02-13 07:00:00'],
    ['2023-02-13 07:00:00', '2023-02-13 19:00:00'],
    ['2023-02-13 19:00:00', '2023-02-14 07:00:00'],
    ['2023-02-14 07:00:00', '2023-02-14 19:00:00'],
    ['2023-02-14 19:00:00', '2023-02-15 07:00:00'],
    ['2023-02-15 07:00:00', '2023-02-15 19:00:00'],
    ['2023-02-15 19:00:00', '2023-02-16 07:00:00'],
    ['2023-02-16 07:00:00', '2023-02-16 19:00:00'],
    ['2023-02-16 19:00:00', '2023-02-17 07:00:00'],
    ['2023-02-17 07:00:00', '2023-02-17 19:00:00'],
    ['2023-02-17 19:00:00', '2023-02-18 07:00:00'],
    ['2023-02-18 07:00:00', '2023-02-18 19:00:00'],
    ['2023-02-18 19:00:00', '2023-02-19 07:00:00'],
    ['2023-02-19 07:00:00', '2023-02-19 19:00:00'],
    ['2023-02-19 19:00:00', '2023-02-20 07:00:00'],
    ['2023-02-20 07:00:00', '2023-02-20 19:00:00'],
    ['2023-02-20 19:00:00', '2023-02-21 07:00:00'],
    ['2023-02-21 07:00:00', '2023-02-21 19:00:00'],
    ['2023-02-21 19:00:00', '2023-02-22 07:00:00'],
    ['2023-02-22 07:00:00', '2023-02-22 19:00:00'],
    ['2023-02-22 19:00:00', '2023-02-23 07:00:00'],
    ['2023-02-23 07:00:00', '2023-02-23 19:00:00'],
    ['2023-02-23 19:00:00', '2023-02-24 07:00:00'],
    ['2023-02-24 07:00:00', '2023-02-24 19:00:00'],
    ['2023-02-24 19:00:00', '2023-02-25 07:00:00'],
    ['2023-02-25 07:00:00', '2023-02-25 19:00:00'],
    ['2023-02-25 19:00:00', '2023-02-26 07:00:00'],
    ['2023-02-26 07:00:00', '2023-02-26 19:00:00'],
    ['2023-02-26 19:00:00', '2023-02-27 07:00:00'],
    ['2023-02-27 07:00:00', '2023-02-27 19:00:00'],
    ['2023-02-27 19:00:00', '2023-02-28 07:00:00'],
    ['2023-02-28 07:00:00', '2023-02-28 19:00:00'],
    ['2023-02-28 19:00:00', '2023-03-01 07:00:00'],
    ['2023-03-01 07:00:00', '2023-03-01 19:00:00'],
    ['2023-03-01 19:00:00', '2023-03-02 07:00:00'],
    ['2023-03-02 07:00:00', '2023-03-02 19:00:00'],
    ['2023-03-02 19:00:00', '2023-03-03 07:00:00'],
    ['2023-03-03 07:00:00', '2023-03-03 19:00:00'],
    ['2023-03-03 19:00:00', '2023-03-04 07:00:00'],
    ['2023-03-04 07:00:00', '2023-03-04 19:00:00'],#here!!
    ['2023-04-19 07:00:00', '2023-04-19 19:00:00'],
    ['2023-04-19 19:00:00', '2023-04-20 07:00:00'],
    ['2023-04-20 07:00:00', '2023-04-20 19:00:00'],
    ['2023-04-20 19:00:00', '2023-04-21 07:00:00'],
    ['2023-04-21 07:00:00', '2023-04-21 19:00:00'],
    ['2023-04-21 19:00:00', '2023-04-22 07:00:00'],
    ['2023-04-22 07:00:00', '2023-04-22 19:00:00'],
    ['2023-04-22 19:00:00', '2023-04-23 07:00:00'],
    ['2023-04-23 07:00:00', '2023-04-23 19:00:00'],
    ['2023-04-23 19:00:00', '2023-04-24 07:00:00'],
    ['2023-04-24 07:00:00', '2023-04-24 19:00:00'],
    ['2023-04-24 19:00:00', '2023-04-25 07:00:00'],
    ['2023-04-25 07:00:00', '2023-04-25 19:00:00'],
    ['2023-04-25 19:00:00', '2023-04-26 07:00:00'],
    ['2023-04-26 07:00:00', '2023-04-26 19:00:00'],
    ['2023-04-26 19:00:00', '2023-04-27 07:00:00'],
    ['2023-04-27 07:00:00', '2023-04-27 19:00:00'],
    ['2023-04-27 19:00:00', '2023-04-28 07:00:00'],
    ['2023-04-28 07:00:00', '2023-04-28 19:00:00'],
    ['2023-04-28 19:00:00', '2023-04-29 07:00:00'],
    ['2023-04-29 07:00:00', '2023-04-29 19:00:00'],
    ['2023-04-29 19:00:00', '2023-04-30 07:00:00'],
    ['2023-04-30 07:00:00', '2023-04-30 19:00:00'],
    ['2023-04-30 19:00:00', '2023-05-01 07:00:00'],
    ['2023-05-01 07:00:00', '2023-05-01 19:00:00'],
    ['2023-05-01 19:00:00', '2023-05-02 07:00:00'],
    ['2023-05-02 07:00:00', '2023-05-02 19:00:00'],
    ['2023-05-02 19:00:00', '2023-05-03 07:00:00'],
    ['2023-05-03 07:00:00', '2023-05-03 19:00:00'],
    ['2023-05-03 19:00:00', '2023-05-04 07:00:00'],
    ['2023-05-04 07:00:00', '2023-05-04 19:00:00'],
    ['2023-05-04 19:00:00', '2023-05-05 07:00:00'],
    ['2023-05-05 07:00:00', '2023-05-05 19:00:00'],
    ['2023-05-05 19:00:00', '2023-05-06 07:00:00'],
    ['2023-05-06 07:00:00', '2023-05-06 19:00:00'],
    ['2023-05-06 19:00:00', '2023-05-07 07:00:00'],
    ['2023-05-07 07:00:00', '2023-05-07 19:00:00'],
    ['2023-05-07 19:00:00', '2023-05-08 07:00:00'],
    ['2023-05-08 07:00:00', '2023-05-08 19:00:00'],
    ['2023-05-08 19:00:00', '2023-05-09 07:00:00'],
    ['2023-05-09 07:00:00', '2023-05-09 19:00:00'],
    ['2023-05-09 19:00:00', '2023-05-10 07:00:00'],
    ['2023-05-10 07:00:00', '2023-05-10 19:00:00'],
    ['2023-05-10 19:00:00', '2023-05-11 07:00:00'],
    ['2023-05-11 07:00:00', '2023-05-11 19:00:00'],
    ['2023-05-11 19:00:00', '2023-05-12 07:00:00'],
    ['2023-05-12 07:00:00', '2023-05-12 19:00:00'],
    ['2023-05-12 19:00:00', '2023-05-13 07:00:00'],
    ['2023-05-13 07:00:00', '2023-05-13 19:00:00'],
    ['2023-05-13 19:00:00', '2023-05-14 07:00:00'],
    ['2023-05-14 07:00:00', '2023-05-14 19:00:00'],
    ['2023-05-14 19:00:00', '2023-05-15 07:00:00'],
    ['2023-05-15 07:00:00', '2023-05-15 19:00:00'],
    ['2023-05-15 19:00:00', '2023-05-16 07:00:00'],#here
    ['2023-06-29 19:00:00', '2023-06-30 07:00:00'],
    ['2023-06-30 07:00:00', '2023-06-30 19:00:00'],
    ['2023-06-30 19:00:00', '2023-07-01 07:00:00'],
    ['2023-07-01 07:00:00', '2023-07-01 19:00:00'],
    ['2023-07-01 19:00:00', '2023-07-02 07:00:00'],
    ['2023-07-02 07:00:00', '2023-07-02 19:00:00'],
    ['2023-07-02 19:00:00', '2023-07-03 07:00:00'],
    ['2023-07-03 07:00:00', '2023-07-03 19:00:00'],
    ['2023-07-03 19:00:00', '2023-07-04 07:00:00'],
    ['2023-07-04 07:00:00', '2023-07-04 19:00:00'],
    ['2023-07-04 19:00:00', '2023-07-05 07:00:00'],
    ['2023-07-05 07:00:00', '2023-07-05 19:00:00'],
    ['2023-07-05 19:00:00', '2023-07-06 07:00:00'],
    ['2023-07-06 07:00:00', '2023-07-06 19:00:00'],
    ['2023-07-06 19:00:00', '2023-07-07 07:00:00'],
    ['2023-07-07 07:00:00', '2023-07-07 19:00:00'],
    ['2023-07-07 19:00:00', '2023-07-08 07:00:00'],
    ['2023-07-08 07:00:00', '2023-07-08 19:00:00'],
    ['2023-07-08 19:00:00', '2023-07-09 07:00:00'],
    ['2023-07-09 07:00:00', '2023-07-09 19:00:00'],
    ['2023-07-09 19:00:00', '2023-07-10 07:00:00'],
    ['2023-07-10 07:00:00', '2023-07-10 19:00:00'],
    ['2023-07-10 19:00:00', '2023-07-11 07:00:00'],
    ['2023-07-11 07:00:00', '2023-07-11 19:00:00'],
    ['2023-07-11 19:00:00', '2023-07-12 07:00:00'],
    ['2023-07-12 07:00:00', '2023-07-12 19:00:00'],
    ['2023-07-12 19:00:00', '2023-07-13 07:00:00'],
    ['2023-07-13 07:00:00', '2023-07-13 19:00:00'],
    ['2023-07-13 19:00:00', '2023-07-14 07:00:00'],
    ['2023-07-14 07:00:00', '2023-07-14 19:00:00'],
    ['2023-07-14 19:00:00', '2023-07-15 07:00:00'],
    ['2023-07-15 07:00:00', '2023-07-15 19:00:00'],
    ['2023-07-15 19:00:00', '2023-07-16 07:00:00'],
    ['2023-07-16 07:00:00', '2023-07-16 19:00:00'],#here
    ['2023-07-18 07:00:00', '2023-07-18 19:00:00'],
    ['2023-07-18 19:00:00', '2023-07-19 07:00:00'],
    ['2023-07-19 07:00:00', '2023-07-19 19:00:00'],
    ['2023-07-19 19:00:00', '2023-07-20 07:00:00'],
    ['2023-07-20 07:00:00', '2023-07-20 19:00:00'],
    ['2023-07-20 19:00:00', '2023-07-21 07:00:00'],
    ['2023-07-21 07:00:00', '2023-07-21 19:00:00'],
    ['2023-07-21 19:00:00', '2023-07-22 07:00:00'],
    ['2023-07-22 07:00:00', '2023-07-22 19:00:00'],
    ['2023-07-22 19:00:00', '2023-07-23 07:00:00'],
    ['2023-07-23 07:00:00', '2023-07-23 19:00:00'],
    ['2023-07-23 19:00:00', '2023-07-24 07:00:00'],
    ['2023-07-24 07:00:00', '2023-07-24 19:00:00'],
    ['2023-07-24 19:00:00', '2023-07-25 07:00:00'],
    ['2023-07-25 07:00:00', '2023-07-25 19:00:00'],
    ['2023-07-25 19:00:00', '2023-07-26 07:00:00'],
    ['2023-07-26 07:00:00', '2023-07-26 19:00:00'],
    ['2023-07-26 19:00:00', '2023-07-27 07:00:00'],
    ['2023-07-27 07:00:00', '2023-07-27 19:00:00'],
    ['2023-07-27 19:00:00', '2023-07-28 07:00:00'],#here
    ['2023-08-22 19:00:00', '2023-08-23 07:00:00'],
    ['2023-08-23 07:00:00', '2023-08-23 19:00:00'],
    ['2023-08-23 19:00:00', '2023-08-24 07:00:00'],
    ['2023-08-24 07:00:00', '2023-08-24 19:00:00'],
    ['2023-08-24 19:00:00', '2023-08-25 07:00:00'],
    ['2023-08-25 07:00:00', '2023-08-25 19:00:00'],
    ['2023-08-25 19:00:00', '2023-08-26 07:00:00'],
    ['2023-08-26 07:00:00', '2023-08-26 19:00:00'],
    ['2023-08-26 19:00:00', '2023-08-27 07:00:00'],
    ['2023-08-27 07:00:00', '2023-08-27 19:00:00'],
    ['2023-08-27 19:00:00', '2023-08-28 07:00:00'],
    ['2023-08-28 07:00:00', '2023-08-28 19:00:00'],
    ['2023-08-28 19:00:00', '2023-08-29 07:00:00'],
    ['2023-08-29 07:00:00', '2023-08-29 19:00:00'],
    ['2023-08-29 19:00:00', '2023-08-30 07:00:00'],
    ['2023-08-30 07:00:00', '2023-08-30 19:00:00'],
    ['2023-08-30 19:00:00', '2023-08-31 07:00:00'],
    ['2023-08-31 07:00:00', '2023-08-31 19:00:00'],#here
    ['2023-09-02 07:00:00', '2023-09-02 19:00:00'],
    ['2023-09-02 19:00:00', '2023-09-03 07:00:00'],
    ['2023-09-03 07:00:00', '2023-09-03 19:00:00'],
    ['2023-09-03 19:00:00', '2023-09-04 07:00:00'],
    ['2023-09-04 07:00:00', '2023-09-04 19:00:00'],
    ['2023-09-04 19:00:00', '2023-09-05 07:00:00'],
    ['2023-09-05 07:00:00', '2023-09-05 19:00:00'],
    ['2023-09-05 19:00:00', '2023-09-06 07:00:00'],
    ['2023-09-06 07:00:00', '2023-09-06 19:00:00'],
    ['2023-09-06 19:00:00', '2023-09-07 07:00:00'],
    ['2023-09-07 07:00:00', '2023-09-07 19:00:00'],
    ['2023-09-07 19:00:00', '2023-09-08 07:00:00'],
    ['2023-09-08 07:00:00', '2023-09-08 19:00:00'],
    ['2023-09-08 19:00:00', '2023-09-09 07:00:00'],
    ['2023-09-09 07:00:00', '2023-09-09 19:00:00'],
    ['2023-09-09 19:00:00', '2023-09-10 07:00:00'],
    ['2023-09-10 07:00:00', '2023-09-10 19:00:00'],
    ['2023-09-10 19:00:00', '2023-09-11 07:00:00'],
    ['2023-09-11 07:00:00', '2023-09-11 19:00:00'],
    ['2023-09-11 19:00:00', '2023-09-12 07:00:00'],
    ['2023-09-12 07:00:00', '2023-09-12 19:00:00'],
    ['2023-09-12 19:00:00', '2023-09-13 07:00:00'],
    ['2023-09-13 07:00:00', '2023-09-13 19:00:00'],
    ['2023-09-13 19:00:00', '2023-09-14 07:00:00'],
    ['2023-09-14 07:00:00', '2023-09-14 19:00:00'],
    ['2023-09-14 19:00:00', '2023-09-15 07:00:00'],
    ['2023-09-15 07:00:00', '2023-09-15 19:00:00'],
    ['2023-09-15 19:00:00', '2023-09-16 07:00:00'],
    ['2023-09-16 07:00:00', '2023-09-16 19:00:00'],
    ['2023-09-16 19:00:00', '2023-09-17 07:00:00'],
    ['2023-09-17 07:00:00', '2023-09-17 19:00:00'],#here
    ['2023-09-19 07:00:00', '2023-09-19 19:00:00'],
    ['2023-09-19 19:00:00', '2023-09-20 07:00:00'],
    ['2023-09-20 07:00:00', '2023-09-20 19:00:00'],
    ['2023-09-20 19:00:00', '2023-09-21 07:00:00'],
    ['2023-09-21 07:00:00', '2023-09-21 19:00:00'],
    ['2023-09-21 19:00:00', '2023-09-22 07:00:00'],
    ['2023-09-22 07:00:00', '2023-09-22 19:00:00'],
    ['2023-09-22 19:00:00', '2023-09-23 07:00:00'],
    ['2023-09-23 07:00:00', '2023-09-23 19:00:00'],
    ['2023-09-23 19:00:00', '2023-09-24 07:00:00'],
    ['2023-09-24 07:00:00', '2023-09-24 19:00:00'],
    ['2023-09-24 19:00:00', '2023-09-25 07:00:00'],
    ['2023-09-25 07:00:00', '2023-09-25 19:00:00'],
    ['2023-09-25 19:00:00', '2023-09-26 07:00:00'],
    ['2023-09-26 07:00:00', '2023-09-26 19:00:00'],
    ['2023-09-26 19:00:00', '2023-09-27 07:00:00'],
    ['2023-09-27 07:00:00', '2023-09-27 19:00:00'],
    ['2023-09-27 19:00:00', '2023-09-28 07:00:00'],
    ['2023-09-28 07:00:00', '2023-09-28 19:00:00'],
    ['2023-09-28 19:00:00', '2023-09-29 07:00:00'],
    ['2023-09-29 07:00:00', '2023-09-29 19:00:00'],
    ['2023-09-29 19:00:00', '2023-09-30 07:00:00'],
    ['2023-09-30 07:00:00', '2023-09-30 19:00:00'],
    ['2023-09-30 19:00:00', '2023-10-01 07:00:00'],
    ['2023-10-01 07:00:00', '2023-10-01 19:00:00'],
    ['2023-10-01 19:00:00', '2023-10-02 07:00:00'],
    ['2023-10-02 07:00:00', '2023-10-02 19:00:00'],
    ['2023-10-02 19:00:00', '2023-10-03 07:00:00'],
    ['2023-10-03 07:00:00', '2023-10-03 19:00:00'],
    ['2023-10-03 19:00:00', '2023-10-04 07:00:00'],
    ['2023-10-04 07:00:00', '2023-10-04 19:00:00'],
    ['2023-10-04 19:00:00', '2023-10-05 07:00:00'],
    ['2023-10-05 07:00:00', '2023-10-05 19:00:00'],
    ['2023-10-05 19:00:00', '2023-10-06 07:00:00'],
    ['2023-10-06 07:00:00', '2023-10-06 19:00:00'],
    ['2023-10-06 19:00:00', '2023-10-07 07:00:00'],
    ['2023-10-07 07:00:00', '2023-10-07 19:00:00'],
    ['2023-10-07 19:00:00', '2023-10-08 07:00:00'],
    ['2023-10-08 07:00:00', '2023-10-08 19:00:00'],
    ['2023-10-08 19:00:00', '2023-10-09 07:00:00'],
    ['2023-10-09 07:00:00', '2023-10-09 19:00:00'],
    ['2023-10-09 19:00:00', '2023-10-10 07:00:00'],
    ['2023-10-10 07:00:00', '2023-10-10 19:00:00'],
    ['2023-10-10 19:00:00', '2023-10-11 07:00:00'],
    ['2023-10-11 07:00:00', '2023-10-11 19:00:00'],
    ['2023-10-11 19:00:00', '2023-10-12 07:00:00'],
    ['2023-10-12 07:00:00', '2023-10-12 19:00:00'],
    ['2023-10-12 19:00:00', '2023-10-13 07:00:00'],
    ['2023-10-13 07:00:00', '2023-10-13 19:00:00'],
    ['2023-10-13 19:00:00', '2023-10-14 07:00:00'],
    ['2023-10-14 07:00:00', '2023-10-14 19:00:00'],
    ['2023-10-14 19:00:00', '2023-10-15 07:00:00'],
    ['2023-10-15 07:00:00', '2023-10-15 19:00:00'],
    ['2023-10-15 19:00:00', '2023-10-16 07:00:00'],
    ['2023-10-16 07:00:00', '2023-10-16 19:00:00'],#here
    ['2023-10-18 07:00:00', '2023-10-18 19:00:00'],
    ['2023-10-18 19:00:00', '2023-10-19 07:00:00'],
    ['2023-10-19 07:00:00', '2023-10-19 19:00:00'],
    ['2023-10-19 19:00:00', '2023-10-20 07:00:00'],
    ['2023-10-20 07:00:00', '2023-10-20 19:00:00'],
    ['2023-10-20 19:00:00', '2023-10-21 07:00:00'],
    ['2023-10-21 07:00:00', '2023-10-21 19:00:00'],
    ['2023-10-21 19:00:00', '2023-10-22 07:00:00'],
    ['2023-10-22 07:00:00', '2023-10-22 19:00:00'],
    ['2023-10-22 19:00:00', '2023-10-23 07:00:00'],
    ['2023-10-23 07:00:00', '2023-10-23 19:00:00'],
    ['2023-10-23 19:00:00', '2023-10-24 07:00:00'],
    ['2023-10-24 07:00:00', '2023-10-24 19:00:00'],
    ['2023-10-24 19:00:00', '2023-10-25 07:00:00'],
    ['2023-10-25 07:00:00', '2023-10-25 19:00:00'],
    ['2023-10-25 19:00:00', '2023-10-26 07:00:00'],
    ['2023-10-26 07:00:00', '2023-10-26 19:00:00'],
    ['2023-10-26 19:00:00', '2023-10-27 07:00:00'],
    ['2023-10-27 07:00:00', '2023-10-27 19:00:00'],
    ['2023-10-27 19:00:00', '2023-10-28 07:00:00'],
    ['2023-10-28 07:00:00', '2023-10-28 19:00:00'],
    ['2023-10-28 19:00:00', '2023-10-29 07:00:00'],
    ['2023-10-29 07:00:00', '2023-10-29 19:00:00'],#here
    ['2023-10-31 07:00:00', '2023-10-31 19:00:00'],
    ['2023-10-31 19:00:00', '2023-11-01 07:00:00'],
    ['2023-11-01 07:00:00', '2023-11-01 19:00:00'],
    ['2023-11-01 19:00:00', '2023-11-02 07:00:00'],
    ['2023-11-02 07:00:00', '2023-11-02 19:00:00'],
    ['2023-11-02 19:00:00', '2023-11-03 07:00:00'],
    ['2023-11-03 07:00:00', '2023-11-03 19:00:00'],
    ['2023-11-03 19:00:00', '2023-11-04 07:00:00'],
    ['2023-11-04 07:00:00', '2023-11-04 19:00:00'],
    ['2023-11-04 19:00:00', '2023-11-05 07:00:00'],
    ['2023-11-05 07:00:00', '2023-11-05 19:00:00'],
    ['2023-11-05 19:00:00', '2023-11-06 07:00:00'],
    ['2023-11-06 07:00:00', '2023-11-06 19:00:00'],
    ['2023-11-06 19:00:00', '2023-11-07 07:00:00'],
    ['2023-11-07 07:00:00', '2023-11-07 19:00:00'],
    ['2023-11-07 19:00:00', '2023-11-08 07:00:00'],
    ['2023-11-08 07:00:00', '2023-11-08 19:00:00'],
    ['2023-11-08 19:00:00', '2023-11-09 07:00:00'],
    ['2023-11-09 07:00:00', '2023-11-09 19:00:00'],
    ['2023-11-09 19:00:00', '2023-11-10 07:00:00'],
    ['2023-11-10 07:00:00', '2023-11-10 19:00:00'],
    ['2023-11-10 19:00:00', '2023-11-11 07:00:00'],
    ['2023-11-11 07:00:00', '2023-11-11 19:00:00'],
    ['2023-11-11 19:00:00', '2023-11-12 07:00:00'],
    ['2023-11-12 07:00:00', '2023-11-12 19:00:00'],
    ['2023-11-12 19:00:00', '2023-11-13 07:00:00'],
    ['2023-11-13 07:00:00', '2023-11-13 19:00:00'],
    ['2023-11-13 19:00:00', '2023-11-14 07:00:00'],
    ['2023-11-14 07:00:00', '2023-11-14 19:00:00'],
    ['2023-11-14 19:00:00', '2023-11-15 07:00:00'],
    ['2023-11-15 07:00:00', '2023-11-15 19:00:00'],
    ['2023-11-15 19:00:00', '2023-11-16 07:00:00'],
    ['2023-11-16 07:00:00', '2023-11-16 19:00:00'],
    ['2023-11-16 19:00:00', '2023-11-17 07:00:00'],
    ['2023-11-17 07:00:00', '2023-11-17 19:00:00'],
    ['2023-11-17 19:00:00', '2023-11-18 07:00:00'],
    ['2023-11-18 07:00:00', '2023-11-18 19:00:00'],
    ['2023-11-18 19:00:00', '2023-11-19 07:00:00'],
    ['2023-11-19 07:00:00', '2023-11-19 19:00:00'],
    ['2023-11-19 19:00:00', '2023-11-20 07:00:00'],
    ['2023-11-20 07:00:00', '2023-11-20 19:00:00'],#here
    ['2023-11-22 07:00:00', '2023-11-22 19:00:00'],
    ['2023-11-22 19:00:00', '2023-11-23 07:00:00'],
    ['2023-11-23 07:00:00', '2023-11-23 19:00:00'],
    ['2023-11-23 19:00:00', '2023-11-24 07:00:00'],
    ['2023-11-24 07:00:00', '2023-11-24 19:00:00'],
    ['2023-11-24 19:00:00', '2023-11-25 07:00:00'],
    ['2023-11-25 07:00:00', '2023-11-25 19:00:00'],
    ['2023-11-25 19:00:00', '2023-11-26 07:00:00'],
    ['2023-11-26 07:00:00', '2023-11-26 19:00:00'],
    ['2023-11-26 19:00:00', '2023-11-27 07:00:00'],
    ['2023-11-27 07:00:00', '2023-11-27 19:00:00'],
    ['2023-11-27 19:00:00', '2023-11-28 07:00:00'],
    ['2023-11-28 07:00:00', '2023-11-28 19:00:00'],
    ['2023-11-28 19:00:00', '2023-11-29 07:00:00'],
    ['2023-11-29 07:00:00', '2023-11-29 19:00:00'],
    ['2023-11-29 19:00:00', '2023-11-30 07:00:00'],
    ['2023-11-30 07:00:00', '2023-11-30 19:00:00'],
    ['2023-11-30 19:00:00', '2023-12-01 07:00:00'],
    ['2023-12-01 07:00:00', '2023-12-01 19:00:00'],
    ['2023-12-01 19:00:00', '2023-12-02 07:00:00'],
    ['2023-12-02 07:00:00', '2023-12-02 19:00:00'],
    ['2023-12-02 19:00:00', '2023-12-03 07:00:00'],
    ['2023-12-03 07:00:00', '2023-12-03 19:00:00'],
    ['2023-12-03 19:00:00', '2023-12-04 07:00:00'],
    ['2023-12-04 07:00:00', '2023-12-04 19:00:00'],
    ['2023-12-04 19:00:00', '2023-12-05 07:00:00'],
    ['2023-12-05 07:00:00', '2023-12-05 19:00:00'],
    ['2023-12-05 19:00:00', '2023-12-06 07:00:00'],
    ['2023-12-06 07:00:00', '2023-12-06 19:00:00'],
    ['2023-12-06 19:00:00', '2023-12-07 07:00:00'],
    ['2023-12-07 07:00:00', '2023-12-07 19:00:00'],
    ['2023-12-07 19:00:00', '2023-12-08 07:00:00'],
    ['2023-12-08 07:00:00', '2023-12-08 19:00:00'],
    ['2023-12-08 19:00:00', '2023-12-09 07:00:00'],
    ['2023-12-09 07:00:00', '2023-12-09 19:00:00'],
    ['2023-12-09 19:00:00', '2023-12-10 07:00:00'],
    ['2023-12-10 07:00:00', '2023-12-10 19:00:00'],
    ['2023-12-10 19:00:00', '2023-12-11 07:00:00'],
    ['2023-12-11 07:00:00', '2023-12-11 19:00:00'],
    ['2023-12-11 19:00:00', '2023-12-12 07:00:00'],
    ['2023-12-12 07:00:00', '2023-12-12 19:00:00'],
    ['2023-12-12 19:00:00', '2023-12-13 07:00:00'],
    ['2023-12-13 07:00:00', '2023-12-13 19:00:00'],
    ['2023-12-13 19:00:00', '2023-12-14 07:00:00'],
    ['2023-12-14 07:00:00', '2023-12-14 19:00:00'],#here
    ['2023-12-16 07:00:00', '2023-12-16 19:00:00'],
    ['2023-12-16 19:00:00', '2023-12-17 07:00:00'],
    ['2023-12-17 07:00:00', '2023-12-17 19:00:00'],#here
    ['2023-12-19 07:00:00', '2023-12-19 19:00:00'],
    ['2023-12-19 19:00:00', '2023-12-20 07:00:00'],
    ['2023-12-20 07:00:00', '2023-12-20 19:00:00'],
    ['2023-12-20 19:00:00', '2023-12-21 07:00:00'],
    ['2023-12-21 07:00:00', '2023-12-21 19:00:00'],
    ['2023-12-21 19:00:00', '2023-12-22 07:00:00'],
    ['2023-12-22 07:00:00', '2023-12-22 19:00:00'],
    ['2023-12-22 19:00:00', '2023-12-23 07:00:00'],
    ['2023-12-23 07:00:00', '2023-12-23 19:00:00'],
    ['2023-12-23 19:00:00', '2023-12-24 07:00:00'],
    ['2023-12-24 07:00:00', '2023-12-24 19:00:00'],
    ['2023-12-24 19:00:00', '2023-12-25 07:00:00'],
    ['2023-12-25 07:00:00', '2023-12-25 19:00:00'],
    ['2023-12-25 19:00:00', '2023-12-26 07:00:00'],
    ['2023-12-26 07:00:00', '2023-12-26 19:00:00'],
    ['2023-12-26 19:00:00', '2023-12-27 07:00:00'],
    ['2023-12-27 07:00:00', '2023-12-27 19:00:00'],
    ['2023-12-27 19:00:00', '2023-12-28 07:00:00'],
    ['2023-12-28 07:00:00', '2023-12-28 19:00:00'],
    ['2023-12-28 19:00:00', '2023-12-29 07:00:00'],
    ['2023-12-29 07:00:00', '2023-12-29 19:00:00'],
    ['2023-12-29 19:00:00', '2023-12-30 07:00:00'],
    ['2023-12-30 07:00:00', '2023-12-30 19:00:00'],
    ['2023-12-30 19:00:00', '2023-12-31 07:00:00'],
    ['2023-12-31 07:00:00', '2023-12-31 19:00:00'],
    ['2023-12-31 19:00:00', '2024-01-01 07:00:00'],
    ['2024-01-01 07:00:00', '2024-01-01 19:00:00'],
    ['2024-01-01 19:00:00', '2024-01-02 07:00:00'],
    ['2024-01-02 07:00:00', '2024-01-02 19:00:00'],
    ['2024-01-02 19:00:00', '2024-01-03 07:00:00'],
    ['2024-01-03 07:00:00', '2024-01-03 19:00:00'],
    ['2024-01-03 19:00:00', '2024-01-04 07:00:00'],
    ['2024-01-04 07:00:00', '2024-01-04 19:00:00'],
    ['2024-01-04 19:00:00', '2024-01-05 07:00:00'],
    ['2024-01-05 07:00:00', '2024-01-05 19:00:00'],
    ['2024-01-05 19:00:00', '2024-01-06 07:00:00'],
    ['2024-01-06 07:00:00', '2024-01-06 19:00:00'],
    ['2024-01-06 19:00:00', '2024-01-07 07:00:00'],
    ['2024-01-07 07:00:00', '2024-01-07 19:00:00'],
    ['2024-01-07 19:00:00', '2024-01-08 07:00:00'],
    ['2024-01-08 07:00:00', '2024-01-08 19:00:00'],
    ['2024-01-08 19:00:00', '2024-01-09 07:00:00'],
    ['2024-01-09 07:00:00', '2024-01-09 19:00:00'],
    ['2024-01-09 19:00:00', '2024-01-10 07:00:00'],
    ['2024-01-10 07:00:00', '2024-01-10 19:00:00'],
    ['2024-01-10 19:00:00', '2024-01-11 07:00:00'],
    ['2024-01-11 07:00:00', '2024-01-11 19:00:00'],
    ['2024-01-11 19:00:00', '2024-01-12 07:00:00'],
    ['2024-01-12 07:00:00', '2024-01-12 19:00:00'],
    ['2024-01-12 19:00:00', '2024-01-13 07:00:00'],
    ['2024-01-13 07:00:00', '2024-01-13 19:00:00'],
    ['2024-01-13 19:00:00', '2024-01-14 07:00:00'],
    ['2024-01-14 07:00:00', '2024-01-14 19:00:00'],#here
    ['2024-01-16 07:00:00', '2024-01-16 19:00:00'],
    ['2024-01-16 19:00:00', '2024-01-17 07:00:00'],
    ['2024-01-17 07:00:00', '2024-01-17 19:00:00'],
    ['2024-01-17 19:00:00', '2024-01-18 07:00:00'],
    ['2024-01-18 07:00:00', '2024-01-18 19:00:00'],
    ['2024-01-18 19:00:00', '2024-01-19 07:00:00'],
    ['2024-01-19 07:00:00', '2024-01-19 19:00:00'],
    ['2024-01-19 19:00:00', '2024-01-20 07:00:00'],
    ['2024-01-20 07:00:00', '2024-01-20 19:00:00'],
    ['2024-01-20 19:00:00', '2024-01-21 07:00:00'],
    ['2024-01-21 07:00:00', '2024-01-21 19:00:00'],
    ['2024-01-21 19:00:00', '2024-01-22 07:00:00'],
    ['2024-01-22 07:00:00', '2024-01-22 19:00:00'],
    ['2024-01-22 19:00:00', '2024-01-23 07:00:00'],
    ['2024-01-23 07:00:00', '2024-01-23 19:00:00'],
    ['2024-01-23 19:00:00', '2024-01-24 07:00:00'],
    ['2024-01-24 07:00:00', '2024-01-24 19:00:00'],
    ['2024-01-24 19:00:00', '2024-01-25 07:00:00'],
    ['2024-01-25 07:00:00', '2024-01-25 19:00:00'],
    ['2024-01-25 19:00:00', '2024-01-26 07:00:00'],
    ['2024-01-26 07:00:00', '2024-01-26 19:00:00'],
    ['2024-01-26 19:00:00', '2024-01-27 07:00:00'],
    ['2024-01-27 07:00:00', '2024-01-27 19:00:00'],
    ['2024-01-27 19:00:00', '2024-01-28 07:00:00'],
    ['2024-01-28 07:00:00', '2024-01-28 19:00:00'],
    ['2024-01-28 19:00:00', '2024-01-29 07:00:00'],
    ['2024-01-29 07:00:00', '2024-01-29 19:00:00'],
    ['2024-01-29 19:00:00', '2024-01-30 07:00:00'],
    ['2024-01-30 07:00:00', '2024-01-30 19:00:00'],
    ['2024-01-30 19:00:00', '2024-01-31 07:00:00'],
    ['2024-01-31 07:00:00', '2024-01-31 19:00:00'],
    ['2024-01-31 19:00:00', '2024-02-01 07:00:00'],
    ['2024-02-01 07:00:00', '2024-02-01 19:00:00'],
    ['2024-02-01 19:00:00', '2024-02-02 07:00:00'],
    ['2024-02-02 07:00:00', '2024-02-02 19:00:00'],
    ['2024-02-02 19:00:00', '2024-02-03 07:00:00'],
    ['2024-02-03 07:00:00', '2024-02-03 19:00:00'],
    ['2024-02-03 19:00:00', '2024-02-04 07:00:00'],
    ['2024-02-04 07:00:00', '2024-02-04 19:00:00'],
    ['2024-02-04 19:00:00', '2024-02-05 07:00:00'],
    ['2024-02-05 07:00:00', '2024-02-05 19:00:00'],
    ['2024-02-05 19:00:00', '2024-02-06 07:00:00'],
    ['2024-02-06 07:00:00', '2024-02-06 19:00:00'],
    ['2024-02-06 19:00:00', '2024-02-07 07:00:00'],
    ['2024-02-07 07:00:00', '2024-02-07 19:00:00'],
    ['2024-02-07 19:00:00', '2024-02-08 07:00:00'],
    ['2024-02-08 07:00:00', '2024-02-08 19:00:00'],
    ['2024-02-08 19:00:00', '2024-02-09 07:00:00'],
    ['2024-02-09 07:00:00', '2024-02-09 19:00:00'],
    ['2024-02-09 19:00:00', '2024-02-10 07:00:00'],
    ['2024-02-10 07:00:00', '2024-02-10 19:00:00'],
    ['2024-02-10 19:00:00', '2024-02-11 07:00:00'],
    ['2024-02-11 07:00:00', '2024-02-11 19:00:00'],
    ['2024-02-11 19:00:00', '2024-02-12 07:00:00'],
    ['2024-02-12 07:00:00', '2024-02-12 19:00:00'],
    ['2024-02-12 19:00:00', '2024-02-13 07:00:00'],
    ['2024-02-13 07:00:00', '2024-02-13 19:00:00'],
    ['2024-02-13 19:00:00', '2024-02-14 07:00:00'],
    ['2024-02-14 07:00:00', '2024-02-14 19:00:00'],
    ['2024-02-14 19:00:00', '2024-02-15 07:00:00'],
    ['2024-02-15 07:00:00', '2024-02-15 19:00:00'],
    ['2024-02-15 19:00:00', '2024-02-16 07:00:00'],
    ['2024-02-16 07:00:00', '2024-02-16 19:00:00'],
    ['2024-02-16 19:00:00', '2024-02-17 07:00:00'],
    ['2024-02-17 07:00:00', '2024-02-17 19:00:00'],
    ['2024-02-17 19:00:00', '2024-02-18 07:00:00'],
    ['2024-02-18 07:00:00', '2024-02-18 19:00:00'],
    ['2024-02-18 19:00:00', '2024-02-19 07:00:00'],
    ['2024-02-19 07:00:00', '2024-02-19 19:00:00'],
    ['2024-02-19 19:00:00', '2024-02-20 07:00:00'],
    ['2024-02-20 07:00:00', '2024-02-20 19:00:00'],
    ['2024-02-20 19:00:00', '2024-02-21 07:00:00'],
    ['2024-02-21 07:00:00', '2024-02-21 19:00:00'],
    ['2024-02-21 19:00:00', '2024-02-22 07:00:00'],
    ['2024-02-22 07:00:00', '2024-02-22 19:00:00'],
    ['2024-02-22 19:00:00', '2024-02-23 07:00:00'],
    ['2024-02-23 07:00:00', '2024-02-23 19:00:00'],
    ['2024-02-23 19:00:00', '2024-02-24 07:00:00'],
    ['2024-02-24 07:00:00', '2024-02-24 19:00:00'],
    ['2024-02-24 19:00:00', '2024-02-25 07:00:00'],
    ['2024-02-25 07:00:00', '2024-02-25 19:00:00'],#here
    ['2024-02-27 07:00:00', '2024-02-27 19:00:00'],
    ['2024-02-27 19:00:00', '2024-02-28 07:00:00'],
    ['2024-02-28 07:00:00', '2024-02-28 19:00:00'],
    ['2024-02-28 19:00:00', '2024-02-29 07:00:00'],
    ['2024-02-29 07:00:00', '2024-02-29 19:00:00'],
    ['2024-02-29 19:00:00', '2024-03-01 07:00:00'],
    ['2024-03-01 07:00:00', '2024-03-01 19:00:00'],
    ['2024-03-01 19:00:00', '2024-03-02 07:00:00'],
    ['2024-03-02 07:00:00', '2024-03-02 19:00:00'],
    ['2024-03-02 19:00:00', '2024-03-03 07:00:00'],
    ['2024-03-03 07:00:00', '2024-03-03 19:00:00'],
    ['2024-03-03 19:00:00', '2024-03-04 07:00:00'],
    ['2024-03-04 07:00:00', '2024-03-04 19:00:00'],
    ['2024-03-04 19:00:00', '2024-03-05 07:00:00'],
    ['2024-03-05 07:00:00', '2024-03-05 19:00:00'],
    ['2024-03-05 19:00:00', '2024-03-06 07:00:00'],
    ['2024-03-06 07:00:00', '2024-03-06 19:00:00'],
    ['2024-03-06 19:00:00', '2024-03-07 07:00:00'],
    ['2024-03-07 07:00:00', '2024-03-07 19:00:00'],
    ['2024-03-07 19:00:00', '2024-03-08 07:00:00'],
    ['2024-03-08 07:00:00', '2024-03-08 19:00:00'],
    ['2024-03-08 19:00:00', '2024-03-09 07:00:00'],
    ['2024-03-09 07:00:00', '2024-03-09 19:00:00'],
    ['2024-03-09 19:00:00', '2024-03-10 07:00:00'],
    ['2024-03-10 07:00:00', '2024-03-10 19:00:00'],
    ['2024-03-10 19:00:00', '2024-03-11 07:00:00'],
    ['2024-03-11 07:00:00', '2024-03-11 19:00:00'],
    ['2024-03-11 19:00:00', '2024-03-12 07:00:00'],
    ['2024-03-12 07:00:00', '2024-03-12 19:00:00'],
    ['2024-03-12 19:00:00', '2024-03-13 07:00:00'],
    ['2024-03-13 07:00:00', '2024-03-13 19:00:00'],
    ['2024-03-13 19:00:00', '2024-03-14 07:00:00'],
    ['2024-03-14 07:00:00', '2024-03-14 19:00:00'],
    ['2024-03-14 19:00:00', '2024-03-15 07:00:00'],
    ['2024-03-15 07:00:00', '2024-03-15 19:00:00'],
    ['2024-03-15 19:00:00', '2024-03-16 07:00:00'],
    ['2024-03-16 07:00:00', '2024-03-16 19:00:00'],
    ['2024-03-16 19:00:00', '2024-03-17 07:00:00'],
    ['2024-03-17 07:00:00', '2024-03-17 19:00:00'],
    ['2024-03-17 19:00:00', '2024-03-18 07:00:00'],
    ['2024-03-18 07:00:00', '2024-03-18 19:00:00'],#here
    ['2024-03-20 07:00:00', '2024-03-20 19:00:00'],
    ['2024-03-20 19:00:00', '2024-03-21 07:00:00'],
    ['2024-03-21 07:00:00', '2024-03-21 19:00:00'],
    ['2024-03-21 19:00:00', '2024-03-22 07:00:00'],
    ['2024-03-22 07:00:00', '2024-03-22 19:00:00'],
    ['2024-03-22 19:00:00', '2024-03-23 07:00:00'],
    ['2024-03-23 07:00:00', '2024-03-23 19:00:00'],
    ['2024-03-23 19:00:00', '2024-03-24 07:00:00'],
    ['2024-03-24 07:00:00', '2024-03-24 19:00:00'],
    ['2024-03-24 19:00:00', '2024-03-25 07:00:00'],
    ['2024-03-25 07:00:00', '2024-03-25 19:00:00'],#here
    ['2024-03-27 07:00:00', '2024-03-27 19:00:00'],
    ['2024-03-27 19:00:00', '2024-03-28 07:00:00'],
    ['2024-03-28 07:00:00', '2024-03-28 19:00:00'],
    ['2024-03-28 19:00:00', '2024-03-29 07:00:00'],
    ['2024-03-29 07:00:00', '2024-03-29 19:00:00'],
    ['2024-03-29 19:00:00', '2024-03-30 07:00:00'],
    ['2024-03-30 07:00:00', '2024-03-30 19:00:00'],
    ['2024-04-01 07:00:00', '2024-04-01 19:00:00'],#here
    ['2024-04-01 19:00:00', '2024-04-02 07:00:00'],
    ['2024-04-02 07:00:00', '2024-04-02 19:00:00'],
    ['2024-04-02 19:00:00', '2024-04-03 07:00:00'],
    ['2024-04-03 07:00:00', '2024-04-03 19:00:00'],
    ['2024-04-03 19:00:00', '2024-04-04 07:00:00'],
    ['2024-04-04 07:00:00', '2024-04-04 19:00:00'],
    ['2024-04-04 19:00:00', '2024-04-05 07:00:00'],
    ['2024-04-05 07:00:00', '2024-04-05 19:00:00'],
    ['2024-04-05 19:00:00', '2024-04-06 07:00:00'],
    ['2024-04-06 07:00:00', '2024-04-06 19:00:00'],
    ['2024-04-06 19:00:00', '2024-04-07 07:00:00'],
    ['2024-04-07 07:00:00', '2024-04-07 19:00:00'],
    ['2024-04-07 19:00:00', '2024-04-08 07:00:00'],
    ['2024-04-08 07:00:00', '2024-04-08 19:00:00'],
    ['2024-04-08 19:00:00', '2024-04-09 07:00:00'],
    ['2024-04-09 07:00:00', '2024-04-09 19:00:00'],
    ['2024-04-09 19:00:00', '2024-04-10 07:00:00'],
    ['2024-04-10 07:00:00', '2024-04-10 19:00:00'],
    ['2024-04-10 19:00:00', '2024-04-11 07:00:00'],
    ['2024-04-11 07:00:00', '2024-04-11 19:00:00'],
    ['2024-04-11 19:00:00', '2024-04-12 07:00:00'],
    ['2024-04-12 07:00:00', '2024-04-12 19:00:00'],
    ['2024-04-12 19:00:00', '2024-04-13 07:00:00'],
    ['2024-04-13 07:00:00', '2024-04-13 19:00:00'],
    ['2024-04-13 19:00:00', '2024-04-14 07:00:00'],
    ['2024-04-14 07:00:00', '2024-04-14 19:00:00'],
    ['2024-04-14 19:00:00', '2024-04-15 07:00:00'],
    ['2024-04-15 07:00:00', '2024-04-15 19:00:00'],
    ['2024-04-15 19:00:00', '2024-04-16 07:00:00'],
    ['2024-04-16 07:00:00', '2024-04-16 19:00:00'],
    ['2024-04-16 19:00:00', '2024-04-17 07:00:00'],
    ['2024-04-17 07:00:00', '2024-04-17 19:00:00'],#here
    ['2024-04-19 07:00:00', '2024-04-19 19:00:00'],
    ['2024-04-19 19:00:00', '2024-04-20 07:00:00'],
    ['2024-04-20 07:00:00', '2024-04-20 19:00:00'],
    ['2024-04-20 19:00:00', '2024-04-21 07:00:00'],
    ['2024-04-21 07:00:00', '2024-04-21 19:00:00'],
    ['2024-04-21 19:00:00', '2024-04-22 07:00:00'],
    ['2024-04-22 07:00:00', '2024-04-22 19:00:00'],
    ['2024-04-22 19:00:00', '2024-04-23 07:00:00'],
    ['2024-04-23 07:00:00', '2024-04-23 19:00:00'],#here
    ['2024-04-25 07:00:00', '2024-04-25 19:00:00'],
    ['2024-04-25 19:00:00', '2024-04-26 07:00:00'],
    ['2024-04-26 07:00:00', '2024-04-26 19:00:00'],
    ['2024-04-26 19:00:00', '2024-04-27 07:00:00'],
    ['2024-04-27 07:00:00', '2024-04-27 19:00:00'],
    ['2024-04-27 19:00:00', '2024-04-28 07:00:00'],
    ['2024-04-28 07:00:00', '2024-04-28 19:00:00'],
    ['2024-04-28 19:00:00', '2024-04-29 07:00:00'],
    ['2024-04-29 07:00:00', '2024-04-29 19:00:00'],
    ['2024-04-29 19:00:00', '2024-04-30 07:00:00'],
    ['2024-04-30 07:00:00', '2024-04-30 19:00:00'],
    ['2024-04-30 19:00:00', '2024-05-01 07:00:00'],
    ['2024-05-01 07:00:00', '2024-05-01 19:00:00'],
    ['2024-05-01 19:00:00', '2024-05-02 07:00:00'],
    ['2024-05-02 07:00:00', '2024-05-02 19:00:00'],
    ['2024-05-02 19:00:00', '2024-05-03 07:00:00'],
    ['2024-05-03 07:00:00', '2024-05-03 19:00:00'],
    ['2024-05-03 19:00:00', '2024-05-04 07:00:00'],
    ['2024-05-04 07:00:00', '2024-05-04 19:00:00'],
    ['2024-05-04 19:00:00', '2024-05-05 07:00:00'],
    ['2024-05-05 07:00:00', '2024-05-05 19:00:00'],
    ['2024-05-05 19:00:00', '2024-05-06 07:00:00'],
    ['2024-05-06 07:00:00', '2024-05-06 19:00:00'],
    ['2024-05-06 19:00:00', '2024-05-07 07:00:00'],
    ['2024-05-07 07:00:00', '2024-05-07 19:00:00'],
    ['2024-05-07 19:00:00', '2024-05-08 07:00:00'],
    ['2024-05-08 07:00:00', '2024-05-08 19:00:00'],
    ['2024-05-08 19:00:00', '2024-05-09 07:00:00'],
    ['2024-05-09 07:00:00', '2024-05-09 19:00:00'],
    ['2024-05-09 19:00:00', '2024-05-10 07:00:00'],
    ['2024-05-10 07:00:00', '2024-05-10 19:00:00'],
    ['2024-05-10 19:00:00', '2024-05-11 07:00:00'],
    ['2024-05-11 07:00:00', '2024-05-11 19:00:00'],
    ['2024-05-11 19:00:00', '2024-05-12 07:00:00'],
    ['2024-05-12 07:00:00', '2024-05-12 19:00:00'],
    ['2024-05-12 19:00:00', '2024-05-13 07:00:00'],
    ['2024-05-13 07:00:00', '2024-05-13 19:00:00'],
    ['2024-05-13 19:00:00', '2024-05-14 07:00:00'],
    ['2024-05-14 07:00:00', '2024-05-14 19:00:00'],#here
    ['2024-05-16 07:00:00', '2024-05-16 19:00:00'],
    ['2024-05-16 19:00:00', '2024-05-17 07:00:00'],
    ['2024-05-17 07:00:00', '2024-05-17 19:00:00'],
    ['2024-05-17 19:00:00', '2024-05-18 07:00:00'],
    ['2024-05-18 07:00:00', '2024-05-18 19:00:00'],
    ['2024-05-18 19:00:00', '2024-05-19 07:00:00'],
    ['2024-05-19 07:00:00', '2024-05-19 19:00:00'],
    ['2024-05-19 19:00:00', '2024-05-20 07:00:00'],
    ['2024-05-20 07:00:00', '2024-05-20 19:00:00'],
    ['2024-05-20 19:00:00', '2024-05-21 07:00:00'],
    ['2024-05-21 07:00:00', '2024-05-21 19:00:00'],
    ['2024-05-21 19:00:00', '2024-05-22 07:00:00'],
    ['2024-05-22 07:00:00', '2024-05-22 19:00:00'],
    ['2024-05-22 19:00:00', '2024-05-23 07:00:00'],
    ['2024-05-23 07:00:00', '2024-05-23 19:00:00'],
    ['2024-05-23 19:00:00', '2024-05-24 07:00:00'],
    ['2024-05-24 07:00:00', '2024-05-24 19:00:00'],
    ['2024-05-24 19:00:00', '2024-05-25 07:00:00'],
    ['2024-05-25 07:00:00', '2024-05-25 19:00:00'],
    ['2024-05-25 19:00:00', '2024-05-26 07:00:00'],
    ['2024-05-26 07:00:00', '2024-05-26 19:00:00'],
    ['2024-05-26 19:00:00', '2024-05-27 07:00:00'],
    ['2024-05-27 07:00:00', '2024-05-27 19:00:00'],
    ['2024-05-27 19:00:00', '2024-05-28 07:00:00'],
    ['2024-05-28 07:00:00', '2024-05-28 19:00:00'],
    ['2024-05-28 19:00:00', '2024-05-29 07:00:00'],
    ['2024-05-29 07:00:00', '2024-05-29 19:00:00'],#here
    ['2024-05-31 07:00:00', '2024-05-31 19:00:00'],
    ['2024-05-31 19:00:00', '2024-06-01 07:00:00'],
    ['2024-06-01 07:00:00', '2024-06-01 19:00:00'],
    ['2024-06-01 19:00:00', '2024-06-02 07:00:00'],
    ['2024-06-02 07:00:00', '2024-06-02 19:00:00'],
    ['2024-06-02 19:00:00', '2024-06-03 07:00:00'],
    ['2024-06-03 07:00:00', '2024-06-03 19:00:00'],
    ['2024-06-03 19:00:00', '2024-06-04 07:00:00'],
    ['2024-06-04 07:00:00', '2024-06-04 19:00:00'],
    ['2024-06-04 19:00:00', '2024-06-05 07:00:00'],
    ['2024-06-05 07:00:00', '2024-06-05 19:00:00'],
    ['2024-06-05 19:00:00', '2024-06-06 07:00:00'],
    ['2024-06-06 07:00:00', '2024-06-06 19:00:00'],
    ['2024-06-06 19:00:00', '2024-06-07 07:00:00'],
    ['2024-06-07 07:00:00', '2024-06-07 19:00:00'],
    ['2024-06-07 19:00:00', '2024-06-08 07:00:00'],
    ['2024-06-08 07:00:00', '2024-06-08 19:00:00'],
    ['2024-06-08 19:00:00', '2024-06-09 07:00:00'],
    ['2024-06-09 07:00:00', '2024-06-09 19:00:00'],
    ['2024-06-09 19:00:00', '2024-06-10 07:00:00'],
    ['2024-06-10 07:00:00', '2024-06-10 19:00:00'],
    ['2024-06-10 19:00:00', '2024-06-11 07:00:00'],
    ['2024-06-11 07:00:00', '2024-06-11 19:00:00'],
    ['2024-06-11 19:00:00', '2024-06-12 07:00:00'],
    ['2024-06-12 07:00:00', '2024-06-12 19:00:00'],
    ['2024-06-12 19:00:00', '2024-06-13 07:00:00'],
    ['2024-06-13 07:00:00', '2024-06-13 19:00:00'],
    ['2024-06-13 19:00:00', '2024-06-14 07:00:00'],
    ['2024-06-14 07:00:00', '2024-06-14 19:00:00'],
    ['2024-06-14 19:00:00', '2024-06-15 07:00:00'],
    ['2024-06-15 07:00:00', '2024-06-15 19:00:00'],
    ['2024-06-15 19:00:00', '2024-06-16 07:00:00'],
    ['2024-06-16 07:00:00', '2024-06-16 19:00:00'],
    ['2024-06-16 19:00:00', '2024-06-17 07:00:00'],
    ['2024-06-17 07:00:00', '2024-06-17 19:00:00'],#here
    ['2024-06-19 07:00:00', '2024-06-19 19:00:00'],
    ['2024-06-19 19:00:00', '2024-06-20 07:00:00'],
    ['2024-06-20 07:00:00', '2024-06-20 19:00:00'],
    ['2024-06-20 19:00:00', '2024-06-21 07:00:00'],
    ['2024-06-21 07:00:00', '2024-06-21 19:00:00'],
    ['2024-06-21 19:00:00', '2024-06-22 07:00:00'],
    ['2024-06-22 07:00:00', '2024-06-22 19:00:00'],
    ['2024-06-22 19:00:00', '2024-06-23 07:00:00'],
    ['2024-06-23 07:00:00', '2024-06-23 19:00:00'],
    ['2024-06-23 19:00:00', '2024-06-24 07:00:00'],
    ['2024-06-24 07:00:00', '2024-06-24 19:00:00'],
    ['2024-06-24 19:00:00', '2024-06-25 07:00:00'],
    ['2024-06-25 07:00:00', '2024-06-25 19:00:00'],
    ['2024-06-25 19:00:00', '2024-06-26 07:00:00'],
    ['2024-06-28 19:00:00', '2024-06-29 07:00:00'],#here
    ['2024-06-29 07:00:00', '2024-06-29 19:00:00'],
    ['2024-06-29 19:00:00', '2024-06-30 07:00:00'],
    ['2024-06-30 07:00:00', '2024-06-30 19:00:00'],
    ['2024-06-30 19:00:00', '2024-07-01 07:00:00'],
    ['2024-07-01 07:00:00', '2024-07-01 19:00:00'],
    ['2024-07-01 19:00:00', '2024-07-02 07:00:00'],
    ['2024-07-02 07:00:00', '2024-07-02 19:00:00'],
    ['2024-07-02 19:00:00', '2024-07-03 07:00:00'],
    ['2024-07-03 07:00:00', '2024-07-03 19:00:00'],
    ['2024-07-03 19:00:00', '2024-07-04 07:00:00'],
    ['2024-07-04 07:00:00', '2024-07-04 19:00:00'],
    ['2024-07-04 19:00:00', '2024-07-05 07:00:00'],
    ['2024-07-05 07:00:00', '2024-07-05 19:00:00'],
    ['2024-07-05 19:00:00', '2024-07-06 07:00:00'],
    ['2024-07-06 07:00:00', '2024-07-06 19:00:00'],
    ['2024-07-06 19:00:00', '2024-07-07 07:00:00'],
    ['2024-07-07 07:00:00', '2024-07-07 19:00:00'],
    ['2024-07-07 19:00:00', '2024-07-08 07:00:00'],
    ['2024-07-08 07:00:00', '2024-07-08 19:00:00'],
    ['2024-07-08 19:00:00', '2024-07-09 07:00:00'],
    ['2024-07-09 07:00:00', '2024-07-09 19:00:00'],
    ['2024-07-09 19:00:00', '2024-07-10 07:00:00'],
    ['2024-07-10 07:00:00', '2024-07-10 19:00:00'],
    ['2024-07-10 19:00:00', '2024-07-11 07:00:00'],
    ['2024-07-11 07:00:00', '2024-07-11 19:00:00'],
    ['2024-07-11 19:00:00', '2024-07-12 07:00:00'],
    ['2024-07-12 07:00:00', '2024-07-12 19:00:00'],
    ['2024-07-12 19:00:00', '2024-07-13 07:00:00'],
    ['2024-07-13 07:00:00', '2024-07-13 19:00:00'],
    ['2024-07-13 19:00:00', '2024-07-14 07:00:00'],
    ['2024-07-14 07:00:00', '2024-07-14 19:00:00'],
    ['2024-07-14 19:00:00', '2024-07-15 07:00:00'],
    ['2024-07-15 07:00:00', '2024-07-15 19:00:00'],
    ['2024-07-15 19:00:00', '2024-07-16 07:00:00'],
    ['2024-07-16 07:00:00', '2024-07-16 19:00:00'],
    ['2024-07-16 19:00:00', '2024-07-17 07:00:00'],
    ['2024-07-17 07:00:00', '2024-07-17 19:00:00'],#here
    ['2024-07-19 07:00:00', '2024-07-19 19:00:00'],
    ['2024-07-19 19:00:00', '2024-07-20 07:00:00'],
    ['2024-07-20 07:00:00', '2024-07-20 19:00:00'],
    ['2024-07-20 19:00:00', '2024-07-21 07:00:00'],
    ['2024-07-21 07:00:00', '2024-07-21 19:00:00'],
    ['2024-07-21 19:00:00', '2024-07-22 07:00:00'],
    ['2024-07-22 07:00:00', '2024-07-22 19:00:00'],
    ['2024-07-22 19:00:00', '2024-07-23 07:00:00'],
    ['2024-07-23 07:00:00', '2024-07-23 19:00:00'],
    ['2024-07-23 19:00:00', '2024-07-24 07:00:00'],
    ['2024-07-24 07:00:00', '2024-07-24 19:00:00'],
    ['2024-07-24 19:00:00', '2024-07-25 07:00:00'],
    ['2024-07-25 07:00:00', '2024-07-25 19:00:00'],
    ['2024-07-25 19:00:00', '2024-07-26 07:00:00'],
    ['2024-07-26 07:00:00', '2024-07-26 19:00:00'],
    ['2024-07-26 19:00:00', '2024-07-27 07:00:00'],
    ['2024-07-27 07:00:00', '2024-07-27 19:00:00'],
    ['2024-07-27 19:00:00', '2024-07-28 07:00:00'],
    ['2024-07-28 07:00:00', '2024-07-28 19:00:00'],
    ['2024-07-28 19:00:00', '2024-07-29 07:00:00'],
    ['2024-07-29 07:00:00', '2024-07-29 19:00:00'],
    ['2024-07-29 19:00:00', '2024-07-30 07:00:00'],
    ['2024-07-30 07:00:00', '2024-07-30 19:00:00'],#here
    ['2024-08-04 19:00:00', '2024-08-05 07:00:00'],
    ['2024-08-05 07:00:00', '2024-08-05 19:00:00'],
    ['2024-08-05 19:00:00', '2024-08-06 07:00:00'],
    ['2024-08-06 07:00:00', '2024-08-06 19:00:00'],
    ['2024-08-06 19:00:00', '2024-08-07 07:00:00'],
    ['2024-08-07 07:00:00', '2024-08-07 19:00:00'],
    ['2024-08-07 19:00:00', '2024-08-08 07:00:00'],
    ['2024-08-08 07:00:00', '2024-08-08 19:00:00'],
    ['2024-08-08 19:00:00', '2024-08-09 07:00:00'],
    ['2024-08-09 07:00:00', '2024-08-09 19:00:00'],
    ['2024-08-09 19:00:00', '2024-08-10 07:00:00'],
    ['2024-08-10 07:00:00', '2024-08-10 19:00:00'],
    ['2024-08-10 19:00:00', '2024-08-11 07:00:00'],
    ['2024-08-11 07:00:00', '2024-08-11 19:00:00'],
    ['2024-08-11 19:00:00', '2024-08-12 07:00:00'],
    ['2024-08-12 07:00:00', '2024-08-12 19:00:00'],
    ['2024-08-12 19:00:00', '2024-08-13 07:00:00'],
    ['2024-08-13 07:00:00', '2024-08-13 19:00:00'],
    ['2024-08-13 19:00:00', '2024-08-14 07:00:00'],
    ['2024-08-14 07:00:00', '2024-08-14 19:00:00'],
    ['2024-08-14 19:00:00', '2024-08-15 07:00:00'],
    ['2024-08-15 07:00:00', '2024-08-15 19:00:00'],
    ['2024-08-15 19:00:00', '2024-08-16 07:00:00'],
    ['2024-08-16 07:00:00', '2024-08-16 19:00:00'],
    ['2024-08-16 19:00:00', '2024-08-17 07:00:00'],
    ['2024-08-17 07:00:00', '2024-08-17 19:00:00'],
    ['2024-08-17 19:00:00', '2024-08-18 07:00:00'],
    ['2024-08-18 07:00:00', '2024-08-18 19:00:00'],#here
    ['2024-08-20 07:00:00', '2024-08-20 19:00:00'],
    ['2024-08-20 19:00:00', '2024-08-21 07:00:00'],
    ['2024-08-21 07:00:00', '2024-08-21 19:00:00'],
    ['2024-08-21 19:00:00', '2024-08-22 07:00:00'],
    ['2024-08-22 07:00:00', '2024-08-22 19:00:00'],
    ['2024-08-22 19:00:00', '2024-08-23 07:00:00'],
    ['2024-08-23 07:00:00', '2024-08-23 19:00:00'],
    ['2024-08-23 19:00:00', '2024-08-24 07:00:00'],
    ['2024-08-24 07:00:00', '2024-08-24 19:00:00'],
    ['2024-08-24 19:00:00', '2024-08-25 07:00:00'],
    ['2024-08-25 07:00:00', '2024-08-25 19:00:00']
], 
    #agg_func=['mean', 'std']
 )

In [ ]:
list=[
    ['2022-09-21 19:00:00', '2022-09-22 07:00:00'],
    ['2022-09-22 07:00:00', '2022-09-22 19:00:00'],
    ['2022-09-22 19:00:00', '2022-09-23 07:00:00'],
    ['2022-09-23 07:00:00', '2022-09-23 19:00:00'],
    ['2022-09-23 19:00:00', '2022-09-24 07:00:00'],
    ['2022-09-24 07:00:00', '2022-09-24 19:00:00'],
    ['2022-09-24 19:00:00', '2022-09-25 07:00:00'],
    ['2022-09-25 07:00:00', '2022-09-25 19:00:00'],
    ['2022-09-25 19:00:00', '2022-09-26 07:00:00'],
    ['2022-09-26 07:00:00', '2022-09-26 19:00:00'],
    ['2022-09-26 19:00:00', '2022-09-27 07:00:00'],
    ['2022-09-27 07:00:00', '2022-09-27 19:00:00'],
    ['2022-09-27 19:00:00', '2022-09-28 07:00:00'],
    ['2022-09-28 07:00:00', '2022-09-28 19:00:00'],
    ['2022-09-28 19:00:00', '2022-09-29 07:00:00'],
    ['2022-09-29 07:00:00', '2022-09-29 19:00:00'],
    ['2022-09-29 19:00:00', '2022-09-30 07:00:00'],
    ['2022-09-30 07:00:00', '2022-09-30 19:00:00'],
    ['2022-09-30 19:00:00', '2022-10-01 07:00:00'],
    ['2022-10-01 07:00:00', '2022-10-01 19:00:00'],
    ['2022-10-01 19:00:00', '2022-10-02 07:00:00'],
    ['2022-10-02 07:00:00', '2022-10-02 19:00:00'],
    ['2022-10-02 19:00:00', '2022-10-03 07:00:00'],
    ['2022-10-03 07:00:00', '2022-10-03 19:00:00'],
    ['2022-10-03 19:00:00', '2022-10-04 07:00:00'],
    ['2022-10-04 07:00:00', '2022-10-04 19:00:00'],
    ['2022-10-04 19:00:00', '2022-10-05 07:00:00'],
    ['2022-10-05 07:00:00', '2022-10-05 19:00:00'],
    ['2022-10-05 19:00:00', '2022-10-06 07:00:00'],
    ['2022-10-06 07:00:00', '2022-10-06 19:00:00'],
    ['2022-10-06 19:00:00', '2022-10-07 07:00:00'],
    ['2022-10-07 07:00:00', '2022-10-07 19:00:00'],
    ['2022-10-07 19:00:00', '2022-10-08 07:00:00'],
    ['2022-10-08 07:00:00', '2022-10-08 19:00:00'],
    ['2022-10-08 19:00:00', '2022-10-09 07:00:00'],
    ['2022-10-09 07:00:00', '2022-10-09 19:00:00'],
    ['2022-10-09 19:00:00', '2022-10-10 07:00:00'],
    ['2022-10-10 07:00:00', '2022-10-10 19:00:00'],
    ['2022-10-10 19:00:00', '2022-10-11 07:00:00'],
    ['2022-10-11 07:00:00', '2022-10-11 19:00:00'],
    ['2022-10-11 19:00:00', '2022-10-12 07:00:00'],
    ['2022-10-12 07:00:00', '2022-10-12 19:00:00'],
    ['2022-10-12 19:00:00', '2022-10-13 07:00:00'],
    ['2022-10-13 07:00:00', '2022-10-13 19:00:00'],
    ['2022-10-13 19:00:00', '2022-10-14 07:00:00'],
    ['2022-10-14 07:00:00', '2022-10-14 19:00:00'],
    ['2022-10-14 19:00:00', '2022-10-15 07:00:00'],
    ['2022-10-15 07:00:00', '2022-10-15 19:00:00'],
    ['2022-10-15 19:00:00', '2022-10-16 07:00:00'],
    ['2022-10-16 07:00:00', '2022-10-16 19:00:00'],
    ['2022-10-16 19:00:00', '2022-10-17 07:00:00'],
    ['2022-10-17 07:00:00', '2022-10-17 19:00:00'],
    ['2022-10-19 07:00:00', '2022-10-19 19:00:00'],#here
    ['2022-10-19 19:00:00', '2022-10-20 07:00:00'],
    ['2022-10-20 07:00:00', '2022-10-20 19:00:00'],
    ['2022-10-20 19:00:00', '2022-10-21 07:00:00'],
    ['2022-10-21 07:00:00', '2022-10-21 19:00:00'],
    ['2022-10-21 19:00:00', '2022-10-22 07:00:00'],
    ['2022-10-22 07:00:00', '2022-10-22 19:00:00'],
    ['2022-10-22 19:00:00', '2022-10-23 07:00:00'],
    ['2022-10-23 07:00:00', '2022-10-23 19:00:00'],
    ['2022-10-23 19:00:00', '2022-10-24 07:00:00'],
    ['2022-10-24 07:00:00', '2022-10-24 19:00:00'],
    ['2022-10-24 19:00:00', '2022-10-25 07:00:00'],
    ['2022-10-25 07:00:00', '2022-10-25 19:00:00'],
    ['2022-10-25 19:00:00', '2022-10-26 07:00:00'],
    ['2022-10-26 07:00:00', '2022-10-26 19:00:00'],
    ['2022-10-26 19:00:00', '2022-10-27 07:00:00'],
    ['2022-10-27 07:00:00', '2022-10-27 19:00:00'],
    ['2022-10-27 19:00:00', '2022-10-28 07:00:00'],
    ['2022-10-28 07:00:00', '2022-10-28 19:00:00'],
    ['2022-10-28 19:00:00', '2022-10-29 07:00:00'],
    ['2022-10-29 07:00:00', '2022-10-29 19:00:00'],
    ['2022-10-29 19:00:00', '2022-10-30 07:00:00'],
    ['2022-10-30 07:00:00', '2022-10-30 19:00:00'],
    ['2022-10-30 19:00:00', '2022-10-31 07:00:00'],
    ['2022-10-31 07:00:00', '2022-10-31 19:00:00'],
    ['2022-10-31 19:00:00', '2022-11-01 07:00:00'],
    ['2022-11-01 07:00:00', '2022-11-01 19:00:00'],
    ['2022-11-01 19:00:00', '2022-11-02 07:00:00'],
    ['2022-11-02 07:00:00', '2022-11-02 19:00:00'],
    ['2022-11-02 19:00:00', '2022-11-03 07:00:00'],
    ['2022-11-03 07:00:00', '2022-11-03 19:00:00'],
    ['2022-11-03 19:00:00', '2022-11-04 07:00:00'],
    ['2022-11-04 07:00:00', '2022-11-04 19:00:00'],
    ['2022-11-04 19:00:00', '2022-11-05 07:00:00'],
    ['2022-11-05 07:00:00', '2022-11-05 19:00:00'],
    ['2022-11-05 19:00:00', '2022-11-06 07:00:00'],
    ['2022-11-06 07:00:00', '2022-11-06 19:00:00'],
    ['2022-11-06 19:00:00', '2022-11-07 07:00:00'],
    ['2022-11-07 07:00:00', '2022-11-07 19:00:00'],
    ['2022-11-07 19:00:00', '2022-11-08 07:00:00'],
    ['2022-11-08 07:00:00', '2022-11-08 19:00:00'],
    ['2022-11-08 19:00:00', '2022-11-09 07:00:00'],
    ['2022-11-09 07:00:00', '2022-11-09 19:00:00'],
    ['2022-11-09 19:00:00', '2022-11-10 07:00:00'],
    ['2022-11-10 07:00:00', '2022-11-10 19:00:00'],
    ['2022-11-10 19:00:00', '2022-11-11 07:00:00'],
    ['2022-11-11 07:00:00', '2022-11-11 19:00:00'],
    ['2022-11-11 19:00:00', '2022-11-12 07:00:00'],
    ['2022-11-12 07:00:00', '2022-11-12 19:00:00'],
    ['2022-11-12 19:00:00', '2022-11-13 07:00:00'],
    ['2022-11-13 07:00:00', '2022-11-13 19:00:00'],
    ['2022-11-13 19:00:00', '2022-11-14 07:00:00'],
    ['2022-11-14 07:00:00', '2022-11-14 19:00:00'],
    ['2022-11-14 19:00:00', '2022-11-15 07:00:00'],
    ['2022-11-15 07:00:00', '2022-11-15 19:00:00'],
    ['2022-11-15 19:00:00', '2022-11-16 07:00:00'],
    ['2022-11-16 07:00:00', '2022-11-16 19:00:00'],
    ['2022-11-16 19:00:00', '2022-11-17 07:00:00'],
    ['2022-11-17 07:00:00', '2022-11-17 19:00:00'],
    ['2022-11-19 07:00:00', '2022-11-19 19:00:00'],#here
    ['2022-11-19 19:00:00', '2022-11-20 07:00:00'],
    ['2022-11-20 07:00:00', '2022-11-20 19:00:00'],
    ['2022-11-20 19:00:00', '2022-11-21 07:00:00'],
    ['2022-11-21 07:00:00', '2022-11-21 19:00:00'],
    ['2022-11-21 19:00:00', '2022-11-22 07:00:00'],
    ['2022-11-22 07:00:00', '2022-11-22 19:00:00'],
    ['2022-11-22 19:00:00', '2022-11-23 07:00:00'],
    ['2022-11-23 07:00:00', '2022-11-23 19:00:00'],
    ['2022-11-23 19:00:00', '2022-11-24 07:00:00'],
    ['2022-11-24 07:00:00', '2022-11-24 19:00:00'],
    ['2022-11-24 19:00:00', '2022-11-25 07:00:00'],
    ['2022-11-25 07:00:00', '2022-11-25 19:00:00'],
    ['2022-11-25 19:00:00', '2022-11-26 07:00:00'],
    ['2022-11-26 07:00:00', '2022-11-26 19:00:00'],
    ['2022-11-26 19:00:00', '2022-11-27 07:00:00'],
    ['2022-11-27 07:00:00', '2022-11-27 19:00:00'],
    ['2022-11-27 19:00:00', '2022-11-28 07:00:00'],
    ['2022-11-28 07:00:00', '2022-11-28 19:00:00'],
    ['2022-11-28 19:00:00', '2022-11-29 07:00:00'],
    ['2022-11-29 07:00:00', '2022-11-29 19:00:00'],
    ['2022-11-29 19:00:00', '2022-11-30 07:00:00'],
    ['2022-11-30 07:00:00', '2022-11-30 19:00:00'],
    ['2022-11-30 19:00:00', '2022-12-01 07:00:00'],
    ['2022-12-01 07:00:00', '2022-12-01 19:00:00'],
    ['2022-12-01 19:00:00', '2022-12-02 07:00:00'],
    ['2022-12-02 07:00:00', '2022-12-02 19:00:00'],
    ['2022-12-02 19:00:00', '2022-12-03 07:00:00'],
    ['2022-12-03 07:00:00', '2022-12-03 19:00:00'],
    ['2022-12-03 19:00:00', '2022-12-04 07:00:00'],
    ['2022-12-04 07:00:00', '2022-12-04 19:00:00'],
    ['2022-12-04 19:00:00', '2022-12-05 07:00:00'],
    ['2022-12-05 07:00:00', '2022-12-05 19:00:00'],
    ['2022-12-05 19:00:00', '2022-12-06 07:00:00'],
    ['2022-12-06 07:00:00', '2022-12-06 19:00:00'],
    ['2022-12-06 19:00:00', '2022-12-07 07:00:00'],
    ['2022-12-07 07:00:00', '2022-12-07 19:00:00'],
    ['2022-12-07 19:00:00', '2022-12-08 07:00:00'],
    ['2022-12-08 07:00:00', '2022-12-08 19:00:00'],
    ['2022-12-08 19:00:00', '2022-12-09 07:00:00'],
    ['2022-12-09 07:00:00', '2022-12-09 19:00:00'],
    ['2022-12-09 19:00:00', '2022-12-10 07:00:00'],
    ['2022-12-10 07:00:00', '2022-12-10 19:00:00'],
    ['2022-12-10 19:00:00', '2022-12-11 07:00:00'],
    ['2022-12-11 07:00:00', '2022-12-11 19:00:00'],
    ['2022-12-11 19:00:00', '2022-12-12 07:00:00'],
    ['2022-12-12 07:00:00', '2022-12-12 19:00:00'],
    ['2022-12-12 19:00:00', '2022-12-13 07:00:00'],
    ['2022-12-13 07:00:00', '2022-12-13 19:00:00'],
    ['2022-12-13 19:00:00', '2022-12-14 07:00:00'],
    ['2022-12-14 07:00:00', '2022-12-14 19:00:00'],
    ['2022-12-14 19:00:00', '2022-12-15 07:00:00'],
    ['2022-12-15 07:00:00', '2022-12-15 19:00:00'],
    ['2022-12-15 19:00:00', '2022-12-16 07:00:00'],
    ['2022-12-16 07:00:00', '2022-12-16 19:00:00'],
    ['2022-12-16 19:00:00', '2022-12-17 07:00:00'],
    ['2022-12-17 07:00:00', '2022-12-17 19:00:00'],
    ['2022-12-17 19:00:00', '2022-12-18 07:00:00'],
    ['2022-12-18 07:00:00', '2022-12-18 19:00:00'],
    ['2022-12-18 19:00:00', '2022-12-19 07:00:00'],
    ['2022-12-19 07:00:00', '2022-12-19 19:00:00'],
    ['2022-12-19 19:00:00', '2022-12-20 07:00:00'],
    ['2022-12-20 07:00:00', '2022-12-20 19:00:00'],
    ['2022-12-20 19:00:00', '2022-12-21 07:00:00'],
    ['2022-12-21 07:00:00', '2022-12-21 19:00:00'],
    ['2022-12-21 19:00:00', '2022-12-22 07:00:00'],
    ['2022-12-22 07:00:00', '2022-12-22 19:00:00'],
    ['2022-12-22 19:00:00', '2022-12-23 07:00:00'],
    ['2022-12-23 07:00:00', '2022-12-23 19:00:00'],
    ['2022-12-23 19:00:00', '2022-12-24 07:00:00'],
    ['2022-12-24 07:00:00', '2022-12-24 19:00:00'],
    ['2022-12-24 19:00:00', '2022-12-25 07:00:00'],
    ['2022-12-25 07:00:00', '2022-12-25 19:00:00'],
    ['2022-12-25 19:00:00', '2022-12-26 07:00:00'],
    ['2022-12-26 07:00:00', '2022-12-26 19:00:00'],
    ['2022-12-26 19:00:00', '2022-12-27 07:00:00'],
    ['2022-12-27 07:00:00', '2022-12-27 19:00:00'],
    ['2022-12-29 07:00:00', '2022-12-29 19:00:00'],#here
    ['2022-12-29 19:00:00', '2022-12-30 07:00:00'],
    ['2022-12-30 07:00:00', '2022-12-30 19:00:00'],
    ['2022-12-30 19:00:00', '2022-12-31 07:00:00'],
    ['2022-12-31 07:00:00', '2022-12-31 19:00:00'],
    ['2022-12-31 19:00:00', '2023-01-01 07:00:00'],
    ['2023-01-01 07:00:00', '2023-01-01 19:00:00'],
    ['2023-01-01 19:00:00', '2023-01-02 07:00:00'],
    ['2023-01-02 07:00:00', '2023-01-02 19:00:00'],
    ['2023-01-02 19:00:00', '2023-01-03 07:00:00'],
    ['2023-01-03 07:00:00', '2023-01-03 19:00:00'],
    ['2023-01-03 19:00:00', '2023-01-04 07:00:00'],
    ['2023-01-04 07:00:00', '2023-01-04 19:00:00'],
    ['2023-01-04 19:00:00', '2023-01-05 07:00:00'],
    ['2023-01-05 07:00:00', '2023-01-05 19:00:00'],
    ['2023-01-05 19:00:00', '2023-01-06 07:00:00'],
    ['2023-01-06 07:00:00', '2023-01-06 19:00:00'],
    ['2023-01-06 19:00:00', '2023-01-07 07:00:00'],
    ['2023-01-07 07:00:00', '2023-01-07 19:00:00'],
    ['2023-01-07 19:00:00', '2023-01-08 07:00:00'],
    ['2023-01-08 07:00:00', '2023-01-08 19:00:00'],
    ['2023-01-08 19:00:00', '2023-01-09 07:00:00'],
    ['2023-01-09 07:00:00', '2023-01-09 19:00:00'],
    ['2023-01-09 19:00:00', '2023-01-10 07:00:00'],
    ['2023-01-10 07:00:00', '2023-01-10 19:00:00'],
    ['2023-01-10 19:00:00', '2023-01-11 07:00:00'],
    ['2023-01-11 07:00:00', '2023-01-11 19:00:00'],
    ['2023-01-11 19:00:00', '2023-01-12 07:00:00'],
    ['2023-01-12 07:00:00', '2023-01-12 19:00:00'],
    ['2023-01-12 19:00:00', '2023-01-13 07:00:00'],
    ['2023-01-13 07:00:00', '2023-01-13 19:00:00'],
    ['2023-01-13 19:00:00', '2023-01-14 07:00:00'],
    ['2023-01-14 07:00:00', '2023-01-14 19:00:00'],
    ['2023-01-14 19:00:00', '2023-01-15 07:00:00'],
    ['2023-01-15 07:00:00', '2023-01-15 19:00:00'],
    ['2023-01-15 19:00:00', '2023-01-16 07:00:00'],
    ['2023-01-16 07:00:00', '2023-01-16 19:00:00'],
    ['2023-01-16 19:00:00', '2023-01-17 07:00:00'],
    ['2023-01-17 07:00:00', '2023-01-17 19:00:00'],
    ['2023-01-19 07:00:00', '2023-01-19 19:00:00'],#here
    ['2023-01-19 19:00:00', '2023-01-20 07:00:00'],
    ['2023-01-20 07:00:00', '2023-01-20 19:00:00'],
    ['2023-01-20 19:00:00', '2023-01-21 07:00:00'],
    ['2023-01-21 07:00:00', '2023-01-21 19:00:00'],
    ['2023-01-21 19:00:00', '2023-01-22 07:00:00'],
    ['2023-01-22 07:00:00', '2023-01-22 19:00:00'],
    ['2023-01-22 19:00:00', '2023-01-23 07:00:00'],
    ['2023-01-23 07:00:00', '2023-01-23 19:00:00'],
    ['2023-01-23 19:00:00', '2023-01-24 07:00:00'],
    ['2023-01-24 07:00:00', '2023-01-24 19:00:00'],
    ['2023-01-24 19:00:00', '2023-01-25 07:00:00'],
    ['2023-01-25 07:00:00', '2023-01-25 19:00:00'],
    ['2023-01-25 19:00:00', '2023-01-26 07:00:00'],
    ['2023-01-26 07:00:00', '2023-01-26 19:00:00'],
    ['2023-01-26 19:00:00', '2023-01-27 07:00:00'],
    ['2023-01-27 07:00:00', '2023-01-27 19:00:00'],
    ['2023-01-27 19:00:00', '2023-01-28 07:00:00'],
    ['2023-01-28 07:00:00', '2023-01-28 19:00:00'],
    ['2023-01-28 19:00:00', '2023-01-29 07:00:00'],
    ['2023-01-29 07:00:00', '2023-01-29 19:00:00'],
    ['2023-01-29 19:00:00', '2023-01-30 07:00:00'],
    ['2023-01-30 07:00:00', '2023-01-30 19:00:00'],
    ['2023-01-30 19:00:00', '2023-01-31 07:00:00'],
    ['2023-01-31 07:00:00', '2023-01-31 19:00:00'],
    ['2023-01-31 19:00:00', '2023-02-01 07:00:00'],
    ['2023-02-01 07:00:00', '2023-02-01 19:00:00'],
    ['2023-02-01 19:00:00', '2023-02-02 07:00:00'],
    ['2023-02-02 07:00:00', '2023-02-02 19:00:00'],
    ['2023-02-02 19:00:00', '2023-02-03 07:00:00'],
    ['2023-02-03 07:00:00', '2023-02-03 19:00:00'],
    ['2023-02-03 19:00:00', '2023-02-04 07:00:00'],
    ['2023-02-04 07:00:00', '2023-02-04 19:00:00'],
    ['2023-02-04 19:00:00', '2023-02-05 07:00:00'],
    ['2023-02-05 07:00:00', '2023-02-05 19:00:00'],
    ['2023-02-05 19:00:00', '2023-02-06 07:00:00'],
    ['2023-02-06 07:00:00', '2023-02-06 19:00:00'],
    ['2023-02-06 19:00:00', '2023-02-07 07:00:00'],
    ['2023-02-07 07:00:00', '2023-02-07 19:00:00'],
    ['2023-02-07 19:00:00', '2023-02-08 07:00:00'],
    ['2023-02-08 07:00:00', '2023-02-08 19:00:00'],
    ['2023-02-08 19:00:00', '2023-02-09 07:00:00'],
    ['2023-02-09 07:00:00', '2023-02-09 19:00:00'],
    ['2023-02-09 19:00:00', '2023-02-10 07:00:00'],
    ['2023-02-10 07:00:00', '2023-02-10 19:00:00'],
    ['2023-02-10 19:00:00', '2023-02-11 07:00:00'],
    ['2023-02-11 07:00:00', '2023-02-11 19:00:00'],
    ['2023-02-11 19:00:00', '2023-02-12 07:00:00'],
    ['2023-02-12 07:00:00', '2023-02-12 19:00:00'],
    ['2023-02-12 19:00:00', '2023-02-13 07:00:00'],
    ['2023-02-13 07:00:00', '2023-02-13 19:00:00'],
    ['2023-02-13 19:00:00', '2023-02-14 07:00:00'],
    ['2023-02-14 07:00:00', '2023-02-14 19:00:00'],
    ['2023-02-14 19:00:00', '2023-02-15 07:00:00'],
    ['2023-02-15 07:00:00', '2023-02-15 19:00:00'],
    ['2023-02-15 19:00:00', '2023-02-16 07:00:00'],
    ['2023-02-16 07:00:00', '2023-02-16 19:00:00'],
    ['2023-02-16 19:00:00', '2023-02-17 07:00:00'],
    ['2023-02-17 07:00:00', '2023-02-17 19:00:00'],
    ['2023-02-17 19:00:00', '2023-02-18 07:00:00'],
    ['2023-02-18 07:00:00', '2023-02-18 19:00:00'],
    ['2023-02-18 19:00:00', '2023-02-19 07:00:00'],
    ['2023-02-19 07:00:00', '2023-02-19 19:00:00'],
    ['2023-02-19 19:00:00', '2023-02-20 07:00:00'],
    ['2023-02-20 07:00:00', '2023-02-20 19:00:00'],
    ['2023-02-20 19:00:00', '2023-02-21 07:00:00'],
    ['2023-02-21 07:00:00', '2023-02-21 19:00:00'],
    ['2023-02-21 19:00:00', '2023-02-22 07:00:00'],
    ['2023-02-22 07:00:00', '2023-02-22 19:00:00'],
    ['2023-02-22 19:00:00', '2023-02-23 07:00:00'],
    ['2023-02-23 07:00:00', '2023-02-23 19:00:00'],
    ['2023-02-23 19:00:00', '2023-02-24 07:00:00'],
    ['2023-02-24 07:00:00', '2023-02-24 19:00:00'],
    ['2023-02-24 19:00:00', '2023-02-25 07:00:00'],
    ['2023-02-25 07:00:00', '2023-02-25 19:00:00'],
    ['2023-02-25 19:00:00', '2023-02-26 07:00:00'],
    ['2023-02-26 07:00:00', '2023-02-26 19:00:00'],
    ['2023-02-26 19:00:00', '2023-02-27 07:00:00'],
    ['2023-02-27 07:00:00', '2023-02-27 19:00:00'],
    ['2023-02-27 19:00:00', '2023-02-28 07:00:00'],
    ['2023-02-28 07:00:00', '2023-02-28 19:00:00'],
    ['2023-02-28 19:00:00', '2023-03-01 07:00:00'],
    ['2023-03-01 07:00:00', '2023-03-01 19:00:00'],
    ['2023-03-01 19:00:00', '2023-03-02 07:00:00'],
    ['2023-03-02 07:00:00', '2023-03-02 19:00:00'],
    ['2023-03-02 19:00:00', '2023-03-03 07:00:00'],
    ['2023-03-03 07:00:00', '2023-03-03 19:00:00'],
    ['2023-03-03 19:00:00', '2023-03-04 07:00:00'],
    ['2023-03-04 07:00:00', '2023-03-04 19:00:00'],#here!!
    ['2023-04-19 07:00:00', '2023-04-19 19:00:00'],
    ['2023-04-19 19:00:00', '2023-04-20 07:00:00'],
    ['2023-04-20 07:00:00', '2023-04-20 19:00:00'],
    ['2023-04-20 19:00:00', '2023-04-21 07:00:00'],
    ['2023-04-21 07:00:00', '2023-04-21 19:00:00'],
    ['2023-04-21 19:00:00', '2023-04-22 07:00:00'],
    ['2023-04-22 07:00:00', '2023-04-22 19:00:00'],
    ['2023-04-22 19:00:00', '2023-04-23 07:00:00'],
    ['2023-04-23 07:00:00', '2023-04-23 19:00:00'],
    ['2023-04-23 19:00:00', '2023-04-24 07:00:00'],
    ['2023-04-24 07:00:00', '2023-04-24 19:00:00'],
    ['2023-04-24 19:00:00', '2023-04-25 07:00:00'],
    ['2023-04-25 07:00:00', '2023-04-25 19:00:00'],
    ['2023-04-25 19:00:00', '2023-04-26 07:00:00'],
    ['2023-04-26 07:00:00', '2023-04-26 19:00:00'],
    ['2023-04-26 19:00:00', '2023-04-27 07:00:00'],
    ['2023-04-27 07:00:00', '2023-04-27 19:00:00'],
    ['2023-04-27 19:00:00', '2023-04-28 07:00:00'],
    ['2023-04-28 07:00:00', '2023-04-28 19:00:00'],
    ['2023-04-28 19:00:00', '2023-04-29 07:00:00'],
    ['2023-04-29 07:00:00', '2023-04-29 19:00:00'],
    ['2023-04-29 19:00:00', '2023-04-30 07:00:00'],
    ['2023-04-30 07:00:00', '2023-04-30 19:00:00'],
    ['2023-04-30 19:00:00', '2023-05-01 07:00:00'],
    ['2023-05-01 07:00:00', '2023-05-01 19:00:00'],
    ['2023-05-01 19:00:00', '2023-05-02 07:00:00'],
    ['2023-05-02 07:00:00', '2023-05-02 19:00:00'],
    ['2023-05-02 19:00:00', '2023-05-03 07:00:00'],
    ['2023-05-03 07:00:00', '2023-05-03 19:00:00'],
    ['2023-05-03 19:00:00', '2023-05-04 07:00:00'],
    ['2023-05-04 07:00:00', '2023-05-04 19:00:00'],
    ['2023-05-04 19:00:00', '2023-05-05 07:00:00'],
    ['2023-05-05 07:00:00', '2023-05-05 19:00:00'],
    ['2023-05-05 19:00:00', '2023-05-06 07:00:00'],
    ['2023-05-06 07:00:00', '2023-05-06 19:00:00'],
    ['2023-05-06 19:00:00', '2023-05-07 07:00:00'],
    ['2023-05-07 07:00:00', '2023-05-07 19:00:00'],
    ['2023-05-07 19:00:00', '2023-05-08 07:00:00'],
    ['2023-05-08 07:00:00', '2023-05-08 19:00:00'],
    ['2023-05-08 19:00:00', '2023-05-09 07:00:00'],
    ['2023-05-09 07:00:00', '2023-05-09 19:00:00'],
    ['2023-05-09 19:00:00', '2023-05-10 07:00:00'],
    ['2023-05-10 07:00:00', '2023-05-10 19:00:00'],
    ['2023-05-10 19:00:00', '2023-05-11 07:00:00'],
    ['2023-05-11 07:00:00', '2023-05-11 19:00:00'],
    ['2023-05-11 19:00:00', '2023-05-12 07:00:00'],
    ['2023-05-12 07:00:00', '2023-05-12 19:00:00'],
    ['2023-05-12 19:00:00', '2023-05-13 07:00:00'],
    ['2023-05-13 07:00:00', '2023-05-13 19:00:00'],
    ['2023-05-13 19:00:00', '2023-05-14 07:00:00'],
    ['2023-05-14 07:00:00', '2023-05-14 19:00:00'],
    ['2023-05-14 19:00:00', '2023-05-15 07:00:00'],
    ['2023-05-15 07:00:00', '2023-05-15 19:00:00'],
    ['2023-05-15 19:00:00', '2023-05-16 07:00:00'],#here
    ['2023-06-29 19:00:00', '2023-06-30 07:00:00'],
    ['2023-06-30 07:00:00', '2023-06-30 19:00:00'],
    ['2023-06-30 19:00:00', '2023-07-01 07:00:00'],
    ['2023-07-01 07:00:00', '2023-07-01 19:00:00'],
    ['2023-07-01 19:00:00', '2023-07-02 07:00:00'],
    ['2023-07-02 07:00:00', '2023-07-02 19:00:00'],
    ['2023-07-02 19:00:00', '2023-07-03 07:00:00'],
    ['2023-07-03 07:00:00', '2023-07-03 19:00:00'],
    ['2023-07-03 19:00:00', '2023-07-04 07:00:00'],
    ['2023-07-04 07:00:00', '2023-07-04 19:00:00'],
    ['2023-07-04 19:00:00', '2023-07-05 07:00:00'],
    ['2023-07-05 07:00:00', '2023-07-05 19:00:00'],
    ['2023-07-05 19:00:00', '2023-07-06 07:00:00'],
    ['2023-07-06 07:00:00', '2023-07-06 19:00:00'],
    ['2023-07-06 19:00:00', '2023-07-07 07:00:00'],
    ['2023-07-07 07:00:00', '2023-07-07 19:00:00'],
    ['2023-07-07 19:00:00', '2023-07-08 07:00:00'],
    ['2023-07-08 07:00:00', '2023-07-08 19:00:00'],
    ['2023-07-08 19:00:00', '2023-07-09 07:00:00'],
    ['2023-07-09 07:00:00', '2023-07-09 19:00:00'],
    ['2023-07-09 19:00:00', '2023-07-10 07:00:00'],
    ['2023-07-10 07:00:00', '2023-07-10 19:00:00'],
    ['2023-07-10 19:00:00', '2023-07-11 07:00:00'],
    ['2023-07-11 07:00:00', '2023-07-11 19:00:00'],
    ['2023-07-11 19:00:00', '2023-07-12 07:00:00'],
    ['2023-07-12 07:00:00', '2023-07-12 19:00:00'],
    ['2023-07-12 19:00:00', '2023-07-13 07:00:00'],
    ['2023-07-13 07:00:00', '2023-07-13 19:00:00'],
    ['2023-07-13 19:00:00', '2023-07-14 07:00:00'],
    ['2023-07-14 07:00:00', '2023-07-14 19:00:00'],
    ['2023-07-14 19:00:00', '2023-07-15 07:00:00'],
    ['2023-07-15 07:00:00', '2023-07-15 19:00:00'],
    ['2023-07-15 19:00:00', '2023-07-16 07:00:00'],
    ['2023-07-16 07:00:00', '2023-07-16 19:00:00'],#here
    ['2023-07-18 07:00:00', '2023-07-18 19:00:00'],
    ['2023-07-18 19:00:00', '2023-07-19 07:00:00'],
    ['2023-07-19 07:00:00', '2023-07-19 19:00:00'],
    ['2023-07-19 19:00:00', '2023-07-20 07:00:00'],
    ['2023-07-20 07:00:00', '2023-07-20 19:00:00'],
    ['2023-07-20 19:00:00', '2023-07-21 07:00:00'],
    ['2023-07-21 07:00:00', '2023-07-21 19:00:00'],
    ['2023-07-21 19:00:00', '2023-07-22 07:00:00'],
    ['2023-07-22 07:00:00', '2023-07-22 19:00:00'],
    ['2023-07-22 19:00:00', '2023-07-23 07:00:00'],
    ['2023-07-23 07:00:00', '2023-07-23 19:00:00'],
    ['2023-07-23 19:00:00', '2023-07-24 07:00:00'],
    ['2023-07-24 07:00:00', '2023-07-24 19:00:00'],
    ['2023-07-24 19:00:00', '2023-07-25 07:00:00'],
    ['2023-07-25 07:00:00', '2023-07-25 19:00:00'],
    ['2023-07-25 19:00:00', '2023-07-26 07:00:00'],
    ['2023-07-26 07:00:00', '2023-07-26 19:00:00'],
    ['2023-07-26 19:00:00', '2023-07-27 07:00:00'],
    ['2023-07-27 07:00:00', '2023-07-27 19:00:00'],
    ['2023-07-27 19:00:00', '2023-07-28 07:00:00'],#here
    ['2023-08-22 19:00:00', '2023-08-23 07:00:00'],
    ['2023-08-23 07:00:00', '2023-08-23 19:00:00'],
    ['2023-08-23 19:00:00', '2023-08-24 07:00:00'],
    ['2023-08-24 07:00:00', '2023-08-24 19:00:00'],
    ['2023-08-24 19:00:00', '2023-08-25 07:00:00'],
    ['2023-08-25 07:00:00', '2023-08-25 19:00:00'],
    ['2023-08-25 19:00:00', '2023-08-26 07:00:00'],
    ['2023-08-26 07:00:00', '2023-08-26 19:00:00'],
    ['2023-08-26 19:00:00', '2023-08-27 07:00:00'],
    ['2023-08-27 07:00:00', '2023-08-27 19:00:00'],
    ['2023-08-27 19:00:00', '2023-08-28 07:00:00'],
    ['2023-08-28 07:00:00', '2023-08-28 19:00:00'],
    ['2023-08-28 19:00:00', '2023-08-29 07:00:00'],
    ['2023-08-29 07:00:00', '2023-08-29 19:00:00'],
    ['2023-08-29 19:00:00', '2023-08-30 07:00:00'],
    ['2023-08-30 07:00:00', '2023-08-30 19:00:00'],
    ['2023-08-30 19:00:00', '2023-08-31 07:00:00'],
    ['2023-08-31 07:00:00', '2023-08-31 19:00:00'],#here
    ['2023-09-02 07:00:00', '2023-09-02 19:00:00'],
    ['2023-09-02 19:00:00', '2023-09-03 07:00:00'],
    ['2023-09-03 07:00:00', '2023-09-03 19:00:00'],
    ['2023-09-03 19:00:00', '2023-09-04 07:00:00'],
    ['2023-09-04 07:00:00', '2023-09-04 19:00:00'],
    ['2023-09-04 19:00:00', '2023-09-05 07:00:00'],
    ['2023-09-05 07:00:00', '2023-09-05 19:00:00'],
    ['2023-09-05 19:00:00', '2023-09-06 07:00:00'],
    ['2023-09-06 07:00:00', '2023-09-06 19:00:00'],
    ['2023-09-06 19:00:00', '2023-09-07 07:00:00'],
    ['2023-09-07 07:00:00', '2023-09-07 19:00:00'],
    ['2023-09-07 19:00:00', '2023-09-08 07:00:00'],
    ['2023-09-08 07:00:00', '2023-09-08 19:00:00'],
    ['2023-09-08 19:00:00', '2023-09-09 07:00:00'],
    ['2023-09-09 07:00:00', '2023-09-09 19:00:00'],
    ['2023-09-09 19:00:00', '2023-09-10 07:00:00'],
    ['2023-09-10 07:00:00', '2023-09-10 19:00:00'],
    ['2023-09-10 19:00:00', '2023-09-11 07:00:00'],
    ['2023-09-11 07:00:00', '2023-09-11 19:00:00'],
    ['2023-09-11 19:00:00', '2023-09-12 07:00:00'],
    ['2023-09-12 07:00:00', '2023-09-12 19:00:00'],
    ['2023-09-12 19:00:00', '2023-09-13 07:00:00'],
    ['2023-09-13 07:00:00', '2023-09-13 19:00:00'],
    ['2023-09-13 19:00:00', '2023-09-14 07:00:00'],
    ['2023-09-14 07:00:00', '2023-09-14 19:00:00'],
    ['2023-09-14 19:00:00', '2023-09-15 07:00:00'],
    ['2023-09-15 07:00:00', '2023-09-15 19:00:00'],
    ['2023-09-15 19:00:00', '2023-09-16 07:00:00'],
    ['2023-09-16 07:00:00', '2023-09-16 19:00:00'],
    ['2023-09-16 19:00:00', '2023-09-17 07:00:00'],
    ['2023-09-17 07:00:00', '2023-09-17 19:00:00'],#here
    ['2023-09-19 07:00:00', '2023-09-19 19:00:00'],
    ['2023-09-19 19:00:00', '2023-09-20 07:00:00'],
    ['2023-09-20 07:00:00', '2023-09-20 19:00:00'],
    ['2023-09-20 19:00:00', '2023-09-21 07:00:00'],
    ['2023-09-21 07:00:00', '2023-09-21 19:00:00'],
    ['2023-09-21 19:00:00', '2023-09-22 07:00:00'],
    ['2023-09-22 07:00:00', '2023-09-22 19:00:00'],
    ['2023-09-22 19:00:00', '2023-09-23 07:00:00'],
    ['2023-09-23 07:00:00', '2023-09-23 19:00:00'],
    ['2023-09-23 19:00:00', '2023-09-24 07:00:00'],
    ['2023-09-24 07:00:00', '2023-09-24 19:00:00'],
    ['2023-09-24 19:00:00', '2023-09-25 07:00:00'],
    ['2023-09-25 07:00:00', '2023-09-25 19:00:00'],
    ['2023-09-25 19:00:00', '2023-09-26 07:00:00'],
    ['2023-09-26 07:00:00', '2023-09-26 19:00:00'],
    ['2023-09-26 19:00:00', '2023-09-27 07:00:00'],
    ['2023-09-27 07:00:00', '2023-09-27 19:00:00'],
    ['2023-09-27 19:00:00', '2023-09-28 07:00:00'],
    ['2023-09-28 07:00:00', '2023-09-28 19:00:00'],
    ['2023-09-28 19:00:00', '2023-09-29 07:00:00'],
    ['2023-09-29 07:00:00', '2023-09-29 19:00:00'],
    ['2023-09-29 19:00:00', '2023-09-30 07:00:00'],
    ['2023-09-30 07:00:00', '2023-09-30 19:00:00'],
    ['2023-09-30 19:00:00', '2023-10-01 07:00:00'],
    ['2023-10-01 07:00:00', '2023-10-01 19:00:00'],
    ['2023-10-01 19:00:00', '2023-10-02 07:00:00'],
    ['2023-10-02 07:00:00', '2023-10-02 19:00:00'],
    ['2023-10-02 19:00:00', '2023-10-03 07:00:00'],
    ['2023-10-03 07:00:00', '2023-10-03 19:00:00'],
    ['2023-10-03 19:00:00', '2023-10-04 07:00:00'],
    ['2023-10-04 07:00:00', '2023-10-04 19:00:00'],
    ['2023-10-04 19:00:00', '2023-10-05 07:00:00'],
    ['2023-10-05 07:00:00', '2023-10-05 19:00:00'],
    ['2023-10-05 19:00:00', '2023-10-06 07:00:00'],
    ['2023-10-06 07:00:00', '2023-10-06 19:00:00'],
    ['2023-10-06 19:00:00', '2023-10-07 07:00:00'],
    ['2023-10-07 07:00:00', '2023-10-07 19:00:00'],
    ['2023-10-07 19:00:00', '2023-10-08 07:00:00'],
    ['2023-10-08 07:00:00', '2023-10-08 19:00:00'],
    ['2023-10-08 19:00:00', '2023-10-09 07:00:00'],
    ['2023-10-09 07:00:00', '2023-10-09 19:00:00'],
    ['2023-10-09 19:00:00', '2023-10-10 07:00:00'],
    ['2023-10-10 07:00:00', '2023-10-10 19:00:00'],
    ['2023-10-10 19:00:00', '2023-10-11 07:00:00'],
    ['2023-10-11 07:00:00', '2023-10-11 19:00:00'],
    ['2023-10-11 19:00:00', '2023-10-12 07:00:00'],
    ['2023-10-12 07:00:00', '2023-10-12 19:00:00'],
    ['2023-10-12 19:00:00', '2023-10-13 07:00:00'],
    ['2023-10-13 07:00:00', '2023-10-13 19:00:00'],
    ['2023-10-13 19:00:00', '2023-10-14 07:00:00'],
    ['2023-10-14 07:00:00', '2023-10-14 19:00:00'],
    ['2023-10-14 19:00:00', '2023-10-15 07:00:00'],
    ['2023-10-15 07:00:00', '2023-10-15 19:00:00'],
    ['2023-10-15 19:00:00', '2023-10-16 07:00:00'],
    ['2023-10-16 07:00:00', '2023-10-16 19:00:00'],#here
    ['2023-10-18 07:00:00', '2023-10-18 19:00:00'],
    ['2023-10-18 19:00:00', '2023-10-19 07:00:00'],
    ['2023-10-19 07:00:00', '2023-10-19 19:00:00'],
    ['2023-10-19 19:00:00', '2023-10-20 07:00:00'],
    ['2023-10-20 07:00:00', '2023-10-20 19:00:00'],
    ['2023-10-20 19:00:00', '2023-10-21 07:00:00'],
    ['2023-10-21 07:00:00', '2023-10-21 19:00:00'],
    ['2023-10-21 19:00:00', '2023-10-22 07:00:00'],
    ['2023-10-22 07:00:00', '2023-10-22 19:00:00'],
    ['2023-10-22 19:00:00', '2023-10-23 07:00:00'],
    ['2023-10-23 07:00:00', '2023-10-23 19:00:00'],
    ['2023-10-23 19:00:00', '2023-10-24 07:00:00'],
    ['2023-10-24 07:00:00', '2023-10-24 19:00:00'],
    ['2023-10-24 19:00:00', '2023-10-25 07:00:00'],
    ['2023-10-25 07:00:00', '2023-10-25 19:00:00'],
    ['2023-10-25 19:00:00', '2023-10-26 07:00:00'],
    ['2023-10-26 07:00:00', '2023-10-26 19:00:00'],
    ['2023-10-26 19:00:00', '2023-10-27 07:00:00'],
    ['2023-10-27 07:00:00', '2023-10-27 19:00:00'],
    ['2023-10-27 19:00:00', '2023-10-28 07:00:00'],
    ['2023-10-28 07:00:00', '2023-10-28 19:00:00'],
    ['2023-10-28 19:00:00', '2023-10-29 07:00:00'],
    ['2023-10-29 07:00:00', '2023-10-29 19:00:00'],#here
    ['2023-10-31 07:00:00', '2023-10-31 19:00:00'],
    ['2023-10-31 19:00:00', '2023-11-01 07:00:00'],
    ['2023-11-01 07:00:00', '2023-11-01 19:00:00'],
    ['2023-11-01 19:00:00', '2023-11-02 07:00:00'],
    ['2023-11-02 07:00:00', '2023-11-02 19:00:00'],
    ['2023-11-02 19:00:00', '2023-11-03 07:00:00'],
    ['2023-11-03 07:00:00', '2023-11-03 19:00:00'],
    ['2023-11-03 19:00:00', '2023-11-04 07:00:00'],
    ['2023-11-04 07:00:00', '2023-11-04 19:00:00'],
    ['2023-11-04 19:00:00', '2023-11-05 07:00:00'],
    ['2023-11-05 07:00:00', '2023-11-05 19:00:00'],
    ['2023-11-05 19:00:00', '2023-11-06 07:00:00'],
    ['2023-11-06 07:00:00', '2023-11-06 19:00:00'],
    ['2023-11-06 19:00:00', '2023-11-07 07:00:00'],
    ['2023-11-07 07:00:00', '2023-11-07 19:00:00'],
    ['2023-11-07 19:00:00', '2023-11-08 07:00:00'],
    ['2023-11-08 07:00:00', '2023-11-08 19:00:00'],
    ['2023-11-08 19:00:00', '2023-11-09 07:00:00'],
    ['2023-11-09 07:00:00', '2023-11-09 19:00:00'],
    ['2023-11-09 19:00:00', '2023-11-10 07:00:00'],
    ['2023-11-10 07:00:00', '2023-11-10 19:00:00'],
    ['2023-11-10 19:00:00', '2023-11-11 07:00:00'],
    ['2023-11-11 07:00:00', '2023-11-11 19:00:00'],
    ['2023-11-11 19:00:00', '2023-11-12 07:00:00'],
    ['2023-11-12 07:00:00', '2023-11-12 19:00:00'],
    ['2023-11-12 19:00:00', '2023-11-13 07:00:00'],
    ['2023-11-13 07:00:00', '2023-11-13 19:00:00'],
    ['2023-11-13 19:00:00', '2023-11-14 07:00:00'],
    ['2023-11-14 07:00:00', '2023-11-14 19:00:00'],
    ['2023-11-14 19:00:00', '2023-11-15 07:00:00'],
    ['2023-11-15 07:00:00', '2023-11-15 19:00:00'],
    ['2023-11-15 19:00:00', '2023-11-16 07:00:00'],
    ['2023-11-16 07:00:00', '2023-11-16 19:00:00'],
    ['2023-11-16 19:00:00', '2023-11-17 07:00:00'],
    ['2023-11-17 07:00:00', '2023-11-17 19:00:00'],
    ['2023-11-17 19:00:00', '2023-11-18 07:00:00'],
    ['2023-11-18 07:00:00', '2023-11-18 19:00:00'],
    ['2023-11-18 19:00:00', '2023-11-19 07:00:00'],
    ['2023-11-19 07:00:00', '2023-11-19 19:00:00'],
    ['2023-11-19 19:00:00', '2023-11-20 07:00:00'],
    ['2023-11-20 07:00:00', '2023-11-20 19:00:00'],#here
    ['2023-11-22 07:00:00', '2023-11-22 19:00:00'],
    ['2023-11-22 19:00:00', '2023-11-23 07:00:00'],
    ['2023-11-23 07:00:00', '2023-11-23 19:00:00'],
    ['2023-11-23 19:00:00', '2023-11-24 07:00:00'],
    ['2023-11-24 07:00:00', '2023-11-24 19:00:00'],
    ['2023-11-24 19:00:00', '2023-11-25 07:00:00'],
    ['2023-11-25 07:00:00', '2023-11-25 19:00:00'],
    ['2023-11-25 19:00:00', '2023-11-26 07:00:00'],
    ['2023-11-26 07:00:00', '2023-11-26 19:00:00'],
    ['2023-11-26 19:00:00', '2023-11-27 07:00:00'],
    ['2023-11-27 07:00:00', '2023-11-27 19:00:00'],
    ['2023-11-27 19:00:00', '2023-11-28 07:00:00'],
    ['2023-11-28 07:00:00', '2023-11-28 19:00:00'],
    ['2023-11-28 19:00:00', '2023-11-29 07:00:00'],
    ['2023-11-29 07:00:00', '2023-11-29 19:00:00'],
    ['2023-11-29 19:00:00', '2023-11-30 07:00:00'],
    ['2023-11-30 07:00:00', '2023-11-30 19:00:00'],
    ['2023-11-30 19:00:00', '2023-12-01 07:00:00'],
    ['2023-12-01 07:00:00', '2023-12-01 19:00:00'],
    ['2023-12-01 19:00:00', '2023-12-02 07:00:00'],
    ['2023-12-02 07:00:00', '2023-12-02 19:00:00'],
    ['2023-12-02 19:00:00', '2023-12-03 07:00:00'],
    ['2023-12-03 07:00:00', '2023-12-03 19:00:00'],
    ['2023-12-03 19:00:00', '2023-12-04 07:00:00'],
    ['2023-12-04 07:00:00', '2023-12-04 19:00:00'],
    ['2023-12-04 19:00:00', '2023-12-05 07:00:00'],
    ['2023-12-05 07:00:00', '2023-12-05 19:00:00'],
    ['2023-12-05 19:00:00', '2023-12-06 07:00:00'],
    ['2023-12-06 07:00:00', '2023-12-06 19:00:00'],
    ['2023-12-06 19:00:00', '2023-12-07 07:00:00'],
    ['2023-12-07 07:00:00', '2023-12-07 19:00:00'],
    ['2023-12-07 19:00:00', '2023-12-08 07:00:00'],
    ['2023-12-08 07:00:00', '2023-12-08 19:00:00'],
    ['2023-12-08 19:00:00', '2023-12-09 07:00:00'],
    ['2023-12-09 07:00:00', '2023-12-09 19:00:00'],
    ['2023-12-09 19:00:00', '2023-12-10 07:00:00'],
    ['2023-12-10 07:00:00', '2023-12-10 19:00:00'],
    ['2023-12-10 19:00:00', '2023-12-11 07:00:00'],
    ['2023-12-11 07:00:00', '2023-12-11 19:00:00'],
    ['2023-12-11 19:00:00', '2023-12-12 07:00:00'],
    ['2023-12-12 07:00:00', '2023-12-12 19:00:00'],
    ['2023-12-12 19:00:00', '2023-12-13 07:00:00'],
    ['2023-12-13 07:00:00', '2023-12-13 19:00:00'],
    ['2023-12-13 19:00:00', '2023-12-14 07:00:00'],
    ['2023-12-14 07:00:00', '2023-12-14 19:00:00'],#here
    ['2023-12-16 07:00:00', '2023-12-16 19:00:00'],
    ['2023-12-16 19:00:00', '2023-12-17 07:00:00'],
    ['2023-12-17 07:00:00', '2023-12-17 19:00:00'],#here
    ['2023-12-19 07:00:00', '2023-12-19 19:00:00'],
    ['2023-12-19 19:00:00', '2023-12-20 07:00:00'],
    ['2023-12-20 07:00:00', '2023-12-20 19:00:00'],
    ['2023-12-20 19:00:00', '2023-12-21 07:00:00'],
    ['2023-12-21 07:00:00', '2023-12-21 19:00:00'],
    ['2023-12-21 19:00:00', '2023-12-22 07:00:00'],
    ['2023-12-22 07:00:00', '2023-12-22 19:00:00'],
    ['2023-12-22 19:00:00', '2023-12-23 07:00:00'],
    ['2023-12-23 07:00:00', '2023-12-23 19:00:00'],
    ['2023-12-23 19:00:00', '2023-12-24 07:00:00'],
    ['2023-12-24 07:00:00', '2023-12-24 19:00:00'],
    ['2023-12-24 19:00:00', '2023-12-25 07:00:00'],
    ['2023-12-25 07:00:00', '2023-12-25 19:00:00'],
    ['2023-12-25 19:00:00', '2023-12-26 07:00:00'],
    ['2023-12-26 07:00:00', '2023-12-26 19:00:00'],
    ['2023-12-26 19:00:00', '2023-12-27 07:00:00'],
    ['2023-12-27 07:00:00', '2023-12-27 19:00:00'],
    ['2023-12-27 19:00:00', '2023-12-28 07:00:00'],
    ['2023-12-28 07:00:00', '2023-12-28 19:00:00'],
    ['2023-12-28 19:00:00', '2023-12-29 07:00:00'],
    ['2023-12-29 07:00:00', '2023-12-29 19:00:00'],
    ['2023-12-29 19:00:00', '2023-12-30 07:00:00'],
    ['2023-12-30 07:00:00', '2023-12-30 19:00:00'],
    ['2023-12-30 19:00:00', '2023-12-31 07:00:00'],
    ['2023-12-31 07:00:00', '2023-12-31 19:00:00'],
    ['2023-12-31 19:00:00', '2024-01-01 07:00:00'],
    ['2024-01-01 07:00:00', '2024-01-01 19:00:00'],
    ['2024-01-01 19:00:00', '2024-01-02 07:00:00'],
    ['2024-01-02 07:00:00', '2024-01-02 19:00:00'],
    ['2024-01-02 19:00:00', '2024-01-03 07:00:00'],
    ['2024-01-03 07:00:00', '2024-01-03 19:00:00'],
    ['2024-01-03 19:00:00', '2024-01-04 07:00:00'],
    ['2024-01-04 07:00:00', '2024-01-04 19:00:00'],
    ['2024-01-04 19:00:00', '2024-01-05 07:00:00'],
    ['2024-01-05 07:00:00', '2024-01-05 19:00:00'],
    ['2024-01-05 19:00:00', '2024-01-06 07:00:00'],
    ['2024-01-06 07:00:00', '2024-01-06 19:00:00'],
    ['2024-01-06 19:00:00', '2024-01-07 07:00:00'],
    ['2024-01-07 07:00:00', '2024-01-07 19:00:00'],
    ['2024-01-07 19:00:00', '2024-01-08 07:00:00'],
    ['2024-01-08 07:00:00', '2024-01-08 19:00:00'],
    ['2024-01-08 19:00:00', '2024-01-09 07:00:00'],
    ['2024-01-09 07:00:00', '2024-01-09 19:00:00'],
    ['2024-01-09 19:00:00', '2024-01-10 07:00:00'],
    ['2024-01-10 07:00:00', '2024-01-10 19:00:00'],
    ['2024-01-10 19:00:00', '2024-01-11 07:00:00'],
    ['2024-01-11 07:00:00', '2024-01-11 19:00:00'],
    ['2024-01-11 19:00:00', '2024-01-12 07:00:00'],
    ['2024-01-12 07:00:00', '2024-01-12 19:00:00'],
    ['2024-01-12 19:00:00', '2024-01-13 07:00:00'],
    ['2024-01-13 07:00:00', '2024-01-13 19:00:00'],
    ['2024-01-13 19:00:00', '2024-01-14 07:00:00'],
    ['2024-01-14 07:00:00', '2024-01-14 19:00:00'],#here
    ['2024-01-16 07:00:00', '2024-01-16 19:00:00'],
    ['2024-01-16 19:00:00', '2024-01-17 07:00:00'],
    ['2024-01-17 07:00:00', '2024-01-17 19:00:00'],
    ['2024-01-17 19:00:00', '2024-01-18 07:00:00'],
    ['2024-01-18 07:00:00', '2024-01-18 19:00:00'],
    ['2024-01-18 19:00:00', '2024-01-19 07:00:00'],
    ['2024-01-19 07:00:00', '2024-01-19 19:00:00'],
    ['2024-01-19 19:00:00', '2024-01-20 07:00:00'],
    ['2024-01-20 07:00:00', '2024-01-20 19:00:00'],
    ['2024-01-20 19:00:00', '2024-01-21 07:00:00'],
    ['2024-01-21 07:00:00', '2024-01-21 19:00:00'],
    ['2024-01-21 19:00:00', '2024-01-22 07:00:00'],
    ['2024-01-22 07:00:00', '2024-01-22 19:00:00'],
    ['2024-01-22 19:00:00', '2024-01-23 07:00:00'],
    ['2024-01-23 07:00:00', '2024-01-23 19:00:00'],
    ['2024-01-23 19:00:00', '2024-01-24 07:00:00'],
    ['2024-01-24 07:00:00', '2024-01-24 19:00:00'],
    ['2024-01-24 19:00:00', '2024-01-25 07:00:00'],
    ['2024-01-25 07:00:00', '2024-01-25 19:00:00'],
    ['2024-01-25 19:00:00', '2024-01-26 07:00:00'],
    ['2024-01-26 07:00:00', '2024-01-26 19:00:00'],
    ['2024-01-26 19:00:00', '2024-01-27 07:00:00'],
    ['2024-01-27 07:00:00', '2024-01-27 19:00:00'],
    ['2024-01-27 19:00:00', '2024-01-28 07:00:00'],
    ['2024-01-28 07:00:00', '2024-01-28 19:00:00'],
    ['2024-01-28 19:00:00', '2024-01-29 07:00:00'],
    ['2024-01-29 07:00:00', '2024-01-29 19:00:00'],
    ['2024-01-29 19:00:00', '2024-01-30 07:00:00'],
    ['2024-01-30 07:00:00', '2024-01-30 19:00:00'],
    ['2024-01-30 19:00:00', '2024-01-31 07:00:00'],
    ['2024-01-31 07:00:00', '2024-01-31 19:00:00'],
    ['2024-01-31 19:00:00', '2024-02-01 07:00:00'],
    ['2024-02-01 07:00:00', '2024-02-01 19:00:00'],
    ['2024-02-01 19:00:00', '2024-02-02 07:00:00'],
    ['2024-02-02 07:00:00', '2024-02-02 19:00:00'],
    ['2024-02-02 19:00:00', '2024-02-03 07:00:00'],
    ['2024-02-03 07:00:00', '2024-02-03 19:00:00'],
    ['2024-02-03 19:00:00', '2024-02-04 07:00:00'],
    ['2024-02-04 07:00:00', '2024-02-04 19:00:00'],
    ['2024-02-04 19:00:00', '2024-02-05 07:00:00'],
    ['2024-02-05 07:00:00', '2024-02-05 19:00:00'],
    ['2024-02-05 19:00:00', '2024-02-06 07:00:00'],
    ['2024-02-06 07:00:00', '2024-02-06 19:00:00'],
    ['2024-02-06 19:00:00', '2024-02-07 07:00:00'],
    ['2024-02-07 07:00:00', '2024-02-07 19:00:00'],
    ['2024-02-07 19:00:00', '2024-02-08 07:00:00'],
    ['2024-02-08 07:00:00', '2024-02-08 19:00:00'],
    ['2024-02-08 19:00:00', '2024-02-09 07:00:00'],
    ['2024-02-09 07:00:00', '2024-02-09 19:00:00'],
    ['2024-02-09 19:00:00', '2024-02-10 07:00:00'],
    ['2024-02-10 07:00:00', '2024-02-10 19:00:00'],
    ['2024-02-10 19:00:00', '2024-02-11 07:00:00'],
    ['2024-02-11 07:00:00', '2024-02-11 19:00:00'],
    ['2024-02-11 19:00:00', '2024-02-12 07:00:00'],
    ['2024-02-12 07:00:00', '2024-02-12 19:00:00'],
    ['2024-02-12 19:00:00', '2024-02-13 07:00:00'],
    ['2024-02-13 07:00:00', '2024-02-13 19:00:00'],
    ['2024-02-13 19:00:00', '2024-02-14 07:00:00'],
    ['2024-02-14 07:00:00', '2024-02-14 19:00:00'],
    ['2024-02-14 19:00:00', '2024-02-15 07:00:00'],
    ['2024-02-15 07:00:00', '2024-02-15 19:00:00'],
    ['2024-02-15 19:00:00', '2024-02-16 07:00:00'],
    ['2024-02-16 07:00:00', '2024-02-16 19:00:00'],
    ['2024-02-16 19:00:00', '2024-02-17 07:00:00'],
    ['2024-02-17 07:00:00', '2024-02-17 19:00:00'],
    ['2024-02-17 19:00:00', '2024-02-18 07:00:00'],
    ['2024-02-18 07:00:00', '2024-02-18 19:00:00'],
    ['2024-02-18 19:00:00', '2024-02-19 07:00:00'],
    ['2024-02-19 07:00:00', '2024-02-19 19:00:00'],
    ['2024-02-19 19:00:00', '2024-02-20 07:00:00'],
    ['2024-02-20 07:00:00', '2024-02-20 19:00:00'],
    ['2024-02-20 19:00:00', '2024-02-21 07:00:00'],
    ['2024-02-21 07:00:00', '2024-02-21 19:00:00'],
    ['2024-02-21 19:00:00', '2024-02-22 07:00:00'],
    ['2024-02-22 07:00:00', '2024-02-22 19:00:00'],
    ['2024-02-22 19:00:00', '2024-02-23 07:00:00'],
    ['2024-02-23 07:00:00', '2024-02-23 19:00:00'],
    ['2024-02-23 19:00:00', '2024-02-24 07:00:00'],
    ['2024-02-24 07:00:00', '2024-02-24 19:00:00'],
    ['2024-02-24 19:00:00', '2024-02-25 07:00:00'],
    ['2024-02-25 07:00:00', '2024-02-25 19:00:00'],#here
    ['2024-02-27 07:00:00', '2024-02-27 19:00:00'],
    ['2024-02-27 19:00:00', '2024-02-28 07:00:00'],
    ['2024-02-28 07:00:00', '2024-02-28 19:00:00'],
    ['2024-02-28 19:00:00', '2024-02-29 07:00:00'],
    ['2024-02-29 07:00:00', '2024-02-29 19:00:00'],
    ['2024-02-29 19:00:00', '2024-03-01 07:00:00'],
    ['2024-03-01 07:00:00', '2024-03-01 19:00:00'],
    ['2024-03-01 19:00:00', '2024-03-02 07:00:00'],
    ['2024-03-02 07:00:00', '2024-03-02 19:00:00'],
    ['2024-03-02 19:00:00', '2024-03-03 07:00:00'],
    ['2024-03-03 07:00:00', '2024-03-03 19:00:00'],
    ['2024-03-03 19:00:00', '2024-03-04 07:00:00'],
    ['2024-03-04 07:00:00', '2024-03-04 19:00:00'],
    ['2024-03-04 19:00:00', '2024-03-05 07:00:00'],
    ['2024-03-05 07:00:00', '2024-03-05 19:00:00'],
    ['2024-03-05 19:00:00', '2024-03-06 07:00:00'],
    ['2024-03-06 07:00:00', '2024-03-06 19:00:00'],
    ['2024-03-06 19:00:00', '2024-03-07 07:00:00'],
    ['2024-03-07 07:00:00', '2024-03-07 19:00:00'],
    ['2024-03-07 19:00:00', '2024-03-08 07:00:00'],
    ['2024-03-08 07:00:00', '2024-03-08 19:00:00'],
    ['2024-03-08 19:00:00', '2024-03-09 07:00:00'],
    ['2024-03-09 07:00:00', '2024-03-09 19:00:00'],
    ['2024-03-09 19:00:00', '2024-03-10 07:00:00'],
    ['2024-03-10 07:00:00', '2024-03-10 19:00:00'],
    ['2024-03-10 19:00:00', '2024-03-11 07:00:00'],
    ['2024-03-11 07:00:00', '2024-03-11 19:00:00'],
    ['2024-03-11 19:00:00', '2024-03-12 07:00:00'],
    ['2024-03-12 07:00:00', '2024-03-12 19:00:00'],
    ['2024-03-12 19:00:00', '2024-03-13 07:00:00'],
    ['2024-03-13 07:00:00', '2024-03-13 19:00:00'],
    ['2024-03-13 19:00:00', '2024-03-14 07:00:00'],
    ['2024-03-14 07:00:00', '2024-03-14 19:00:00'],
    ['2024-03-14 19:00:00', '2024-03-15 07:00:00'],
    ['2024-03-15 07:00:00', '2024-03-15 19:00:00'],
    ['2024-03-15 19:00:00', '2024-03-16 07:00:00'],
    ['2024-03-16 07:00:00', '2024-03-16 19:00:00'],
    ['2024-03-16 19:00:00', '2024-03-17 07:00:00'],
    ['2024-03-17 07:00:00', '2024-03-17 19:00:00'],
    ['2024-03-17 19:00:00', '2024-03-18 07:00:00'],
    ['2024-03-18 07:00:00', '2024-03-18 19:00:00'],#here
    ['2024-03-20 07:00:00', '2024-03-20 19:00:00'],
    ['2024-03-20 19:00:00', '2024-03-21 07:00:00'],
    ['2024-03-21 07:00:00', '2024-03-21 19:00:00'],
    ['2024-03-21 19:00:00', '2024-03-22 07:00:00'],
    ['2024-03-22 07:00:00', '2024-03-22 19:00:00'],
    ['2024-03-22 19:00:00', '2024-03-23 07:00:00'],
    ['2024-03-23 07:00:00', '2024-03-23 19:00:00'],
    ['2024-03-23 19:00:00', '2024-03-24 07:00:00'],
    ['2024-03-24 07:00:00', '2024-03-24 19:00:00'],
    ['2024-03-24 19:00:00', '2024-03-25 07:00:00'],
    ['2024-03-25 07:00:00', '2024-03-25 19:00:00'],#here
    ['2024-03-27 07:00:00', '2024-03-27 19:00:00'],
    ['2024-03-27 19:00:00', '2024-03-28 07:00:00'],
    ['2024-03-28 07:00:00', '2024-03-28 19:00:00'],
    ['2024-03-28 19:00:00', '2024-03-29 07:00:00'],
    ['2024-03-29 07:00:00', '2024-03-29 19:00:00'],
    ['2024-03-29 19:00:00', '2024-03-30 07:00:00'],
    ['2024-03-30 07:00:00', '2024-03-30 19:00:00'],
    ['2024-04-01 07:00:00', '2024-04-01 19:00:00'],#here
    ['2024-04-01 19:00:00', '2024-04-02 07:00:00'],
    ['2024-04-02 07:00:00', '2024-04-02 19:00:00'],
    ['2024-04-02 19:00:00', '2024-04-03 07:00:00'],
    ['2024-04-03 07:00:00', '2024-04-03 19:00:00'],
    ['2024-04-03 19:00:00', '2024-04-04 07:00:00'],
    ['2024-04-04 07:00:00', '2024-04-04 19:00:00'],
    ['2024-04-04 19:00:00', '2024-04-05 07:00:00'],
    ['2024-04-05 07:00:00', '2024-04-05 19:00:00'],
    ['2024-04-05 19:00:00', '2024-04-06 07:00:00'],
    ['2024-04-06 07:00:00', '2024-04-06 19:00:00'],
    ['2024-04-06 19:00:00', '2024-04-07 07:00:00'],
    ['2024-04-07 07:00:00', '2024-04-07 19:00:00'],
    ['2024-04-07 19:00:00', '2024-04-08 07:00:00'],
    ['2024-04-08 07:00:00', '2024-04-08 19:00:00'],
    ['2024-04-08 19:00:00', '2024-04-09 07:00:00'],
    ['2024-04-09 07:00:00', '2024-04-09 19:00:00'],
    ['2024-04-09 19:00:00', '2024-04-10 07:00:00'],
    ['2024-04-10 07:00:00', '2024-04-10 19:00:00'],
    ['2024-04-10 19:00:00', '2024-04-11 07:00:00'],
    ['2024-04-11 07:00:00', '2024-04-11 19:00:00'],
    ['2024-04-11 19:00:00', '2024-04-12 07:00:00'],
    ['2024-04-12 07:00:00', '2024-04-12 19:00:00'],
    ['2024-04-12 19:00:00', '2024-04-13 07:00:00'],
    ['2024-04-13 07:00:00', '2024-04-13 19:00:00'],
    ['2024-04-13 19:00:00', '2024-04-14 07:00:00'],
    ['2024-04-14 07:00:00', '2024-04-14 19:00:00'],
    ['2024-04-14 19:00:00', '2024-04-15 07:00:00'],
    ['2024-04-15 07:00:00', '2024-04-15 19:00:00'],
    ['2024-04-15 19:00:00', '2024-04-16 07:00:00'],
    ['2024-04-16 07:00:00', '2024-04-16 19:00:00'],
    ['2024-04-16 19:00:00', '2024-04-17 07:00:00'],
    ['2024-04-17 07:00:00', '2024-04-17 19:00:00'],#here
    ['2024-04-19 07:00:00', '2024-04-19 19:00:00'],
    ['2024-04-19 19:00:00', '2024-04-20 07:00:00'],
    ['2024-04-20 07:00:00', '2024-04-20 19:00:00'],
    ['2024-04-20 19:00:00', '2024-04-21 07:00:00'],
    ['2024-04-21 07:00:00', '2024-04-21 19:00:00'],
    ['2024-04-21 19:00:00', '2024-04-22 07:00:00'],
    ['2024-04-22 07:00:00', '2024-04-22 19:00:00'],
    ['2024-04-22 19:00:00', '2024-04-23 07:00:00'],
    ['2024-04-23 07:00:00', '2024-04-23 19:00:00'],#here
    ['2024-04-25 07:00:00', '2024-04-25 19:00:00'],
    ['2024-04-25 19:00:00', '2024-04-26 07:00:00'],
    ['2024-04-26 07:00:00', '2024-04-26 19:00:00'],
    ['2024-04-26 19:00:00', '2024-04-27 07:00:00'],
    ['2024-04-27 07:00:00', '2024-04-27 19:00:00'],
    ['2024-04-27 19:00:00', '2024-04-28 07:00:00'],
    ['2024-04-28 07:00:00', '2024-04-28 19:00:00'],
    ['2024-04-28 19:00:00', '2024-04-29 07:00:00'],
    ['2024-04-29 07:00:00', '2024-04-29 19:00:00'],
    ['2024-04-29 19:00:00', '2024-04-30 07:00:00'],
    ['2024-04-30 07:00:00', '2024-04-30 19:00:00'],
    ['2024-04-30 19:00:00', '2024-05-01 07:00:00'],
    ['2024-05-01 07:00:00', '2024-05-01 19:00:00'],
    ['2024-05-01 19:00:00', '2024-05-02 07:00:00'],
    ['2024-05-02 07:00:00', '2024-05-02 19:00:00'],
    ['2024-05-02 19:00:00', '2024-05-03 07:00:00'],
    ['2024-05-03 07:00:00', '2024-05-03 19:00:00'],
    ['2024-05-03 19:00:00', '2024-05-04 07:00:00'],
    ['2024-05-04 07:00:00', '2024-05-04 19:00:00'],
    ['2024-05-04 19:00:00', '2024-05-05 07:00:00'],
    ['2024-05-05 07:00:00', '2024-05-05 19:00:00'],
    ['2024-05-05 19:00:00', '2024-05-06 07:00:00'],
    ['2024-05-06 07:00:00', '2024-05-06 19:00:00'],
    ['2024-05-06 19:00:00', '2024-05-07 07:00:00'],
    ['2024-05-07 07:00:00', '2024-05-07 19:00:00'],
    ['2024-05-07 19:00:00', '2024-05-08 07:00:00'],
    ['2024-05-08 07:00:00', '2024-05-08 19:00:00'],
    ['2024-05-08 19:00:00', '2024-05-09 07:00:00'],
    ['2024-05-09 07:00:00', '2024-05-09 19:00:00'],
    ['2024-05-09 19:00:00', '2024-05-10 07:00:00'],
    ['2024-05-10 07:00:00', '2024-05-10 19:00:00'],
    ['2024-05-10 19:00:00', '2024-05-11 07:00:00'],
    ['2024-05-11 07:00:00', '2024-05-11 19:00:00'],
    ['2024-05-11 19:00:00', '2024-05-12 07:00:00'],
    ['2024-05-12 07:00:00', '2024-05-12 19:00:00'],
    ['2024-05-12 19:00:00', '2024-05-13 07:00:00'],
    ['2024-05-13 07:00:00', '2024-05-13 19:00:00'],
    ['2024-05-13 19:00:00', '2024-05-14 07:00:00'],
    ['2024-05-14 07:00:00', '2024-05-14 19:00:00'],#here
    ['2024-05-16 07:00:00', '2024-05-16 19:00:00'],
    ['2024-05-16 19:00:00', '2024-05-17 07:00:00'],
    ['2024-05-17 07:00:00', '2024-05-17 19:00:00'],
    ['2024-05-17 19:00:00', '2024-05-18 07:00:00'],
    ['2024-05-18 07:00:00', '2024-05-18 19:00:00'],
    ['2024-05-18 19:00:00', '2024-05-19 07:00:00'],
    ['2024-05-19 07:00:00', '2024-05-19 19:00:00'],
    ['2024-05-19 19:00:00', '2024-05-20 07:00:00'],
    ['2024-05-20 07:00:00', '2024-05-20 19:00:00'],
    ['2024-05-20 19:00:00', '2024-05-21 07:00:00'],
    ['2024-05-21 07:00:00', '2024-05-21 19:00:00'],
    ['2024-05-21 19:00:00', '2024-05-22 07:00:00'],
    ['2024-05-22 07:00:00', '2024-05-22 19:00:00'],
    ['2024-05-22 19:00:00', '2024-05-23 07:00:00'],
    ['2024-05-23 07:00:00', '2024-05-23 19:00:00'],
    ['2024-05-23 19:00:00', '2024-05-24 07:00:00'],
    ['2024-05-24 07:00:00', '2024-05-24 19:00:00'],
    ['2024-05-24 19:00:00', '2024-05-25 07:00:00'],
    ['2024-05-25 07:00:00', '2024-05-25 19:00:00'],
    ['2024-05-25 19:00:00', '2024-05-26 07:00:00'],
    ['2024-05-26 07:00:00', '2024-05-26 19:00:00'],
    ['2024-05-26 19:00:00', '2024-05-27 07:00:00'],
    ['2024-05-27 07:00:00', '2024-05-27 19:00:00'],
    ['2024-05-27 19:00:00', '2024-05-28 07:00:00'],
    ['2024-05-28 07:00:00', '2024-05-28 19:00:00'],
    ['2024-05-28 19:00:00', '2024-05-29 07:00:00'],
    ['2024-05-29 07:00:00', '2024-05-29 19:00:00'],#here
    ['2024-05-31 07:00:00', '2024-05-31 19:00:00'],
    ['2024-05-31 19:00:00', '2024-06-01 07:00:00'],
    ['2024-06-01 07:00:00', '2024-06-01 19:00:00'],
    ['2024-06-01 19:00:00', '2024-06-02 07:00:00'],
    ['2024-06-02 07:00:00', '2024-06-02 19:00:00'],
    ['2024-06-02 19:00:00', '2024-06-03 07:00:00'],
    ['2024-06-03 07:00:00', '2024-06-03 19:00:00'],
    ['2024-06-03 19:00:00', '2024-06-04 07:00:00'],
    ['2024-06-04 07:00:00', '2024-06-04 19:00:00'],
    ['2024-06-04 19:00:00', '2024-06-05 07:00:00'],
    ['2024-06-05 07:00:00', '2024-06-05 19:00:00'],
    ['2024-06-05 19:00:00', '2024-06-06 07:00:00'],
    ['2024-06-06 07:00:00', '2024-06-06 19:00:00'],
    ['2024-06-06 19:00:00', '2024-06-07 07:00:00'],
    ['2024-06-07 07:00:00', '2024-06-07 19:00:00'],
    ['2024-06-07 19:00:00', '2024-06-08 07:00:00'],
    ['2024-06-08 07:00:00', '2024-06-08 19:00:00'],
    ['2024-06-08 19:00:00', '2024-06-09 07:00:00'],
    ['2024-06-09 07:00:00', '2024-06-09 19:00:00'],
    ['2024-06-09 19:00:00', '2024-06-10 07:00:00'],
    ['2024-06-10 07:00:00', '2024-06-10 19:00:00'],
    ['2024-06-10 19:00:00', '2024-06-11 07:00:00'],
    ['2024-06-11 07:00:00', '2024-06-11 19:00:00'],
    ['2024-06-11 19:00:00', '2024-06-12 07:00:00'],
    ['2024-06-12 07:00:00', '2024-06-12 19:00:00'],
    ['2024-06-12 19:00:00', '2024-06-13 07:00:00'],
    ['2024-06-13 07:00:00', '2024-06-13 19:00:00'],
    ['2024-06-13 19:00:00', '2024-06-14 07:00:00'],
    ['2024-06-14 07:00:00', '2024-06-14 19:00:00'],
    ['2024-06-14 19:00:00', '2024-06-15 07:00:00'],
    ['2024-06-15 07:00:00', '2024-06-15 19:00:00'],
    ['2024-06-15 19:00:00', '2024-06-16 07:00:00'],
    ['2024-06-16 07:00:00', '2024-06-16 19:00:00'],
    ['2024-06-16 19:00:00', '2024-06-17 07:00:00'],
    ['2024-06-17 07:00:00', '2024-06-17 19:00:00'],#here
    ['2024-06-19 07:00:00', '2024-06-19 19:00:00'],
    ['2024-06-19 19:00:00', '2024-06-20 07:00:00'],
    ['2024-06-20 07:00:00', '2024-06-20 19:00:00'],
    ['2024-06-20 19:00:00', '2024-06-21 07:00:00'],
    ['2024-06-21 07:00:00', '2024-06-21 19:00:00'],
    ['2024-06-21 19:00:00', '2024-06-22 07:00:00'],
    ['2024-06-22 07:00:00', '2024-06-22 19:00:00'],
    ['2024-06-22 19:00:00', '2024-06-23 07:00:00'],
    ['2024-06-23 07:00:00', '2024-06-23 19:00:00'],
    ['2024-06-23 19:00:00', '2024-06-24 07:00:00'],
    ['2024-06-24 07:00:00', '2024-06-24 19:00:00'],
    ['2024-06-24 19:00:00', '2024-06-25 07:00:00'],
    ['2024-06-25 07:00:00', '2024-06-25 19:00:00'],
    ['2024-06-25 19:00:00', '2024-06-26 07:00:00'],
    ['2024-06-28 19:00:00', '2024-06-29 07:00:00'],#here
    ['2024-06-29 07:00:00', '2024-06-29 19:00:00'],
    ['2024-06-29 19:00:00', '2024-06-30 07:00:00'],
    ['2024-06-30 07:00:00', '2024-06-30 19:00:00'],
    ['2024-06-30 19:00:00', '2024-07-01 07:00:00'],
    ['2024-07-01 07:00:00', '2024-07-01 19:00:00'],
    ['2024-07-01 19:00:00', '2024-07-02 07:00:00'],
    ['2024-07-02 07:00:00', '2024-07-02 19:00:00'],
    ['2024-07-02 19:00:00', '2024-07-03 07:00:00'],
    ['2024-07-03 07:00:00', '2024-07-03 19:00:00'],
    ['2024-07-03 19:00:00', '2024-07-04 07:00:00'],
    ['2024-07-04 07:00:00', '2024-07-04 19:00:00'],
    ['2024-07-04 19:00:00', '2024-07-05 07:00:00'],
    ['2024-07-05 07:00:00', '2024-07-05 19:00:00'],
    ['2024-07-05 19:00:00', '2024-07-06 07:00:00'],
    ['2024-07-06 07:00:00', '2024-07-06 19:00:00'],
    ['2024-07-06 19:00:00', '2024-07-07 07:00:00'],
    ['2024-07-07 07:00:00', '2024-07-07 19:00:00'],
    ['2024-07-07 19:00:00', '2024-07-08 07:00:00'],
    ['2024-07-08 07:00:00', '2024-07-08 19:00:00'],
    ['2024-07-08 19:00:00', '2024-07-09 07:00:00'],
    ['2024-07-09 07:00:00', '2024-07-09 19:00:00'],
    ['2024-07-09 19:00:00', '2024-07-10 07:00:00'],
    ['2024-07-10 07:00:00', '2024-07-10 19:00:00'],
    ['2024-07-10 19:00:00', '2024-07-11 07:00:00'],
    ['2024-07-11 07:00:00', '2024-07-11 19:00:00'],
    ['2024-07-11 19:00:00', '2024-07-12 07:00:00'],
    ['2024-07-12 07:00:00', '2024-07-12 19:00:00'],
    ['2024-07-12 19:00:00', '2024-07-13 07:00:00'],
    ['2024-07-13 07:00:00', '2024-07-13 19:00:00'],
    ['2024-07-13 19:00:00', '2024-07-14 07:00:00'],
    ['2024-07-14 07:00:00', '2024-07-14 19:00:00'],
    ['2024-07-14 19:00:00', '2024-07-15 07:00:00'],
    ['2024-07-15 07:00:00', '2024-07-15 19:00:00'],
    ['2024-07-15 19:00:00', '2024-07-16 07:00:00'],
    ['2024-07-16 07:00:00', '2024-07-16 19:00:00'],
    ['2024-07-16 19:00:00', '2024-07-17 07:00:00'],
    ['2024-07-17 07:00:00', '2024-07-17 19:00:00'],#here
    ['2024-07-19 07:00:00', '2024-07-19 19:00:00'],
    ['2024-07-19 19:00:00', '2024-07-20 07:00:00'],
    ['2024-07-20 07:00:00', '2024-07-20 19:00:00'],
    ['2024-07-20 19:00:00', '2024-07-21 07:00:00'],
    ['2024-07-21 07:00:00', '2024-07-21 19:00:00'],
    ['2024-07-21 19:00:00', '2024-07-22 07:00:00'],
    ['2024-07-22 07:00:00', '2024-07-22 19:00:00'],
    ['2024-07-22 19:00:00', '2024-07-23 07:00:00'],
    ['2024-07-23 07:00:00', '2024-07-23 19:00:00'],
    ['2024-07-23 19:00:00', '2024-07-24 07:00:00'],
    ['2024-07-24 07:00:00', '2024-07-24 19:00:00'],
    ['2024-07-24 19:00:00', '2024-07-25 07:00:00'],
    ['2024-07-25 07:00:00', '2024-07-25 19:00:00'],
    ['2024-07-25 19:00:00', '2024-07-26 07:00:00'],
    ['2024-07-26 07:00:00', '2024-07-26 19:00:00'],
    ['2024-07-26 19:00:00', '2024-07-27 07:00:00'],
    ['2024-07-27 07:00:00', '2024-07-27 19:00:00'],
    ['2024-07-27 19:00:00', '2024-07-28 07:00:00'],
    ['2024-07-28 07:00:00', '2024-07-28 19:00:00'],
    ['2024-07-28 19:00:00', '2024-07-29 07:00:00'],
    ['2024-07-29 07:00:00', '2024-07-29 19:00:00'],
    ['2024-07-29 19:00:00', '2024-07-30 07:00:00'],
    ['2024-07-30 07:00:00', '2024-07-30 19:00:00'],#here
    ['2024-08-04 19:00:00', '2024-08-05 07:00:00'],
    ['2024-08-05 07:00:00', '2024-08-05 19:00:00'],
    ['2024-08-05 19:00:00', '2024-08-06 07:00:00'],
    ['2024-08-06 07:00:00', '2024-08-06 19:00:00'],
    ['2024-08-06 19:00:00', '2024-08-07 07:00:00'],
    ['2024-08-07 07:00:00', '2024-08-07 19:00:00'],
    ['2024-08-07 19:00:00', '2024-08-08 07:00:00'],
    ['2024-08-08 07:00:00', '2024-08-08 19:00:00'],
    ['2024-08-08 19:00:00', '2024-08-09 07:00:00'],
    ['2024-08-09 07:00:00', '2024-08-09 19:00:00'],
    ['2024-08-09 19:00:00', '2024-08-10 07:00:00'],
    ['2024-08-10 07:00:00', '2024-08-10 19:00:00'],
    ['2024-08-10 19:00:00', '2024-08-11 07:00:00'],
    ['2024-08-11 07:00:00', '2024-08-11 19:00:00'],
    ['2024-08-11 19:00:00', '2024-08-12 07:00:00'],
    ['2024-08-12 07:00:00', '2024-08-12 19:00:00'],
    ['2024-08-12 19:00:00', '2024-08-13 07:00:00'],
    ['2024-08-13 07:00:00', '2024-08-13 19:00:00'],
    ['2024-08-13 19:00:00', '2024-08-14 07:00:00'],
    ['2024-08-14 07:00:00', '2024-08-14 19:00:00'],
    ['2024-08-14 19:00:00', '2024-08-15 07:00:00'],
    ['2024-08-15 07:00:00', '2024-08-15 19:00:00'],
    ['2024-08-15 19:00:00', '2024-08-16 07:00:00'],
    ['2024-08-16 07:00:00', '2024-08-16 19:00:00'],
    ['2024-08-16 19:00:00', '2024-08-17 07:00:00'],
    ['2024-08-17 07:00:00', '2024-08-17 19:00:00'],
    ['2024-08-17 19:00:00', '2024-08-18 07:00:00'],
    ['2024-08-18 07:00:00', '2024-08-18 19:00:00'],#here
    ['2024-08-20 07:00:00', '2024-08-20 19:00:00'],
    ['2024-08-20 19:00:00', '2024-08-21 07:00:00'],
    ['2024-08-21 07:00:00', '2024-08-21 19:00:00'],
    ['2024-08-21 19:00:00', '2024-08-22 07:00:00'],
    ['2024-08-22 07:00:00', '2024-08-22 19:00:00'],
    ['2024-08-22 19:00:00', '2024-08-23 07:00:00'],
    ['2024-08-23 07:00:00', '2024-08-23 19:00:00'],
    ['2024-08-23 19:00:00', '2024-08-24 07:00:00'],
    ['2024-08-24 07:00:00', '2024-08-24 19:00:00'],
    ['2024-08-24 19:00:00', '2024-08-25 07:00:00'],
    ['2024-08-25 07:00:00', '2024-08-25 19:00:00']]

In [ ]:
# convert "list" into a dataframe
df = pd.DataFrame(list, columns = ['start', 'end'])
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])

In [ ]:
# convert datetime to string
df['start'] = df['start'].dt.strftime('%Y-%m-%d')
df['end'] = df['end'].dt.strftime('%Y-%m-%d')

In [ ]:
df.head()

In [ ]:
df['range'] = df['start'] + '/' + df['end']

In [ ]:
#drop the start and end columns
df = df.drop(['start', 'end'], axis=1)

In [ ]:
# trasform the range column into a list
list = df['range'].tolist()

In [ ]:
a['date'] = list

In [ ]:
a.head()

In [ ]:
a.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_sept22_ago24.csv')

In [ ]:
b=raw.light.summary_statistics_per_time_bin(
    bins=[
        ['2024-09-23 07:00:00','2024-09-23 19:00:00'],
        ['2024-09-23 19:00:00','2024-09-24 07:00:00'],
        ['2024-09-24 07:00:00','2024-09-24 19:00:00'],
        ['2024-09-24 19:00:00','2024-09-25 07:00:00'],
        ['2024-09-25 07:00:00','2024-09-25 19:00:00'],
        ['2024-09-25 19:00:00','2024-09-26 07:00:00'],
        ['2024-09-26 07:00:00','2024-09-26 19:00:00'],
        ['2024-09-26 19:00:00','2024-09-27 07:00:00'],
        ['2024-09-27 07:00:00','2024-09-27 19:00:00'],
        ['2024-09-27 19:00:00','2024-09-28 07:00:00'],
        ['2024-09-28 07:00:00','2024-09-28 19:00:00'],
        ['2024-09-28 19:00:00','2024-09-29 07:00:00'],
        ['2024-09-29 07:00:00','2024-09-29 19:00:00'],
        ['2024-09-29 19:00:00','2024-09-30 07:00:00'],
        ['2024-09-30 07:00:00','2024-09-30 19:00:00'],
        ['2024-09-30 19:00:00','2024-10-01 07:00:00'],
        ['2024-10-01 07:00:00','2024-10-01 19:00:00'],
        ['2024-10-01 19:00:00','2024-10-02 07:00:00'],
        ['2024-10-02 07:00:00','2024-10-02 19:00:00'],
        ['2024-10-02 19:00:00','2024-10-03 07:00:00'],
        ['2024-10-03 07:00:00','2024-10-03 19:00:00'],
        ['2024-10-03 19:00:00','2024-10-04 07:00:00'],
        ['2024-10-04 07:00:00','2024-10-04 19:00:00'],
        ['2024-10-04 19:00:00','2024-10-05 07:00:00'],
        ['2024-10-05 07:00:00','2024-10-05 19:00:00'],
        ['2024-10-05 19:00:00','2024-10-06 07:00:00'],
        ['2024-10-06 07:00:00','2024-10-06 19:00:00'],
        ['2024-10-06 19:00:00','2024-10-07 07:00:00'],
        ['2024-10-07 07:00:00','2024-10-07 19:00:00'],
        ['2024-10-07 19:00:00','2024-10-08 07:00:00'],
        ['2024-10-08 07:00:00','2024-10-08 19:00:00'],
        ['2024-10-08 19:00:00','2024-10-09 07:00:00'],
        ['2024-10-09 07:00:00','2024-10-09 19:00:00'],
        ['2024-10-09 19:00:00','2024-10-10 07:00:00'],
        ['2024-10-10 07:00:00','2024-10-10 19:00:00'],
        ['2024-10-10 19:00:00','2024-10-11 07:00:00'],
        ['2024-10-11 07:00:00','2024-10-11 19:00:00'],
        ['2024-10-11 19:00:00','2024-10-12 07:00:00'],
        ['2024-10-12 07:00:00','2024-10-12 19:00:00'],
        ['2024-10-12 19:00:00','2024-10-13 07:00:00'],
        ['2024-10-13 07:00:00','2024-10-13 19:00:00'],
        ['2024-10-13 19:00:00','2024-10-14 07:00:00'],
        ['2024-10-14 07:00:00','2024-10-14 19:00:00'],
        ['2024-10-14 19:00:00','2024-10-15 07:00:00'],
        ['2024-10-15 07:00:00','2024-10-15 19:00:00'],
        ['2024-10-15 19:00:00','2024-10-16 07:00:00'],
        ['2024-10-16 07:00:00','2024-10-16 19:00:00'],
        ['2024-10-16 19:00:00','2024-10-17 07:00:00'],
        ['2024-10-17 07:00:00','2024-10-17 19:00:00'],
        ['2024-10-17 19:00:00','2024-10-18 07:00:00'],
        ['2024-10-18 07:00:00','2024-10-18 19:00:00'],
        ['2024-10-18 19:00:00','2024-10-19 07:00:00'],
        ['2024-10-19 07:00:00','2024-10-19 19:00:00'],
        ['2024-10-19 19:00:00','2024-10-20 07:00:00'],
        ['2024-10-20 07:00:00','2024-10-20 19:00:00'],
        ['2024-10-22 07:00:00','2024-10-22 19:00:00'],
        ['2024-10-22 19:00:00','2024-10-23 07:00:00'],
        ['2024-10-23 07:00:00','2024-10-23 19:00:00'],
        ['2024-10-23 19:00:00','2024-10-24 07:00:00'],
        ['2024-10-24 07:00:00','2024-10-24 19:00:00'],
        ['2024-10-24 19:00:00','2024-10-25 07:00:00'],
        ['2024-10-25 07:00:00','2024-10-25 19:00:00'],
        ['2024-10-25 19:00:00','2024-10-26 07:00:00'],
        ['2024-10-26 07:00:00','2024-10-26 19:00:00'],
        ['2024-10-26 19:00:00','2024-10-27 07:00:00'],
        ['2024-10-27 07:00:00','2024-10-27 19:00:00'],
        ['2024-10-27 19:00:00','2024-10-28 07:00:00'],
        ['2024-10-28 07:00:00','2024-10-28 19:00:00'],
        ['2024-10-28 19:00:00','2024-10-29 07:00:00'],
        ['2024-10-29 07:00:00','2024-10-29 19:00:00'],
        ['2024-10-29 19:00:00','2024-10-30 07:00:00'],
        ['2024-10-30 07:00:00','2024-10-30 19:00:00'],
        ['2024-10-30 19:00:00','2024-10-31 07:00:00'],
        ['2024-10-31 07:00:00','2024-10-31 19:00:00'],
        ['2024-10-31 19:00:00','2024-11-01 07:00:00'],
        ['2024-11-01 07:00:00','2024-11-01 19:00:00'],
        ['2024-11-01 19:00:00','2024-11-02 07:00:00'],
        ['2024-11-02 07:00:00','2024-11-02 19:00:00'],
        ['2024-11-02 19:00:00','2024-11-03 07:00:00'],
        ['2024-11-03 07:00:00','2024-11-03 19:00:00'],
        ['2024-11-03 19:00:00','2024-11-04 07:00:00'],
        ['2024-11-04 07:00:00','2024-11-04 19:00:00'],
        ['2024-11-04 19:00:00','2024-11-05 07:00:00'],
        ['2024-11-05 07:00:00','2024-11-05 19:00:00'],
        ['2024-11-05 19:00:00','2024-11-06 07:00:00'],
        ['2024-11-06 07:00:00','2024-11-06 19:00:00'],
        ['2024-11-06 19:00:00','2024-11-07 07:00:00'],
        ['2024-11-07 07:00:00','2024-11-07 19:00:00'],
        ['2024-11-07 19:00:00','2024-11-08 07:00:00'],
        ['2024-11-08 07:00:00','2024-11-08 19:00:00'],
        ['2024-11-08 19:00:00','2024-11-09 07:00:00'],
        ['2024-11-09 07:00:00','2024-11-09 19:00:00'],
        ['2024-11-09 19:00:00','2024-11-10 07:00:00'],
        ['2024-11-10 07:00:00','2024-11-10 19:00:00'],
        ['2024-11-10 19:00:00','2024-11-11 07:00:00'],
        ['2024-11-11 07:00:00','2024-11-11 19:00:00'],
        ['2024-11-11 19:00:00','2024-11-12 07:00:00'],
        ['2024-11-12 07:00:00','2024-11-12 19:00:00'],
        ['2024-11-12 19:00:00','2024-11-13 07:00:00'],
        ['2024-11-13 07:00:00','2024-11-13 19:00:00'],
        ['2024-11-13 19:00:00','2024-11-14 07:00:00'],
        ['2024-11-14 07:00:00','2024-11-14 19:00:00'],
        ['2024-11-14 19:00:00','2024-11-15 07:00:00'],
        ['2024-11-15 07:00:00','2024-11-15 19:00:00'],
        ['2024-11-15 19:00:00','2024-11-16 07:00:00'],
        ['2024-11-16 07:00:00','2024-11-16 19:00:00'],
        ['2024-11-16 19:00:00','2024-11-17 07:00:00'],
        ['2024-11-17 07:00:00','2024-11-17 19:00:00'],
        ['2024-11-19 07:00:00','2024-11-19 19:00:00'],
        ['2024-11-19 19:00:00','2024-11-20 07:00:00'],
        ['2024-11-20 07:00:00','2024-11-20 19:00:00'],
        ['2024-11-20 19:00:00','2024-11-21 07:00:00'],
        ['2024-11-21 07:00:00','2024-11-21 19:00:00'],
        ['2024-11-21 19:00:00','2024-11-22 07:00:00'],
        ['2024-11-22 07:00:00','2024-11-22 19:00:00'],
        ['2024-11-22 19:00:00','2024-11-23 07:00:00'],
        ['2024-11-23 07:00:00','2024-11-23 19:00:00'],
        ['2024-11-23 19:00:00','2024-11-24 07:00:00'],
        ['2024-11-24 07:00:00','2024-11-24 19:00:00'],
        ['2024-11-24 19:00:00','2024-11-25 07:00:00'],
        ['2024-11-25 07:00:00','2024-11-25 19:00:00'],
        ['2024-11-25 19:00:00','2024-11-26 07:00:00'],
        ['2024-11-26 07:00:00','2024-11-26 19:00:00'],
        ['2024-11-26 19:00:00','2024-11-27 07:00:00'],
        ['2024-11-27 07:00:00','2024-11-27 19:00:00'],
        ['2024-11-27 19:00:00','2024-11-28 07:00:00'],
        ['2024-11-28 07:00:00','2024-11-28 19:00:00'],
        ['2024-11-28 19:00:00','2024-11-29 07:00:00'],
        ['2024-11-29 07:00:00','2024-11-29 19:00:00'],
        ['2024-11-29 19:00:00','2024-11-30 07:00:00'],
        ['2024-11-30 07:00:00','2024-11-30 19:00:00'],
        ['2024-11-30 19:00:00','2024-12-01 07:00:00'],
        ['2024-12-01 07:00:00','2024-12-01 19:00:00'],
        ['2024-12-01 19:00:00','2024-12-02 07:00:00'],
        ['2024-12-02 07:00:00','2024-12-02 19:00:00'],
        ['2024-12-04 07:00:00','2024-12-04 19:00:00'],
        ['2024-12-04 19:00:00','2024-12-05 07:00:00'],
        ['2024-12-05 07:00:00','2024-12-05 19:00:00'],
        ['2024-12-05 19:00:00','2024-12-06 07:00:00'],
        ['2024-12-06 07:00:00','2024-12-06 19:00:00'],
        ['2024-12-06 19:00:00','2024-12-07 07:00:00'],
        ['2024-12-07 07:00:00','2024-12-07 19:00:00'],
        ['2024-12-07 19:00:00','2024-12-08 07:00:00'],
        ['2024-12-08 07:00:00','2024-12-08 19:00:00'],
        ['2024-12-08 19:00:00','2024-12-09 07:00:00'],
        ['2024-12-09 07:00:00','2024-12-09 19:00:00'],
        ['2024-12-09 19:00:00','2024-12-10 07:00:00'],
        ['2024-12-10 07:00:00','2024-12-10 19:00:00'],
        ['2024-12-10 19:00:00','2024-12-11 07:00:00'],
        ['2024-12-11 07:00:00','2024-12-11 19:00:00'],
        ['2024-12-11 19:00:00','2024-12-12 07:00:00'],
        ['2024-12-12 07:00:00','2024-12-12 19:00:00'],
        ['2024-12-12 19:00:00','2024-12-13 07:00:00'],
        ['2024-12-13 07:00:00','2024-12-13 19:00:00'],
        ['2024-12-13 19:00:00','2024-12-14 07:00:00'],
        ['2024-12-14 07:00:00','2024-12-14 19:00:00'],
        ['2024-12-14 19:00:00','2024-12-15 07:00:00'],
        ['2024-12-15 07:00:00','2024-12-15 19:00:00'],
        ['2024-12-15 19:00:00','2024-12-16 07:00:00'],
        ['2024-12-16 07:00:00','2024-12-16 19:00:00'],
        ['2024-12-16 19:00:00','2024-12-17 07:00:00'],
        ['2024-12-17 07:00:00','2024-12-17 19:00:00'],
        ['2024-12-17 19:00:00','2024-12-18 07:00:00'],
        ['2024-12-18 07:00:00','2024-12-18 19:00:00'],
        ['2024-12-18 19:00:00','2024-12-19 07:00:00'],
        ['2024-12-19 07:00:00','2024-12-19 19:00:00'],
        ['2024-12-19 19:00:00','2024-12-20 07:00:00'],
        ['2024-12-20 07:00:00','2024-12-20 19:00:00'],
        ['2024-12-20 19:00:00','2024-12-21 07:00:00'],
        ['2024-12-21 07:00:00','2024-12-21 19:00:00'],
        ['2024-12-21 19:00:00','2024-12-22 07:00:00'],
        ['2024-12-22 07:00:00','2024-12-22 19:00:00'],
        ['2024-12-22 19:00:00','2024-12-23 07:00:00'],
        ['2024-12-23 07:00:00','2024-12-23 19:00:00'],
        ['2024-12-23 19:00:00','2024-12-24 07:00:00'],
        ['2024-12-24 07:00:00','2024-12-24 19:00:00'],
        ['2024-12-24 19:00:00','2024-12-25 07:00:00'],
        ['2024-12-25 07:00:00','2024-12-25 19:00:00'],
        ['2024-12-25 19:00:00','2024-12-26 07:00:00'],
        ['2024-12-26 07:00:00','2024-12-26 19:00:00'],
        ['2024-12-26 19:00:00','2024-12-27 07:00:00'],
        ['2024-12-27 07:00:00','2024-12-27 19:00:00'],
        ['2024-12-27 19:00:00','2024-12-28 07:00:00'],
        ['2024-12-28 07:00:00','2024-12-28 19:00:00'],
        ['2024-12-28 19:00:00','2024-12-29 07:00:00'],
        ['2024-12-29 07:00:00','2024-12-29 19:00:00'],
        ['2024-12-29 19:00:00','2024-12-30 07:00:00'],
        ['2024-12-30 07:00:00','2024-12-30 19:00:00'],
        ['2024-12-30 19:00:00','2024-12-31 07:00:00']
    ], 
    #agg_func=['mean', 'min', 'max']
 )

In [ ]:
# add the date column to the dataframe "a" 
b['date'] = ['2024-09-23/2024-09-23', '2024-09-23/2024-09-24', '2024-09-24/2024-09-24', '2024-09-24/2024-09-25', '2024-09-25/2024-09-25', '2024-09-25/2024-09-26', '2024-09-26/2024-09-26', '2024-09-26/2024-09-27', '2024-09-27/2024-09-27', '2024-09-27/2024-09-28', '2024-09-28/2024-09-28', '2024-09-28/2024-09-29', '2024-09-29/2024-09-29', '2024-09-29/2024-09-30', '2024-09-30/2024-09-30', '2024-09-30/2024-10-01', '2024-10-01/2024-10-01', '2024-10-01/2024-10-02', '2024-10-02/2024-10-02', '2024-10-02/2024-10-03', '2024-10-03/2024-10-03', '2024-10-03/2024-10-04', '2024-10-04/2024-10-04', '2024-10-04/2024-10-05', '2024-10-05/2024-10-05', '2024-10-05/2024-10-06', '2024-10-06/2024-10-06', '2024-10-06/2024-10-07', '2024-10-07/2024-10-07', '2024-10-07/2024-10-08', '2024-10-08/2024-10-08', '2024-10-08/2024-10-09', '2024-10-09/2024-10-09', '2024-10-09/2024-10-10', '2024-10-10/2024-10-10', '2024-10-10/2024-10-11', '2024-10-11/2024-10-11', '2024-10-11/2024-10-12', 
             '2024-10-12/2024-10-12', '2024-10-12/2024-10-13', '2024-10-13/2024-10-13', '2024-10-13/2024-10-14', '2024-10-14/2024-10-14', '2024-10-14/2024-10-15', '2024-10-15/2024-10-15', '2024-10-15/2024-10-16', '2024-10-16/2024-10-16', '2024-10-16/2024-10-17', '2024-10-17/2024-10-17', '2024-10-17/2024-10-18', '2024-10-18/2024-10-18', '2024-10-18/2024-10-19', '2024-10-19/2024-10-19', '2024-10-19/2024-10-20', '2024-10-20/2024-10-20', '2024-10-22/2024-10-22', '2024-10-22/2024-10-23', '2024-10-23/2024-10-23', '2024-10-23/2024-10-24', '2024-10-24/2024-10-24', '2024-10-24/2024-10-25', '2024-10-25/2024-10-25', '2024-10-25/2024-10-26', '2024-10-26/2024-10-26', '2024-10-26/2024-10-27', '2024-10-27/2024-10-27', '2024-10-27/2024-10-28', '2024-10-28/2024-10-28', '2024-10-28/2024-10-29', '2024-10-29/2024-10-29', '2024-10-29/2024-10-30', '2024-10-30/2024-10-30', '2024-10-30/2024-10-31', '2024-10-31/2024-10-31', '2024-10-31/2024-11-01', '2024-11-01/2024-11-01', 
             '2024-11-01/2024-11-02', '2024-11-02/2024-11-02', '2024-11-02/2024-11-03', '2024-11-03/2024-11-03', '2024-11-03/2024-11-04', '2024-11-04/2024-11-04', '2024-11-04/2024-11-05', '2024-11-05/2024-11-05', '2024-11-05/2024-11-06', '2024-11-06/2024-11-06', '2024-11-06/2024-11-07', '2024-11-07/2024-11-07', '2024-11-07/2024-11-08', '2024-11-08/2024-11-08', '2024-11-08/2024-11-09', '2024-11-09/2024-11-09', '2024-11-09/2024-11-10', '2024-11-10/2024-11-10', '2024-11-10/2024-11-11', '2024-11-11/2024-11-11', '2024-11-11/2024-11-12', '2024-11-12/2024-11-12', '2024-11-12/2024-11-13', '2024-11-13/2024-11-13', '2024-11-13/2024-11-14', '2024-11-14/2024-11-14', '2024-11-14/2024-11-15', '2024-11-15/2024-11-15', '2024-11-15/2024-11-16', '2024-11-16/2024-11-16', '2024-11-16/2024-11-17', '2024-11-17/2024-11-17', '2024-11-19/2024-11-19', '2024-11-19/2024-11-20', '2024-11-20/2024-11-20', '2024-11-20/2024-11-21', '2024-11-21/2024-11-21', '2024-11-21/2024-11-22', 
             '2024-11-22/2024-11-22', '2024-11-22/2024-11-23', '2024-11-23/2024-11-23', '2024-11-23/2024-11-24', '2024-11-24/2024-11-24', '2024-11-24/2024-11-25', '2024-11-25/2024-11-25', '2024-11-25/2024-11-26', '2024-11-26/2024-11-26', '2024-11-26/2024-11-27', '2024-11-27/2024-11-27', '2024-11-27/2024-11-28', '2024-11-28/2024-11-28', '2024-11-28/2024-11-29', '2024-11-29/2024-11-29', '2024-11-29/2024-11-30', '2024-11-30/2024-11-30', '2024-11-30/2024-12-01', '2024-12-01/2024-12-01', '2024-12-01/2024-12-02', '2024-12-02/2024-12-02', '2024-12-04/2024-12-04', '2024-12-04/2024-12-05', '2024-12-05/2024-12-05', '2024-12-05/2024-12-06', '2024-12-06/2024-12-06', '2024-12-06/2024-12-07', '2024-12-07/2024-12-07', '2024-12-07/2024-12-08', '2024-12-08/2024-12-08', '2024-12-08/2024-12-09', '2024-12-09/2024-12-09', '2024-12-09/2024-12-10', '2024-12-10/2024-12-10', '2024-12-10/2024-12-11', '2024-12-11/2024-12-11', '2024-12-11/2024-12-12', '2024-12-12/2024-12-12', 
             '2024-12-12/2024-12-13', '2024-12-13/2024-12-13', '2024-12-13/2024-12-14', '2024-12-14/2024-12-14', '2024-12-14/2024-12-15', '2024-12-15/2024-12-15', '2024-12-15/2024-12-16', '2024-12-16/2024-12-16', '2024-12-16/2024-12-17', '2024-12-17/2024-12-17', '2024-12-17/2024-12-18', '2024-12-18/2024-12-18', '2024-12-18/2024-12-19', '2024-12-19/2024-12-19', '2024-12-19/2024-12-20', '2024-12-20/2024-12-20', '2024-12-20/2024-12-21', '2024-12-21/2024-12-21', '2024-12-21/2024-12-22', '2024-12-22/2024-12-22', '2024-12-22/2024-12-23', '2024-12-23/2024-12-23', '2024-12-23/2024-12-24', '2024-12-24/2024-12-24', '2024-12-24/2024-12-25', '2024-12-25/2024-12-25', '2024-12-25/2024-12-26', '2024-12-26/2024-12-26', '2024-12-26/2024-12-27', '2024-12-27/2024-12-27', '2024-12-27/2024-12-28', '2024-12-28/2024-12-28', '2024-12-28/2024-12-29', '2024-12-29/2024-12-29', '2024-12-29/2024-12-30', '2024-12-30/2024-12-30', '2024-12-30/2024-12-31']

In [ ]:
b.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\light_sept24_dec24.csv')

TIME ABOVE THRESHOLD PER DAY

In [ ]:
# Calculate TATp
daily_TATp_10 = raw.light.TATp(threshold=1, oformat='minute') # 3.398 to 2500 lux; 2.69 to 490 lux
daily_TATp_100 = raw.light.TATp(threshold=2, oformat='minute') 
daily_TATp_500 = raw.light.TATp(threshold=2.69, oformat='minute')
daily_TATp_500 = raw.light.TATp(threshold=2.69, oformat='minute') 
daily_TATp_1000 = raw.light.TATp(threshold=3, oformat='minute')
daily_TATp_2500 = raw.light.TATp(threshold=3.398, oformat='minute') 

In [ ]:
# Convert daily TAT into a pandas DataFrame
light_TATp_10 = pd.DataFrame(daily_TATp_10)
light_TATp_100 = pd.DataFrame(daily_TATp_100)
light_TATp_500 = pd.DataFrame(daily_TATp_500)
light_TATp_1000 = pd.DataFrame(daily_TATp_1000)
light_TATp_2500 = pd.DataFrame(daily_TATp_2500)

In [ ]:
# convert index in column
light_TATp_10.reset_index(inplace=True)
light_TATp_100.reset_index(inplace=True)
light_TATp_500.reset_index(inplace=True)
light_TATp_1000.reset_index(inplace=True)
light_TATp_2500.reset_index(inplace=True)

In [ ]:
# Save to a CSV file
light_TATp_10.to_csv(fpath + '\\1daily_TATp_10.csv', index=False)
light_TATp_100.to_csv(fpath + '\\1daily_TATp_100.csv', index=False)
light_TATp_500.to_csv(fpath + '\\1daily_TATp_500.csv', index=False)
light_TATp_1000.to_csv(fpath + '\\1daily_TATp_1000.csv', index=False)
light_TATp_2500.to_csv(fpath + '\\1daily_TATp_2500.csv', index=False)

# Save for part2 (sept-dec2024)
#light_TATp_10.to_csv(fpath + '\\2daily_TATp_10.csv', index=False)
#light_TATp_100.to_csv(fpath + '\\2daily_TATp_100.csv', index=False)
#light_TATp_500.to_csv(fpath + '\\2daily_TATp_500.csv', index=False)
#light_TATp_1000.to_csv(fpath + '\\2daily_TATp_1000.csv', index=False)
#light_TATp_2500.to_csv(fpath + '\\2daily_TATp_2500.csv', index=False)

In [ ]:
raw.light.LMX(length='10h',lowest=False)

In [ ]:
# Check if start_time and end_time are None
if raw.light.start_time is None or raw.light.stop_time is None:
    # Manually set start_time and end_time based on the data index
    start_time = raw.light.data.index[0]  # First timestamp in the data
    end_time = raw.light.data.index[-1]   # Last timestamp in the data
else:
    start_time = raw.light.start_time
    end_time = raw.light.end_time

In [ ]:
# Initialize lists to store daily M10 and L5 values
daily_m10 = []
daily_l5 = []
daily_m10_start = []
daily_l5_start = []
dates = []

# Get the start and end times of the recording
start_time = raw.light.data.index[0]  # First timestamp in the data
end_time = raw.light.data.index[-1]   # Last timestamp in the data

# Iterate over each day in the dataset
current_time = start_time
while current_time < end_time:
    # Define the start and end of the current day
    day_start = current_time
    day_end = day_start + pd.Timedelta(days=1)
    
    # Extract the light data for the current day using pandas filtering
    day_data = raw.light.data.loc[day_start:day_end]
    
    # Check if there is data for the current day
    if len(day_data) > 0:
        # Calculate M10 for the current day
        m10_series = day_data.rolling(window='10h').mean().max()  # M10 level (as a Series)
        m10 = m10_series.iloc[0]  # Extract the raw value from the Series
        m10_start_series = day_data.rolling(window='10h').mean().idxmax()  # M10 start time (as a Series)
        m10_start = m10_start_series.iloc[0]  # Extract the raw timestamp from the Series
        
        # Calculate L5 for the current day
        l5_series = day_data.rolling(window='5h').mean().min()  # L5 level (as a Series)
        l5 = l5_series.iloc[0]  # Extract the raw value from the Series
        l5_start_series = day_data.rolling(window='5h').mean().idxmin()  # L5 start time (as a Series)
        l5_start = l5_start_series.iloc[0]  # Extract the raw timestamp from the Series
        
        # Append results to lists
        daily_m10.append(m10)
        daily_m10_start.append(m10_start)
        daily_l5.append(l5)
        daily_l5_start.append(l5_start)
        dates.append(day_start.date())
    else:
        # If no data is available for the day, append NaN or None
        daily_m10.append(None)
        daily_m10_start.append(None)
        daily_l5.append(None)
        daily_l5_start.append(None)
        dates.append(day_start.date())
    
    # Move to the next day
    current_time = day_end

# Create a DataFrame to store the daily results
daily_results = pd.DataFrame({
    'Date': dates,  # List of dates
    'M10_Level': daily_m10,  # M10 level for each day (raw value)
    'M10_Start': daily_m10_start,  # M10 start time for each day (raw timestamp)
    'L5_Level': daily_l5,  # L5 level for each day (raw value)
    'L5_Start': daily_l5_start  # L5 start time for each day (raw timestamp)
})

# Save the results to a CSV file
daily_results.to_csv('daily_m10_l5_results.csv', index=False)

# Print the results
print(daily_results)

In [ ]:
raw.light.MLiTp(threshold=np.log10(500+1))

In [ ]:
mlitp_value = raw.light.MLiTp(threshold=np.log10(500 + 1))

In [ ]:
divmod(773.05,60)

In [ ]:
#convert the index in column
mlitp_value.reset_index(inplace=True)

In [ ]:
mlitp_value.rename(columns={'index': 'date', 'whitelight': 'MLiTp'}, inplace=True)

In [ ]:
# Convert the value in the column "MLiTp" in mlitp_value into a time of day, simply divide it by the number of minutes per hour
mlitp_value['MLiTp'] = mlitp_value['MLiTp'] / 60


In [ ]:
mlitp_value.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\mlitp_sept22_ago24.csv', index=False)
#mlitp_value.to_csv('C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara\\mlitp_sept24_dec24.csv', index=False)